# LSM-ECG thesis: the redesign (X to Y ladder, E0 to E3)
Implements the plan *LSM-ECG thesis: redesign plan, design and criteria*. Built on the replication notebook: its data, encoders, folds, readout, run cache and runner cells are reused unchanged, so the protocol is identical and every cached run is read, not rerun. The new code is the `lsm2` package (simulator, liquids, levels 1 to 4, gain rule, Maass check, ladder), embedded below and shipped next to this notebook with its tests.

**Stage 1** (one machine, `RUN_STAGE1 = True`): level 1 checks (1a tests, 1b Brian2, 1c this card, 1d time grid, 1e Maass's script, 1f regression), the Maass check, the ladder with its confirmation, the input size of every front end and a speed test. It stops with an error at any stop point of the plan, and writes `checks/redesign_calibration.json`.

**Stage 2** (every machine): copy the calibration file into each machine's `WORK_DIR/checks/`, set `UNIT_SHARES` (the same on every machine, weights from each machine's speed test) and `THIS_MACHINE`, and run. Each machine first runs level 1c on its own card (once), then only its own units.

**Stage 3** (one machine): `MERGE_DIRS = [every machine's output]` with a new, empty `WORK_DIR` combines them; a normal run on the merged folder rebuilds `summary_redesign.md` and `tables/redesign/`.

Point `WORK_DIR` at the replication's output folder (or list a copy in `RESUME_DIRS`): the comparisons read its cached runs.

## 0. Constants (edit here)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CONSTANTS — the only cell you normally need to edit                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
# The redesign (plan "LSM-ECG thesis: redesign plan, design and criteria"). Runs on top of the replication's
# output folder: point WORK_DIR at it (or RESUME_DIRS at a copy), so every cached run is read, never rerun.

# ---- Machine -----------------------------------------------------------
MACHINE = "local"                  # "kaggle" or "local" (your own Linux machine; set its paths below)

MACHINES = {
    "kaggle": {
        "INPUT_DIR":   "/kaggle/input/datasets/tienthinh22/mitdb-1-0-0/mit-bih-arrhythmia-database-1.0.0",
        "WORK_DIR":    "/kaggle/working",
        "SCRATCH_DIR": "/kaggle/working/scratch",
        "RESUME_DIRS": ["/kaggle/input/datasets/tienthinh22/mitdb-common"],
        "SESSION_BUDGET_HOURS": 11.3,
        "DISK_LIMIT_GB": 19.0,
    },
    "local": {
        "INPUT_DIR":   "~/lsm_ecg/mitdb-1.0.0/mit-bih-arrhythmia-database-1.0.0",
        "WORK_DIR":    "~/lsm_ecg/output",          # the replication's output folder (reused in place)
        "SCRATCH_DIR": None,
        "RESUME_DIRS": [],
        "SESSION_BUDGET_HOURS": 240.0,
        "DISK_LIMIT_GB": 250.0,
    },
}
_M = MACHINES[MACHINE]
INPUT_DIR            = _M["INPUT_DIR"]
WORK_DIR             = _M["WORK_DIR"]
SCRATCH_DIR          = _M["SCRATCH_DIR"] or WORK_DIR.rstrip("/") + "/scratch"
RESUME_DIRS          = _M["RESUME_DIRS"]
AUTO_RESUME_ROOT     = "/kaggle/input"
SESSION_BUDGET_HOURS = _M["SESSION_BUDGET_HOURS"]
DISK_LIMIT_GB        = _M["DISK_LIMIT_GB"]

# ---- Stages (plan, Stages, compute and stop points) ---------------------
# Stage 1 (one machine): level 1 checks, the Maass check, the X to Y ladder, the confirmation and the speed test.
#   It writes checks/redesign_calibration.json; copy that file into every machine's WORK_DIR/checks before stage 2.
# Stage 2 (every machine): the classification runs E1 to E3, split into units (see UNIT_SHARES).
# Stage 3 (one machine): MERGE_DIRS = every machine's output folder; then the hybrid-rule runs and the analysis.
RUN_STAGE1           = True        # does nothing once the calibration file exists (delete it to redo stage 1)
RUN_STAGE2           = True        # needs the calibration file
STAGE1_STEPS         = ["level1", "maass_check", "ladder", "speed"]   # resumable: each step saves its result
LEVEL1E_PROCESSES    = None        # CPU processes for (d)'s Brian2 script in 1e (None = every core)
D_REPLACEMENT        = "fixed"     # (d)'s mean replacement of bad U, D, F draws: "fixed" (works as (d) intends) or
                                   # "as_written" (Brian2 reads its boolean index as 0/1, so negative draws stay)
X_UPDATE             = "equations" # X's reset: "equations" = (a)'s Eqs. 17-19, as the plan says; "authors_code" = the
                                   # authors' snnTorch neuron (its threshold test subtracts 1 more while the stored
                                   # voltage is above 1). Only "authors_code" can reproduce the cached runs (check 1f).
MAASS_SCRIPT_URL     = "https://raw.githubusercontent.com/brian-team/brian2/master/examples/frompapers/Maass_Natschlaeger_Markram_2002.py"   # (d), Brian2 examples
MAASS_SCRIPT_PATH    = None        # or a local copy of (d)'s maass2002.py (used instead of downloading)

# ---- Several machines in stage 2 ----------------------------------------
# Units (plan, Stages): one split of one seed for a robust-normalisation front end, or all splits of one seed for a
# standard-normalisation front end. Every machine computes the same assignment from UNIT_SHARES (machine name ->
# weight, e.g. its measured beats per second from the stage-1 speed test) and runs only its own units.
UNIT_SHARES          = {"this": 1.0}
THIS_MACHINE         = "this"

# ---- Session control ---------------------------------------------------
SELF_TEST            = False
SMOKE_TEST           = False       # True = tiny end-to-end test (few beats, 1 seed, coarse grid), its own WORK_DIR
BENCHMARK            = False
ANALYSIS_RESERVE_MIN = 25
RETRY_FAILED         = True
OLD_ANALYSIS         = False       # True = also rebuild the replication's tables and figures

# ---- Software ----------------------------------------------------------
PIN_TORCH_240  = False             # the redesign runs on the installed PyTorch
AUTHORS_REPO   = "https://github.com/Richard97Code/LSM-ECG-Classification.git"
AUTHORS_COMMIT = "02e45c4f3232886aed16a40d77de1f851ef4fcdd"

# ---- Experiment scope --------------------------------------------------
SEEDS = [2026, 69, 420]             # classification seeds (plan, ground rules)

# The replication's groups (all finished; disabled here: their cached runs are read for comparisons).
OLD_GROUPS = ["checks", "brian2", "main_final", "ablation_final", "rr_final", "baseline_final", "gain_final",
              "main_cv", "gain_cv", "ablation_cv", "rr_cv", "baseline_cv", "nn_tune", "retune", "kstale_final",
              "kstale_cv", "huang_final", "huang_cv", "meanrr_final", "meanrr_cv", "elmrr_final", "elmrr_cv",
              "rrliqmean_final", "rrliqmean_cv", "meanrr_nn", "hybrid", "working_diag", "working_final", "working_cv"]
NEW_GROUPS = [
    "e1",              # final liquid, its recurrence-off twin, X at g = 1: film + 3 snapshots (config 11)
    "e1_ablation",     # memory ablation: final liquid and its twin on beats with the early input replaced
    "e1_controls",     # no-liquid, fading-memory and snapshot-ELM controls at the 3 snapshots
    "e2",              # all 24 configurations, film readout, the final liquid at config 11's g
    "e3",              # RR family: film + RR features, RR in the liquid, RR + mean-RR ratio in the liquid
    "e3_partner",      # only if check 1f fails: the paper's fixed liquid + RR in the liquid, redone in the new code
    "e3_hybrid",       # Huang's SVEB rule on the base, RR-readout and RR-liquid runs (after the merge)
]
GROUP_ORDER = OLD_GROUPS + NEW_GROUPS
GROUPS_ENABLED = {g: (g in NEW_GROUPS) for g in GROUP_ORDER}

SHARE = "BOTH"; SHARE_GROUPS = {"A": [], "B": []}     # the replication's two-machine split (not used here)
MERGE_DIRS = []          # stage 3: every machine's output folder; the combined result goes to WORK_DIR (a new folder)

# replication settings that its cells still read (unchanged values)
GAINS = [1, 2, 3, 5, 8, 13, 21, 34]; DENSE_LAMBDAS = [1.5, 2.5]; RETUNE_TRIALS = 30
RETUNE_GRID = {"gain": [1, 2, 3, 5, 8, 13], "lambda_val": [0.8, 1.5, 2.5], "num_levels": [50, 100, 200, 400]}
NN_EPOCHS = 15; BRIAN2_BEATS = 10; NN_TUNE_TRIALS = 30; NN_PARAM_CAP = 35000; NN_TUNE_MAX_EPOCHS = 30
NN_TUNE_CHECKPOINTS = [5, 10, 15, 20, 25, 30]
HUANG_MATRICES = 15; HUANG_DIM = 50; HUANG_C = 1.0; HUANG_GAMMAS = [1.3, 1 / (2 * 1.3 ** 2)]
HUANG_THRESHOLD = 0.8; HUANG_SVM_CACHE_MB = 500; HYBRID_RULES = ["replace", "add"]
WORKING_LAMBDA = 2.0; WORKING_GAINS = [1, 3, 10, 30, 100, 300, 1000]; WORKING_DIAG_BEATS = 300
WORKING_T_PERTURB = 150; WORKING_MAX_FIRING = 0.2
SAVE_STATES = []

# ---- Performance -------------------------------------------------------
SIM_BATCH_CV    = 4096
SIM_BATCH_FINAL = 2000
NEW_SIM_BATCH   = 500      # beats per GPU batch for the new simulator: the same on every machine (plan, determinism)
PIPELINE        = True
FIT_WORKERS     = "auto"
CORE_SEARCH     = False
RAM_RESERVE_GB  = 4.0
READOUT_JOBS    = 4


## 1. Environment

In [ ]:
NOTEBOOK_BUILD = "054173effa51"   # fingerprint of this build's code cells (do not edit)
# Environment: installs, imports, versions, device. Smoke-test overrides.
import os, sys, subprocess, time, json, platform, shutil, glob, hashlib, traceback, math, re, gc, random
import importlib, importlib.util, datetime, itertools, warnings, contextlib, io, heapq, collections
warnings.filterwarnings("ignore")
SESSION_START = time.time()
ON_KAGGLE = os.path.isdir("/kaggle/working") and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
NOTEBOOK_BUILD = globals().get("NOTEBOOK_BUILD", "dev")      # fingerprint of the code cells, set by the build
if SHARE is None:
    SHARE = "BOTH"
if SHARE != "BOTH" and SHARE not in SHARE_GROUPS:
    raise ValueError(f"SHARE must be 'BOTH', 'A' or 'B', not {SHARE!r}")
MERGE_MODE = bool(MERGE_DIRS)
if MERGE_MODE and SHARE != "BOTH":
    print(f"merge mode: SHARE = {SHARE!r} ignored (the merged result is checked against the whole plan)")
    SHARE = "BOTH"
if MACHINE == "kaggle" and not ON_KAGGLE:
    print("NOTE: MACHINE = 'kaggle' but this does not look like a Kaggle session (paths may not exist).")
INPUT_DIR, WORK_DIR, SCRATCH_DIR = (os.path.abspath(os.path.expanduser(p)) for p in (INPUT_DIR, WORK_DIR, SCRATCH_DIR))
if isinstance(RESUME_DIRS, (list, tuple)):
    RESUME_DIRS = [os.path.abspath(os.path.expanduser(p)) for p in RESUME_DIRS]
os.makedirs(os.path.join(WORK_DIR, "logs"), exist_ok=True)
import threading
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic cuBLAS (plan, determinism)
SESSION_TAG = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
LOG_PATH = os.path.join(WORK_DIR, "logs", f"session_{SESSION_TAG}.log")

class _Tee:
    """Everything the notebook prints (all cells, all threads, warnings and tracebacks) also goes to LOG_PATH,
    flushed line by line, so `tail -f WORK_DIR/logs/latest.log` shows it live from another terminal."""
    def __init__(self, stream, path):
        self._s, self._f, self._lock = stream, open(path, "a", buffering=1), threading.Lock()
    def write(self, text):
        with self._lock:
            try:
                self._f.write(text); self._f.flush()
            except Exception:
                pass
        return self._s.write(text)
    def flush(self):
        self._s.flush()
    def __getattr__(self, name):
        return getattr(self._s, name)

if getattr(sys.stdout, "_lsm_tee", False):           # notebook re-run in the same kernel: switch to the new log
    for _t in (sys.stdout, sys.stderr):
        _t._f = open(LOG_PATH, "a", buffering=1)
else:
    sys.stdout, sys.stderr = _Tee(sys.stdout, LOG_PATH), _Tee(sys.stderr, LOG_PATH)
    sys.stdout._lsm_tee = sys.stderr._lsm_tee = True
try:                                                    # a crash's traceback goes into the log file too
    def _exc_to_log(shell, etype, value, tb, tb_offset=None):
        with open(LOG_PATH, "a") as f:
            f.write("\n" + "".join(traceback.format_exception(etype, value, tb)))
        shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    get_ipython().set_custom_exc((Exception,), _exc_to_log)
except NameError:
    pass
_latest = os.path.join(WORK_DIR, "logs", "latest.log")
try:
    if os.path.islink(_latest) or os.path.exists(_latest):
        os.remove(_latest)
    os.symlink(os.path.basename(LOG_PATH), _latest)
except OSError:
    pass
print(f"session {SESSION_TAG}: everything printed is also written to {LOG_PATH} (and {_latest})")

N_CPU = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
if PIPELINE:
    # one fit per core: the maths libraries must not start their own thread pools inside each fit
    for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ[_v] = "1"

import importlib.metadata as _md

def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

def _pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-3000:]

# Kaggle's core packages must not change version mid-session (a NumPy swap after it is loaded breaks imports),
# so every install is constrained to the versions already present; pip then picks compatible package versions.
CORE = ("numpy", "scipy", "pandas", "matplotlib", "torch")
CORE_BEFORE = {p: _ver(p) for p in CORE}
INSTALL_REPORT = []
if not ON_KAGGLE:
    # Your own machine: you install the packages (in a virtual environment); here we only check.
    _need = {"numpy": None, "scipy": None, "pandas": None, "matplotlib": None, "torch": None, "snntorch": "0.9.4",
             "wfdb": None, "tqdm": None, "scikit-learn": "1.7.2", "joblib": None, "threadpoolctl": None, "brian2": None}
    _miss = [p for p in _need if _ver(p) is None]
    if _miss:
        raise RuntimeError(f"missing packages {_miss}: pip install them in the virtual environment, activate it, then run the notebook")
    for p, want in _need.items():
        if want and _ver(p) != want:
            print(f"WARNING: {p} {_ver(p)} installed, {want} expected")
    INSTALL_REPORT.append({"package": "(local machine: installed by hand)", "status": "checked", "tail": ""})
    if _ver("brian2") is None:
        print("WARNING: Brian2 is not installed; the brian2 check will be marked FAILED, everything else runs")
if ON_KAGGLE:
    os.makedirs(SCRATCH_DIR, exist_ok=True)
    def _constraints(skip=()):
        path = os.path.join(SCRATCH_DIR, "pip_constraints.txt")
        with open(path, "w") as f:
            f.write("\n".join(f"{p}=={v}" for p, v in CORE_BEFORE.items() if v and p not in skip) + "\n")
        return path
    if PIN_TORCH_240 and sys.version_info[:2] <= (3, 12):
        code, out = _pip("torch==2.4.0", "-c", _constraints(skip=("torch",)))
        INSTALL_REPORT.append({"package": "torch==2.4.0", "status": "ok" if code == 0 else "FAILED", "tail": out})
    elif PIN_TORCH_240:
        INSTALL_REPORT.append({"package": "torch==2.4.0", "status": "skipped",
                               "tail": f"Python {platform.python_version()} has no torch 2.4.0; using Kaggle's torch {CORE_BEFORE['torch']}"})
    for pkg, need in [("snntorch==0.9.4", "required"), ("wfdb", "required"), ("tqdm", "required"),
                      ("scikit-learn==1.7.2", "preferred"), ("brian2", "optional")]:
        code, out = _pip(pkg, "-c", _constraints())
        INSTALL_REPORT.append({"package": pkg, "need": need, "status": "ok" if code == 0 else "FAILED", "tail": out})
    for r in INSTALL_REPORT:
        print(f"install {r['package']:<22} -> {r['status']}" + (f"  ({r['tail']})" if r["status"] == "skipped" else ""))
    bad = [r for r in INSTALL_REPORT if r["status"] == "FAILED" and r.get("need") == "required"]
    if bad:
        raise RuntimeError("Required install failed (is Internet ON in the notebook settings?):\n" + bad[0]["tail"])
    for r in INSTALL_REPORT:
        if r["status"] == "FAILED" and r.get("need") == "preferred":
            print(f"WARNING: {r['package']} could not be installed; using Kaggle's scikit-learn {_ver('scikit-learn')}")
        if r["status"] == "FAILED" and r.get("need") == "optional":
            print("WARNING: Brian2 could not be installed; the brian2 check will be marked FAILED, everything else runs")
    changed = {p: (CORE_BEFORE[p], _ver(p)) for p in CORE if _ver(p) != CORE_BEFORE[p]
               and not (p == "torch" and any(r["package"] == "torch==2.4.0" and r["status"] == "ok" for r in INSTALL_REPORT))}
    if changed:
        raise RuntimeError(f"pip changed core packages {changed}. Restart the session and run again.")

import numpy as np, pandas as pd, torch, snntorch, sklearn, scipy, wfdb, joblib
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold
from scipy import stats as sstats
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_PINNED = torch.__version__.split("+")[0] == "2.4.0"

def gpu_supported(cap, arch_list):
    """Can this PyTorch build run on a GPU of compute capability cap=(major, minor)? A GPU runs code built for its own
    major version and a lower or equal minor (sm_61 runs sm_60), or PTX (compute_XY) at or below its capability."""
    for a in arch_list:
        kind, _, num = a.partition("_")
        if not num.isdigit():
            continue
        maj, mnr = int(num[:-1]), int(num[-1])
        if kind == "sm" and maj == cap[0] and mnr <= cap[1]:
            return True
        if kind == "compute" and (maj, mnr) <= tuple(cap):
            return True
    return False

if DEVICE.type == "cuda":
    _cap = torch.cuda.get_device_capability(0)
    _gpu = torch.cuda.get_device_name(0)
    _archs = torch.cuda.get_arch_list()
    _hint = (f"Install a PyTorch build that includes it: on Python <= 3.12 'pip install torch==2.4.0' (the authors' version); "
             f"otherwise a CUDA 12.6 build, 'pip install \"torch<2.15\" --index-url https://download.pytorch.org/whl/cu126'. "
             f"Older GPUs (e.g. Pascal sm_6x) are not in the CUDA 12.8+ / 13.x builds.")
    if not gpu_supported(_cap, _archs):
        raise RuntimeError(f"PyTorch {torch.__version__} (CUDA {torch.version.cuda}) has no code for this GPU: {_gpu}, "
                           f"compute capability {_cap[0]}.{_cap[1]}; this build supports {_archs}. {_hint}")
    try:
        torch.ones(8, device=DEVICE).add_(1).sum().item()          # a real kernel launch, not just the device query
    except Exception as ex:
        raise RuntimeError(f"PyTorch cannot run on {_gpu} (capability {_cap[0]}.{_cap[1]}): {ex}. {_hint}")
    print(f"GPU: {_gpu}, compute capability {_cap[0]}.{_cap[1]}: supported by PyTorch {torch.__version__} (CUDA {torch.version.cuda})")
# Plain float32 everywhere: Ampere+ GPUs (e.g. RTX 3090) would otherwise run convolutions in TF32 (a T4 cannot)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)   # plan: PyTorch's deterministic algorithms on
if PIPELINE:
    from threadpoolctl import threadpool_limits
    threadpool_limits(1)                 # same as the environment variables above, in case NumPy was already loaded
    torch.set_num_threads(max(1, N_CPU // 4) if DEVICE.type == "cpu" else 1)
import threading
READOUT_EPOCHS = 15            # authors' value (part of the replication, not a tuning knob)

def _ram_gb():
    try:
        with open("/proc/meminfo") as f:
            return round(int(f.readline().split()[1]) / 1e6, 1)
    except Exception:
        return None

ENV = {
    "python": platform.python_version(), "platform": platform.platform(),
    "torch": torch.__version__, "torch_pinned_2_4_0": TORCH_PINNED, "snntorch": snntorch.__version__,
    "sklearn": sklearn.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
    "pandas": pd.__version__, "wfdb": wfdb.__version__, "device": str(DEVICE),
    "cpu_count": N_CPU, "ram_gb": _ram_gb(), "on_kaggle": ON_KAGGLE, "machine": MACHINE, "smoke_test": SMOKE_TEST,
    "pipeline": PIPELINE, "cpu_model": next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "?")
                 if os.path.exists("/proc/cpuinfo") else platform.processor(),
}
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    ENV.update({"gpu": p.name, "gpu_sm_count": p.multi_processor_count, "gpu_capability": f"{p.major}.{p.minor}",
                "gpu_mem_gb": round(p.total_memory / 1e9, 1), "cuda": torch.version.cuda})
try:
    import brian2
    ENV["brian2"] = brian2.__version__
except Exception as e:
    ENV["brian2"] = f"unavailable ({e.__class__.__name__})"

if SMOKE_TEST:
    SEEDS = SEEDS[:1]
    GAINS = [1, 21]
    DENSE_LAMBDAS = DENSE_LAMBDAS[:1]
    RETUNE_TRIALS = 2
    NN_EPOCHS = 1
    BRIAN2_BEATS = 2
    NN_TUNE_TRIALS = 2
    NN_TUNE_MAX_EPOCHS = 2
    NN_TUNE_CHECKPOINTS = [1, 2]
    HUANG_MATRICES = 2
    WORKING_DIAG_BEATS = 40
    SAVE_STATES = [s for s in SAVE_STATES if s["seed"] in SEEDS]
SMOKE_CONFIGS   = [11, 12, 13, 22]       # used only when SMOKE_TEST
SMOKE_CV_SPLITS = [(1, 1)]               # used only when SMOKE_TEST
SMOKE_PER_CLASS = {0: 10, 1: 5, 2: 5}    # beats kept per record and class when SMOKE_TEST

for k, v in ENV.items():
    print(f"{k:>20}: {v}")
if PIN_TORCH_240 and not TORCH_PINNED:
    print("NOTE: torch 2.4.0 is not active; the exact-replication comparison will still run but cannot match.")


## 2. The authors' code (pinned commit)

In [ ]:
# The authors' code, cloned at a pinned commit. Their functions are used directly;
# bugs are patched only inside the "fixed" liquid modes (see the liquid cell).
REPO_DIR = os.path.join(SCRATCH_DIR, "authors_repo")
os.makedirs(SCRATCH_DIR, exist_ok=True)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "-q", AUTHORS_REPO, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", AUTHORS_COMMIT], check=True)
ENV["authors_commit"] = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                                       capture_output=True, text=True).stdout.strip()
assert ENV["authors_commit"] == AUTHORS_COMMIT, ENV["authors_commit"]
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.preprocessing import signals as A_sig
from src.preprocessing import stratification as A_strat
from src.encoding import spike_coding as A_enc
from src.utils.reproducibility import seed_everything

def _load_script(name, rel):
    spec = importlib.util.spec_from_file_location(name, os.path.join(REPO_DIR, rel))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)          # only imports + definitions; their main blocks are guarded
    return mod

A03 = _load_script("authors_cv_script", "scripts/03_run_24_configurations_cv.py")
LSM3D = A03.LiquidStateMachine3D           # identical class in scripts 02 and 03 (checked)

AUTHORS_CONFIGS = pd.read_csv(os.path.join(REPO_DIR, "configs/all_24_configurations.csv"))
ENC_SHORT = {"multithreshold": "multi", "temporal_rate": "rate", "bounded_delay_latency": "latency"}
CONFIGS = {int(r.config_id): {"config_id": int(r.config_id), "filter": r.filtering, "norm": r.normalization,
                              "encoder": ENC_SHORT[r.encoding], "readout": "svm" if r.readout_loss == "hinge" else "lr",
                              "authors_csv": r.source_result_file}
           for r in AUTHORS_CONFIGS.itertuples()}
print(f"authors' code at {ENV['authors_commit'][:10]}; {len(CONFIGS)} configurations loaded")


## 3. The redesign's code (lsm2)

In [ ]:
# The redesign's code: the lsm2 package and its tests (level 1a), written to SCRATCH_DIR and imported.
# The same files are shipped next to this notebook (lsm2/, tests/); this cell is generated from them.
LSM2_FILES = {}
LSM2_FILES['lsm2/__init__.py'] = r''''''
LSM2_FILES['lsm2/brian2_ref.py'] = r'''"""Level 1b and 1e references in Brian2.

brian2_ours: Brian2 runs our per-sub-step update as explicit code, in the same order of operations (level 1b).
  Brian2 step k+1 computes our sub-step k (the same one-step shift the replication's Brian2 check used):
    'start'      neurons: currents, integrate, threshold, reset (our steps 2-4), using the pushes that arrived
    'groups'     synapses: recover / fade over one sub-step (our step 1, first half), for our next sub-step
    'thresholds' our sub-step's spikes are emitted (closed loop) or replayed from a recording (open loop)
    'synapses'   arrivals for our next sub-step: dynamic synapses update u, push, R (our step 1, second half)
  A delay of d sub-steps is a Brian2 delay of d - 1 steps; input spikes of sample t are delivered in Brian2
  step t*n so they act in our sub-step t*n.
"""
import numpy as np


def _b2():
    import brian2 as b2
    b2.BrianLogger.log_level_error()
    try:
        b2.prefs.codegen.target = "cython"
        b2.start_scope()
        b2.NeuronGroup(1, "v:1").v = 0        # probe that cython compiles here
    except Exception:
        b2.prefs.codegen.target = "numpy"
    return b2


def brian2_ours(liq, x, u0, T, g=1.0, in_scale=1.0, open_loop=None, target="numpy"):
    """One beat. x: (T_in, C) input spikes per input step; u0: (N,); open_loop: (T*n, N) presynaptic spikes
    (None = closed loop). Returns traces: u, I1, I2, I3 (T*n, N), synapse states R, u at the end of each input
    step (T, S), and the spike raster (T*n, N)."""
    import brian2 as b2
    b2.BrianLogger.log_level_error()
    b2.prefs.codegen.target = target
    b2.start_scope()
    n, N, h = liq.n_sub, liq.N, liq.h_ms * 1e-3
    b2.defaultclock.dt = h * b2.second
    fine = liq.cur_a is not None
    dyn = liq.syn_model == "markram"
    eqs = """
    u : 1
    ref : 1
    IE : 1
    II : 1
    Iin : 1
    accE : 1
    accI : 1
    accIn : 1
    lastE : 1
    lastI : 1
    lastIn : 1
    spiked : 1
    beta : 1 (constant)
    tap : 1 (constant)
    gap : 1 (constant)
    a_in : 1 (constant)
    """
    G = b2.NeuronGroup(N, eqs, threshold="spiked > 0.5", reset="", method="euler", name="liquid")
    G.u = u0
    G.beta, G.tap, G.gap = liq.beta, liq.tap, liq.gap.astype(float)
    G.a_in = liq.in_a if liq.in_a is not None else 0.0
    aE = float(liq.cur_a[0]) if fine else 0.0
    aI = float(liq.cur_a[1]) if fine else 0.0
    reset = ("u_new - spk" if liq.reset_mode == "subtract" else f"spk*{liq.reset_value!r} + (1 - spk)*u_new")
    thr = "u_new - int(u > 1.0) > 1.0" if liq.snn_quirk else ("u_new > 1.0" if liq.strict else "u_new >= 1.0")
    code = f"""
    active = int(t > 0.5*dt)
    blocked = int(ref > 0.5)
    IE = active*({aE!r}*IE + (1 - {aE!r})*accE) + (1 - active)*IE
    II = active*({aI!r}*II + (1 - {aI!r})*accI) + (1 - active)*II
    Iin = active*(a_in*Iin + (1 - a_in)*accIn) + (1 - active)*Iin
    total = {'IE + II + Iin' if fine else 'accE + accI + accIn'}
    u_new = beta*u + total + tap
    {'u_new = blocked*u + (1 - blocked)*u_new' if liq.clamp else ''}
    spk = int({thr})*(1 - blocked)*active
    u_next = {reset}
    u = active*u_next + (1 - active)*u
    ref = active*(spk*(gap - 1) + (1 - spk)*clip(ref - 1, 0, 1e9)) + (1 - active)*ref
    lastE = {'IE' if fine else 'accE'}
    lastI = {'II' if fine else 'accI'}
    lastIn = {'Iin' if fine else 'accIn'}
    accE = 0
    accI = 0
    accIn = 0
    spiked = spk
    """
    G.run_regularly(code, when="start", name="update")
    objs = [G]
    # input: channel spikes of sample t delivered at Brian2 step t*n
    xs = np.asarray(x)
    tt, cc = np.nonzero(xs[:min(len(xs), T)])
    Win = liq.Win * in_scale
    if len(tt):
        inp = b2.SpikeGeneratorGroup(xs.shape[1], cc, (tt * n) * h * b2.second, name="inp")
        ci, nj = np.nonzero(Win)
        Sin = b2.Synapses(inp, G, "w : 1 (constant)", on_pre="accIn_post += w", name="sin")
        Sin.connect(i=ci, j=nj); Sin.w = Win[ci, nj]
        objs += [inp, Sin]
    # presynaptic source
    if open_loop is not None:
        kk, ii = np.nonzero(np.asarray(open_loop))
        src = b2.SpikeGeneratorGroup(N, ii, (kk + 1) * h * b2.second, name="src")
        objs.append(src)
    else:
        src = G
    syn_mons = []
    if liq.S:
        for ti in range(4):
            m = np.nonzero(liq.typ == ti)[0]
            if not len(m):
                continue
            acc = "accE_post" if ti < 2 else "accI_post"
            d = int(liq.delay[ti])
            if dyn:
                model = """
                wg : 1 (constant)
                U : 1 (constant)
                eD : 1 (constant)
                eF : 1 (constant)
                R : 1
                q : 1
                """
                on_pre = f"""
                q = q + U*(1 - q)
                {acc} += wg*q*R
                R = R - q*R
                """
            else:
                model, on_pre = "wg : 1 (constant)", f"{acc} += wg"
            S = b2.Synapses(src, G, model, on_pre=on_pre, delay=(d - 1) * h * b2.second, name=f"syn{ti}")
            S.connect(i=liq.pre[m], j=liq.post[m])
            if dyn:
                S.wg = liq.w[m] * g / liq.Ubar[ti]
                S.U, S.eD, S.eF = liq.U[m], np.exp(-liq.h_ms / liq.D_ms[m]), np.exp(-liq.h_ms / liq.F_ms[m])
                S.R, S.q = 1.0, 0.0
                S.run_regularly("R = 1 - (1 - R)*eD\nq = q*eF", when="groups", name=f"decay{ti}")
                mon = b2.StateMonitor(S, ["R", "q"], record=True, when="end", name=f"smon{ti}")
                syn_mons.append((m, mon))
                objs.append(mon)
            else:
                S.wg = liq.w[m] * g
            objs.append(S)
    M = b2.StateMonitor(G, ["u", "lastE", "lastI", "lastIn"], record=True, when="end", name="mon")
    spm = b2.SpikeMonitor(G, name="spm")
    objs += [M, spm]
    net = b2.Network(*objs)
    net.run((T * n + 1) * h * b2.second)
    out = {k: np.asarray(getattr(M, v))[:, 1:].T for k, v in (("u", "u"), ("I1", "lastE"), ("I2", "lastI"), ("I3", "lastIn"))}
    r = np.zeros((T * n, N), bool)
    k = np.round(np.asarray(spm.t / b2.second) / h).astype(int) - 1
    ok = (k >= 0) & (k < T * n)
    r[k[ok], np.asarray(spm.i)[ok]] = True
    out["raster"] = r
    if dyn and syn_mons:
        R = np.zeros((T, liq.S)); q = np.zeros((T, liq.S))
        last = np.arange(T) * n + n - 1          # Brian2 step k ('end') holds our sub-step k's synapse state
        for m, mon in syn_mons:
            R[:, m] = np.asarray(mon.R).T[last]
            q[:, m] = np.asarray(mon.q).T[last]
        out["R"], out["q"] = R, q
    return out


# ---------------------------------------------------------------- level 1e: (d)'s script, reused unchanged
D_REPLACEMENT_LINES = {
    "synapses.U[synapses.U < 0] = U": "synapses.U[np.nonzero(np.asarray(synapses.U) < 0)[0]] = U",
    "synapses.D[synapses.D / ms <= 0] = D": "synapses.D[np.nonzero(np.asarray(synapses.D / ms) <= 0)[0]] = D",
    "synapses.F[synapses.F / ms <= 0] = F": "synapses.F[np.nonzero(np.asarray(synapses.F / ms) <= 0)[0]] = F",
}


def script_module(path, fix_replacement=True):
    """Load (d)'s maass2002.py (its main block is guarded), so its own functions build and run the liquid.

    (d) means to replace bad U, D, F draws by the mean, but Brian2 reads a boolean array index as the integer
    indices 0 and 1: as written, only synapses 0 and 1 of each group get the mean and every negative draw stays
    (about 2.3% of synapses each for U, D and F; a negative D or F makes x or u grow without bound).
    fix_replacement=True patches those three lines so they do what (d) intends (and what the plan describes)."""
    import types
    src = open(path).read()
    if fix_replacement:
        for a, b in D_REPLACEMENT_LINES.items():
            if a not in src:
                raise RuntimeError(f"(d)'s script changed: line not found: {a}")
            src = src.replace(a, b)
    mod = types.ModuleType("maass2002_d")
    mod.__file__ = path
    exec(compile(src, path, "exec"), mod.__dict__)
    return mod
'''
LSM2_FILES['lsm2/calib.py'] = r'''"""Input size (plan, Input and calibration): the paper's rule, or with factor 5 a bisection on one factor s so the
input-receiving neurons fire, with recurrence off, at the rate Maass's own input-receiving neurons reach in the
Maass check (paper version). Factor 9 fixes Maass's own per-spike size instead."""
import math
import numpy as np

from . import config as C
from .liquid import paper_rule_scale
from .measure import simulate, firing_stats
from .sim import start_voltages


def ain_of(spikes):
    """Mean number of input spikes per sample, summed over channels (the authors' A_in)."""
    return float(np.asarray(spikes, np.float64).sum() / (spikes.shape[0] * spikes.shape[1]))


def input_neuron_rate(ctx, liqs, s, spikes, T=C.BEAT_SAMPLES):
    """Mean rate (Hz) of the input-receiving neurons, recurrence off, averaged over the liquids (seeds)."""
    rates = []
    for liq in liqs:
        u0 = start_voltages(liq, liq.meta.get("seed", 0), range(len(spikes)), 5)
        o = simulate(ctx, liq, 0.0, s, spikes, u0, T, {"neuron_counts": (0, T)}, recurrence=False)
        cnt = o["neuron_counts"][:, liq.in_neurons]
        rates.append(firing_stats(cnt, T, liq.h_ms * liq.n_sub)[0])
    return float(np.mean(rates))


def bisect_scale(ctx, liqs, spikes, target_hz, tol=C.CODE["calib_tol"], iters=C.CODE["calib_iters"], log=print):
    """Smallest bracket on s (log scale) until the rate is within tol of the target."""
    lo, hi = 0.0, 0.05
    r_hi = input_neuron_rate(ctx, liqs, hi, spikes)
    k = 0
    while r_hi < target_hz and k < 30:
        lo, hi = hi, hi * 2
        r_hi = input_neuron_rate(ctx, liqs, hi, spikes)
        k += 1
    if r_hi < target_hz:
        raise RuntimeError(f"input calibration: rate {r_hi:.2f} Hz < target {target_hz:.2f} Hz even at s = {hi}")
    best, best_r = hi, r_hi
    for i in range(iters):
        mid = math.sqrt(lo * hi) if lo > 0 else hi / 2
        r = input_neuron_rate(ctx, liqs, mid, spikes)
        if abs(r - target_hz) < abs(best_r - target_hz):
            best, best_r = mid, r
        if abs(r - target_hz) <= tol * target_hz:
            break
        if r < target_hz:
            lo = mid
        else:
            hi = mid
    log(f"    factor-5 input size s = {best:.5g} (rate {best_r:.3f} Hz, target {target_hz:.3f} Hz); Maass's own 0.12")
    return float(best), float(best_r)


def input_scale(ctx, liqs, spikes_cal, ain, maass_target_hz, log=print):
    """The candidate's input size: (scale, how it was set)."""
    liq = liqs[0]
    if liq.in_scale_fixed is not None:
        return float(liq.in_scale_fixed), {"rule": "fixed (factor 9 or Maass's own)", "scale": float(liq.in_scale_fixed)}
    if 5 in liq.factors:
        if maass_target_hz is None:
            raise RuntimeError("factor 5 needs the Maass check's input-neuron rate")
        s, r = bisect_scale(ctx, liqs, spikes_cal, maass_target_hz, log=log)
        return s, {"rule": "factor 5 calibration", "scale": s, "rate_hz": r, "target_hz": maass_target_hz}
    s = paper_rule_scale(ain, liq)
    fill_E = C.I_TARGET / (1 - float(liq.beta[liq.is_exc][0] ** liq.n_sub))
    fill_I = C.I_TARGET / (1 - float(liq.beta[~liq.is_exc][0] ** liq.n_sub))
    return s, {"rule": "paper's rule", "scale": s, "A_in": ain, "P_in": 1.0 if 6 in liq.factors else C.X["P_in"],
               "fill_E": fill_E, "fill_I": fill_I}
'''
LSM2_FILES['lsm2/config.py'] = r'''"""Every fixed number of the redesign plan, with its source or reason.

Sources (as labelled in the plan):
  (a) Perez-Lopez et al. 2026 and the authors' code, commit 02e45c4
  (c) Maass, Natschlager & Markram 2002, Section 5 and Appendix B
  (d) Schmitt 2022, Brian2 reproduction of (c)'s Figure 2 (maass2002.py)
  (e) Legenstein & Maass 2007
Numbers marked "plan" are the plan's own choices; the reason is in the plan (section named).
Numbers marked "code" were not fixed by the plan; their reason is given here and listed in NOTES.md.
"""
import math

# ---------------------------------------------------------------- time
FS = 360.0                          # MIT-BIH sampling rate (Hz), (a)
SAMPLE_MS = 1000.0 / FS             # 2.7778 ms, one ECG sample = one step of (a)'s liquid
BEAT_SAMPLES = 577                  # (a): beats are R-0.8 s .. R+0.8 s at 360 Hz
SIM_SAMPLES = 650                   # (a): num_sim_steps = 650 (input is zero after the beat)
R_SAMPLE = 288                      # plan, Readouts: the R peak is sample 288 of the 577-sample window
WIN, NWIN = 20, 32                  # (a): spike counts in 32 windows of 20 samples (film readout)
SNAPSHOT_SAMPLES = (288, 342, 396)  # plan, Readouts: R, R + 150 ms, R + 300 ms
STATE_TAU_MS = 30.0                 # (c): liquid states are spikes filtered with a 30 ms exponential
N_SUB_REFERENCE = 28                # plan, Time grid: 28 sub-steps per sample (0.0992 ms) is the reference grid

# ---------------------------------------------------------------- X: (a)'s liquid, both bugs fixed
X = dict(
    betaE=0.95, betaI=0.80,         # (a) leak per 2.778 ms step (about 54 / 12 ms)
    threshold=1.0,                  # (a) Uthr
    rho=0.9,                        # (a) target fill of the paper's input rule
    excit_ratio=0.8,                # (a) and (c)
    C={"EE": 0.3, "EI": 0.2, "IE": 0.4, "II": 0.1},   # (a) and (c)
    lam=0.8,                        # (a)
    mu_rsv={"EE": 0.020, "EI": 0.025, "IE": 0.050, "II": 0.040},   # (a) type means
    P_in=0.1,                       # (a) code line 691
    dims=(7, 7, 7),                 # (a) 343 neurons
)
I_TARGET = X["rho"] * (1 - X["betaE"]) * X["threshold"]     # 0.045: (a)'s rule, mean input per step
TYPES = ("EE", "EI", "IE", "II")    # pre-post

# ---------------------------------------------------------------- Maass (c), in the paper's units (threshold 1)
MAASS = dict(
    tau_m_ms=30.0,                  # (c) membrane time constant
    v_thresh_mV=15.0, v_reset_mV=13.5, I_b_nA=13.5, R_Mohm=1.0,   # (c); background 13.5 mV of 15
    refrac_ms={"E": 3.0, "I": 2.0},  # (c)
    tau_s_ms={"E": 3.0, "I": 6.0},   # (c) synaptic current decay by presynaptic type
    tau_in_ms=3.0,                   # plan: input synapses are excitatory, so 3 ms for every target
    delay_ms={"EE": 1.5, "EI": 0.8, "IE": 0.8, "II": 0.8},   # (c)
    A_nA={"EE": 30.0, "EI": 60.0, "IE": -19.0, "II": -19.0},  # (c)
    U={"EE": 0.5, "EI": 0.05, "IE": 0.25, "II": 0.32},          # (c)
    D_s={"EE": 1.1, "EI": 0.125, "IE": 0.7, "II": 0.144},       # (c)
    F_s={"EE": 0.05, "EI": 1.2, "IE": 0.02, "II": 0.06},        # (c)
    udf_spread=0.5,                  # (c): SD 50% of the mean
    lam=2.0,                         # (c)
    in_nA={"E": 18.0, "I": 9.0},     # (c) input amplitudes
    in_fraction=0.3,                 # (c) input injected into 30% of neurons
    dims=(15, 3, 3),                 # (c) 135-neuron column
    input_rate_hz=20.0,              # (d) (and (c)'s Figure 4 task)
)
TAP = MAASS["I_b_nA"] * MAASS["R_Mohm"] / MAASS["v_thresh_mV"]    # 0.9 of threshold
RESET_TAP = MAASS["v_reset_mV"] / MAASS["v_thresh_mV"]            # 0.9

def _push(A_nA, tau_s_ms, U=1.0):
    """Voltage push (in thresholds) of one current pulse A*exp(-t/tau_s) through R into a 30 ms membrane:
    A * U * tau_s * R / (tau_m * V_thresh), the tap's convention (plan, factors 9 and 10)."""
    return A_nA * 1e-9 * U * tau_s_ms * 1e-3 * MAASS["R_Mohm"] * 1e6 / (MAASS["tau_m_ms"] * 1e-3 * MAASS["v_thresh_mV"] * 1e-3)

_pre = {"EE": "E", "EI": "E", "IE": "I", "II": "I"}
MAASS_PUSH = {t: abs(_push(MAASS["A_nA"][t], MAASS["tau_s_ms"][_pre[t]], MAASS["U"][t])) for t in TYPES}
# EE 0.1, EI 0.02, IE 0.0633, II 0.0811 (plan, factor 3: ratios 1 / 0.2 / 0.63 / 0.81; factor 10: EE 0.1)
MAASS_RATIOS = {t: MAASS_PUSH[t] / MAASS_PUSH["EE"] for t in TYPES}
MAASS_IN_PUSH = {k: _push(MAASS["in_nA"][k], MAASS["tau_in_ms"]) for k in ("E", "I")}   # 0.12 / 0.06 (factor 9)
MAASS_EE_PUSH = MAASS_PUSH["EE"]   # 0.1 (factor 10)

# ---------------------------------------------------------------- the 11 factors (plan table)
FACTOR_NAMES = {
    1: "reach lambda", 2: "code-only normalisations", 3: "E/I balance", 4: "leak", 5: "operating point",
    6: "input wiring", 7: "fine timing", 8: "dynamic synapses", 9: "input size", 10: "weight scale",
    11: "size and shape",
}
ALL_FACTORS = frozenset(range(1, 12))
LEAK_MS_Y = 30.0                    # factor 4: (c)'s 30 ms for E and I

# ---------------------------------------------------------------- gain grid (plan, Choosing the recurrent gain; decision 5)
G_LOW = (0.1, 0.3)
G_COARSE = (1, 3, 10, 30, 100, 300, 1000)
G_EXTEND_STEP, G_EXTEND_MAX = 3.0, 3
G_FINE_STEP, G_FINE_K = 1.4, 3      # g* x 1.4^k, k = -3..3 (1.4^3 ~ 2.7 splits a 3x step into three)

# ---------------------------------------------------------------- level thresholds (plan, Correctness criteria)
TOL_1A = 1e-9                       # 1a, 1b open loop (float64)
TOL_1C = 1e-4                       # 1c float32 vs float64
LEVEL2_SE = 4.5                     # level 2 (decision 24)
BAND_HZ = (2.0, 40.0)               # level 3 firing band (decision 7)
HIGH_HZ, HIGH_FRAC = 100.0, 0.05    # at most 5% of neurons above 100 Hz (decision 7)
N_CAL_BEATS = 500                   # plan: 500 beats for level 3 and the rank scores (decision 8)
GEN_BEATS, GEN_COPIES = 4, 125      # decision 8: 4 beats x 125 jittered copies
JITTER_MS = 4.0                     # decision 8: (e)'s 4 ms jitter for its 30-channel patterns
STATE_READ_SAMPLE = 396             # decision 8: states read at R + 300 ms
START_TEST_SAMPLE = 576             # level 3: starting-state rule judged at sample 576
MEMORY_T_MS = (0, 25, 50, 100, 200, 400)   # level 4 lags
MEMORY_PASS_T_MS = 100              # level 4 pass lag (plan: well past 30-54 ms membranes and 3-6 ms currents)
MEMORY_MIN_GAIN = 0.05              # level 4 (decision 9)
MEMORY_WARMUP = 100                 # level 4: 278 ms warm-up
SURROGATE_TOL = 0.10                # level 4: surrogate spikes per sample within 10% of ECG's
E1D_N_TRY_MAX = 28                  # 1d: n = 1, 2, 3, ... up to the reference grid
E1E_T_CRIT = 4.2                    # 1e: t at 8 df, two-sided 0.3%
E1E_FRAC = 0.95                     # 1e: 95% of time points
E1E_RUNS = 5                        # 1e: 5 runs per side

SEEDS_SEARCH = (1, 2, 3)            # code: liquid seeds for the ladder search (the plan fixes 3 seeds, not their values)
SEEDS_CONFIRM = (101, 202, 303)     # plan, confirmation
SEEDS_RETRY = (404, 505, 606)       # plan, confirmation retry
SEEDS_TEST_REDESIGN = (707, 808, 909)   # plan, Maass check redesign
SEEDS_CLASSIFY = (2026, 69, 420)    # plan, ground rules

# ---------------------------------------------------------------- code choices the plan does not fix (see NOTES.md)
CODE = dict(
    twin_beats=100,          # twin-run cross-check per g and seed: a reported cross-check, 100 beats give a stable mean ratio
    twin_early=(289, 306),   # samples of the first 50 ms after the extra spike
    twin_late=(559, 576),    # samples of the last 50 ms of the beat
    memory_sequences=500,    # level 4: same count as level 3's beats; 250 train / 250 test sequences
    ridge_alphas=tuple(10.0 ** k for k in range(-3, 5)),   # level 4 inner-CV penalty grid
    ridge_inner_folds=5,     # level 4 inner CV (grouped by sequence)
    calib_tol=0.01,          # factor 5: bisection stops at 1% of the target rate (well inside seed-to-seed spread)
    calib_iters=40,          # factor 5: bisection cap
    search_split_seed=0,     # DS1 patients halved by a fixed random permutation of the 22 records
    sim_batch=500,           # beats per GPU batch (fixed on every machine: plan, determinism)
    x_update="equations",    # X's reset: "equations" = (a)'s Eqs. 17-19 (the plan); "authors_code" = (a)'s snnTorch
                             # Leaky, whose threshold test subtracts 1 more while the stored voltage is above 1
)
'''
LSM2_FILES['lsm2/data.py'] = r'''"""Label-free inputs for the search: the DS1 halves, the 500 calibration beats, the generalisation and twin sets,
the memory test's random surrogate, and Poisson trains for the Maass check."""
from dataclasses import dataclass
import numpy as np

from . import config as C
from .measure import level3_sets, surrogate_walks


def multithreshold(X, num_levels=200):
    """The authors' Multi-threshold encoder (detect_edges='both', include_endpoints=True): a spike on threshold j
    at sample t when x crosses it between t-1 and t. X: (n, T) in [0, 1] -> (n, T, K) uint8."""
    X = np.asarray(X)
    thr = np.linspace(0.0, 1.0, num_levels, dtype=X.dtype)
    above = X[:, :, None] >= thr
    out = np.zeros(above.shape, np.uint8)
    out[:, 1:] = (above[:, 1:] != above[:, :-1])
    return out


def split_records(records, seed=C.CODE["search_split_seed"]):
    """DS1's patients halved without labels: a fixed random permutation of the record list."""
    rec = sorted(records)
    perm = np.random.default_rng(seed).permutation(len(rec))
    half = len(rec) // 2
    return sorted(rec[i] for i in perm[:half]), sorted(rec[i] for i in perm[half:])


@dataclass
class SearchData:
    name: str
    sets: dict                 # sep (500, T, C), gen (500, T, C), twin (100, T, C)
    ain: float                 # input spikes per sample (A_in) on this half, labels unused
    mem_spikes: np.ndarray     # (500, T, C) surrogate encoded
    mem_targets: np.ndarray    # (500, T) input spike count per sample
    surrogate_ain: float
    real_spikes: np.ndarray = None    # secondary memory check on real beats
    real_targets: np.ndarray = None
    meta: dict = None


def make_search_data(name, beats, encode, seed, n_cal=None, n_mem=None):
    """beats: (n, 577) normalised config-11 beats of one half, in a fixed order (labels never used).
    encode: (m, 577) -> (m, 577, C) spikes."""
    n_cal = n_cal or C.N_CAL_BEATS
    n_mem = n_mem or C.CODE["memory_sequences"]
    rng = np.random.default_rng([seed, 51])
    pick = np.sort(rng.choice(len(beats), n_cal + C.GEN_BEATS, replace=False))
    perm = rng.permutation(pick)
    sep_idx, gen_idx = np.sort(perm[:n_cal]), np.sort(perm[n_cal:])
    sep = encode(beats[sep_idx])
    gen_base = encode(beats[gen_idx])
    sets = level3_sets(sep, gen_base, seed_sets=seed)
    # A_in over the whole half (the authors' quantity: input spikes per sample, summed over channels)
    tot, cnt = 0.0, 0
    for s in range(0, len(beats), 2000):
        e = encode(beats[s:s + 2000]); tot += float(e.sum(dtype=np.int64)); cnt += e.shape[0] * e.shape[1]
    ain = tot / cnt
    steps = np.abs(np.diff(beats, axis=1)).ravel()
    walks = surrogate_walks(steps, n_mem, beats.shape[1], seed)
    mem = encode(walks)
    targets = mem.sum(2).astype(np.int64)
    s_ain = float(mem.sum(dtype=np.int64) / (mem.shape[0] * mem.shape[1]))
    # secondary check: real beats, target = deviation from the R-aligned mean beat
    mean_beat = beats.mean(0)
    real_targets = beats[sep_idx] - mean_beat[None, :]
    return SearchData(name=name, sets=sets, ain=ain, mem_spikes=mem, mem_targets=targets, surrogate_ain=s_ain,
                      real_spikes=sep, real_targets=real_targets,
                      meta={"n_beats_half": int(len(beats)), "sep_idx": sep_idx.tolist(), "gen_idx": gen_idx.tolist(),
                            "surrogate_vs_ecg": s_ain / ain})


def poisson_trains(n, T, rate_hz, seed, stream, step_ms=C.SAMPLE_MS):
    """n Poisson trains of T samples, counts per sample (one channel)."""
    rng = np.random.default_rng([seed, stream])
    return rng.poisson(rate_hz * step_ms / 1000.0, (n, T, 1)).astype(np.uint8)


def make_maass_data(seed, T=C.BEAT_SAMPLES, n_cal=None, n_mem=None):
    """The Maass check's inputs: 20 Hz Poisson trains of beat length stand in for beats."""
    n_cal = n_cal or C.N_CAL_BEATS
    n_mem = n_mem or C.CODE["memory_sequences"]
    r = C.MAASS["input_rate_hz"]
    sep = poisson_trains(n_cal, T, r, seed, 61)
    gen_base = poisson_trains(C.GEN_BEATS, T, r, seed, 62)
    sets = level3_sets(sep, gen_base, seed_sets=seed)
    mem = poisson_trains(n_mem, T, r, seed, 63)
    return SearchData(name="maass_poisson", sets=sets, ain=float(sep.mean() * 1), mem_spikes=mem,
                      mem_targets=mem[:, :, 0].astype(np.int64), surrogate_ain=float(mem.mean()), meta={"rate_hz": r})
'''
LSM2_FILES['lsm2/gain.py'] = r'''"""Choosing the recurrent gain g (plan, Choosing the recurrent gain; decision 5).

Grid: g = 0 (control), 0.1 and 0.3, a coarse pass at 1, 3, ..., 1000 times the base weights, extensions in 3x
steps at an edge (at most 3), then a fine pass g* x 1.4^k, k = -3..3. Each setting is measured on all seeds and
averaged. Rule: among settings above 0 inside the firing band, the largest mean gap; a setting whose mean gap lies
within the range of the best setting's seed values is tied with it, and the smallest tied g is taken."""
import numpy as np

from . import config as C


def _agg(per_seed):
    keys = [k for k in per_seed[0] if isinstance(per_seed[0][k], (int, float, bool, np.floating, np.integer))]
    m = {k: float(np.nanmean([float(r[k]) for r in per_seed])) for k in keys}
    m["gap_seeds"] = [float(r["gap"]) for r in per_seed]
    m["rate_seeds"] = [float(r["rate_hz"]) for r in per_seed]
    m["in_band"] = bool(C.BAND_HZ[0] <= m["rate_hz"] <= C.BAND_HZ[1] and m["frac_above_100hz"] <= C.HIGH_FRAC)
    m["per_seed"] = per_seed
    return m


def choose(table):
    """table: {g: aggregated measures}. Returns (chosen g or None, best g or None)."""
    cand = [g for g in table if g > 0 and table[g]["in_band"]]
    if not cand:
        return None, None
    best = max(cand, key=lambda g: (table[g]["gap"], -g))
    lo, hi = min(table[best]["gap_seeds"]), max(table[best]["gap_seeds"])
    eps = 1e-12 * max(1.0, abs(lo), abs(hi))                 # the mean of equal seed values may round
    tied = [g for g in cand if lo - eps <= table[g]["gap"] <= hi + eps] + [best]
    return min(tied), best


def edge_status(table, g):
    """'upper' / 'lower' if g sits at that edge of the measured grid (g = 0 excluded), else None. A setting next
    to one outside the firing band counts as inside the grid (plan, Grid)."""
    gs = sorted(x for x in table if x > 0)
    i = gs.index(g)
    if i == len(gs) - 1:
        return "upper"
    if i == 0:
        return "lower"
    return None


def band_limited(table, g):
    gs = sorted(x for x in table if x > 0)
    i = gs.index(g)
    up = i + 1 < len(gs) and not table[gs[i + 1]]["in_band"]
    down = i > 0 and not table[gs[i - 1]]["in_band"]
    return bool(up or down)


def sweep(measure, seeds, log=print, coarse=None):
    """measure(g, seed) -> level-3 measures at g (measure.measure_at_g). Returns the sweep record."""
    table, order = {}, []

    def run(gs, stage):
        for g in gs:
            g = float(g)
            if g in table:
                continue
            table[g] = _agg([measure(g, s) for s in seeds])
            table[g]["stage"] = stage
            order.append(g)
            t = table[g]
            log(f"    g={g:<10.4g} abs={t['abs_g']:.4g} rate={t['rate_hz']:.2f} Hz in_band={t['in_band']} "
                f"sep={t['sep']:.2f} gen={t['gen']:.2f} gap={t['gap']:.2f}"
                + (f" twin={t['twin_ratio']:.2f}" if 'twin_ratio' in t else ""))

    run([0.0] + list(C.G_LOW) + list(coarse or C.G_COARSE), "coarse")
    chosen, best = choose(table)
    ext = 0
    while best is not None and ext < C.G_EXTEND_MAX:
        e = edge_status(table, best)
        if e is None:
            break
        gs = sorted(x for x in table if x > 0)
        nxt = gs[-1] * C.G_EXTEND_STEP if e == "upper" else gs[0] / C.G_EXTEND_STEP
        run([nxt], f"extension {ext + 1}")
        ext += 1
        chosen, best = choose(table)
    at_edge = best is not None and edge_status(table, best) is not None
    if best is not None and not at_edge:
        run([best * C.G_FINE_STEP ** k for k in range(-C.G_FINE_K, C.G_FINE_K + 1) if k != 0], "fine")
        chosen, best = choose(table)
        at_edge = edge_status(table, best) is not None
    rec = {"table": table, "order": order, "chosen_g": chosen, "best_g": best, "extensions": ext,
           "no_g_in_band": best is None, "peak_at_edge": bool(at_edge),
           "band_limited": bool(chosen is not None and band_limited(table, chosen))}
    if chosen is not None:
        gs = sorted(x for x in table if x > 0)
        i = gs.index(chosen)
        rec["next_higher_g"] = gs[i + 1] if i + 1 < len(gs) else None
        rec["next_higher_gap_lower"] = (rec["next_higher_g"] is not None
                                        and table[rec["next_higher_g"]]["gap"] < table[chosen]["gap"])
    rec["gap_peak_inside_grid"] = bool(best is not None and not at_edge)
    return rec
'''
LSM2_FILES['lsm2/ladder.py'] = r'''"""The X to Y ladder (plan, The X to Y ladder): test X and Y; forward from X adding factors 1 to 11 in order
until a candidate passes all four levels; backward loop removing each added factor, newest first, until a full
pass removes nothing; confirmation on the other half of DS1 with fresh seeds and frozen g and input size."""
import json
import time
import numpy as np

from . import config as C
from .liquid import build_candidate
from .measure import measure_at_g, memory_curve
from .calib import input_scale
from . import gain, level2


def key_of(factors):
    return "+".join(str(f) for f in sorted(factors)) or "X"


class Ladder:
    def __init__(self, ctx, search, confirm, n_fine, maass_target_hz, seeds=C.SEEDS_SEARCH, n_channels=200,
                 log=print, on_record=None, cache=None):
        self.ctx, self.search, self.confirm_data = ctx, search, confirm
        self.n_fine, self.maass_target = n_fine, maass_target_hz
        self.seeds, self.n_channels, self.log = tuple(seeds), n_channels, log
        self.records = dict(cache or {})
        self.on_record = on_record
        self.path = []

    def n_sub_for(self, factors, y_grid=None):
        f = frozenset(factors)
        if f == C.ALL_FACTORS:
            return y_grid or C.N_SUB_REFERENCE          # Y itself always on 28 sub-steps
        return self.n_fine if 7 in f else 1

    # ------------------------------------------------------------------ one candidate
    def evaluate(self, factors, data=None, seeds=None, frozen=None, y_grid=None, tag=None, n_channels=None):
        """All four levels for one candidate. frozen = {"g": ..., "in_scale": ...} for the confirmation
        (no sweep; level 3 is the band and the starting-state rule at the frozen g)."""
        f = frozenset(factors)
        data = data or self.search
        seeds = tuple(seeds or self.seeds)
        n = self.n_sub_for(f, y_grid)
        rkey = tag or f"{key_of(f)}|n{n}|{data.name}|{','.join(map(str, seeds))}"
        if rkey in self.records:
            return self.records[rkey]
        t0 = time.time()
        self.log(f"candidate {key_of(f)} (n = {n}, {data.name}, seeds {list(seeds)})")
        liqs = {s: build_candidate(f, s, n, n_channels or self.n_channels) for s in seeds}
        for s, l in liqs.items():
            l.meta["seed"] = s
        rec = {"factors": sorted(f), "key": key_of(f), "n_sub": n, "data": data.name, "seeds": list(seeds)}
        # level 2
        l2 = {s: level2.check(liqs[s]) for s in seeds}
        rec["level2"] = {"pass": all(v["pass"] for v in l2.values()),
                         "failed_items": {s: [i["item"] for i in v["items"] if not i["pass"]] for s, v in l2.items()},
                         "report": {s: v["report"] for s, v in l2.items()}}
        # input size
        if frozen is not None:
            scale, how = frozen["in_scale"], {"rule": "frozen", "scale": frozen["in_scale"]}
        else:
            scale, how = input_scale(self.ctx, list(liqs.values()), data.sets["sep"], data.ain, self.maass_target, log=self.log)
        rec["input"] = how
        # gain
        if frozen is None:
            measure = lambda g, s: measure_at_g(self.ctx, liqs[s], g, scale, data.sets, s)
            sw = gain.sweep(measure, seeds, log=self.log)
            rec["sweep"] = {"table": {repr(g): {k: v for k, v in t.items() if k != "per_seed"} for g, t in sw["table"].items()},
                            "per_seed": {repr(g): t["per_seed"] for g, t in sw["table"].items()},
                            **{k: v for k, v in sw.items() if k != "table"}}
            g_chosen = sw["chosen_g"]
            peak_ok = sw["gap_peak_inside_grid"]
        else:
            g_chosen, peak_ok, sw = frozen["g"], True, None
        if 10 in f:
            g_use = {s: liqs[s].g_maass for s in seeds}      # Maass's own scale instead of the rule's g
        else:
            g_use = {s: g_chosen for s in seeds}
        rec["g_rule"] = g_chosen
        rec["g_used"] = {str(s): g_use[s] for s in seeds}
        rec["abs_g_used"] = {str(s): (liqs[s].ee_mean_push(g_use[s]) if g_use[s] is not None else None) for s in seeds}
        rec["band_limited"] = bool(sw["band_limited"]) if sw else False
        # level 3 at the g used
        if g_chosen is None and 10 not in f:
            rec["level3"] = {"pass": False, "reason": "no setting inside the firing band"}
        else:
            rows = []
            for s in seeds:
                m = measure_at_g(self.ctx, liqs[s], g_use[s], scale, data.sets, s, with_start_test=True, with_twins=False)
                rows.append({"seed": s, "rate_hz": m["rate_hz"], "frac_above_100hz": m["frac_above_100hz"],
                             "in_band": m["in_band"], "start_rule": m["start_rule_pass"],
                             "start_same": m["start_same"], "start_diff": m["start_diff"], "gap": m["gap"]})
            ok = all(r["in_band"] and r["start_rule"] for r in rows) and peak_ok
            rec["level3"] = {"pass": bool(ok), "rows": rows, "gap_peak_inside_grid": bool(peak_ok)}
        # level 4 (memory curve reported for every candidate)
        if all(g_use[s] is not None for s in seeds):
            on = {s: memory_curve(self.ctx, liqs[s], g_use[s], scale, data.mem_spikes, data.mem_targets, s) for s in seeds}
            off = {s: memory_curve(self.ctx, liqs[s], g_use[s], scale, data.mem_spikes, data.mem_targets, s,
                                   recurrence=False) for s in seeds}
            L = C.MEMORY_PASS_T_MS
            gains = {s: on[s][f"r2_{L}"] - off[s][f"r2_{L}"] for s in seeds}
            surrogate_ok = abs(data.surrogate_ain / data.ain - 1) <= C.SURROGATE_TOL
            real = {}
            if data.real_spikes is not None:          # secondary check (reported): real beats, deviation from the mean beat
                for s in seeds:
                    a = memory_curve(self.ctx, liqs[s], g_use[s], scale, data.real_spikes, data.real_targets, s)
                    b = memory_curve(self.ctx, liqs[s], g_use[s], scale, data.real_spikes, data.real_targets, s, recurrence=False)
                    real[str(s)] = {"on": a, "off": b}
            rec["level4"] = {"pass": bool(surrogate_ok and all(np.isfinite(v) and v >= C.MEMORY_MIN_GAIN for v in gains.values())),
                             "gain_100ms": {str(s): v for s, v in gains.items()},
                             "on": {str(s): v for s, v in on.items()}, "off": {str(s): v for s, v in off.items()},
                             "surrogate_spikes_vs_ecg": data.surrogate_ain / data.ain, "surrogate_ok": bool(surrogate_ok),
                             "firing_on_surrogate_hz": {str(s): on[s]["rate_hz"] for s in seeds},
                             "real_beats_reported": real,
                             "firing_on_ecg_hz": {str(r["seed"]): r["rate_hz"] for r in rec["level3"].get("rows", [])}}
        else:
            rec["level4"] = {"pass": False, "reason": "no g to judge at"}
        rec["in_scale"] = scale
        rec["pass"] = bool(rec["level2"]["pass"] and rec["level3"]["pass"] and rec["level4"]["pass"])
        rec["first_failed_level"] = next((lv for lv in ("level2", "level3", "level4") if not rec[lv]["pass"]), None)
        rec["seconds"] = time.time() - t0
        self.log(f"  -> {key_of(f)}: {'PASS' if rec['pass'] else 'fail at ' + rec['first_failed_level']} "
                 f"(g {g_chosen}, {rec['seconds']:.0f} s)")
        self.records[rkey] = rec
        if self.on_record:
            self.on_record(rkey, rec)
        return rec

    # ------------------------------------------------------------------ search
    def forward(self, start_k=0):
        for k in range(start_k, 12):
            f = frozenset(range(1, k + 1))
            rec = self.evaluate(f)
            self.path.append(("forward", key_of(f), rec["pass"]))
            if rec["pass"]:
                return f, k
        return None, None

    def backward(self, f):
        cur = frozenset(f)
        while True:
            removed = False
            for fac in sorted(cur, reverse=True):
                cand = cur - {fac}
                rec = self.evaluate(cand)
                self.path.append(("backward", key_of(cand), rec["pass"]))
                if rec["pass"]:
                    cur = cand
                    removed = True
            if not removed:
                return cur

    def confirm(self, f, seeds, y_grid=None):
        rec = self.evaluate(f, y_grid=y_grid)
        frozen = {"g": rec["g_rule"], "in_scale": rec["in_scale"]}
        c = self.evaluate(f, data=self.confirm_data, seeds=seeds, frozen=frozen, y_grid=y_grid,
                          tag=f"confirm|{key_of(f)}|n{self.n_sub_for(f, y_grid)}|{','.join(map(str, seeds))}")
        return c

    def run(self):
        out = {"steps": []}
        rx = self.evaluate(frozenset())
        ry = self.evaluate(C.ALL_FACTORS)
        out["X"], out["Y"] = rx["pass"], ry["pass"]
        if rx["pass"]:
            final, k = frozenset(), 0
        else:
            final, k = self.forward(1)
        if final is None:
            out.update(final=None, stop="no forward candidate passed all four levels")
            out["path"] = self.path
            return out
        final = self.backward(final)
        attempts = 1
        seeds = C.SEEDS_CONFIRM
        while True:
            c = self.confirm(final, seeds)
            out["steps"].append({"confirm": key_of(final), "seeds": list(seeds), "pass": c["pass"]})
            if c["pass"]:
                break
            if final == C.ALL_FACTORS or attempts >= 2:
                out.update(final=None, stop="confirmation failed" + (" (Y itself)" if final == C.ALL_FACTORS else " twice"),
                           attempts=attempts, path=self.path)
                return out
            attempts += 1
            f2, k = self.forward(k + 1)                   # resume forward once from the last forward candidate
            if f2 is None:
                out.update(final=None, stop="confirmation failed; no later forward candidate passed", attempts=attempts,
                           path=self.path)
                return out
            final = self.backward(f2)
            seeds = C.SEEDS_RETRY
        rec = self.evaluate(final)
        out.update(final=sorted(final), final_key=key_of(final), attempts=attempts, path=self.path,
                   g=rec["g_rule"], maass_scale=10 in final, in_scale=rec["in_scale"], n_sub=rec["n_sub"])
        if final == C.ALL_FACTORS:                         # Y: also tested on n; if it passes there too, use n
            yn = self.evaluate(final, y_grid=self.n_fine)
            ycn = None
            if yn["pass"]:
                ycn = self.confirm(final, seeds, y_grid=self.n_fine)
            if yn["pass"] and ycn["pass"]:
                out.update(n_sub=self.n_fine, g=yn["g_rule"], in_scale=yn["in_scale"], y_on_n=True)
        return out
'''
LSM2_FILES['lsm2/level1.py'] = r'''"""Levels 1b to 1f (plan, Correctness criteria). 1a is the test suite (tests/test_level1a.py)."""
import math
import time
import numpy as np
import torch

from . import config as C
from .liquid import build_maass_script
from .sim import Simulator, start_voltages
from .measure import Ctx, measure_at_g, simulate
from . import brian2_ref


def _ours_trace(liq, x, u0, T, g, in_scale, device="cpu", dtype=torch.float64, open_loop=None, force_own=False):
    sm = Simulator(liq, device=device, dtype=dtype, g=g, in_scale=in_scale)
    ol = None if open_loop is None else open_loop[:, None, :]
    o = sm.run(x[None], u0[None], T, rec={"raster": True}, open_loop=ol, trace=True,
               own_spikes=ol if force_own else None)
    tr = {k: np.array(o["trace"][k])[:, 0, :] for k in ("u", "I1", "I2", "I3") if o["trace"][k]}
    if o["trace"]["syn"]:
        tr["R"] = np.array([s["R"][0] for s in o["trace"]["syn"]])
        tr["q"] = np.array([s["u"][0] for s in o["trace"]["syn"]])
    tr["raster"] = o["raster"][:, 0, :].cpu().numpy()
    tr["flips"] = o.get("flips", 0)
    return tr


def _isi(raster, h_ms):
    out = []
    for i in range(raster.shape[1]):
        k = np.nonzero(raster[:, i])[0]
        if len(k) > 1:
            out.append(np.diff(k) * h_ms)
    return np.concatenate(out) if out else np.zeros(0)


def _ks(a, b):
    if len(a) == 0 or len(b) == 0:
        return float("nan")
    from scipy.stats import ks_2samp
    return float(ks_2samp(a, b).statistic)


def level1b(liq, beats, g, in_scale, seed, T=C.SIM_SAMPLES, log=print, target="numpy", n_null=5):
    """Open loop: Brian2 and our float64 code get the same recorded presynaptic and input spikes; voltages,
    currents and synapse states must agree within 1e-9. Closed loop: reported (rate difference and ISI KS
    distance next to n_null Brian2-vs-Brian2 pairs from different starting states)."""
    rows, worst = [], 0.0
    h = liq.h_ms
    for b, x in enumerate(beats):
        u0 = start_voltages(liq, seed, [b], 11, random_start=True)[0]
        ours = _ours_trace(liq, x, u0, T, g, in_scale)
        t0 = time.time()
        br = brian2_ref.brian2_ours(liq, x, u0, T, g=g, in_scale=in_scale, open_loop=ours["raster"], target=target)
        d = {k: float(np.abs(ours[k] - br[k]).max()) for k in ("u", "I1", "I2", "I3", "R", "q") if k in ours and k in br}
        d["spike_mismatches"] = int((ours["raster"] != br["raster"]).sum())
        worst = max(worst, max(v for k, v in d.items() if k != "spike_mismatches"))
        rows.append({"beat": b, **d, "spikes": int(ours["raster"].sum()), "brian2_s": time.time() - t0})
    open_pass = worst <= C.TOL_1A and all(r["spike_mismatches"] == 0 for r in rows)
    # closed loop (reported)
    closed = []
    for b, x in enumerate(beats[:max(1, n_null)]):
        u0 = start_voltages(liq, seed, [b], 11, random_start=True)[0]
        ours = _ours_trace(liq, x, u0, T, g, in_scale)
        b1 = brian2_ref.brian2_ours(liq, x, u0, T, g=g, in_scale=in_scale, target=target)
        u0b = start_voltages(liq, seed, [b], 12, random_start=True)[0]
        b2 = brian2_ref.brian2_ours(liq, x, u0b, T, g=g, in_scale=in_scale, target=target)
        dur = T * liq.n_sub * h / 1000.0
        rate = lambda r: r.sum() / (liq.N * dur)
        closed.append({"beat": b, "ours_vs_brian2_rate_diff_hz": float(rate(ours["raster"]) - rate(b1["raster"])),
                       "ours_vs_brian2_ks": _ks(_isi(ours["raster"], h), _isi(b1["raster"], h)),
                       "ours_vs_brian2_identical_spikes": bool((ours["raster"] == b1["raster"]).all()),
                       "null_brian2_pair_rate_diff_hz": float(rate(b1["raster"]) - rate(b2["raster"])),
                       "null_brian2_pair_ks": _ks(_isi(b1["raster"], h), _isi(b2["raster"], h))})
    log(f"  1b open loop: worst difference {worst:.2e} (tolerance {C.TOL_1A:g}), "
        f"spike mismatches {sum(r['spike_mismatches'] for r in rows)} -> {'PASS' if open_pass else 'FAIL'}")
    return {"pass": bool(open_pass), "worst_abs_diff": worst, "open_loop": rows, "closed_loop_reported": closed}


def level1c(liq, beats, g, in_scale, seed, device, T=C.SIM_SAMPLES, log=print):
    """The open-loop test of 1b on this machine's card: float32 on the device vs float64, within 1e-4."""
    worst, rows = 0.0, []
    for b, x in enumerate(beats):
        u0 = start_voltages(liq, seed, [b], 11, random_start=True)[0]
        ref = _ours_trace(liq, x, u0, T, g, in_scale)
        # every spike (presynaptic and each neuron's own) is the float64 run's, so only arithmetic can differ;
        # threshold decisions float32 would have taken differently are counted (reported)
        f32 = _ours_trace(liq, x, u0, T, g, in_scale, device=device, dtype=torch.float32, open_loop=ref["raster"],
                          force_own=True)
        d = {k: float(np.abs(ref[k] - f32[k]).max()) for k in ("u", "I1", "I2", "I3", "R", "q") if k in ref}
        flips = int(f32["flips"])
        worst = max(worst, max(d.values()))
        rows.append({"beat": b, **d, "spike_flips": flips})
    ok = worst <= C.TOL_1C
    log(f"  1c {device}: worst float32 vs float64 difference {worst:.2e} (tolerance {C.TOL_1C:g}) -> {'PASS' if ok else 'FAIL'}")
    return {"pass": bool(ok), "worst_abs_diff": worst, "device": str(device), "rows": rows}


def _grid_measures(ctx, build, n, g_of, in_scale_of, sets, seeds, T=C.BEAT_SAMPLES):
    out = {}
    for s in seeds:
        liq = build(s, n)
        m = measure_at_g(ctx, liq, g_of(liq), in_scale_of(liq), sets, s, T=T, with_twins=False)
        out[s] = {k: m[k] for k in ("rate_hz", "sep", "gen")}
    return out


def grid_effect_ok(test, ref):
    """For firing, separation and generalisation: |mean over seeds of (test - reference)| is no larger than the
    range (largest minus smallest) across the seeds on the reference grid."""
    res = {}
    for k in ("rate_hz", "sep", "gen"):
        diff = float(np.mean([test[s][k] - ref[s][k] for s in ref]))
        vals = [ref[s][k] for s in ref]
        rng = float(max(vals) - min(vals))
        res[k] = {"effect": diff, "reference_range": rng, "pass": bool(abs(diff) <= rng)}
    return all(v["pass"] for v in res.values()), res


def level1d(ctx, build_X, build_Y, gX, inX, gY, inY, sets, seeds, log=print, n_max=None):
    """X on the fine (reference) grid vs X on the paper's grid; Y at Maass's own weights on n = 1, 2, 3, ...
    vs the 28-sub-step reference; the smallest passing n is used for every candidate with factor 7."""
    nref = C.N_SUB_REFERENCE
    n_max = n_max or C.E1D_N_TRY_MAX
    xr = _grid_measures(ctx, build_X, nref, gX, inX, sets, seeds)
    xp = _grid_measures(ctx, build_X, 1, gX, inX, sets, seeds)
    x_ok, x_res = grid_effect_ok(xp, xr)
    log(f"  1d X: paper grid vs 28 sub-steps -> {'PASS' if x_ok else 'FAIL'} {x_res}")
    yr = _grid_measures(ctx, build_Y, nref, gY, inY, sets, seeds)
    tried, n_pass = [], None
    for n in range(1, n_max + 1):
        if n == nref:
            n_pass = nref
            break
        yn = _grid_measures(ctx, build_Y, n, gY, inY, sets, seeds)
        ok, res = grid_effect_ok(yn, yr)
        tried.append({"n": n, "pass": ok, "detail": res})
        log(f"  1d Y: n = {n} vs 28 -> {'PASS' if ok else 'fail'}")
        if ok:
            n_pass = n
            break
    return {"pass": bool(x_ok), "x_detail": x_res, "n": n_pass, "y_tried": tried, "y_reference": yr}


# ---------------------------------------------------------------- 1e: (d)'s Figure 2 vs our code at its settings
def _d_pairs(mod, seed, n_pairs):
    np.random.seed(seed)
    mod.N_PAIRS = n_pairs
    pairs = mod.collect_stimulus_pairs()
    pairs[0] = [[mod.generate_poisson(mod.DURATION / mod.ms, mod.STIMULUS_POISSON_RATE / mod.Hz / 1e3)] * 2
                for _ in range(n_pairs)]
    return pairs


def _to_bins(spike_times_ms, dt_ms, n_steps):
    x = np.zeros((n_steps, 1), np.uint8)
    k = ((np.asarray(spike_times_ms) + 1e-3 * dt_ms) / dt_ms).astype(int)
    k = k[(k >= 0) & (k < n_steps)]
    np.add.at(x, (k, 0), 1)
    return x


def ours_fig2_run(ctx, mod, seed, n_pairs, d_replacement="fixed"):
    """One run of (d)'s protocol with our code in the script's settings at its 0.1 ms step: a fresh liquid,
    fresh stimulus pairs, initial voltages redrawn for every simulation; returns {distance: mean curve}."""
    dt = float(mod.DT / mod.ms)
    n_steps = len(mod.TS)
    liq = build_maass_script(seed, n_sub=1, step_ms=dt, d_replacement=d_replacement)
    pairs = _d_pairs(mod, seed, n_pairs)
    curves = {}
    for dist, pp in pairs.items():
        xu = np.stack([_to_bins(a, dt, n_steps) for a, _ in pp])
        xv = np.stack([_to_bins(b, dt, n_steps) for _, b in pp])
        uu = start_voltages(liq, seed, range(len(pp)), 21 + int(dist * 100), random_start=True)
        uv = start_voltages(liq, seed, range(len(pp)), 61 + int(dist * 100), random_start=True)
        su = simulate(ctx, liq, 1.0, 1.0, xu, uu, n_steps, {"states_every": (0, n_steps)})["states_every"]
        sv = simulate(ctx, liq, 1.0, 1.0, xv, uv, n_steps, {"states_every": (0, n_steps)})["states_every"]
        curves[dist] = ((su.astype(np.float64) - sv) ** 2).mean(2).mean(0)    # (d): mean over neurons, then pairs
    return curves


def _d_sim_worker(args):
    import brian2_d_globals as G
    return G.mod.sim(G.net, args)


def d_fig2_run(mod, seed, n_pairs, processes=None):
    """One run of (d)'s own script (its functions, unchanged): build the liquid, collect pairs, simulate."""
    import multiprocessing as mp
    import sys, types
    import gc, brian2
    gc.collect(); brian2.start_scope()
    np.random.seed(seed)
    neurons = mod.get_neurons()
    N_exc = int(0.8 * len(neurons))
    exc, inh = neurons[:N_exc], neurons[N_exc:]
    exc.tau_refrac = 3 * mod.ms; exc.tau_stimulus = 3 * mod.ms
    inh.tau_refrac = 2 * mod.ms; inh.tau_stimulus = 6 * mod.ms
    second, nA = mod.second, mod.nA
    syn = [mod.get_synapses("ee_synapses", exc, exc, C=0.3, l=2, tau_I=3 * mod.ms, A=30 * nA, U=0.5, D=1.1 * second, F=0.05 * second, delay=1.5 * mod.ms),
           mod.get_synapses("ei_synapses", exc, inh, C=0.2, l=2, tau_I=3 * mod.ms, A=60 * nA, U=0.05, D=0.125 * second, F=1.2 * second, delay=0.8 * mod.ms),
           mod.get_synapses("ie_synapses", inh, exc, C=0.4, l=2, tau_I=6 * mod.ms, A=-19 * nA, U=0.25, D=0.7 * second, F=0.02 * second, delay=0.8 * mod.ms),
           mod.get_synapses("ii_synapses", inh, inh, C=0.1, l=2, tau_I=6 * mod.ms, A=-19 * nA, U=0.32, D=0.144 * second, F=0.06 * second, delay=0.8 * mod.ms)]
    stim = mod.SpikeGeneratorGroup(1, [], [] * mod.ms, name="stimulus")
    se = mod.Synapses(stim, exc, "A : ampere (shared, constant)", on_pre="I_stimulus += A"); se.connect(p=1); se.A = 18 * nA
    si = mod.Synapses(stim, inh, "A : ampere (shared, constant)", on_pre="I_stimulus += A"); si.connect(p=1); si.A = 9 * nA
    me = mod.SpikeMonitor(exc, name="spike_monitor_exc"); mi = mod.SpikeMonitor(inh, name="spike_monitor_inh")
    mod.defaultclock.dt = mod.DT
    net = mod.Network([neurons, *syn, se, si, stim, me, mi])
    net.store()
    mod.neurons = neurons                      # (d)'s sim() reads the global `neurons`
    pairs = _d_pairs(mod, seed, n_pairs)
    g = types.ModuleType("brian2_d_globals"); g.mod, g.net = mod, net
    sys.modules["brian2_d_globals"] = g
    curves = {}
    ctx = mp.get_context("fork")
    for dist, pp in pairs.items():                 # as in (d): a fresh pool for every input distance
        with ctx.Pool(processes) as pool:
            su = pool.map(_d_sim_worker, [p[0] for p in pp])
            sv = pool.map(_d_sim_worker, [p[1] for p in pp])
        curves[dist] = np.mean([mod.euclidian_distance(a, b) for a, b in zip(su, sv)], axis=0)
    return curves


def compare_fig2(ours_runs, d_runs, t_crit=C.E1E_T_CRIT, frac=C.E1E_FRAC):
    """Our mean curve vs the script's: within t_crit combined standard errors at >= frac of time points,
    for every input distance."""
    res = {}
    for dist in d_runs[0]:
        A = np.stack([r[dist] for r in ours_runs]); B = np.stack([r[dist] for r in d_runs])
        diff = A.mean(0) - B.mean(0)
        se = np.sqrt(A.var(0, ddof=1) / len(A) + B.var(0, ddof=1) / len(B))
        ok = np.where(se > 0, np.abs(diff) <= t_crit * se, np.abs(diff) <= 1e-12)
        res[float(dist)] = {"frac_within": float(ok.mean()), "pass": bool(ok.mean() >= frac),
                            "max_abs_diff": float(np.abs(diff).max())}
    return all(v["pass"] for v in res.values()), res


def level1e(ctx, script_path, seeds, n_pairs=200, processes=None, log=print, skip_brian2=False, d_replacement="fixed"):
    """5 runs each side (plan: each run of the script is 1,600 simulations). d_replacement: "fixed" patches (d)'s
    mean replacement of bad U, D, F draws so it works as intended (see brian2_ref.script_module); "as_written"
    runs (d) unchanged and our code with the same behaviour."""
    mod = brian2_ref.script_module(script_path, fix_replacement=(d_replacement == "fixed"))
    ours = []
    for s in seeds:
        t0 = time.time(); ours.append(ours_fig2_run(ctx, mod, s, n_pairs, d_replacement))
        log(f"  1e ours run seed {s}: {time.time() - t0:.0f} s")
    if skip_brian2:
        return {"pass": None, "ours": ours}
    d = []
    for s in seeds:
        t0 = time.time(); d.append(d_fig2_run(mod, 10_000 + s, n_pairs, processes))
        log(f"  1e Brian2 script run seed {s}: {time.time() - t0:.0f} s")
    ok, res = compare_fig2(ours, d)
    log(f"  1e: {'PASS' if ok else 'FAIL'} {res}")
    return {"pass": bool(ok), "by_distance": res, "ours_curves": ours, "d_curves": d, "d_replacement": d_replacement}
'''
LSM2_FILES['lsm2/level2.py'] = r'''"""Level 2, faithful: is each candidate the liquid its switches describe? Connection counts by type, input neurons,
sample means of weights and of U, D, F, signs, each neuron's net excitatory and inhibitory input. Counts and means
within 4.5 standard errors of their targets; the U, D, F targets and their standard errors come from the drawing
procedure itself (10 million draws per type); fractions as set; every sign correct."""
import functools
import numpy as np

from . import config as C
from .liquid import _type_matrix, _draw_udf, candidate_settings, round_half_up

N_DRAWS = 10_000_000


@functools.lru_cache(maxsize=None)
def udf_targets(bad_rule="redraw"):
    """Mean and SD of U, D, F per type under the drawing procedure (10 million draws per type)."""
    out = {}
    for name, scale, cap in (("U", 1.0, True), ("D_s", 1000.0, False), ("F_s", 1000.0, False)):
        for ti, t in enumerate(C.TYPES):
            T = np.full(N_DRAWS, ti)
            v = _draw_udf(np.random.default_rng([99, ti, len(name)]), np.random.default_rng([98, ti, len(name)]),
                          T, {k: x * scale for k, x in C.MAASS[name].items()}, cap and bad_rule == "redraw", bad_rule)
            out[(name, t)] = (float(v.mean()), float(v.std()))
    return out


def _z(obs, target, se):
    return float((obs - target) / se) if se > 0 else (0.0 if abs(obs - target) < 1e-12 else float("inf"))


def check(liq, n_channels=None):
    """Level 2 for a ladder candidate (and Y). Returns {"pass": bool, "items": [...]}."""
    f = liq.factors
    s = candidate_settings(f)
    items = []
    def add(name, obs, target, se=None, exact=False, ok=None):
        if ok is None:
            if exact:
                ok = abs(obs - target) <= 1e-9 * max(1.0, abs(target))
                z = None
            else:
                z = _z(obs, target, se)
                ok = abs(z) <= C.LEVEL2_SE
        else:
            z = None
        items.append({"item": name, "observed": float(obs), "target": float(target),
                      "se": None if se is None else float(se), "z": z, "pass": bool(ok)})
    N, E = liq.N, liq.is_exc
    add("neurons", N, int(np.prod(s["dims"])), exact=True)
    add("excitatory neurons", E.sum(), int(N * C.X["excit_ratio"]), exact=True)
    # connections by type: expected count and binomial SE from the realised types and positions
    T = _type_matrix(E)
    Cm = np.vectorize(lambda t: C.X["C"][C.TYPES[t]])(T)
    d = np.sqrt(((liq.pos[:, None, :] - liq.pos[None, :, :]) ** 2).sum(-1))
    p = np.clip(Cm * np.exp(-(d / s["lam"]) ** 2), 0, 1)
    np.fill_diagonal(p, 0)
    for ti, t in enumerate(C.TYPES):
        m = T == ti
        add(f"{t} connections", (liq.typ == ti).sum(), p[m].sum(), np.sqrt((p[m] * (1 - p[m])).sum()))
    add("self-connections", int((liq.pre == liq.post).sum()), 0, exact=True)
    # recurrent weights: the Gamma draws before (a)'s fan-in division
    fan_in = np.maximum(np.bincount(liq.post, minlength=N), 1)
    raw = np.abs(liq.w) * (fan_in[liq.post] if s["normalise"] else 1.0)
    for ti, t in enumerate(C.TYPES):
        m = liq.typ == ti
        if m.sum():
            mu = s["mu"][t]
            add(f"{t} weight mean (Gamma draws)", raw[m].mean(), mu, mu / np.sqrt(m.sum()))
            add(f"{t} sign", float(np.all(np.sign(liq.w[m]) == (1 if ti < 2 else -1))), 1.0, exact=True)
    # U, D, F
    if liq.syn_model == "markram":
        tg = udf_targets("redraw")
        for name, arr in (("U", liq.U), ("D_s", liq.D_ms), ("F_s", liq.F_ms)):
            for ti, t in enumerate(C.TYPES):
                m = liq.typ == ti
                if m.sum():
                    mean, sd = tg[(name, t)]
                    add(f"{t} {name.split('_')[0]} mean", arr[m].mean(), mean, sd / np.sqrt(m.sum()))
            add(f"{name.split('_')[0]} positive", float((arr > 0).all()), 1.0, exact=True)
        add("U at most 1", float((liq.U <= 1).all()), 1.0, exact=True)
    # input
    Cn = liq.Win.shape[0]
    mask = liq.Win > 0
    add("input weights non-negative", float((liq.Win >= 0).all()), 1.0, exact=True)
    if s["input30"]:
        k = round_half_up(C.MAASS["in_fraction"] * N)
        add("input-receiving neurons", liq.in_neurons.sum(), k, exact=True)
        add("each input neuron hears every channel", float(mask[:, liq.in_neurons].all()), 1.0, exact=True)
        add("other neurons get no input", float(not mask[:, ~liq.in_neurons].any()), 1.0, exact=True)
        n_in = int(mask.sum())
    else:
        P = C.X["P_in"]
        n_in = int(mask.sum())
        add("input synapses", n_in, Cn * N * P, np.sqrt(Cn * N * P * (1 - P)))
    split = liq.in_split
    for typ_name, sel, m_type in (("E", E, split[0]), ("I", ~E, split[1])):
        cols = mask[:, sel]
        if not cols.any():
            continue
        if s["normalise"]:
            cm = liq.Win[:, sel].sum(0)[cols.any(0)] / cols.sum(0)[cols.any(0)]
            dev = float(np.abs(cm / m_type - 1).max())             # (a)'s eps of 1e-8 leaves ~1e-8 relative
            add(f"input column means onto {typ_name} ((a)'s rescale), largest relative deviation", dev, 0.0, ok=dev < 1e-6)
        else:
            v = liq.Win[:, sel][cols]
            add(f"input weight mean onto {typ_name}", v.mean(), m_type, m_type / np.sqrt(v.size))
    # net excitatory and inhibitory input per neuron (reported)
    exc_in = np.bincount(liq.post[liq.typ < 2], weights=liq.w[liq.typ < 2], minlength=N)
    inh_in = np.bincount(liq.post[liq.typ >= 2], weights=liq.w[liq.typ >= 2], minlength=N)
    report = {"net_exc_input_mean": float(exc_in.mean()), "net_inh_input_mean": float(inh_in.mean()),
              "neurons_without_recurrent_input": int((np.bincount(liq.post, minlength=N) == 0).sum()),
              "input_synapses": n_in}
    return {"pass": all(i["pass"] for i in items), "items": items, "report": report}
'''
LSM2_FILES['lsm2/liquid.py'] = r'''"""Building liquids: every ladder candidate (X plus any set of the 11 factors), Maass's two versions for the
Maass check (script and paper), and X converted from the authors' own class (for 1f and E1's paper liquid).

A Liquid holds the realised network and every time constant already converted to its grid (n sub-steps per
input step). Input weights are stored as a base matrix; the input size (paper's rule, factor-5 calibration or
factor 9) is a separate scalar `in_scale`, and the recurrent gain g a scalar given to the simulator.
"""
from dataclasses import dataclass, field
import math
import numpy as np

from . import config as C

TYPE_INDEX = {t: i for i, t in enumerate(C.TYPES)}     # EE 0, EI 1, IE 2, II 3


def round_half_up(x):
    return int(math.floor(x + 0.5))


def steps_of(ms, h_ms):
    """A time in ms as a whole number of sub-steps: rounded to the nearest, at least 1 (plan, Time grid)."""
    return max(1, int(round(ms / h_ms)))


@dataclass
class Liquid:
    # structure
    N: int
    dims: tuple
    is_exc: np.ndarray          # (N,) bool
    pos: np.ndarray             # (N, 3)
    pre: np.ndarray             # (S,) int
    post: np.ndarray            # (S,) int
    typ: np.ndarray             # (S,) int 0..3 (EE, EI, IE, II)
    w: np.ndarray               # (S,) signed base weight: a push in thresholds ("ours"), or J amplitude ("brian2")
    U: np.ndarray               # (S,) utilisation (dynamic synapses), else ones
    D_ms: np.ndarray            # (S,)
    F_ms: np.ndarray            # (S,)
    Ubar: np.ndarray            # (4,) nominal type means of U
    Win: np.ndarray             # (C, N) base input weights, >= 0
    in_neurons: np.ndarray      # (N,) bool: neurons with any input synapse
    # time grid and neuron model
    n_sub: int                  # sub-steps per input step
    h_ms: float                 # sub-step length
    beta: np.ndarray            # (N,) leak per sub-step
    tap: np.ndarray             # (N,) tap per sub-step ("ours"); 0 without factor 5
    reset_mode: str             # "subtract" or "value"
    reset_value: float
    start_lo: float             # starting voltages drawn uniformly in [start_lo, start_hi]; equal = fixed
    start_hi: float
    rest: float                 # resting level (0, or 0.9 with the tap): lower end of level 3's random starts
    gap: np.ndarray             # (N,) int minimum gap between spikes, in sub-steps
    clamp: bool                 # voltage clamped during the blocked sub-steps
    strict: bool                # threshold test u > 1 (Brian2 script) instead of u >= 1 (ours, (a)'s H)
    schedule: str               # "ours" or "brian2"
    syn_model: str              # "static", "markram" (two-state, (c)'s form) or "tsodyks3" ((d)'s three-state)
    delay: np.ndarray           # (4,) int sub-steps by type
    cur_a: object               # None (push at once) or (2,) decay per sub-step for E- and I-presynaptic currents
    in_a: object                # None or (N,) decay per sub-step of each neuron's input current
    # Brian2-schedule constants (script version only)
    J_b: float = 0.0
    kappa: object = None        # (N,) exact weight of the decaying input current in the membrane update
    syn_tau_ms: object = None   # (4,) current decay per type (tsodyks3)
    # bookkeeping
    factors: frozenset = frozenset()
    snn_quirk: bool = False     # (a)'s code (snnTorch Leaky, reset_delay=False): the threshold test subtracts 1 more
                                # when the stored voltage is above 1; u > 1 strict. Default: (a)'s Eqs. 17-19
    g_maass: float = None       # g at Maass's own scale (factor 10)
    in_scale_fixed: float = None   # input size fixed by the design (factor 9, Maass versions)
    in_split: tuple = (1.0, 1.0)   # base input means onto E / I neurons (2:1 with factors 5 or 9)
    meta: dict = field(default_factory=dict)

    @property
    def S(self):
        return len(self.pre)

    def ee_mean_push(self, g=1.0):
        """Mean isolated EE push per presynaptic spike, in thresholds (the plan's absolute g scale)."""
        m = self.typ == 0
        if not m.any():
            return 0.0
        iso = np.abs(self.w[m]) * (self.U[m] / self.Ubar[0] if self.syn_model == "markram" else 1.0)
        return float(g * iso.mean())


# ---------------------------------------------------------------- shared pieces
def _grid(dims):
    return np.stack(np.meshgrid(*[np.arange(d) for d in dims], indexing="ij"), -1).reshape(-1, 3).astype(np.float64)


def _type_matrix(is_exc):
    pe, qe = is_exc[:, None], is_exc[None, :]
    T = np.where(pe & qe, 0, np.where(pe & ~qe, 1, np.where(~pe & qe, 2, 3)))
    return T


def _draw_udf(rng_n, rng_u, T, mean_of_type, cap, bad_rule):
    """U, D or F for every (pre, post) pair: Normal(mean, 0.5 mean); draws at or below 0 replaced by
    uniform(0, 2 mean) ("redraw", plan decision 17) or by the mean ("mean", (d)); U capped at 1 if cap."""
    mean = np.vectorize(lambda t: mean_of_type[C.TYPES[t]])(T).astype(np.float64)
    z = rng_n.standard_normal(T.shape)
    v = mean + C.MAASS["udf_spread"] * mean * z
    repl = rng_u.random(T.shape) * 2 * mean
    if bad_rule == "redraw":
        v = np.where(v <= 0, repl, v)
    else:      # (d): U < 0 -> mean; D, F <= 0 -> mean
        v = np.where(v <= 0, mean, v)
    if cap:
        v = np.minimum(v, 1.0)
    return v


# ---------------------------------------------------------------- ladder candidates
def candidate_settings(factors):
    f = frozenset(factors)
    return {
        "dims": C.MAASS["dims"] if 11 in f else C.X["dims"],
        "lam": C.MAASS["lam"] if 1 in f else C.X["lam"],
        "mu": ({t: C.X["mu_rsv"]["EE"] * C.MAASS_RATIOS[t] for t in C.TYPES} if 3 in f else dict(C.X["mu_rsv"])),
        "normalise": 2 not in f,
        "tau_E_ms": C.LEAK_MS_Y if 4 in f else -C.SAMPLE_MS / math.log(C.X["betaE"]),
        "tau_I_ms": C.LEAK_MS_Y if 4 in f else -C.SAMPLE_MS / math.log(C.X["betaI"]),
        "tap": 5 in f,
        "input30": 6 in f,
        "fine": 7 in f,
        "dynamic": 8 in f,
        "maass_input": 9 in f,
        "maass_scale": 10 in f,
    }


def build_candidate(factors, seed, n_sub, n_channels, n_sub_input_step_ms=C.SAMPLE_MS):
    """A ladder candidate: X ((a), bugs fixed) with the given factors switched on.
    n_sub: sub-steps per sample (1 without factor 7 unless a grid check asks otherwise)."""
    f = frozenset(factors)
    s = candidate_settings(f)
    N = int(np.prod(s["dims"]))
    h = n_sub_input_step_ms / n_sub
    # independent streams, so candidates with the same seed share their random numbers where they can
    r_types, r_conn, r_w = (np.random.default_rng([seed, k]) for k in (1, 2, 3))
    r_un, r_uu, r_dn, r_du, r_fn, r_fu = (np.random.default_rng([seed, k]) for k in (4, 5, 6, 7, 8, 9))
    r_inm, r_inw, r_inn = (np.random.default_rng([seed, k]) for k in (10, 11, 12))
    n_exc = int(N * C.X["excit_ratio"])
    is_exc = np.zeros(N, bool)
    is_exc[r_types.permutation(N)[:n_exc]] = True
    pos = _grid(s["dims"])
    T = _type_matrix(is_exc)
    Cm = np.vectorize(lambda t: C.X["C"][C.TYPES[t]])(T)
    d = np.sqrt(((pos[:, None, :] - pos[None, :, :]) ** 2).sum(-1))
    p = np.clip(Cm * np.exp(-(d / s["lam"]) ** 2), 0, 1)
    conn = r_conn.random((N, N)) < p
    np.fill_diagonal(conn, False)                            # no self-connections ((a); kept in Y)
    mu = np.vectorize(lambda t: s["mu"][C.TYPES[t]])(T)
    Wmag = r_w.standard_exponential((N, N)) * mu             # Gamma(shape 1, scale mu): spread = mean
    if s["normalise"]:
        fan_in = np.maximum(conn.sum(0, keepdims=True), 1)   # (a) lines 168-169: divide by the receiver's count
        Wmag = Wmag / fan_in
    W = np.where(is_exc[:, None], Wmag, -Wmag)
    pre, post = np.nonzero(conn)
    order = np.lexsort((post, pre, T[pre, post]))           # sorted by type, then pre, then post
    pre, post = pre[order], post[order]
    typ = T[pre, post]
    if s["dynamic"]:
        Ufull = _draw_udf(r_un, r_uu, T, C.MAASS["U"], True, "redraw")
        Dfull = _draw_udf(r_dn, r_du, T, {k: v * 1000 for k, v in C.MAASS["D_s"].items()}, False, "redraw")
        Ffull = _draw_udf(r_fn, r_fu, T, {k: v * 1000 for k, v in C.MAASS["F_s"].items()}, False, "redraw")
        U, D, F = Ufull[pre, post], Dfull[pre, post], Ffull[pre, post]
    else:
        U, D, F = np.ones(len(pre)), np.full(len(pre), np.inf), np.full(len(pre), np.inf)
    Ubar = np.array([C.MAASS["U"][t] for t in C.TYPES])
    # input wiring and base weights
    if s["input30"]:
        k = round_half_up(C.MAASS["in_fraction"] * N)
        in_n = np.zeros(N, bool)
        in_n[r_inn.permutation(N)[:k]] = True
        mask = np.repeat(in_n[None, :], n_channels, 0)
    else:
        mask = r_inm.random((n_channels, N)) < C.X["P_in"]
        in_n = mask.any(0)
    split = (1.0, 0.5) if (s["tap"] or s["maass_input"]) else (1.0, 1.0)
    m_in = np.where(is_exc, split[0], split[1])[None, :]
    Win = r_inw.standard_exponential((n_channels, N)) * m_in * mask
    if s["normalise"]:                                       # (a) lines 71-75: each column's mean active weight = m
        cnt = mask.sum(0, keepdims=True)
        col = Win.sum(0, keepdims=True) / (cnt + 1e-8)
        Win = Win * (m_in / (col + 1e-8))
    # time constants on the grid
    tau = np.where(is_exc, s["tau_E_ms"], s["tau_I_ms"])
    beta = np.exp(-h / tau)
    tap = C.TAP * (1 - beta) if s["tap"] else np.zeros(N)
    if s["fine"]:
        gap = np.where(is_exc, steps_of(C.MAASS["refrac_ms"]["E"], h), steps_of(C.MAASS["refrac_ms"]["I"], h))
        delay = np.array([steps_of(C.MAASS["delay_ms"][t], h) for t in C.TYPES])
        cur_a = np.exp(-h / np.array([C.MAASS["tau_s_ms"]["E"], C.MAASS["tau_s_ms"]["I"]]))
        in_a = np.full(N, math.exp(-h / C.MAASS["tau_in_ms"]))
        clamp = True
    else:
        # X: the 2.78 ms gap and the one-sample delay, in ms, converted to the grid in use
        gap = np.full(N, steps_of(C.SAMPLE_MS, h))
        delay = np.full(4, steps_of(C.SAMPLE_MS, h))
        cur_a, in_a, clamp = None, None, False
    liq = Liquid(
        N=N, dims=tuple(s["dims"]), is_exc=is_exc, pos=pos, pre=pre, post=post, typ=typ, w=W[pre, post].astype(np.float64),
        U=U, D_ms=D, F_ms=F, Ubar=Ubar, Win=Win.astype(np.float64), in_neurons=in_n,
        n_sub=n_sub, h_ms=h, beta=beta, tap=tap,
        reset_mode="value" if s["tap"] else "subtract", reset_value=C.RESET_TAP if s["tap"] else 0.0,
        start_lo=C.RESET_TAP if s["tap"] else 0.0, start_hi=1.0 if s["tap"] else 0.0,
        rest=C.TAP if s["tap"] else 0.0,
        gap=gap.astype(np.int64), clamp=clamp, strict=False, schedule="ours",
        syn_model="markram" if s["dynamic"] else "static", delay=delay.astype(np.int64), cur_a=cur_a, in_a=in_a,
        factors=f, in_split=split,
        meta={"kind": "candidate", "seed": int(seed), "n_channels": int(n_channels), "settings": s},
    )
    if liq.reset_mode == "subtract" and C.CODE["x_update"] == "authors_code":
        liq.snn_quirk, liq.strict = True, True
    m = typ == 0
    liq.g_maass = C.MAASS_EE_PUSH / float(np.abs(liq.w[m]).mean()) if m.any() else None
    if s["maass_input"]:
        liq.in_scale_fixed = C.MAASS_IN_PUSH["E"]           # base means 1 / 0.5 -> 0.12 / 0.06 per input spike
    return liq


def p_in_effective(liq):
    """Connection probability the paper's rule uses: P_in = 0.1, or 1 for the input neurons with factor 6."""
    return 1.0 if 6 in liq.factors else C.X["P_in"]


def paper_rule_scale(ain, liq):
    """(a)'s rule: mu_in = I_target / (A_in P_in), always with X's excitatory leak (plan, Input size)."""
    return C.I_TARGET / (ain * p_in_effective(liq))


# ---------------------------------------------------------------- Maass's liquid, two versions (Maass check)
def build_maass_paper(seed, n_sub=None, n_channels=1):
    """Our reading of (c), the version Y copies: Y with all 11 factors and one Poisson input channel."""
    n_sub = n_sub or C.N_SUB_REFERENCE
    liq = build_candidate(C.ALL_FACTORS, seed, n_sub, n_channels)
    liq.meta["kind"] = "maass_paper"
    return liq


def build_maass_script(seed, n_sub=None, step_ms=C.SAMPLE_MS, d_replacement="fixed"):
    """The liquid of (d)'s Brian2 script, in threshold units, with its six settings (plan, level 1e):
    input onto I neurons decays over 6 ms, bad U, D, F draws replaced by the mean, U not capped, no input delay,
    an arriving spike acts after that step's state update (Brian2 schedule), each step's synaptic current held
    constant (exact neuron update with step-constant synaptic current). Also from (d): input to all 135 neurons
    at a fixed 18 / 9 nA, self-connections allowed, E neurons are the first 108 indices with shuffled positions,
    three-state synapses (used resources recover only once the current has ended)."""
    n_sub = n_sub or C.N_SUB_REFERENCE
    N = int(np.prod(C.MAASS["dims"]))
    h = step_ms / n_sub
    rng = np.random.default_rng([seed, 1000])
    idx = rng.permutation(N)
    pos = np.stack([idx % 3, (idx // 3) % 3, idx // 9], 1).astype(np.float64)
    n_exc = int(0.8 * N)
    is_exc = np.zeros(N, bool); is_exc[:n_exc] = True
    T = _type_matrix(is_exc)
    Cm = np.vectorize(lambda t: C.X["C"][C.TYPES[t]])(T)
    d2 = ((pos[:, None, :] - pos[None, :, :]) ** 2).sum(-1)
    p = Cm * np.exp(-d2 / C.MAASS["lam"] ** 2)
    conn = np.random.default_rng([seed, 1001]).random((N, N)) < p          # self-connections allowed ((d))
    pre, post = np.nonzero(conn)
    order = np.lexsort((post, pre, T[pre, post]))
    pre, post = pre[order], post[order]
    typ = T[pre, post]
    J = C.MAASS["R_Mohm"] / C.MAASS["v_thresh_mV"]                         # nA -> thresholds (1 MOhm / 15 mV)
    A = np.array([C.MAASS["A_nA"][C.TYPES[t]] for t in typ])
    r = np.random.default_rng([seed, 1002])
    w = np.sign(A) * r.standard_exponential(len(pre)) * np.abs(A) * J       # Gamma(1, |A|) with sign ((d))
    def udf(name, scale, k):
        mean = np.array([C.MAASS[name][C.TYPES[t]] for t in typ]) * scale
        v = mean + 0.5 * mean * np.random.default_rng([seed, k]).standard_normal(len(pre))
        if d_replacement == "as_written":       # (d) as written: only the first two synapses of each group
            for ti in range(4):
                first = np.nonzero(typ == ti)[0][:2]
                v[first] = mean[first]
            return v
        return np.where(v < 0, mean, v) if name == "U" else np.where(v <= 0, mean, v)
    U, D, F = udf("U", 1.0, 1003), udf("D_s", 1000.0, 1004), udf("F_s", 1000.0, 1005)
    Ubar = np.array([C.MAASS["U"][t] for t in C.TYPES])
    Win = (np.where(is_exc, C.MAASS["in_nA"]["E"], C.MAASS["in_nA"]["I"]) * J)[None, :]   # J jump per input spike
    tau_m = C.MAASS["tau_m_ms"]
    beta = np.full(N, math.exp(-h / tau_m))
    in_tau = np.where(is_exc, 3.0, 6.0)                                     # (d): tau_stimulus 3 ms (E), 6 ms (I)
    a_in = np.exp(-h / in_tau)
    kappa = in_tau / (in_tau - tau_m) * (np.exp(-h / in_tau) - np.exp(-h / tau_m))   # Brian2's exact solution
    gap = np.where(is_exc, int((C.MAASS["refrac_ms"]["E"] + 1e-3 * h) / h), int((C.MAASS["refrac_ms"]["I"] + 1e-3 * h) / h))
    delay = np.array([int(round(C.MAASS["delay_ms"][t] / h)) for t in C.TYPES])   # Brian2 rounds delays to the grid
    syn_tau = np.array([C.MAASS["tau_s_ms"]["E"], C.MAASS["tau_s_ms"]["E"], C.MAASS["tau_s_ms"]["I"], C.MAASS["tau_s_ms"]["I"]])
    liq = Liquid(
        N=N, dims=C.MAASS["dims"], is_exc=is_exc, pos=pos, pre=pre, post=post, typ=typ, w=w, U=U, D_ms=D, F_ms=F,
        Ubar=Ubar, Win=Win, in_neurons=np.ones(N, bool), n_sub=n_sub, h_ms=h, beta=beta, tap=np.zeros(N),
        reset_mode="value", reset_value=C.RESET_TAP, start_lo=C.RESET_TAP, start_hi=1.0, rest=C.TAP,
        gap=gap.astype(np.int64), clamp=True, strict=True, schedule="brian2", syn_model="tsodyks3",
        delay=delay.astype(np.int64), cur_a=None, in_a=a_in, J_b=C.TAP, kappa=kappa, syn_tau_ms=syn_tau,
        factors=frozenset(), in_scale_fixed=1.0, in_split=(1.0, 0.5),
        meta={"kind": "maass_script", "seed": int(seed), "n_channels": 1, "d_replacement": d_replacement},
    )
    liq.g_maass = 1.0
    return liq


# ---------------------------------------------------------------- X from the authors' class
def from_authors_model(model, n_sub=1, snn_quirk=None):
    """X exactly as (a)'s class built it (weights already include its normalisations and its g_rec):
    used for the 1f regression and for E1's paper liquid at g = 1. Input weights are returned at their size
    (in_scale 1)."""
    import torch
    W = model.rsv_weights.detach().double().cpu().numpy()
    Win = model.input_rsv_weights.detach().double().cpu().numpy()
    is_exc = model.is_excit.detach().cpu().numpy().astype(bool)
    N = W.shape[0]
    T = _type_matrix(is_exc)
    pre, post = np.nonzero(W)
    order = np.lexsort((post, pre, T[pre, post]))
    pre, post = pre[order], post[order]
    h = C.SAMPLE_MS / n_sub
    tau = np.where(is_exc, -C.SAMPLE_MS / math.log(C.X["betaE"]), -C.SAMPLE_MS / math.log(C.X["betaI"]))
    liq = Liquid(
        N=N, dims=tuple(model.reservoir_dim), is_exc=is_exc, pos=_grid(model.reservoir_dim), pre=pre, post=post,
        typ=T[pre, post], w=W[pre, post], U=np.ones(len(pre)), D_ms=np.full(len(pre), np.inf),
        F_ms=np.full(len(pre), np.inf), Ubar=np.array([C.MAASS["U"][t] for t in C.TYPES]), Win=Win,
        in_neurons=(Win != 0).any(0), n_sub=n_sub, h_ms=h, beta=np.exp(-h / tau), tap=np.zeros(N),
        reset_mode="subtract", reset_value=0.0, start_lo=0.0, start_hi=0.0, rest=0.0,
        gap=np.full(N, steps_of(C.SAMPLE_MS, h)), clamp=False, strict=False, schedule="ours", syn_model="static",
        delay=np.full(4, steps_of(C.SAMPLE_MS, h)), cur_a=None, in_a=None, factors=frozenset(),
        in_scale_fixed=1.0, meta={"kind": "authors_X", "n_channels": int(Win.shape[0])},
    )
    if snn_quirk or (snn_quirk is None and C.CODE["x_update"] == "authors_code"):
        liq.snn_quirk, liq.strict = True, True
    return liq
'''
LSM2_FILES['lsm2/maass_check.py'] = r'''"""Testing the test on Maass's liquid (plan, Correctness criteria): levels 3 and 4 on Maass's own 135-neuron
column in two versions (the Brian2 script's, and our reading of the paper, the one Y copies), each with its own
20 Hz Poisson input and its own weights, on the 28-sub-step grid."""
import numpy as np

from . import config as C
from .liquid import build_maass_script, build_maass_paper
from .measure import measure_at_g, memory_curve
from .calib import input_neuron_rate
from . import gain


def level3_verdict(per_seed, sweep_rec):
    """Level 3 at one g: firing band and starting-state rule on every seed, and the gap peaking inside the grid."""
    rows = [{"seed": r["seed"], "in_band": bool(C.BAND_HZ[0] <= r["rate_hz"] <= C.BAND_HZ[1]
                                                 and r["frac_above_100hz"] <= C.HIGH_FRAC),
             "start_rule": bool(r.get("start_rule_pass", False)), "rate_hz": r["rate_hz"]} for r in per_seed]
    ok = all(x["in_band"] and x["start_rule"] for x in rows) and sweep_rec["gap_peak_inside_grid"]
    return bool(ok), rows


def level4_verdict(curves_on, curves_off, lag=C.MEMORY_PASS_T_MS):
    gains = [on[f"r2_{lag}"] - off[f"r2_{lag}"] for on, off in zip(curves_on, curves_off)]
    return bool(all(np.isfinite(gn) and gn >= C.MEMORY_MIN_GAIN for gn in gains)), gains


def run_version(ctx, version, seeds, data, log=print):
    build = build_maass_script if version == "script" else build_maass_paper
    liqs = {s: build(s) for s in seeds}
    for s, l in liqs.items():
        l.meta["seed"] = s
    scale = liqs[seeds[0]].in_scale_fixed
    per = {}

    def measure(g, s):
        r = measure_at_g(ctx, liqs[s], g, scale, data.sets, s, with_start_test=True, extra_ranks=True)
        r["seed"] = s
        per[(g, s)] = r
        return r

    log(f"  Maass check, {version} version: gain sweep on seeds {list(seeds)}")
    sw = gain.sweep(measure, seeds, log=log)
    res = {"version": version, "sweep": sw, "input_scale": scale}
    for label, g in (("own_weights", 1.0), ("rule", sw["chosen_g"])):
        if g is None:
            res[label] = {"g": None, "level3": False, "level4": False}
            continue
        l3, rows = level3_verdict([per[(g, s)] for s in seeds], sw)
        on = [memory_curve(ctx, liqs[s], g, scale, data.mem_spikes, data.mem_targets, s) for s in seeds]
        off = [memory_curve(ctx, liqs[s], g, scale, data.mem_spikes, data.mem_targets, s, recurrence=False) for s in seeds]
        l4, gains = level4_verdict(on, off)
        res[label] = {"g": g, "level3": l3, "level3_rows": rows, "level4": l4, "memory_gain_100ms": gains,
                      "memory_on": on, "memory_off": off}
        log(f"    {version} at g = {g:g} ({label}): level 3 {'pass' if l3 else 'FAIL'}, level 4 "
            f"{'pass' if l4 else 'FAIL'} (R^2 gains at 100 ms {np.round(gains, 3).tolist()})")
    # the test must also reject something: some g on the sweep fails the working-regime rules
    fails = [g for g in sw["table"] if g > 0 and not all(
        (C.BAND_HZ[0] <= per[(g, s)]["rate_hz"] <= C.BAND_HZ[1] and per[(g, s)]["frac_above_100hz"] <= C.HIGH_FRAC
         and per[(g, s)]["start_rule_pass"]) for s in seeds)]
    res["rejects_some_g"] = bool(fails)
    res["rejected_g"] = sorted(fails)
    # what keeps the rank measure: an in-band peak above the smallest g tried, and a lower gap at the next g up
    gs = sorted(g for g in sw["table"] if g > 0)
    ch = sw["chosen_g"]
    res["rank_measure_kept"] = bool(ch is not None and ch > gs[0] and sw.get("next_higher_gap_lower", False))
    if version == "paper":
        res["input_neuron_rate_recurrence_off_hz"] = input_neuron_rate(ctx, list(liqs.values()), scale, data.sets["sep"])
    res["pass"] = bool(res["own_weights"]["level3"] and res["own_weights"]["level4"] and res["rule"]["level3"]
                       and res["rule"]["level4"] and res["rejects_some_g"] and res["rank_measure_kept"])
    return res


def maass_check(ctx, data, seeds=C.SEEDS_SEARCH, log=print):
    out = {v: run_version(ctx, v, seeds, data, log) for v in ("script", "paper")}
    out["pass"] = bool(out["script"]["pass"] and out["paper"]["pass"])
    out["factor5_target_hz"] = out["paper"]["input_neuron_rate_recurrence_off_hz"]
    log(f"  Maass check: {'PASS' if out['pass'] else 'FAIL'}; factor-5 target rate {out['factor5_target_hz']:.3f} Hz")
    return out
'''
LSM2_FILES['lsm2/measure.py'] = r'''"""Label-free measurements: batching beats through the simulator, firing, the rank scores, the starting-state
rule, twin runs (level 3) and the memory test (level 4)."""
import math
import numpy as np
import torch

from . import config as C
from .sim import Simulator, start_voltages


class Ctx:
    """Where and how to simulate: device, precision and a fixed batch size (plan, determinism)."""
    def __init__(self, device="cpu", dtype=torch.float32, batch=C.CODE["sim_batch"], log=print):
        self.device, self.dtype, self.batch, self.log = device, dtype, batch, log


def simulate(ctx, liq, g, in_scale, spikes, u0, T, rec, recurrence=True):
    """spikes: (n, T_in, C) numpy input spikes; u0: (n, N). Returns the simulator outputs as numpy, batched."""
    sm = Simulator(liq, device=ctx.device, dtype=ctx.dtype, g=g, in_scale=in_scale, recurrence=recurrence)
    parts = []
    for s in range(0, len(spikes), ctx.batch):
        o = sm.run(spikes[s:s + ctx.batch], u0[s:s + ctx.batch], T, rec=rec)
        parts.append({k: v.cpu().numpy() if torch.is_tensor(v) else v for k, v in o.items() if k != "u_end"})
    out = {}
    for k in parts[0]:
        if k == "states_at_times":
            out[k] = parts[0][k]
        elif k == "raster":
            out[k] = np.concatenate([p[k] for p in parts], 1)
        else:
            out[k] = np.concatenate([p[k] for p in parts], 0)
    return out


# ---------------------------------------------------------------- rank scores
def effective_rank(M):
    """Roy & Vetterli 2007: exp of the entropy of the normalised singular values, of the centred matrix
    (float64). 0 for an all-zero matrix."""
    M = np.asarray(M, np.float64)
    M = M - M.mean(0, keepdims=True)
    s = np.linalg.svd(M, compute_uv=False)
    tot = s.sum()
    if tot <= 0:
        return 0.0
    p = s / tot
    p = p[p > 0]
    return float(np.exp(-(p * np.log(p)).sum()))


def plain_rank(M):
    """(e)'s plain rank (reported only)."""
    M = np.asarray(M, np.float64)
    return int(np.linalg.matrix_rank(M)) if M.any() else 0


def cutoff_rank(M, frac=0.01):
    """An earlier draft's measure: singular values above 1% of the largest, uncentred (reported only)."""
    s = np.linalg.svd(np.asarray(M, np.float64), compute_uv=False)
    return int((s > frac * s[0]).sum()) if len(s) and s[0] > 0 else 0


# ---------------------------------------------------------------- input sets
def jitter_spikes(spikes, rng, sd_ms=C.JITTER_MS, step_ms=C.SAMPLE_MS):
    """Each encoded spike moved by a Gaussian amount with SD 4 ms, rounded to the nearest sample; spikes moved
    outside the beat are dropped (plan, Generalisation score). spikes: (T, C) counts."""
    T, Cn = spikes.shape
    out = np.zeros((T, Cn), np.uint8)
    tt, cc = np.nonzero(spikes)
    reps = spikes[tt, cc].astype(int)
    tt, cc = np.repeat(tt, reps), np.repeat(cc, reps)
    nt = tt + np.rint(rng.normal(0, sd_ms / step_ms, len(tt))).astype(int)
    ok = (nt >= 0) & (nt < T)
    np.add.at(out, (nt[ok], cc[ok]), 1)
    return out


def twin_spike(spikes, rng, t=C.R_SAMPLE):
    """The same input with one extra spike at sample t (the R peak), on a channel silent there."""
    s = spikes.copy()
    free = np.nonzero(s[t] == 0)[0]
    ch = rng.choice(free) if len(free) else rng.integers(s.shape[1])
    s[t, ch] = min(255, int(s[t, ch]) + 1)
    return s


def level3_sets(sep_spikes, gen_base, seed_sets=0, n_twin=None):
    """sep_spikes: (500, T, C) spikes of 500 different inputs; gen_base: (4, T, C) the 4 inputs that are jittered.
    Returns the separation, generalisation and twin input sets (fixed for every candidate and g)."""
    n_twin = C.CODE["twin_beats"] if n_twin is None else n_twin
    gen = np.stack([jitter_spikes(gen_base[i], np.random.default_rng([seed_sets, 31, i, c]))
                    for i in range(len(gen_base)) for c in range(C.GEN_COPIES)])
    twin = np.stack([twin_spike(sep_spikes[i], np.random.default_rng([seed_sets, 37, i])) for i in range(min(n_twin, len(sep_spikes)))])
    return {"sep": sep_spikes, "gen": gen, "twin": twin}


# ---------------------------------------------------------------- level 3 at one g
def firing_stats(neuron_counts, n_samples, step_ms=C.SAMPLE_MS):
    """Mean rate over all neurons (silent ones included) and the share of neurons above 100 Hz.
    neuron_counts: (beats, N) spike counts over n_samples input steps."""
    dur_s = n_samples * step_ms / 1000.0
    per_neuron = neuron_counts.mean(0) / dur_s
    return float(per_neuron.mean()), float((per_neuron > C.HIGH_HZ).mean())


def measure_at_g(ctx, liq, g, in_scale, sets, seed, T=C.BEAT_SAMPLES, read=C.STATE_READ_SAMPLE,
                 with_start_test=False, with_twins=True, extra_ranks=False):
    """Every level-3 measure at one g for one liquid. Starting voltages: uniform between the resting level and
    the threshold for every candidate, drawn from (seed, stream, input index)."""
    nb = len(sets["sep"])
    u_sep = start_voltages(liq, seed, range(nb), 1, random_start=True)
    rec = {"states_at": [read, C.START_TEST_SAMPLE], "neuron_counts": (0, T)}
    o = simulate(ctx, liq, g, in_scale, sets["sep"], u_sep, T, rec)
    rate, high = firing_stats(o["neuron_counts"], T, liq.h_ms * liq.n_sub)
    sep_states = o["states_at"][:, 0]
    u_gen = start_voltages(liq, seed, range(len(sets["gen"])), 2, random_start=True)
    og = simulate(ctx, liq, g, in_scale, sets["gen"], u_gen, T, {"states_at": [read]})
    gen_states = og["states_at"][:, 0]
    res = {"g": float(g), "abs_g": liq.ee_mean_push(g), "rate_hz": rate, "frac_above_100hz": high,
           "in_band": bool(C.BAND_HZ[0] <= rate <= C.BAND_HZ[1] and high <= C.HIGH_FRAC),
           "sep": effective_rank(sep_states), "gen": effective_rank(gen_states)}
    res["gap"] = res["sep"] - res["gen"]
    if extra_ranks:
        res.update({"sep_plain": plain_rank(sep_states), "gen_plain": plain_rank(gen_states),
                    "sep_cut1": cutoff_rank(sep_states), "gen_cut1": cutoff_rank(gen_states)})
        res["gap_plain"], res["gap_cut1"] = res["sep_plain"] - res["gen_plain"], res["sep_cut1"] - res["gen_cut1"]
    if with_twins and len(sets["twin"]):
        nt = len(sets["twin"])
        u_t = u_sep[:nt]
        rt = {"states_every": (0, T)}
        a = simulate(ctx, liq, g, in_scale, sets["sep"][:nt], u_t, T, rt)["states_every"]
        b = simulate(ctx, liq, g, in_scale, sets["twin"], u_t, T, rt)["states_every"]
        dist = np.sqrt(((a - b) ** 2).sum(-1)).mean(0)          # mean over beats of the state distance, per sample
        e0, e1 = C.CODE["twin_early"]; l0, l1 = C.CODE["twin_late"]
        early, late = dist[e0:e1 + 1].mean(), dist[l0:l1 + 1].mean()
        res["twin_early"], res["twin_late"] = float(early), float(late)
        res["twin_ratio"] = float(late / early) if early > 0 else float("nan")
    if with_start_test:
        u2 = start_voltages(liq, seed, range(nb), 3, random_start=True)
        o2 = simulate(ctx, liq, g, in_scale, sets["sep"], u2, T, {"states_at": [C.START_TEST_SAMPLE]})
        s1, s2 = o["states_at"][:, 1].astype(np.float64), o2["states_at"][:, 0].astype(np.float64)
        same = float(np.sqrt(((s1 - s2) ** 2).sum(1)).mean())
        sq = (s1 ** 2).sum(1)
        D2 = np.maximum(sq[:, None] + sq[None, :] - 2 * s1 @ s1.T, 0)
        iu = np.triu_indices(len(s1), 1)
        diff = float(np.sqrt(D2[iu]).mean())
        res.update({"start_same": same, "start_diff": diff,
                    "start_rule_pass": bool(same < diff or (same == 0 and diff == 0))})
    return res


# ---------------------------------------------------------------- level 4: memory
def surrogate_walks(steps_pool, n_seq, T, seed):
    """Random walks on [0, 1], reflected at the edges, step sizes drawn from ECG's own sample-to-sample steps
    with random signs (plan, level 4). Starting levels uniform on [0, 1]."""
    rng = np.random.default_rng([seed, 41])
    X = np.empty((n_seq, T))
    for i in range(n_seq):
        st = rng.choice(steps_pool, T - 1) * rng.choice([-1.0, 1.0], T - 1)
        v = rng.uniform(0, 1)
        X[i, 0] = v
        for t in range(1, T):
            v = v + st[t - 1]
            while v < 0 or v > 1:
                v = -v if v < 0 else 2 - v
            X[i, t] = v
    return X


class _Stats:
    """Sufficient statistics of (x, y) rows for ridge with an intercept: n, sum x, sum y, sum y^2, X'X, X'y."""
    def __init__(self, N, dev):
        z = lambda *sh: torch.zeros(*sh, dtype=torch.float64, device=dev)
        self.n, self.sx, self.sy, self.syy, self.G, self.b = 0, z(N), z(()), z(()), z(N, N), z(N)

    def add(self, X, y):
        self.n += X.shape[0]; self.sx += X.sum(0); self.sy += y.sum(); self.syy += (y * y).sum()
        self.G += X.T @ X; self.b += X.T @ y

    def __add__(self, o):
        r = _Stats.__new__(_Stats)
        r.n = self.n + o.n
        for k in ("sx", "sy", "syy", "G", "b"):
            setattr(r, k, getattr(self, k) + getattr(o, k))
        return r


def _fit(st, alpha):
    mu, my = st.sx / st.n, st.sy / st.n
    Gc = st.G - st.n * torch.outer(mu, mu)
    bc = st.b - st.n * mu * my
    w = torch.linalg.solve(Gc + alpha * torch.eye(len(mu), dtype=Gc.dtype, device=Gc.device), bc)
    return w, mu, my


def _ss_res(st, w, mu, my):
    """sum over st's rows of (y - my - (x - mu)'w)^2, from its statistics."""
    c = my - mu @ w                                     # prediction = x'w + c
    return float(st.syy - 2 * (w @ st.b) - 2 * c * st.sy + w @ st.G @ w + 2 * c * (w @ st.sx) + st.n * c * c)


def _ss_tot(st, mean=None):
    m = st.sy / st.n if mean is None else mean
    return float(st.syy - 2 * m * st.sy + st.n * m * m)


def memory_curve(ctx, liq, g, in_scale, seq_spikes, target_counts, seed, T=C.BEAT_SAMPLES,
                 lags_ms=C.MEMORY_T_MS, warmup=C.MEMORY_WARMUP, recurrence=True, batch=100,
                 alphas=C.CODE["ridge_alphas"], k=C.CODE["ridge_inner_folds"]):
    """R^2 of reconstructing target[t - T] from the 30 ms-filtered state at t, for each lag: a ridge readout
    trained on the first half of the sequences (penalty by inner CV, folds grouped by sequence) and scored on
    the second half, at every sample after the warm-up. Statistics are accumulated batch by batch."""
    n = len(seq_spikes)
    step_ms = liq.h_ms * liq.n_sub
    half = n // 2
    fold = np.full(n, k)                                      # group k = test sequences
    fold[:half] = np.random.default_rng(0).permutation(half) % k
    lagL = {lag: int(round(lag / step_ms)) for lag in lags_ms}
    dev = ctx.device
    stats = {lag: [_Stats(liq.N, dev) for _ in range(k + 1)] for lag in lags_ms}
    sm = Simulator(liq, device=dev, dtype=ctx.dtype, g=g, in_scale=in_scale, recurrence=recurrence)
    u0 = start_voltages(liq, seed, range(n), 4)               # the liquid's own starting rule, as on ECG
    counts = 0.0
    for s0 in range(0, n, batch):
        o = sm.run(seq_spikes[s0:s0 + batch], u0[s0:s0 + batch], T, rec={"states_every": (0, T), "neuron_counts": (0, T)})
        Z = o["states_every"].to(torch.float64)               # (b, T, N)
        counts += float(o["neuron_counts"].sum())
        tg = torch.as_tensor(np.asarray(target_counts[s0:s0 + batch]), dtype=torch.float64, device=dev)
        for lag, L in lagL.items():
            t0 = max(warmup, L)
            for bi in range(Z.shape[0]):
                X = Z[bi, t0:T]
                y = tg[bi, t0 - L:T - L]
                stats[lag][fold[s0 + bi]].add(X, y)
        del o, Z
    out = {"rate_hz": counts / (n * liq.N * T * step_ms / 1000.0)}
    for lag in lags_ms:
        st = stats[lag]
        train_all = st[0]
        for f in range(1, k):
            train_all = train_all + st[f]
        best, best_a = -np.inf, alphas[0]
        for a in alphas:
            res = tot = 0.0
            for f in range(k):
                tr = None
                for f2 in range(k):
                    if f2 != f:
                        tr = st[f2] if tr is None else tr + st[f2]
                w, mu, my = _fit(tr, a)
                res += _ss_res(st[f], w, mu, my)
                tot += _ss_tot(st[f], float(train_all.sy / train_all.n))
            r2 = 1 - res / tot if tot > 0 else float("nan")
            if r2 > best:
                best, best_a = r2, a
        w, mu, my = _fit(train_all, best_a)
        tot = _ss_tot(st[k])
        out[f"r2_{lag}"] = float(1 - _ss_res(st[k], w, mu, my) / tot) if tot > 0 else float("nan")
        out[f"alpha_{lag}"] = float(best_a)
    return out
'''
LSM2_FILES['lsm2/sim.py'] = r'''"""The simulator: one batched update for X, Y and every candidate between them (each factor is a setting of the
Liquid), plus the Brian2 schedule used only for (d)'s own liquid (level 1e and the Maass check's script version).

Units: thresholds (the voltage threshold is 1). Time: input steps (one ECG sample, or 0.1 ms for 1e), each split
into n sub-steps. Our order within a sub-step (plan, Simulation details):
  1. synapse states recover/fade over the sub-step; spikes whose delay ends now arrive (dynamic synapses
     update u, push, R in (d)'s order)
  2. currents: without fine timing a push arrives all at once; with it, it feeds a current that decays over
     3 ms (from E and from inputs) or 6 ms (from I), scaled so its total equals the push
  3. every neuron integrates: u <- beta u + input + recurrent + tap (not during a clamped refractory period)
  4. threshold test (u >= 1, outside the minimum gap), then reset (subtract 1, or set to 0.9 with the tap)
Input spikes of an input step act in its first sub-step ((a)'s Eq. 16: x[t] W_in in step t); this step's spikes
reach their targets from the next sub-step on (delay >= 1).
"""
import math
import numpy as np
import torch

from . import config as C


def _t(a, device, dtype):
    return torch.as_tensor(np.asarray(a), device=device, dtype=dtype)


class Simulator:
    def __init__(self, liq, device="cpu", dtype=torch.float64, g=1.0, in_scale=None, recurrence=True):
        self.liq, self.device, self.dtype = liq, torch.device(device), dtype
        self.g = float(g) if recurrence else 0.0
        if in_scale is None:
            if liq.in_scale_fixed is None:
                raise ValueError("this liquid needs an input size (paper rule or calibration)")
            in_scale = liq.in_scale_fixed
        self.in_scale = float(in_scale)
        d, t = self.device, dtype
        N = liq.N
        self.N = N
        self.beta = _t(liq.beta, d, t)
        self.tap = _t(liq.tap, d, t)
        self.gap = _t(liq.gap, d, torch.int64)
        self.Win = _t(liq.Win * self.in_scale, d, t)
        self.z_decay = math.exp(-liq.h_ms / C.STATE_TAU_MS)
        self.L = int(max(liq.delay.max(), 1)) + 1           # spike-history ring length
        S = liq.S
        self.S = S
        self.static = liq.syn_model == "static"
        if liq.schedule == "ours":
            if self.static:
                # one dense matrix per (delay, presynaptic class): pushes of the spikes that left `delay` sub-steps ago
                self.groups = {}
                for ti in range(4):
                    m = liq.typ == ti
                    if not m.any():
                        continue
                    key = (int(liq.delay[ti]), 0 if ti < 2 else 1)
                    W = self.groups.get(key)
                    if W is None:
                        W = torch.zeros(N, N, device=d, dtype=t)
                    W[_t(liq.pre[m], d, torch.int64), _t(liq.post[m], d, torch.int64)] += _t(liq.w[m] * self.g, d, t)
                    self.groups[key] = W
            else:
                self._prep_sparse()
                Ubar_s = liq.Ubar[liq.typ]
                self.wg = _t(liq.w * self.g / Ubar_s, d, t)        # push = (w / U-bar) u R
                self.U = _t(liq.U, d, t)
                self.eD = _t(np.exp(-liq.h_ms / liq.D_ms), d, t)
                self.eF = _t(np.exp(-liq.h_ms / liq.F_ms), d, t)
            if liq.cur_a is not None:
                self.aE, self.aI = float(liq.cur_a[0]), float(liq.cur_a[1])
            self.in_a = None if liq.in_a is None else _t(liq.in_a, d, t)
        else:   # Brian2 schedule, (d)'s three-state synapses, currents in voltage-equivalent units
            self._prep_sparse()
            self.wg = _t(liq.w * self.g, d, t)
            self.U = _t(liq.U, d, t)
            tau = liq.syn_tau_ms[liq.typ]
            D = liq.D_ms
            h = liq.h_ms
            self.ey = _t(np.exp(-h / tau), d, t)
            self.eD = _t(np.exp(-h / D), d, t)
            self.cxy = _t(tau * (np.exp(-h / tau) - np.exp(-h / D)) / (D - tau), d, t)
            self.eF = _t(np.exp(-h / liq.F_ms), d, t)
            self.kappa = _t(liq.kappa, d, t)
            self.in_a = _t(liq.in_a, d, t)
            self.J_b = float(liq.J_b)

    def _prep_sparse(self):
        liq, d, t = self.liq, self.device, self.dtype
        self.pre_idx = _t(liq.pre, d, torch.int64)
        self.syn_delay = liq.delay[liq.typ]
        self.delay_groups = [(int(dl), _t(np.nonzero(self.syn_delay == dl)[0], d, torch.int64))
                             for dl in np.unique(self.syn_delay)]
        self.nE = int((liq.typ < 2).sum())                  # synapses are sorted by type: E-presynaptic first
        M = torch.zeros(liq.S, liq.N, device=d, dtype=t)
        if liq.S:
            M[torch.arange(liq.S, device=d), _t(liq.post, d, torch.int64)] = 1.0
        self.M_E, self.M_I, self.M = M[:self.nE], M[self.nE:], M

    # ------------------------------------------------------------------ helpers
    def _arrivals(self, hist, k, B):
        """(B, S) 0/1: which synapses receive a spike in sub-step k."""
        a = torch.zeros(B, self.S, device=self.device, dtype=self.dtype)
        for dl, idx in self.delay_groups:
            if k - dl < 0:
                continue
            src = hist[(k - dl) % self.L]
            a[:, idx] = src[:, self.pre_idx[idx]]
        return a

    def start_state(self, u0):
        return torch.as_tensor(u0, device=self.device, dtype=self.dtype).clone()

    # ------------------------------------------------------------------ main loop
    @torch.no_grad()
    def run(self, x, u0, T, rec=None, open_loop=None, trace=False, own_spikes=None):
        """x: (B, T_in, C) input spikes per input step; u0: (B, N) starting voltages; T: input steps to simulate.
        rec: {"windows": (win, nwin), "states_at": [t, ...], "states_every": (t0, t1), "neuron_counts": (t0, t1),
              "raster": bool}; times are input steps, a state is read at the end of its step.
        open_loop: (T*n, B, N) recorded spikes used as the presynaptic spikes (the liquid's own spikes are still
        computed and returned). own_spikes: (T*n, B, N) recorded spikes that also replace each neuron's own
        threshold decisions (level 1c: pure arithmetic drift, no flips); out["flips"] counts where the threshold
        test would have decided differently. trace: return u and currents every sub-step and synapse states
        every input step."""
        liq, dev, dt = self.liq, self.device, self.dtype
        rec = rec or {}
        x = torch.as_tensor(x, device=dev)
        B, T_in, Cin = x.shape
        N, n = self.N, liq.n_sub
        u = self.start_state(u0)
        ref = torch.zeros(B, N, device=dev, dtype=torch.int64)
        hist = torch.zeros(self.L, B, N, device=dev, dtype=dt)
        z = torch.zeros(B, N, device=dev, dtype=dt)
        ours = liq.schedule == "ours"
        fine = ours and liq.cur_a is not None
        IE = torch.zeros(B, N, device=dev, dtype=dt) if fine else None
        II = torch.zeros(B, N, device=dev, dtype=dt) if fine else None
        Iin = torch.zeros(B, N, device=dev, dtype=dt) if (self.in_a is not None) else None
        if not self.static or not ours:
            R = torch.ones(B, self.S, device=dev, dtype=dt)       # resources ready (x in (d))
            q = torch.zeros(B, self.S, device=dev, dtype=dt)      # utilisation u
            y = torch.zeros(B, self.S, device=dev, dtype=dt) if not ours else None
        out = {}
        win = rec.get("windows")
        if win:
            wl, nw = win
            counts = torch.zeros(B, nw, N, device=dev, dtype=torch.int32)
        st_at = sorted(set(rec.get("states_at", [])))
        st_at_set = {t: i for i, t in enumerate(st_at)}
        if st_at:
            states = torch.zeros(B, len(st_at), N, device=dev, dtype=torch.float32)
        se = rec.get("states_every")
        if se:
            states_every = torch.zeros(B, se[1] - se[0], N, device=dev, dtype=torch.float32)
        nc = rec.get("neuron_counts")
        if nc:
            ncount = torch.zeros(B, N, device=dev, dtype=torch.int64)
        raster = torch.zeros(T * n, B, N, device=dev, dtype=torch.bool) if rec.get("raster") else None
        tr = {"u": [], "I1": [], "I2": [], "I3": [], "syn": []} if trace else None
        if open_loop is not None:
            open_loop = torch.as_tensor(open_loop, device=dev)
        if own_spikes is not None:
            own_spikes = torch.as_tensor(own_spikes, device=dev).bool()
            flips = torch.zeros((), device=dev, dtype=torch.int64)
        zero_in = torch.zeros(B, N, device=dev, dtype=dt)
        k = 0
        for t in range(T):
            step_spk = torch.zeros(B, N, device=dev, dtype=dt) if (win or nc) else None
            if t < T_in:
                in_push = x[:, t, :].to(dt) @ self.Win
            else:
                in_push = zero_in
            for j in range(n):
                blocked = ref > 0
                if ours:
                    # 1. synapses
                    if self.static:
                        pE = pI = None
                        for (dl, cls), W in self.groups.items():
                            if k - dl < 0:
                                continue
                            p = hist[(k - dl) % self.L] @ W
                            if cls == 0:
                                pE = p if pE is None else pE + p
                            else:
                                pI = p if pI is None else pI + p
                        pE = zero_in if pE is None else pE
                        pI = zero_in if pI is None else pI
                    else:
                        R = 1.0 - (1.0 - R) * self.eD
                        q = q * self.eF
                        a = self._arrivals(hist, k, B)
                        q = q + a * self.U * (1.0 - q)
                        push = a * self.wg * q * R
                        R = R - a * q * R
                        pE = push[:, :self.nE] @ self.M_E if self.nE else zero_in
                        pI = push[:, self.nE:] @ self.M_I if self.S > self.nE else zero_in
                    ip = in_push if j == 0 else zero_in
                    # 2. currents
                    if fine:
                        IE = self.aE * IE + (1 - self.aE) * pE
                        II = self.aI * II + (1 - self.aI) * pI
                        Iin = self.in_a * Iin + (1 - self.in_a) * ip
                        total = IE + II + Iin
                    else:
                        total = pE + pI + ip
                    # 3. integrate
                    u_new = self.beta * u + total + self.tap
                    if liq.clamp:
                        u_new = torch.where(blocked, u, u_new)
                    # 4. threshold and reset
                    if liq.snn_quirk:       # snnTorch: the spike test sees u_new - [u_prev > 1]
                        crossed = (u_new - (u > 1.0).to(dt)) > 1.0
                    else:
                        crossed = (u_new > 1.0) if liq.strict else (u_new >= 1.0)
                    spk = crossed & ~blocked
                    if own_spikes is not None:
                        flips += (spk != own_spikes[k]).sum()
                        spk = own_spikes[k]
                    if liq.reset_mode == "subtract":
                        u = u_new - spk.to(dt)
                    else:
                        u = torch.where(spk, torch.full_like(u_new, liq.reset_value), u_new)
                else:
                    # Brian2 schedule of (d)'s network (its sorted objects): synapse state update (exact), the
                    # summed current from the updated y, exact neuron update with that current held constant over
                    # the step and the input current decaying exactly; threshold; arrivals (on_pre); reset.
                    q = q * self.eF
                    R, y = R * self.eD + (1 - self.eD) + y * self.cxy, y * self.ey
                    Jsyn = (y * self.wg) @ self.M if self.S else zero_in
                    u_new = self.beta * u + (1 - self.beta) * (self.J_b + Jsyn) + self.kappa * Iin
                    u_new = torch.where(blocked, u, u_new)
                    Iin = Iin * self.in_a
                    spk = (u_new > 1.0) & ~blocked
                    if own_spikes is not None:
                        flips += (spk != own_spikes[k]).sum()
                        spk = own_spikes[k]
                    if j == 0 and t < T_in:
                        Iin = Iin + in_push
                    hist[k % self.L] = (open_loop[k] if open_loop is not None else spk).to(dt)
                    a = self._arrivals(hist, k, B)
                    q = q + a * self.U * (1.0 - q)
                    y = y + a * q * R
                    R = R - a * q * R
                    u = torch.where(spk, torch.full_like(u_new, liq.reset_value), u_new)
                ref = torch.where(spk, self.gap - 1, torch.clamp(ref - 1, min=0))
                if ours:
                    hist[k % self.L] = (open_loop[k] if open_loop is not None else spk).to(dt)
                spf = spk.to(dt)
                z = z * self.z_decay + spf
                if step_spk is not None:
                    step_spk += spf
                if raster is not None:
                    raster[k] = spk
                if trace:
                    tr["u"].append(u.detach().cpu().numpy().copy())
                    if ours:
                        tr["I1"].append((IE if fine else pE).cpu().numpy().copy())
                        tr["I2"].append((II if fine else pI).cpu().numpy().copy())
                        tr["I3"].append((Iin if fine else ip).cpu().numpy().copy())
                    else:
                        tr["I1"].append(Jsyn.cpu().numpy().copy()); tr["I3"].append(Iin.cpu().numpy().copy())
                k += 1
            # end of input step t
            if win and t < wl * nw:
                counts[:, t // wl] += step_spk.to(torch.int32)
            if nc and nc[0] <= t < nc[1]:
                ncount += step_spk.to(torch.int64)
            if t in st_at_set:
                states[:, st_at_set[t]] = z.float()
            if se and se[0] <= t < se[1]:
                states_every[:, t - se[0]] = z.float()
            if trace and (not self.static or not ours):
                syn = {"R": R.cpu().numpy().copy(), "u": q.cpu().numpy().copy()}
                if not ours:
                    syn["y"] = y.cpu().numpy().copy()
                tr["syn"].append(syn)
        if win:
            out["windows"] = counts
        if st_at:
            out["states_at"] = states
            out["states_at_times"] = st_at
        if se:
            out["states_every"] = states_every
        if nc:
            out["neuron_counts"] = ncount
        if raster is not None:
            out["raster"] = raster
        if trace:
            out["trace"] = tr
        out["u_end"] = u
        if own_spikes is not None:
            out["flips"] = int(flips)
        return out


# ---------------------------------------------------------------- starting states (CPU, from (seed, beat index))
def start_voltages(liq, seed, beat_ids, stream, random_start=None):
    """Starting voltages for each beat, drawn on the CPU from (seed, stream, beat index) so they do not depend
    on batching or the card. random_start=None: the liquid's own rule (0 in X; uniform 0.9..1 with the tap).
    random_start=True: uniform between the resting level and the threshold (level 3's two starting states)."""
    lo, hi = (liq.rest, 1.0) if random_start else (liq.start_lo, liq.start_hi)
    out = np.empty((len(beat_ids), liq.N))
    for i, b in enumerate(beat_ids):
        if hi > lo:
            out[i] = np.random.default_rng([int(seed), int(stream), int(b)]).uniform(lo, hi, liq.N)
        else:
            out[i] = lo
    return out
'''
LSM2_FILES['lsm2/stats.py'] = r'''"""Statistics and decision rules (plan, Statistics and decision rules): the paired Nadeau-Bengio corrected
t-test over the 12 CV splits (variance factor 1/12 + 0.336), Holm within a family, the DS2 difference paired by
seed, and the verdicts meaningful / comparable / inconclusive. Inputs are tables of runs with columns spec_split,
spec_seed, macro_f1 (and DS2 metrics)."""
import math
import numpy as np
from scipy import stats as sstats

NB_RATIO = 0.336          # plan: test/train ratio of the 3x4 patient-grouped CV (the replication used 0.25/0.75)


def per_split_seed(d):
    """Macro-F1 by (split, seed)."""
    return d.groupby(["spec_split", "spec_seed"], dropna=False).macro_f1.mean()

def compare(a, b, require_seed_sign=False):
    """A vs B (Macro-F1): seeds averaged per CV split; paired Nadeau-Bengio corrected t-test over the 12 splits
    with its 95% interval; the DS2 difference paired by seed."""
    if a is None or b is None or a.empty or b.empty:
        return None
    A, B = per_split_seed(a), per_split_seed(b)
    if a["spec_seed"].isna().all() or b["spec_seed"].isna().all():      # a seed-free control pairs with every seed
        A = A.groupby(level=0).mean(); B = B.groupby(level=0).mean()
        Asp, Bsp = A, B
        ds2_seed = None
    else:
        Asp, Bsp = A.groupby(level=0).mean(), B.groupby(level=0).mean()
        ds2_seed = (A.xs("final", level=0) - B.xs("final", level=0)).dropna() if "final" in A.index.get_level_values(0) and "final" in B.index.get_level_values(0) else None
    cv = [s for s in Asp.index if str(s).startswith("cv-") and s in Bsp.index]
    if len(cv) < 3:
        return None
    d = (Asp[cv] - Bsp[cv]).values
    k = len(d)
    var = np.var(d, ddof=1) * (1 / k + NB_RATIO)
    se = math.sqrt(var)
    t = d.mean() / se if se > 0 else np.inf
    p = 2 * sstats.t.sf(abs(t), k - 1)
    q = sstats.t.ppf(0.975, k - 1)
    ds2 = float(Asp.get("final", np.nan) - Bsp.get("final", np.nan)) if ("final" in Asp.index and "final" in Bsp.index) else np.nan
    out = {"n_splits": k, "cv_diff": float(d.mean()), "ci_low": float(d.mean() - q * se), "ci_high": float(d.mean() + q * se),
           "t": float(t), "p": float(p), "ds2_diff": ds2,
           "ds2_diff_by_seed": None if ds2_seed is None else {str(s): float(v) for s, v in ds2_seed.items()}}
    # DS2 without record 232, and per-class SVEB / VEB recall and precision (plan: every comparison)
    fa, fb = a[a.spec_split == "final"], b[b.spec_split == "final"]
    for m in ("macro_f1_no232", "recall_SVEB", "precision_SVEB", "recall_VEB", "precision_VEB"):
        if m in fa and m in fb and len(fa) and len(fb):
            out[f"ds2_diff_{m}"] = float(fa[m].mean() - fb[m].mean())
    if require_seed_sign and not a["spec_seed"].isna().all():
        per_seed = []
        for sd in sorted(set(a.spec_seed.dropna()) & set(b.spec_seed.dropna())):
            xa = a[(a.spec_seed == sd) & a.spec_split.astype(str).str.startswith("cv-")].groupby("spec_split").macro_f1.mean()
            xb = b[(b.spec_seed == sd) & b.spec_split.astype(str).str.startswith("cv-")].groupby("spec_split").macro_f1.mean()
            j = xa.index.intersection(xb.index)
            per_seed.append(float((xa[j] - xb[j]).mean()))
        out["cv_diff_by_seed"] = per_seed
        out["same_sign_all_seeds"] = bool(per_seed and (all(x > 0 for x in per_seed) or all(x < 0 for x in per_seed)))
    return out

def compare_diff(a_on, a_off, b_on, b_off):
    """F7: (on - off, full) - (on - off, ablated), per split (seeds averaged), corrected t-test."""
    parts = [per_split_seed(x).groupby(level=0).mean() if x is not None and not x.empty else None for x in (a_on, a_off, b_on, b_off)]
    if any(p is None for p in parts):
        return None
    D = (parts[0] - parts[1]) - (parts[2] - parts[3])
    cv = [s for s in D.index if str(s).startswith("cv-") and np.isfinite(D[s])]
    if len(cv) < 3:
        return None
    d = D[cv].values; k = len(d)
    se = math.sqrt(np.var(d, ddof=1) * (1 / k + NB_RATIO)); t = d.mean() / se if se > 0 else np.inf
    q = sstats.t.ppf(0.975, k - 1)
    return {"n_splits": k, "cv_diff": float(d.mean()), "ci_low": float(d.mean() - q * se), "ci_high": float(d.mean() + q * se),
            "t": float(t), "p": float(2 * sstats.t.sf(abs(t), k - 1)), "ds2_diff": float(D.get("final", np.nan)),
            "plain_full_minus_ablated_on": float((parts[0] - parts[2])[cv].mean())}

def holm(rows):
    ps = [r["p"] if r else np.nan for r in rows]
    idx = [i for i, p in enumerate(ps) if np.isfinite(p)]
    order = sorted(idx, key=lambda i: ps[i]); m = len(order); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * ps[i]))
        rows[i]["p_holm"] = run
    return rows

def verdict(r, need_seed_sign=False):
    if r is None:
        return "missing"
    meaningful = abs(r["cv_diff"]) >= 0.01 and r.get("p_holm", 1) < 0.05 and np.sign(r["ds2_diff"]) == np.sign(r["cv_diff"])
    if need_seed_sign:
        meaningful = meaningful and r.get("same_sign_all_seeds", False)
    if meaningful:
        return "meaningful"
    if r["ci_low"] >= -0.02 and r["ci_high"] <= 0.02 and abs(r["ds2_diff"]) < 0.02:
        return "comparable"
    return "inconclusive"

def family(name, items, need_seed_sign=False):
    rows = [it[1] for it in items]
    rows = holm([dict(r) if r else None for r in rows])
    out = []
    for (label, _), r in zip(items, rows):
        out.append({"family": name, "comparison": label, **({} if r is None else {k: v for k, v in r.items() if not isinstance(v, (dict, list))}),
                    "verdict": verdict(r, need_seed_sign)})
    return out

'''
LSM2_FILES['lsm2_tests/test_level1a.py'] = r'''"""Level 1a: every switch against its closed-form solution, to 1e-9 in float64 (plan, Correctness criteria).
One neuron under every reset, gap and tap option; one synapse static, dynamic, with and without a decaying
current; every time conversion on both grids."""
import math
import numpy as np
import pytest
import torch

from lsm2 import config as C, liquid as L, sim as S

TOL = C.TOL_1A


def tiny(factors=frozenset(), n_sub=1, N=2, exc=(True, True), w=None, typ=0, gap=None, n_channels=1):
    """A 2-neuron liquid built from a candidate's settings: neuron 0 -> neuron 1 (one synapse of type `typ`)."""
    base = L.build_candidate(factors, 1, n_sub, n_channels)
    liq = base
    is_exc = np.array(exc, bool)
    s = L.candidate_settings(factors)
    h = C.SAMPLE_MS / n_sub
    tau = np.where(is_exc, s["tau_E_ms"], s["tau_I_ms"])
    liq.N, liq.is_exc, liq.pos = N, is_exc, np.zeros((N, 3))
    liq.beta = np.exp(-h / tau)
    liq.tap = C.TAP * (1 - liq.beta) if s["tap"] else np.zeros(N)
    if s["fine"]:
        liq.gap = np.where(is_exc, L.steps_of(3.0, h), L.steps_of(2.0, h)).astype(np.int64)
        liq.in_a = np.full(N, math.exp(-h / 3.0))
    else:
        liq.gap = np.full(N, L.steps_of(C.SAMPLE_MS, h), np.int64)
    if gap is not None:
        liq.gap = np.full(N, gap, np.int64)
    if w is None:
        liq.pre = liq.post = liq.typ = np.zeros(0, np.int64)
        liq.w = liq.U = liq.D_ms = liq.F_ms = np.zeros(0)
    else:
        liq.pre, liq.post, liq.typ = np.array([0]), np.array([1]), np.array([typ])
        liq.w = np.array([w], float)
        tname = C.TYPES[typ]
        liq.U = np.array([C.MAASS["U"][tname]])
        liq.D_ms = np.array([C.MAASS["D_s"][tname] * 1000])
        liq.F_ms = np.array([C.MAASS["F_s"][tname] * 1000])
    liq.Win = np.ones((n_channels, N))
    liq.in_neurons = np.ones(N, bool)
    return liq


def run(liq, x, T, u0, g=1.0, in_scale=1.0, **kw):
    sm = S.Simulator(liq, dtype=torch.float64, g=g, in_scale=in_scale)
    return sm.run(x, u0, T, trace=True, **kw)


def U_trace(o):
    return np.array(o["trace"]["u"])        # (sub-steps, B, N)


# ---------------------------------------------------------------- neuron
@pytest.mark.parametrize("n_sub", [1, 4, 28])
@pytest.mark.parametrize("factors", [frozenset(), frozenset({4})])
def test_leak_matches_exponential_decay(n_sub, factors):
    liq = tiny(factors, n_sub)
    T = 40
    o = run(liq, np.zeros((1, T, 1)), T, np.full((1, 2), 0.7))
    u = U_trace(o)[:, 0, :]
    s = L.candidate_settings(factors)
    t_ms = (np.arange(T * n_sub) + 1) * C.SAMPLE_MS / n_sub
    want = np.stack([0.7 * np.exp(-t_ms / s["tau_E_ms"])] * 2, 1)
    assert np.abs(u - want).max() < TOL


def test_x_leak_is_papers_beta_per_sample():
    liq = tiny(frozenset(), 1, exc=(True, False))
    o = run(liq, np.zeros((1, 10, 1)), 10, np.full((1, 2), 0.5))
    u = U_trace(o)[:, 0, :]
    assert np.abs(u[:, 0] - 0.5 * 0.95 ** np.arange(1, 11)).max() < TOL
    assert np.abs(u[:, 1] - 0.5 * 0.80 ** np.arange(1, 11)).max() < TOL


@pytest.mark.parametrize("n_sub", [1, 4, 28])
def test_tap_settles_at_0_9(n_sub):
    liq = tiny(frozenset({4, 5}), n_sub)
    T = 200
    o = run(liq, np.zeros((1, T, 1)), T, np.full((1, 2), 0.2))
    u = U_trace(o)[:, 0, 0]
    k = np.arange(1, T * n_sub + 1)
    want = 0.9 + (0.2 - 0.9) * np.exp(-k * (C.SAMPLE_MS / n_sub) / 30.0)
    assert np.abs(u - want).max() < TOL
    assert abs(u[-1] - 0.9) < 1e-6


def test_reset_subtract_constant_drive_first_spike_time():
    liq = tiny(frozenset(), 1)
    c = 0.08
    T = 60
    x = np.ones((1, T, 1))
    o = run(liq, x, T, np.zeros((1, 2)), in_scale=c, rec={"raster": True})
    beta = 0.95
    k1 = math.ceil(math.log(1 - (1 - beta) / c) / math.log(beta))   # smallest k with c (1 - beta^k)/(1 - beta) >= 1
    r = o["raster"][:, 0, 0].numpy()
    assert np.nonzero(r)[0][0] == k1 - 1                            # 0-based step index
    u = U_trace(o)[:, 0, 0]
    u_peak = c * (1 - beta ** k1) / (1 - beta)
    assert abs(u[k1 - 1] - (u_peak - 1)) < TOL                      # reset by subtracting 1


def test_reset_to_tap_and_clamp_during_gap():
    n = 4
    liq = tiny(frozenset({4, 5, 7}), n, exc=(True, False))
    T = 30
    x = np.zeros((1, T, 1)); x[0, 2, 0] = 1
    o = run(liq, x, T, np.full((1, 2), 0.9), in_scale=0.5, rec={"raster": True})
    r = o["raster"][:, 0, :].numpy()
    u = U_trace(o)[:, 0, :]
    for nrn, gap in ((0, 4), (1, 3)):          # 3 ms -> 4 sub-steps, 2 ms -> 3 sub-steps on n = 4
        ks = np.nonzero(r[:, nrn])[0]
        assert len(ks) >= 1
        k = ks[0]
        assert abs(u[k, nrn] - 0.9) < TOL                           # reset to 0.9
        assert np.abs(u[k + 1:k + gap, nrn] - 0.9).max() < TOL       # clamped for gap - 1 sub-steps
        assert abs(u[k + gap, nrn] - 0.9) > 1e-6 or not r[k + gap:, nrn].any()


@pytest.mark.parametrize("n_sub,gapE,gapI", [(4, 4, 3), (28, 30, 20)])
def test_minimum_gap_with_fine_timing(n_sub, gapE, gapI):
    liq = tiny(frozenset({4, 5, 7}), n_sub, exc=(True, False))
    T = 30
    x = np.ones((1, T, 1))
    o = run(liq, x, T, np.full((1, 2), 0.9), in_scale=50.0, rec={"raster": True})
    r = o["raster"][:, 0, :].numpy()
    for nrn, gap in ((0, gapE), (1, gapI)):
        isi = np.diff(np.nonzero(r[:, nrn])[0])
        assert isi.min() == gap
    h = C.SAMPLE_MS / n_sub
    assert abs(gapE * h - (2.78 if n_sub == 4 else 2.98)) < 0.01
    assert abs(gapI * h - (2.08 if n_sub == 4 else 1.98)) < 0.01


@pytest.mark.parametrize("n_sub", [1, 4, 28])
def test_x_gap_on_any_grid_without_clamp(n_sub):
    """X keeps its 2.78 ms gap on any grid: n - 1 blocked sub-steps, voltage still integrating."""
    liq = tiny(frozenset(), n_sub)
    T = 20
    x = np.ones((1, T, 1))
    o = run(liq, x, T, np.zeros((1, 2)), in_scale=3.0, rec={"raster": True})
    r = o["raster"][:, 0, 0].numpy()
    isi = np.diff(np.nonzero(r)[0])
    assert isi.min() == n_sub
    if n_sub > 1:
        u = U_trace(o)[:, 0, 0]
        k = np.nonzero(r)[0][0]
        assert u[k + 1] != u[k]                                       # not clamped


# ---------------------------------------------------------------- synapses
@pytest.mark.parametrize("n_sub", [1, 4])
def test_static_synapse_without_current(n_sub):
    """A spike at sub-step k adds w to its target in sub-step k + delay, all at once (X: delay = one sample)."""
    liq = tiny(frozenset(), n_sub, w=0.3)
    T = 15
    k0 = 1 * n_sub + (n_sub // 2)                                     # presynaptic spike, given open loop
    pre = np.zeros((T * n_sub, 1, 2), bool); pre[k0, 0, 0] = True
    o = run(liq, np.zeros((1, T, 1)), T, np.zeros((1, 2)), open_loop=pre)
    u1 = U_trace(o)[:, 0, 1]
    d = n_sub                                                         # one sample, in sub-steps
    beta = math.exp(-(C.SAMPLE_MS / n_sub) / L.candidate_settings(frozenset())["tau_E_ms"])
    m = np.arange(len(u1))
    want = np.where(m >= k0 + d, 0.3 * beta ** (m - (k0 + d)), 0.0)
    assert np.abs(u1 - want).max() < TOL


@pytest.mark.parametrize("n_sub", [4, 28])
@pytest.mark.parametrize("typ,tau_s", [(0, 3.0), (2, 6.0)])
def test_static_synapse_with_decaying_current(n_sub, typ, tau_s):
    """Fine timing: the push w arrives after its delay as a current decaying over tau_s, total = w.
    u_k = (1-a) w (beta^(m+1) - a^(m+1)) / (beta - a), m sub-steps after arrival."""
    exc = (typ < 2, True)
    liq = tiny(frozenset({4, 7}), n_sub, exc=exc, w=0.3 if typ < 2 else -0.3, typ=typ)
    h = C.SAMPLE_MS / n_sub
    liq.delay = np.array([L.steps_of(C.MAASS["delay_ms"][t], h) for t in C.TYPES])
    liq.cur_a = np.exp(-h / np.array([3.0, 6.0]))
    T = 40
    k0 = 5
    pre = np.zeros((T * n_sub, 1, 2), bool); pre[k0, 0, 0] = True
    o = run(liq, np.zeros((1, T, 1)), T, np.zeros((1, 2)), open_loop=pre)
    d = liq.delay[typ]
    a = math.exp(-h / tau_s)
    beta = math.exp(-h / 30.0)
    u1 = U_trace(o)[:, 0, 1]
    m = np.arange(len(u1)) - (k0 + d)
    w = liq.w[0]
    want = np.where(m >= 0, (1 - a) * w * (beta ** (m + 1) - a ** (m + 1)) / (beta - a), 0.0)
    assert np.abs(u1 - want).max() < TOL
    # total push equals w: sum of the current over all sub-steps
    I = np.array(o["trace"]["I1" if typ < 2 else "I2"])[:, 0, 1]
    assert abs(I.sum() - w * (1 - a ** (len(I) - (k0 + d)))) < TOL


@pytest.mark.parametrize("typ", [0, 1, 2, 3])
def test_dynamic_synapse_markram(typ):
    """Factor 8: per spike u <- u + U (1 - u); push = (w / U-bar) u R; R <- R - u R; between spikes R recovers
    towards 1 over D and u fades towards 0 over F (closed form between spikes)."""
    tname = C.TYPES[typ]
    liq = tiny(frozenset({4, 8}), 1, exc=(typ < 2, typ in (0, 2)), w=0.02 if typ < 2 else -0.02, typ=typ)
    liq.syn_model = "markram"
    T = 120
    rng = np.random.default_rng(typ)
    times = np.sort(rng.choice(np.arange(2, 100), 12, replace=False))
    pre = np.zeros((T, 1, 2), bool); pre[times, 0, 0] = True
    o = run(liq, np.zeros((1, T, 1)), T, np.zeros((1, 2)), open_loop=pre)
    ks = times
    U, D, F, Ub = liq.U[0], liq.D_ms[0], liq.F_ms[0], C.MAASS["U"][tname]
    R, uu, last = 1.0, 0.0, None
    pushes = []
    for k in ks + 1:                                               # arrival one sample later
        if last is not None:
            dt = (k - last) * C.SAMPLE_MS
            R = 1 - (1 - R) * math.exp(-dt / D)
            uu = uu * math.exp(-dt / F)
        else:
            R, uu = 1.0, 0.0
        uu = uu + U * (1 - uu)
        pushes.append(liq.w[0] / Ub * uu * R)
        R = R - uu * R
        last = k
    beta = math.exp(-C.SAMPLE_MS / 30.0)
    u1 = U_trace(o)[:, 0, 1]
    want = np.zeros(T)
    for k, p in zip(ks + 1, pushes):
        m = np.arange(T) - k
        want += np.where(m >= 0, p * beta ** np.maximum(m, 0), 0.0)
    assert np.abs(u1 - want).max() < TOL


def test_isolated_dynamic_push_equals_w_U_over_Ubar():
    liq = tiny(frozenset({4, 8}), 1, w=0.1, typ=0)
    liq.syn_model = "markram"
    liq.U = np.array([0.35])
    pre = np.zeros((10, 1, 2), bool); pre[1, 0, 0] = True
    o = run(liq, np.zeros((1, 10, 1)), 10, np.zeros((1, 2)), open_loop=pre)
    u1 = U_trace(o)[:, 0, 1]
    assert abs(u1[2] - 0.1 * 0.35 / 0.5) < TOL


# ---------------------------------------------------------------- time conversions on both grids
@pytest.mark.parametrize("n_sub", [1, 4, 28])
def test_total_input_push_same_on_every_grid(n_sub):
    """One input spike: without fine timing its push arrives at once in its own sample's first sub-step;
    with fine timing as a 3 ms current whose total equals the same push."""
    for factors in (frozenset({4}), frozenset({4, 7})):
        if 7 in factors and n_sub == 1:
            continue
        liq = tiny(factors, n_sub)
        T = 60
        x = np.zeros((1, T, 1)); x[0, 3, 0] = 1
        o = run(liq, x, T, np.zeros((1, 2)), in_scale=0.2)
        I = np.array(o["trace"]["I3"])[:, 0, 0]
        assert np.nonzero(I)[0][0] == 3 * n_sub                       # first sub-step of sample 3
        a = math.exp(-(C.SAMPLE_MS / n_sub) / 3.0)
        tail = a ** (len(I) - 3 * n_sub) if 7 in factors else 0.0
        assert abs(I.sum() - 0.2 * (1 - tail)) < TOL


def test_delay_conversions():
    for n, (dEE, dO) in ((4, (2, 1)), (28, (15, 8))):
        h = C.SAMPLE_MS / n
        assert L.steps_of(1.5, h) == dEE and L.steps_of(0.8, h) == dO
    liq = L.build_candidate(frozenset({7}), 1, 4, 10)
    assert list(liq.delay) == [2, 1, 1, 1]
    liq = L.build_candidate(frozenset(), 1, 4, 10)
    assert list(liq.delay) == [4, 4, 4, 4]            # X: one sample on any grid


def test_constants():
    assert abs(C.MAASS_EE_PUSH - 0.1) < 1e-12
    assert abs(C.MAASS_IN_PUSH["E"] - 0.12) < 1e-12 and abs(C.MAASS_IN_PUSH["I"] - 0.06) < 1e-12
    r = C.MAASS_RATIOS
    assert abs(r["EI"] - 0.2) < 1e-12 and abs(r["IE"] - 0.6333) < 1e-3 and abs(r["II"] - 0.8107) < 1e-3
    assert abs(C.TAP - 0.9) < 1e-12 and abs(C.I_TARGET - 0.045) < 1e-12
    assert abs(math.exp(-C.SAMPLE_MS / 30.0) - 0.912) < 1e-3       # plan: 30 ms is 0.912 per step


# ---------------------------------------------------------------- Brian2 schedule (script version)
def test_script_neuron_exact_solution_with_decaying_input():
    """(d)'s neuron: exact update with the input current decaying over 3 ms (E) or 6 ms (I):
    v(t) = J tau_s / (tau_s - tau_m) (e^(-t/tau_s) - e^(-t/tau_m)) after a jump J (no tap, no recurrence)."""
    liq = L.build_maass_script(1, n_sub=1, step_ms=0.1)
    liq.J_b = 0.0
    sel = [0, liq.N - 1]                                            # one E, one I neuron
    liq.N = 2; liq.is_exc = liq.is_exc[sel]; liq.beta = liq.beta[sel]; liq.kappa = liq.kappa[sel]
    liq.in_a = liq.in_a[sel]; liq.gap = liq.gap[sel]; liq.Win = np.array([[0.5, 0.5]])
    liq.pre = liq.post = liq.typ = np.zeros(0, np.int64); liq.w = liq.U = liq.D_ms = liq.F_ms = np.zeros(0)
    T = 300
    x = np.zeros((1, T, 1)); x[0, 0, 0] = 1
    o = run(liq, x, T, np.zeros((1, 2)))
    u = U_trace(o)[:, 0, :]
    t = np.arange(T) * 0.1                                         # the jump lands after step 0's update
    for i, ts in ((0, 3.0), (1, 6.0)):
        want = 0.5 * ts / (ts - 30.0) * (np.exp(-t / ts) - np.exp(-t / 30.0))
        assert np.abs(u[:, i] - want).max() < TOL


def test_script_three_state_synapse():
    """(d)'s synapse: dx/dt = (1 - x - y)/D, dy/dt = -y/tau; on a spike u += U (1-u), y += u x, x -= u x."""
    liq = L.build_maass_script(1, n_sub=1, step_ms=0.1)
    m = np.nonzero((liq.typ == 0) & (liq.pre != liq.post))[0][0]
    liq.N = 2; liq.is_exc = np.array([True, True]); liq.beta = liq.beta[:2]; liq.kappa = liq.kappa[:2]
    liq.in_a = liq.in_a[:2]; liq.gap = np.array([1, 1]); liq.Win = np.array([[50.0, 0.0]])
    liq.pre, liq.post, liq.typ = np.array([0]), np.array([1]), np.array([0])
    for a in ("w", "U", "D_ms", "F_ms"):
        setattr(liq, a, getattr(liq, a)[[m]])
    T = 400
    x = np.zeros((1, T, 1)); spikes_in = [5, 60, 90, 250]
    x[0, spikes_in, 0] = 1
    o = run(liq, x, T, np.full((1, 2), 0.0), rec={"raster": True})
    ks = np.nonzero(o["raster"][:, 0, 0].numpy())[0]
    U, D, F, tau = liq.U[0], liq.D_ms[0], liq.F_ms[0], 3.0
    xs, ys, us = 1.0, 0.0, 0.0
    h = 0.1
    want_y = []
    arrivals = set(int(k) + liq.delay[0] for k in ks)
    for k in range(T):
        us *= math.exp(-h / F)
        xs = xs * math.exp(-h / D) + 1 - math.exp(-h / D) + ys * tau * (math.exp(-h / tau) - math.exp(-h / D)) / (D - tau)
        ys = ys * math.exp(-h / tau)
        if k in arrivals:
            us += U * (1 - us); ys += us * xs; xs -= us * xs
        want_y.append(ys)
    got_y = np.array([s["y"][0, 0] for s in o["trace"]["syn"]])
    assert np.abs(got_y - np.array(want_y)).max() < TOL


@pytest.mark.parametrize("n_sub", [4, 28])
def test_dynamic_synapse_with_decaying_current(n_sub):
    """Factors 7 + 8: each push of the dynamic synapse arrives after its delay as a 3 ms current."""
    liq = tiny(frozenset({4, 7, 8}), n_sub, w=0.02, typ=0)
    liq.syn_model = "markram"
    h = C.SAMPLE_MS / n_sub
    liq.delay = np.array([L.steps_of(C.MAASS["delay_ms"][t], h) for t in C.TYPES])
    liq.cur_a = np.exp(-h / np.array([3.0, 6.0]))
    T = 60
    times = np.array([7, 30, 41, 90, 95, 150]) * (n_sub // 4)
    pre = np.zeros((T * n_sub, 1, 2), bool); pre[times, 0, 0] = True
    o = run(liq, np.zeros((1, T, 1)), T, np.zeros((1, 2)), open_loop=pre)
    U, D, F, Ub = liq.U[0], liq.D_ms[0], liq.F_ms[0], 0.5
    d = liq.delay[0]
    a, beta = math.exp(-h / 3.0), math.exp(-h / 30.0)
    R, uu, last, want = 1.0, 0.0, None, np.zeros(T * n_sub)
    for k in times + d:
        if last is not None:
            dt = (k - last) * h
            R = 1 - (1 - R) * math.exp(-dt / D); uu *= math.exp(-dt / F)
        uu += U * (1 - uu)
        p = liq.w[0] / Ub * uu * R
        R -= uu * R
        last = k
        m = np.arange(T * n_sub) - k
        want += np.where(m >= 0, (1 - a) * p * (beta ** (m + 1) - a ** (m + 1)) / (beta - a), 0.0)
    assert np.abs(U_trace(o)[:, 0, 1] - want).max() < TOL
'''
LSM2_FILES['lsm2_tests/test_logic.py'] = r'''"""The decision logic on synthetic inputs: the gain rule (grid, extensions, fine pass, tie rule, band edge),
the ladder (forward, backward loop, confirmation and its retry), level 2 catching a wrong liquid, the effective
rank, the 1d and 1e pass rules, and the memory readout's statistics."""
import math
import numpy as np
import pytest
import torch

from lsm2 import config as C, gain, ladder, level2, liquid as L, measure as M, level1


def fake_measure(gap_of, rate_of, seeds=(1, 2, 3), noise=0.0):
    def m(g, s):
        return {"g": g, "abs_g": 0.02 * g, "rate_hz": rate_of(g), "frac_above_100hz": 0.0, "in_band": True,
                "sep": gap_of(g) + 10, "gen": 10.0, "gap": gap_of(g) + noise * (s - 2)}
    return m


def test_interior_peak_and_fine_pass():
    rec = gain.sweep(fake_measure(lambda g: -abs(math.log10(g + 1e-12) - math.log10(20)), lambda g: 5 + g * 0.01),
                     (1, 2, 3), log=lambda *a: None)
    assert rec["extensions"] == 0 and not rec["peak_at_edge"]
    fine = [g for g, t in rec["table"].items() if t["stage"] == "fine"]
    assert len(fine) == 6
    assert abs(rec["chosen_g"] - 20) / 20 < 0.3 and rec["gap_peak_inside_grid"]


def test_upper_edge_extends_at_most_three_times_then_fails():
    rec = gain.sweep(fake_measure(lambda g: math.log(g + 1e-9), lambda g: 10.0), (1, 2, 3), log=lambda *a: None)
    assert rec["extensions"] == 3 and rec["peak_at_edge"] and not rec["gap_peak_inside_grid"]
    assert max(rec["table"]) == pytest.approx(27000)


def test_band_limited_counts_as_inside_and_is_flagged():
    rec = gain.sweep(fake_measure(lambda g: math.log(g + 1e-9), lambda g: 1.0 * g), (1, 2, 3), log=lambda *a: None)
    # rate = g Hz: in band for 2 <= g <= 40; the gap keeps rising, so the band chooses g
    assert rec["extensions"] == 0 and rec["gap_peak_inside_grid"] and rec["band_limited"]
    assert rec["chosen_g"] <= 40 and rec["next_higher_gap_lower"] is False


def test_no_setting_in_band():
    rec = gain.sweep(fake_measure(lambda g: 1.0, lambda g: 0.5), (1, 2, 3), log=lambda *a: None)
    assert rec["chosen_g"] is None and rec["no_g_in_band"] and not rec["gap_peak_inside_grid"]


def test_tie_rule_takes_smallest_tied_g():
    gaps = {0.1: 1, 0.3: 2, 1: 5.0, 3: 5.05, 10: 4.98, 30: 2, 100: 1, 300: 0, 1000: -1}
    table = {g: {"gap": v, "gap_seeds": [v - 0.1, v, v + 0.1], "in_band": True} for g, v in gaps.items()}
    table[0.0] = {"gap": 9, "gap_seeds": [9, 9, 9], "in_band": True}        # g = 0 is never chosen
    ch, best = gain.choose(table)
    assert best == 3 and ch == 1


def test_g0_excluded_and_lower_edge_extends_down():
    rec = gain.sweep(fake_measure(lambda g: -g, lambda g: 10.0), (1, 2, 3), log=lambda *a: None)
    assert min(x for x in rec["table"] if x > 0) == pytest.approx(0.1 / 27)


# ---------------------------------------------------------------- ladder logic
class FakeLadder(ladder.Ladder):
    def __init__(self, passes, confirm_passes):
        super().__init__(None, type("D", (), {"name": "search"})(), type("D", (), {"name": "confirm"})(), 4, 1.0,
                         log=lambda *a: None)
        self.passes, self.confirm_passes, self.calls = passes, confirm_passes, []

    def evaluate(self, factors, data=None, seeds=None, frozen=None, y_grid=None, tag=None):
        f = frozenset(factors)
        self.calls.append((ladder.key_of(f), frozen is not None))
        ok = self.confirm_passes(f, seeds) if frozen is not None else self.passes(f)
        return {"pass": ok, "g_rule": 1.0, "in_scale": 0.1, "n_sub": self.n_sub_for(f, y_grid)}


def test_ladder_forward_backward_minimal_set():
    lad = FakeLadder(lambda f: {3, 5} <= f, lambda f, s: True)
    out = lad.run()
    assert out["final"] == [3, 5]
    fw = [k for d, k, _ in lad.path if d == "forward"]
    assert fw == ["1", "1+2", "1+2+3", "1+2+3+4", "1+2+3+4+5"]           # stops at the first pass


def test_ladder_x_passes():
    out = FakeLadder(lambda f: True, lambda f, s: True).run()
    assert out["final"] == []


def test_ladder_none_passes():
    out = FakeLadder(lambda f: False, lambda f, s: True).run()
    assert out["final"] is None and "no forward candidate" in out["stop"]


def test_ladder_confirmation_retry_once():
    # {3, 5} passes the search but fails confirmation on the first seeds: forward resumes at {1..6}, the backward
    # loop runs again (and removes 6 again, since the search passes without it), then fresh seeds confirm it
    lad = FakeLadder(lambda f: {3, 5} <= f, lambda f, s: tuple(s) == C.SEEDS_RETRY)
    out = lad.run()
    assert out["final"] == [3, 5] and out["attempts"] == 2
    assert ("1+2+3+4+5+6", False) in lad.calls


def test_ladder_retry_keeps_a_needed_factor():
    # the search needs 6 only once 5 is gone; confirmation fails first, passes on the retry seeds
    lad = FakeLadder(lambda f: {3} <= f and (5 in f or 6 in f), lambda f, s: tuple(s) == C.SEEDS_RETRY)
    out = lad.run()
    assert out["final"] in ([3, 5], [3, 6]) and out["attempts"] == 2


def test_ladder_confirmation_fails_twice_stops():
    out = FakeLadder(lambda f: {3, 5} <= f, lambda f, s: False).run()
    assert out["final"] is None and "twice" in out["stop"]


def test_y_always_on_28_and_factor7_on_n():
    lad = FakeLadder(lambda f: False, lambda f, s: False)
    assert lad.n_sub_for(C.ALL_FACTORS) == 28 and lad.n_sub_for({7}) == 4 and lad.n_sub_for({1, 2}) == 1


# ---------------------------------------------------------------- level 2 must reject a wrong liquid
def test_level2_catches_wrong_lambda_and_sign():
    liq = L.build_candidate(frozenset({1}), 1, 1, 50)
    assert level2.check(liq)["pass"]
    wrong = L.build_candidate(frozenset(), 1, 1, 50)          # built with lambda 0.8, claims factor 1
    wrong.factors = frozenset({1})
    assert not level2.check(wrong)["pass"]
    bad = L.build_candidate(frozenset({1}), 1, 1, 50)
    bad.w = np.abs(bad.w)                                       # inhibitory synapses made excitatory
    assert not level2.check(bad)["pass"]


# ---------------------------------------------------------------- measures
def test_effective_rank_basics():
    rng = np.random.default_rng(0)
    assert M.effective_rank(np.zeros((10, 5))) == 0.0
    iso = rng.standard_normal((5000, 4))
    assert abs(M.effective_rank(iso) - 4) < 0.1
    one = np.outer(rng.standard_normal(100), rng.standard_normal(6))
    assert abs(M.effective_rank(one) - 1) < 1e-6
    shifted = iso + 100.0                                       # centring: the mean state does not count
    assert abs(M.effective_rank(shifted) - M.effective_rank(iso)) < 1e-9


def test_jitter_keeps_spikes_and_moves_them():
    rng = np.random.default_rng(0)
    s = np.zeros((577, 3), np.uint8); s[100, 0] = 1; s[300, 2] = 1; s[576, 1] = 1
    j = M.jitter_spikes(s, rng)
    assert j.sum() <= 3 and j.sum() >= 2


def test_grid_effect_rule():
    ref = {1: {"rate_hz": 5, "sep": 10, "gen": 3}, 2: {"rate_hz": 6, "sep": 12, "gen": 4}, 3: {"rate_hz": 7, "sep": 11, "gen": 5}}
    ok, _ = level1.grid_effect_ok({s: {k: v + 1.9 for k, v in r.items()} for s, r in ref.items()}, ref)
    assert ok
    bad, _ = level1.grid_effect_ok({s: {k: v + 2.1 for k, v in r.items()} for s, r in ref.items()}, ref)
    assert not bad


def test_fig2_compare_rule():
    rng = np.random.default_rng(0)
    base = np.linspace(0, 1, 500)
    A = [{0.1: base + 0.01 * rng.standard_normal(500)} for _ in range(5)]
    B = [{0.1: base + 0.01 * rng.standard_normal(500)} for _ in range(5)]
    ok, res = level1.compare_fig2(A, B)
    assert ok and res[0.1]["frac_within"] > 0.95
    Bs = [{0.1: base + 0.2 + 0.01 * rng.standard_normal(500)} for _ in range(5)]
    assert not level1.compare_fig2(A, Bs)[0]


def test_memory_readout_finds_a_delay_line():
    """A liquid replaced by a perfect delay line must give R^2 ~ 1 at its lag and ~ 0 elsewhere."""
    rng = np.random.default_rng(0)
    n, T, lagS = 40, 300, 36
    y = rng.poisson(2.0, (n, T)).astype(float)
    st = [M._Stats(3, "cpu") for _ in range(2)]
    for i in range(n):
        X = np.stack([np.roll(y[i], lagS), np.roll(y[i], 5), rng.standard_normal(T)], 1)[lagS:]
        st[i % 2].add(torch.tensor(X), torch.tensor(y[i][:T - lagS]))
    w, mu, my = M._fit(st[0], 1e-6)
    r2 = 1 - M._ss_res(st[1], w, mu, my) / M._ss_tot(st[1])
    assert r2 > 0.99


AUTHORS = __import__("os").environ.get("AUTHORS_REPO_DIR", "/home/claude/ext/authors_repo")


@pytest.mark.skipif(not __import__("os").path.isdir(AUTHORS), reason="authors' code not available")
def test_x_mode_reproduces_authors_snntorch_loop_with_their_reset():
    """X built by (a)'s class: with x_update = authors_code the new simulator gives the authors' window counts
    exactly; with (a)'s Eqs. 17-19 (default) it differs only through the extra subtraction."""
    import sys, importlib.util
    sys.path.insert(0, AUTHORS)
    from src.utils.reproducibility import seed_everything
    spec = importlib.util.spec_from_file_location("a03t", AUTHORS + "/scripts/03_run_24_configurations_cv.py")
    A03 = importlib.util.module_from_spec(spec); spec.loader.exec_module(A03)
    G = torch.distributions.Gamma
    seed_everything(2026, deterministic=False)
    torch.distributions.Gamma = lambda c, s: G(c, 1.0 / s)
    try:
        m = A03.LiquidStateMachine3D(200, 0.1, 0.045 / (1.224 * 0.1), 0.95, 0.8, (7, 7, 7), 0.8, 1.0, 650,
                                     {"exc_exc": 0.3, "exc_inh": 0.2, "inh_exc": 0.4, "inh_inh": 0.1}, 0.8, 1,
                                     np.array([0.02, 0.025, 0.05, 0.04]), 1, 0.01, "cpu")
    finally:
        torch.distributions.Gamma = G
    rng = np.random.default_rng(0)
    X = np.clip(0.45 + 0.4 * np.sin(np.linspace(0, 6, 577))[None] * rng.uniform(0.5, 1, (20, 1))
                + 0.004 * rng.standard_normal((20, 577)).cumsum(1), 0, 1)
    thr = np.linspace(0, 1, 200); ab = X[:, :, None] >= thr
    x = np.zeros(ab.shape, np.float32); x[:, 1:] = ab[:, 1:] != ab[:, :-1]
    xt = torch.from_numpy(x)
    with torch.no_grad():
        spk = torch.zeros(20, 343); mem = torch.zeros(20, 343); ref = torch.zeros(20, 32, 343)
        for t in range(650):
            xi = xt[:, t] if t < 577 else torch.zeros(20, 200)
            spk, mem = m.lif(xi @ m.input_rsv_weights + spk @ m.rsv_weights, mem)
            if t < 640:
                ref[:, t // 20] += spk
    liq = L.from_authors_model(m, snn_quirk=True)
    o = __import__("lsm2.sim", fromlist=["Simulator"]).Simulator(liq, dtype=torch.float32).run(
        x, np.zeros((20, 343)), 650, rec={"windows": (20, 32)})
    assert int((o["windows"].numpy() != ref.numpy()).sum()) == 0
'''
LSM2_FILES['lsm2_tests/test_script_mode.py'] = r'''"""Our code in the script's settings against (d)'s own Brian2 script on the same realised liquid, the same
stimulus and the same initial voltages: the spike trains must be identical (a deterministic version of 1e)."""
import os
import numpy as np
import pytest
import torch

from lsm2 import config as C, liquid as L, sim as S, brian2_ref

SCRIPT = os.environ.get("MAASS_SCRIPT", "/home/claude/ext/brian2ex/maass2002.py")


@pytest.mark.skipif(not os.path.exists(SCRIPT), reason="(d)'s script not available")
def test_script_mode_reproduces_d_exactly():
    mod = brian2_ref.script_module(SCRIPT)
    mod.prefs.codegen.target = "numpy"
    np.random.seed(5)
    import brian2; brian2.start_scope()
    neurons = mod.get_neurons()
    N_exc = int(0.8 * len(neurons))
    exc, inh = neurons[:N_exc], neurons[N_exc:]
    exc.tau_refrac = 3 * mod.ms; exc.tau_stimulus = 3 * mod.ms
    inh.tau_refrac = 2 * mod.ms; inh.tau_stimulus = 6 * mod.ms
    s, nA, ms = mod.second, mod.nA, mod.ms
    specs = [("ee_synapses", exc, exc, 0.3, 3, 30, 0.5, 1.1, 0.05, 1.5, 0), ("ei_synapses", exc, inh, 0.2, 3, 60, 0.05, 0.125, 1.2, 0.8, 1),
             ("ie_synapses", inh, exc, 0.4, 6, -19, 0.25, 0.7, 0.02, 0.8, 2), ("ii_synapses", inh, inh, 0.1, 6, -19, 0.32, 0.144, 0.06, 0.8, 3)]
    syns = [mod.get_synapses(nm, a, b, C=c, l=2, tau_I=ti * ms, A=A * nA, U=U, D=D * s, F=F * s, delay=dl * ms)
            for nm, a, b, c, ti, A, U, D, F, dl, _ in specs]
    stim = mod.SpikeGeneratorGroup(1, [], [] * ms, name="stimulus")
    se = mod.Synapses(stim, exc, "A : ampere (shared, constant)", on_pre="I_stimulus += A"); se.connect(p=1); se.A = 18 * nA
    si = mod.Synapses(stim, inh, "A : ampere (shared, constant)", on_pre="I_stimulus += A"); si.connect(p=1); si.A = 9 * nA
    me = mod.SpikeMonitor(exc, name="spike_monitor_exc"); mi = mod.SpikeMonitor(inh, name="spike_monitor_inh")
    mod.defaultclock.dt = mod.DT
    net = mod.Network([neurons, *syns, se, si, stim, me, mi])
    v0 = np.random.uniform(13.5, 15, len(neurons))
    neurons.v = v0 * mod.mV
    spikes = np.sort(np.random.uniform(0, 200, 6))
    spikes = np.round(spikes, 1)
    stim.set_spikes([0] * len(spikes), spikes * ms)
    T = 2000                                   # 200 ms at 0.1 ms
    net.run(T * 0.1 * ms)
    b_r = np.zeros((T, len(neurons)), bool)
    for mon, off in ((me, 0), (mi, N_exc)):
        k = np.round(np.asarray(mon.t / ms) / 0.1).astype(int)
        b_r[k, np.asarray(mon.i) + off] = True
    # our liquid with the same realised parameters
    liq = L.build_maass_script(1, n_sub=1, step_ms=0.1)
    pos = np.stack([np.asarray(neurons.x_pos), np.asarray(neurons.y_pos), np.asarray(neurons.z_pos)], 1)
    pre, post, typ, w, U, D, F = [], [], [], [], [], [], []
    for (nm, a, b, *_, ti_), syn in zip(specs, syns):
        oi = 0 if a is exc else N_exc
        oj = 0 if b is exc else N_exc
        pre.append(np.asarray(syn.i) + oi); post.append(np.asarray(syn.j) + oj); typ.append(np.full(len(syn), ti_))
        w.append(np.asarray(syn.A / nA) / 15.0); U.append(np.asarray(syn.U))
        D.append(np.asarray(syn.D / ms)); F.append(np.asarray(syn.F / ms))
    liq.pos = pos
    liq.pre, liq.post, liq.typ = (np.concatenate(v) for v in (pre, post, typ))
    liq.w, liq.U, liq.D_ms, liq.F_ms = (np.concatenate(v) for v in (w, U, D, F))
    x = np.zeros((1, T, 1), np.uint8)
    x[0, np.round(spikes / 0.1).astype(int), 0] = 1
    sm = S.Simulator(liq, dtype=torch.float64, g=1.0, in_scale=1.0)
    o = sm.run(x, (v0 / 15.0)[None], T, rec={"raster": True})
    r = o["raster"][:, 0, :].numpy()
    assert b_r.sum() > 50
    assert (r != b_r).sum() == 0, f"{(r != b_r).sum()} spike mismatches of {b_r.sum()}"
'''
LSM2_FILES['lsm2_tests/test_stats.py'] = r'''"""Statistics and decision rules on synthetic run tables."""
import math
import numpy as np
import pandas as pd
import pytest
from scipy import stats as sstats

from lsm2 import stats as S

SPLITS = [f"cv-r{r}f{f}" for r in (1, 2, 3) for f in (1, 2, 3, 4)]


def runs(base, shift=0.0, seeds=(2026, 69, 420), noise=0.01, seed_free=False, rng=None, ds2=None):
    rng = rng or np.random.default_rng(0)
    rows = []
    for sd in ([None] if seed_free else seeds):
        for i, sp in enumerate(SPLITS + ["final"]):
            v = base[i] + shift + noise * rng.standard_normal()
            if sp == "final" and ds2 is not None:
                v = ds2
            rows.append({"spec_split": sp, "spec_seed": sd, "macro_f1": v, "macro_f1_no232": v,
                         "recall_SVEB": v, "precision_SVEB": v, "recall_VEB": v, "precision_VEB": v})
    return pd.DataFrame(rows)


def test_nb_test_matches_formula():
    rng = np.random.default_rng(1)
    base = 0.6 + 0.05 * rng.standard_normal(13)
    a, b = runs(base, 0.03, rng=rng), runs(base, 0.0, rng=rng)
    r = S.compare(a, b)
    A = a.groupby(["spec_split", "spec_seed"]).macro_f1.mean().groupby(level=0).mean()
    B = b.groupby(["spec_split", "spec_seed"]).macro_f1.mean().groupby(level=0).mean()
    d = (A[SPLITS] - B[SPLITS]).values
    se = math.sqrt(np.var(d, ddof=1) * (1 / 12 + 0.336))
    assert r["n_splits"] == 12 and r["cv_diff"] == pytest.approx(d.mean())
    assert r["t"] == pytest.approx(d.mean() / se)
    assert r["p"] == pytest.approx(2 * sstats.t.sf(abs(d.mean() / se), 11))
    assert r["ci_high"] - r["cv_diff"] == pytest.approx(sstats.t.ppf(0.975, 11) * se)
    assert r["ds2_diff"] == pytest.approx(A["final"] - B["final"])


def test_verdicts():
    big = {"cv_diff": 0.05, "ci_low": 0.03, "ci_high": 0.07, "ds2_diff": 0.04, "p_holm": 0.001, "same_sign_all_seeds": True}
    assert S.verdict(big) == "meaningful"
    assert S.verdict({**big, "ds2_diff": -0.01}) == "inconclusive"            # DS2 must have the same sign
    assert S.verdict({**big, "same_sign_all_seeds": False}, need_seed_sign=True) == "inconclusive"
    assert S.verdict({**big, "cv_diff": 0.008, "ci_low": 0.007, "ci_high": 0.009, "ds2_diff": 0.005}) == "comparable"
    small = {"cv_diff": 0.001, "ci_low": -0.015, "ci_high": 0.017, "ds2_diff": 0.01, "p_holm": 0.8}
    assert S.verdict(small) == "comparable"
    assert S.verdict({**small, "ds2_diff": 0.03}) == "inconclusive"
    assert S.verdict(None) == "missing"


def test_holm():
    rows = S.holm([{"p": 0.01}, {"p": 0.04}, {"p": 0.03}, None])
    assert rows[0]["p_holm"] == pytest.approx(0.03)
    assert rows[2]["p_holm"] == pytest.approx(0.06)
    assert rows[1]["p_holm"] == pytest.approx(0.06)          # monotone


def test_seed_sign_and_seed_free_control():
    rng = np.random.default_rng(2)
    base = 0.6 + 0.05 * rng.standard_normal(13)
    a = runs(base, 0.02, rng=rng, noise=0.001)
    c = runs(base, 0.0, rng=rng, noise=0.001, seed_free=True)
    r = S.compare(a, c, require_seed_sign=True)
    assert r is not None and r["cv_diff"] == pytest.approx(0.02, abs=0.002)


def test_compare_diff_f7():
    rng = np.random.default_rng(3)
    base = 0.6 + 0.05 * rng.standard_normal(13)
    on_full, off_full = runs(base, 0.04, rng=rng, noise=0.001), runs(base, 0.0, rng=rng, noise=0.001)
    on_abl, off_abl = runs(base, 0.01, rng=rng, noise=0.001), runs(base, 0.0, rng=rng, noise=0.001)
    r = S.compare_diff(on_full, off_full, on_abl, off_abl)
    assert r["cv_diff"] == pytest.approx(0.03, abs=0.002) and r["p"] < 1e-6


def test_family_applies_holm():
    rng = np.random.default_rng(4)
    base = 0.6 + 0.05 * rng.standard_normal(13)
    items = [(f"c{i}", S.compare(runs(base, 0.03 * i, rng=rng), runs(base, 0.0, rng=rng))) for i in range(3)]
    fam = S.family("F", items)
    assert [f["comparison"] for f in fam] == ["c0", "c1", "c2"] and all("p_holm" in f for f in fam)
'''

for _rel, _src in LSM2_FILES.items():
    _p = os.path.join(SCRATCH_DIR, _rel)
    os.makedirs(os.path.dirname(_p), exist_ok=True)
    if not os.path.exists(_p) or open(_p).read() != _src:
        with open(_p, "w") as _f:
            _f.write(_src)
open(os.path.join(SCRATCH_DIR, "lsm2", "__init__.py"), "a").close()
if SCRATCH_DIR not in sys.path:
    sys.path.insert(0, SCRATCH_DIR)
for _m in [m for m in sys.modules if m == "lsm2" or m.startswith("lsm2.")]:
    del sys.modules[_m]
import lsm2
print(f"lsm2: {len(LSM2_FILES)} files at {os.path.join(SCRATCH_DIR, 'lsm2')}")


## 4. Utilities: cache, resume, guards, metrics, the authors' readout

In [ ]:
# Utilities: output folders, logging, cache.txt, resume, time/disk guards, metrics, the authors' readout.
CODE_VERSION = "v1"   # part of every run's hash; bump only if experiment code changes meaning
for sub in ["runs", "reservoirs", "states", "tables", "figures", "logs", "checks"]:
    os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)

def elapsed_h():
    return (time.time() - SESSION_START) / 3600

LOCK = threading.RLock()     # cache.txt, the log, speeds and disk counters are shared with the fit workers

def log(*parts):
    line = f"[{elapsed_h():6.2f} h] " + " ".join(str(p) for p in parts)
    with LOCK:
        print(line, flush=True)               # the tee writes it to LOG_PATH

def _json_default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, torch.Tensor): return o.detach().cpu().tolist()
    return str(o)

def atomic_write_text(path, text):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        f.write(text)
    os.replace(tmp, path)

def jdump(obj, path):
    atomic_write_text(path, json.dumps(obj, indent=2, default=_json_default))

def stable_hash(obj, n=8):
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=_json_default).encode()).hexdigest()[:n]

# ---------------- cache.txt ----------------
CACHE_PATH = os.path.join(WORK_DIR, "cache.txt")
MACHINE_ID = f"{MACHINE}|{ENV.get('gpu', DEVICE.type)}|{N_CPU} cores"      # identifies the machine in merged outputs

def out_tag(share, machine_id):
    """Folder-safe name of an output: its share and machine."""
    return re.sub(r"[^A-Za-z0-9.]+", "_", f"{share if share not in (None, 'None') else 'all'}__{machine_id or 'unknown'}").strip("_")

class RunCache:
    """cache.txt = the record of what has run. Rewritten (atomically) after every run.
    Sections: [DONE] run_id, finished_utc, seconds, macro_f1, accuracy, group
              [FAILED] run_id, when_utc, error
              [PENDING] run_id, group"""
    def __init__(self):
        self.done, self.failed, self.pending = {}, {}, []

    def merge_file(self, path):
        if not os.path.exists(path):
            return
        section = None
        for line in open(path):
            line = line.rstrip("\n")
            if line.startswith("[") and line.endswith("]"):
                section = line[1:-1]
                continue
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if section == "DONE":
                self.done[parts[0]] = parts[1:]
                self.failed.pop(parts[0], None)
            elif section == "FAILED" and parts[0] not in self.done:
                self.failed[parts[0]] = parts[1:]

    def is_done(self, rid):
        return rid in self.done

    def mark_done(self, rid, seconds, metrics, group):
        with LOCK:
            self._mark_done(rid, seconds, metrics, group)

    def _mark_done(self, rid, seconds, metrics, group):
        mf1 = f"{metrics['macro_f1']:.4f}" if metrics and "macro_f1" in metrics else "-"
        acc = f"{metrics['accuracy']:.4f}" if metrics and "accuracy" in metrics else "-"
        self.done[rid] = [datetime.datetime.utcnow().isoformat(timespec="seconds"), f"{seconds:.1f}", mf1, acc, group]
        self.failed.pop(rid, None)
        self.write()

    def mark_failed(self, rid, err):
        with LOCK:
            self._mark_failed(rid, err)

    def _mark_failed(self, rid, err):
        self.failed[rid] = [datetime.datetime.utcnow().isoformat(timespec="seconds"), err.replace("\t", " ").replace("\n", " | ")[:500]]
        self.write()

    def write(self):
        with LOCK:
            self._write()

    def _write(self):
        lines = ["# LSM-ECG replication: run cache. Rewritten after every run; read by the next session to skip finished work.",
                 f"# updated_utc: {datetime.datetime.utcnow().isoformat(timespec='seconds')}   session_elapsed_h: {elapsed_h():.2f}",
                 f"# totals: DONE {len(self.done)} | FAILED {len(self.failed)} | PENDING {len(self.pending)}",
                 f"# smoke_test: {SMOKE_TEST}",
                 f"# share: {SHARE}",
                 f"# machine_id: {MACHINE_ID}",
                 f"# code_version: {CODE_VERSION}",
                 f"# notebook_build: {NOTEBOOK_BUILD}",
                 "# DONE columns: run_id, finished_utc, seconds, macro_f1, accuracy, group", "[DONE]"]
        lines += ["\t".join([k] + v) for k, v in self.done.items()]
        lines += ["", "# FAILED columns: run_id, when_utc, error", "[FAILED]"]
        lines += ["\t".join([k] + v) for k, v in self.failed.items()]
        lines += ["", "# PENDING columns: run_id, group", "[PENDING]"]
        lines += [f"{rid}\t{g}" for rid, g in self.pending]
        atomic_write_text(CACHE_PATH, "\n".join(lines) + "\n")

CACHE = RunCache()

# ---------------- resume (previous session) and merge mode (two machines' outputs) ----------------
RESUME_SUBDIRS = ["runs", "reservoirs", "states", "logs", "checks"]   # outputs worth carrying; tables/figures are rebuilt

def _cache_is_smoke(p):
    return any(l.strip() == "# smoke_test: True" for l in itertools.islice(open(p), 12))

def cache_header(p):
    h = {}
    for l in itertools.islice(open(p), 12):
        m = re.match(r"^# (smoke_test|share|machine_id|code_version|notebook_build): (.*)$", l.rstrip("\n"))
        if m:
            h[m.group(1)] = m.group(2).strip()
    return h

def copy_forward(src_root):
    n = 0
    for sub in RESUME_SUBDIRS:
        for root, _, files in os.walk(os.path.join(src_root, sub)):
            rel = os.path.relpath(root, src_root)
            for fn in files:
                dst = os.path.normpath(os.path.join(WORK_DIR, rel, fn))
                if not os.path.exists(dst):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy2(os.path.join(root, fn), dst)
                    n += 1
    return n

if MERGE_MODE:
    for md in MERGE_DIRS:
        if os.path.abspath(md) == os.path.abspath(WORK_DIR):
            raise ValueError("merge mode: WORK_DIR must be a new folder, not one of MERGE_DIRS")
        if not os.path.exists(os.path.join(md, "cache.txt")):
            raise FileNotFoundError(f"merge mode: no cache.txt in {md} (give the output folders themselves)")
    _hdrs = [cache_header(os.path.join(md, "cache.txt")) for md in MERGE_DIRS]
    if len({h.get("notebook_build") for h in _hdrs}) > 1:
        print("WARNING: the folders were made by different notebook builds: "
              + ", ".join(f"{md}: {h.get('notebook_build', '?')}" for md, h in zip(MERGE_DIRS, _hdrs)))

if RESUME_DIRS == "auto":
    RESUME_DIRS = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(AUTO_RESUME_ROOT, "**", "cache.txt"), recursive=True)})
for rd in (MERGE_DIRS if MERGE_MODE else RESUME_DIRS):
    cp = os.path.join(rd, "cache.txt")
    if os.path.abspath(rd) == os.path.abspath(WORK_DIR):
        continue
    if not os.path.exists(cp):
        print(f"resume: no cache.txt in {rd} -> fresh start")
        continue
    if _cache_is_smoke(cp) != SMOKE_TEST:
        msg = f"{rd}: its smoke_test flag differs from this session's SMOKE_TEST={SMOKE_TEST}"
        if MERGE_MODE:
            raise ValueError("merge mode: " + msg)
        print(f"resume: skipping {msg}")
        continue
    before = len(CACHE.done)
    _o = RunCache(); _o.merge_file(cp); n_there = len(_o.done)
    CACHE.merge_file(cp)
    n = copy_forward(rd)
    n_dirs = len(os.listdir(os.path.join(rd, "runs"))) if os.path.isdir(os.path.join(rd, "runs")) else 0
    h = cache_header(cp)
    print(f"{'merged' if MERGE_MODE else 'resumed'} from {rd} (share {h.get('share', '?')}, {h.get('machine_id', '?')}): "
          f"{n_there} finished runs there, {len(CACHE.done) - before} of them new here, {n} files copied")
    if n_dirs < len(CACHE.done) - before - 2:
        print(f"WARNING: its cache.txt lists {len(CACHE.done) - before} finished runs but only {n_dirs} run folders are there. "
              "They will be skipped, but their results will be missing from tables and figures. "
              "Bring the whole output folder, not only cache.txt.")
if os.path.exists(CACHE_PATH) and _cache_is_smoke(CACHE_PATH) != SMOKE_TEST:
    raise RuntimeError(f"{WORK_DIR} holds {'smoke-test' if _cache_is_smoke(CACHE_PATH) else 'real'} results but SMOKE_TEST = "
                       f"{SMOKE_TEST}. Use a separate WORK_DIR for the smoke test so the two never mix.")
CACHE.merge_file(CACHE_PATH)       # this machine's earlier sessions (same folder, reused in place)
print(f"cache: {len(CACHE.done)} done, {len(CACHE.failed)} failed")

# ---------------- time and disk guards ----------------
def seconds_left():
    return SESSION_BUDGET_HOURS * 3600 - (time.time() - SESSION_START) - ANALYSIS_RESERVE_MIN * 60

_DISK = {"bytes": None, "since": 0}
def work_dir_gb(force=False):
    if force or _DISK["bytes"] is None or _DISK["since"] >= 50:
        total = 0
        for root, _, files in os.walk(WORK_DIR):
            for fn in files:
                try:
                    total += os.path.getsize(os.path.join(root, fn))
                except OSError:
                    pass
        _DISK.update(bytes=total, since=0)
    return _DISK["bytes"] / 1e9

def disk_add(nbytes):
    with LOCK:
        if _DISK["bytes"] is not None:
            _DISK["bytes"] += nbytes
            _DISK["since"] += 1

# Throughput model, measured as the session runs (and carried over between sessions in checks/speeds.json).
# Units: beats/s for simulation and encoding; seconds per (training beats x features / 1e9) for a classifier fit;
# seconds per epoch per 1,000 training beats for the CNN/GRU.
_GPU = DEVICE.type == "cuda"
# GPU defaults: roughly what a Kaggle T4 measured (used only until this machine has measured its own)
SPEED = {"sim_beats_per_s": {e: (2000.0 if _GPU else 300.0) for e in ("multi", "rate", "latency")},
         "encode_beats_per_s": {e: (20000.0 if _GPU else 1500.0) for e in ("multi", "rate", "latency")},
         "fit_s_per_Gcell": {"1": 400.0, "2": 220.0, "3": 160.0, "pipe": 400.0}, "fit_threads": float(READOUT_JOBS),
         "nn_s_per_epoch_kbeat": {"cnn": 0.04 if _GPU else 3.0, "gru": 0.025 if _GPU else 8.0},
         "nn_unit_s_per_epoch_kbeat": {"cnn": 0.03 if _GPU else 3.0, "gru": 0.04 if _GPU else 8.0},
         "lsm_job_overhead_s": 3.0, "huang_s_per_run": 1200.0}
SPEED_SAMPLES = {}                # recent measurements of the robust (median) entries
# fit_s_per_Gcell: "1"/"2"/"3" = one fit using that many cores (older runner); "pipe" = one fit per core while all
# the pipeline's workers are busy (includes their competition for memory bandwidth).
# nn_unit_*: seconds per epoch per 1,000 beats for a tuned architecture, divided by its relative cost (see nn_cost).
SPEED_MEASURED = set()            # which entries were measured (vs default guesses)
SPEEDS_PATH = os.path.join(WORK_DIR, "checks", "speeds.json")
if os.path.exists(SPEEDS_PATH):
    try:
        _sp = json.load(open(SPEEDS_PATH))
        if _sp.get("device") == DEVICE.type and _sp.get("gpu") == ENV.get("gpu") and _sp.get("cpu_count") == N_CPU:
            for k, v in _sp["speed"].items():
                if isinstance(SPEED.get(k), dict) and isinstance(v, dict):
                    SPEED[k].update(v)
                elif k in SPEED and not isinstance(SPEED[k], dict) and not isinstance(v, dict):
                    SPEED[k] = v                                               # entries whose format changed are skipped
            SPEED_MEASURED.update(m for m in _sp["measured"] if not (m == "fit_s_per_Gcell"))
            SPEED_SAMPLES.update(_sp.get("samples", {}))
            print("speeds carried over from an earlier session:", sorted(SPEED_MEASURED))
    except Exception as ex:
        print("could not read speeds.json:", ex)

def update_speed(key, value, sub=None, alpha=0.3, robust=False):
    with LOCK:
        if robust:            # median of the last 40 measurements: one odd fit does not move the projection
            tag = f"{key}:{sub}" if sub else key
            xs = SPEED_SAMPLES.setdefault(tag, []); xs.append(float(value)); del xs[:-40]
            value, alpha = float(np.median(xs)), 1.0
        _update_speed(key, value, sub, alpha)

def _update_speed(key, value, sub, alpha):
    tag = f"{key}:{sub}" if sub else key
    if sub is None:
        SPEED[key] = value if tag not in SPEED_MEASURED else alpha * value + (1 - alpha) * SPEED[key]
    else:
        SPEED[key][sub] = value if tag not in SPEED_MEASURED else alpha * value + (1 - alpha) * SPEED[key][sub]
    SPEED_MEASURED.add(tag)

def save_speeds():
    with LOCK:
        jdump({"device": DEVICE.type, "gpu": ENV.get("gpu"), "cpu_count": N_CPU, "speed": SPEED, "samples": SPEED_SAMPLES, "measured": sorted(SPEED_MEASURED)}, SPEEDS_PATH)

# ---------------- metrics ----------------
CLASS_NAMES = ["N", "SVEB", "VEB"]

def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    tp = np.diag(cm).astype(float)
    support, predicted, total = cm.sum(1).astype(float), cm.sum(0).astype(float), cm.sum()
    rec = np.divide(tp, support, out=np.zeros(3), where=support > 0)
    prec = np.divide(tp, predicted, out=np.zeros(3), where=predicted > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(3), where=(prec + rec) > 0)
    fp = predicted - tp
    tn = total - support - fp
    spec = np.divide(tn, tn + fp, out=np.zeros(3), where=(tn + fp) > 0)
    m = {"accuracy": float(tp.sum() / total), "macro_f1": float(f1.mean()), "macro_recall": float(rec.mean()),
         "macro_precision": float(prec.mean()), "n": int(total), "confusion_matrix": cm.tolist()}
    for i, c in enumerate(CLASS_NAMES):
        m.update({f"recall_{c}": float(rec[i]), f"precision_{c}": float(prec[i]), f"f1_{c}": float(f1[i]),
                  f"specificity_{c}": float(spec[i]), f"fpr_{c}": float(1 - spec[i]), f"support_{c}": int(support[i])})
    return m

# ---------------- the authors' readout (exact settings) ----------------
def soft_class_weight(y, p=0.6):
    classes = np.unique(y)
    base = compute_class_weight(class_weight="balanced", classes=classes, y=y).astype(np.float64)
    return dict(zip(classes.astype(int), (base ** p).astype(float)))

CURVE_EVAL_EPOCHS = {1, 3, 5, 10, 15}      # epochs at which the validation curve is measured (CV runs only)

class Cancelled(Exception):
    """Raised inside a fit when the session's time is up; the run stays pending for the next session."""

def prepare_train(Xtr_fn, ntr, prescale):
    """StandardScaler fitted with batched partial_fit (authors' way); optionally the whole scaled training matrix."""
    bs = 4096
    scaler = StandardScaler(with_mean=True, with_std=True)
    for i in range(0, ntr, bs):
        scaler.partial_fit(Xtr_fn(np.arange(i, min(i + bs, ntr))))
    XS = None
    if prescale:
        XS = np.concatenate([scaler.transform(Xtr_fn(np.arange(i, min(i + bs, ntr)))).astype(np.float32) for i in range(0, ntr, bs)])
    return scaler, XS

def fit_readout(Xtr_fn, ntr, ytr, Xev_fn, nev, loss, rng_seed, yev_for_curve=None, curve_sub=4000, prescale=False, n_jobs=None,
                prepared=None, cancel=None, epochs=None):
    """Authors' readout: StandardScaler (batched partial_fit) + SGDClassifier(hinge|log_loss, l2, alpha=1e-4,
    optimal LR, average=True, random_state=2026, soft class weights ^0.6), 15 epochs, batch 4096,
    shuffling with np.random.default_rng(rng_seed). Batches are scaled on the fly as in the authors' code, or, when
    prescale=True and RAM allows, the whole matrix is scaled once (row-wise identical values, faster).
    prepared=(scaler, XS) reuses a scaler / scaled matrix shared by several fits on the same training rows.
    Xtr_fn(positions) / Xev_fn(positions) return float32 feature rows."""
    bs = 4096
    scaler, XS = prepared if prepared is not None else prepare_train(Xtr_fn, ntr, prescale)
    tf = lambda X: scaler.transform(X).astype(np.float32)
    get_tr = (lambda p: XS[p]) if XS is not None else (lambda p: tf(Xtr_fn(p)))
    clf = SGDClassifier(loss=loss, penalty="l2", alpha=1e-4, fit_intercept=True, learning_rate="optimal",
                        max_iter=1, tol=None, class_weight=soft_class_weight(ytr), average=True, random_state=2026,
                        n_jobs=n_jobs)   # n_jobs only parallelises the 3 one-vs-rest problems: results are bit-identical
    classes = np.unique(ytr)
    rng = np.random.default_rng(rng_seed)
    sub = np.sort(np.random.default_rng(0).choice(ntr, min(curve_sub, ntr), replace=False))
    Xsub = get_tr(sub)
    def predict_eval():
        preds, scores = [], []
        for i in range(0, nev, bs):
            Xe = tf(Xev_fn(np.arange(i, min(i + bs, nev))))
            preds.append(clf.predict(Xe)); scores.append(clf.decision_function(Xe).astype(np.float32))
        return np.concatenate(preds), np.concatenate(scores)
    curve = []
    for ep in range(epochs or READOUT_EPOCHS):          # epochs: only the core search uses fewer
        idx = rng.permutation(ntr)
        for i in range(0, ntr, bs):
            if cancel is not None and cancel.is_set():
                raise Cancelled()
            sel = idx[i:i + bs]
            Xb = get_tr(sel)
            if ep == 0 and i == 0:
                clf.partial_fit(Xb, ytr[sel], classes=classes)
            else:
                clf.partial_fit(Xb, ytr[sel])
        row = {"epoch": ep + 1, "train_sub_macro_f1": compute_metrics(ytr[sub], clf.predict(Xsub))["macro_f1"]}
        if loss == "hinge":
            dec = clf.decision_function(Xsub)
            onehot = -np.ones_like(dec); onehot[np.arange(len(sub)), ytr[sub]] = 1
            row["train_sub_hinge_loss"] = float(np.maximum(0, 1 - onehot * dec).mean())
        if yev_for_curve is not None and (ep + 1) in CURVE_EVAL_EPOCHS:
            row["eval_macro_f1"] = compute_metrics(yev_for_curve, predict_eval()[0])["macro_f1"]
        curve.append(row)
    y_pred, scores = predict_eval()
    return scaler, clf, y_pred, scores, curve

# ---------------- saving a run ----------------
def save_run(rid, spec, group, metrics, eval_idx, eval_pid, y_true, y_pred, scores, model_obj,
             curve, liquid_stats, timing, extra=None):
    d = os.path.join(WORK_DIR, "runs", rid)
    os.makedirs(d, exist_ok=True)
    jdump({"run_id": rid, "group": group, "spec": spec, "env": ENV, "code_version": CODE_VERSION,
           "session": SESSION_TAG, **(extra or {})}, os.path.join(d, "config.json"))
    jdump(metrics, os.path.join(d, "metrics.json"))
    np.savez_compressed(os.path.join(d, "predictions.npz"), eval_index=np.asarray(eval_idx, np.int32),
                        patient_id=np.asarray(eval_pid).astype("U4"), y_true=np.asarray(y_true, np.int8),
                        y_pred=np.asarray(y_pred, np.int8),
                        scores=np.asarray(scores, np.float32) if scores is not None else np.zeros((0, 3), np.float32))
    if model_obj is not None:
        if isinstance(model_obj, dict) and model_obj.get("_torch_state"):
            torch.save(model_obj, os.path.join(d, "model.pt"))
        else:
            joblib.dump(model_obj, os.path.join(d, "readout.joblib"), compress=3)
    if curve:
        pd.DataFrame(curve).to_csv(os.path.join(d, "train_curve.csv"), index=False)
    if liquid_stats:
        jdump(liquid_stats, os.path.join(d, "liquid_stats.json"))
    jdump(timing, os.path.join(d, "timing.json"))
    with open(os.path.join(d, "log.txt"), "w") as f:
        f.write(f"{rid}\nmacro_f1={metrics.get('macro_f1')}\naccuracy={metrics.get('accuracy')}\n"
                f"confusion_matrix={metrics.get('confusion_matrix')}\ntiming={timing}\n")
    disk_add(sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d)))


## 5. Data: MIT-BIH → DS1 / DS2, checks against the authors' files

In [ ]:
# Data: locate + checksum MIT-BIH, build DS1/DS2 exactly as the authors do (both filters),
# track each beat's position for RR-interval features, verify against the authors' released files.
def locate_input():
    cands = [INPUT_DIR] + sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/100.hea", recursive=True)})
    for c in cands:
        if os.path.exists(os.path.join(c, "100.hea")) and os.path.exists(os.path.join(c, "234.atr")):
            return c
    raise FileNotFoundError(f"MIT-BIH not found at {INPUT_DIR} or anywhere under /kaggle/input")

DATA_DIR = locate_input()
print("MIT-BIH at:", DATA_DIR)

CHECKS = {}
_recs = pd.read_csv(os.path.join(REPO_DIR, "splits/ds1_ds2_records.csv")).set_index("set")["records"]
DS_RECORDS = {"DS1": _recs["DS1"].split(","), "DS2": _recs["DS2"].split(",")}
DS_RECORDS_ALL = DS_RECORDS["DS1"] + DS_RECORDS["DS2"]
sums_path = os.path.join(DATA_DIR, "SHA256SUMS.txt")
if os.path.exists(sums_path):
    bad, missing, n = [], [], 0
    for line in open(sums_path):
        h, fn = line.strip().split(maxsplit=1)
        fp = os.path.join(DATA_DIR, fn)
        if os.path.isfile(fp):
            n += 1
            if hashlib.sha256(open(fp, "rb").read()).hexdigest() != h:
                bad.append(fn)
        else:
            missing.append(fn)
    used = {f"{r}.{x}" for r in DS_RECORDS_ALL for x in ("dat", "hea", "atr")}
    CHECKS["checksums"] = {"files_listed": n + len(missing), "files_checked": n, "mismatches": bad, "missing": missing,
                           "missing_files_used_by_pipeline": sorted(set(missing) & used)}
    print(f"checksums: {n} files checked, {len(bad)} mismatches, {len(missing)} missing"
          + (f" ({', '.join(missing[:12])}{' ...' if len(missing) > 12 else ''})" if missing else ""))
    if set(missing) & used:
        raise FileNotFoundError(f"record files needed by the pipeline are missing: {sorted(set(missing) & used)}")
else:
    CHECKS["checksums"] = {"files_checked": 0, "note": "SHA256SUMS.txt not present"}


NORMAL_S, SVEB_S, VEB_S = ['N', 'L', 'R', 'e', 'j'], ['A', 'a', 'J', 'S'], ['V', 'E']
BEAT_SYMBOLS = set("NLRBAaJSVrFejnE/fQ?")      # all beat annotations (for RR intervals)
RR_NAMES = ["pre", "post", "local10", "rec_median", "pre_over_local", "post_over_local", "pre_over_rec", "pre_over_post"]

def rr_features(all_beats, pos, fs):
    """RR features (seconds / ratios) for beats at sample positions `pos`, from every beat annotation."""
    all_beats = np.sort(np.asarray(all_beats))
    rr = np.diff(all_beats) / fs
    rec_med = float(np.median(rr))
    i = np.searchsorted(all_beats, pos)
    pre = np.where(i > 0, rr[np.clip(i - 1, 0, len(rr) - 1)], rec_med)
    post = np.where(i < len(all_beats) - 1, rr[np.clip(i, 0, len(rr) - 1)], rec_med)
    csum = np.concatenate([[0], np.cumsum(rr)])
    lo = np.clip(i - 10, 0, None)
    cnt = np.maximum(i - lo, 1)
    local = np.where(i > 0, (csum[np.clip(i, 0, len(rr))] - csum[lo]) / cnt, rec_med)
    f = np.stack([pre, post, local, np.full_like(pre, rec_med), pre / local, post / local, pre / rec_med, pre / post], 1)
    return f.astype(np.float32)

def rr_mean_ratio(all_beats, pos, fs):
    """pre-RR / mean RR of the whole recording (Huang et al. 2014); the same pre-RR as the "pre" column."""
    rr = np.diff(np.sort(np.asarray(all_beats))) / fs
    return (rr_features(all_beats, pos, fs)[:, 0] / rr.mean()).astype(np.float32)

def huang_windows(sig, pos, half=100):
    """Huang et al.: 200 raw (unfiltered) samples, R-100 .. R+99 at 360 Hz, minus their mean, divided by their SD."""
    W = np.stack([sig[p - half:p + half] for p in pos]).astype(np.float64)
    return ((W - W.mean(1, keepdims=True)) / W.std(1, keepdims=True)).astype(np.float32)

def build_set(records, filt):
    Xs, syms, pids, poss, rrs = [], [], [], [], []
    rrms, hws = [], []          # extensions: pre-RR / record mean RR, Huang windows
    for rec in records:
        sig, atr_sym, atr_sample, fs = A03.load_ecg(os.path.join(DATA_DIR, rec), target_fs=360)
        if filt == "median":
            sig_f, _ = A_sig.median_filters(sig, fs=fs, kernel_size_ms1=200, kernel_size_ms2=600, mode='reflect')
        else:   # the authors' (commented-out) morphological line
            _, sig_f, _, _ = A_sig.morph_filter_baseline(sig, fs=360, k1_ratio=0.20, k2_ratio=0.35, k3_ratio=0.005, mode='reflect')
        df_all = pd.DataFrame({'atr_sym': atr_sym, 'atr_sample': atr_sample})
        df = df_all[df_all.atr_sym.isin(NORMAL_S + SVEB_S + VEB_S)].reset_index(drop=True)
        Xp, _, sp, pp = A03.build_XY(sig_f, df, 0.8, 0.8, fs, NORMAL_S, SVEB_S + VEB_S, patient_id=rec)
        L = int(round(0.8 * fs))
        keep = ((df.atr_sample - L) >= 0) & ((df.atr_sample + L + 1) <= len(sig_f))
        pos = df.atr_sample[keep].to_numpy()
        assert len(pos) == len(Xp) and list(df.atr_sym[keep]) == list(sp)
        beats_all = [s for s, y in zip(atr_sample, atr_sym) if y in BEAT_SYMBOLS]
        Xs.append(Xp); syms += list(sp); pids += list(pp); poss.append(pos); rrs.append(rr_features(beats_all, pos, fs))
        rrms.append(rr_mean_ratio(beats_all, pos, fs)); hws.append(huang_windows(sig, pos))
    X = np.vstack(Xs); pos = np.concatenate(poss); rr = np.vstack(rrs); pid = np.array(pids)
    num = A03.Numerical_Labels(syms)
    X_sh, y_sh, pid_sh = A03.Shuffle_data(X, num, pid, seed=42)
    np.random.seed(42); perm = np.random.permutation(len(X))           # the same permutation Shuffle_data used
    assert np.array_equal(X[perm], X_sh)
    with contextlib.redirect_stdout(io.StringIO()):     # silence the authors' "taking all of them" prints
        Xd, yd, pidd = A_strat.Make_Data_Stratified(X_sh, y_sh, pid_sh, num_data=50000, Type_Class='SuperC')
        idx_d, _, _ = A_strat.Make_Data_Stratified(np.arange(len(X))[:, None], y_sh, pid_sh, num_data=50000, Type_Class='SuperC')
    src = perm[np.asarray(idx_d)[:, 0].astype(int)]
    assert np.array_equal(X[src], Xd), "beat-order tracking failed"
    return {"X": np.asarray(Xd, np.float64), "y": np.asarray(yd, int), "pid": np.asarray(pidd).astype(str),
            "pos": pos[src], "rr": rr[src], "sym": np.array(syms)[src],
            "rr_mean_ratio": np.concatenate(rrms)[src], "huang_beats": np.vstack(hws)[src]}

t0 = time.time()
DATA = {f: {ds: build_set(DS_RECORDS[ds], f) for ds in ("DS1", "DS2")} for f in ("morph", "median")}
print(f"datasets built in {time.time()-t0:.0f}s")
for ds in ("DS1", "DS2"):
    assert np.array_equal(DATA["morph"][ds]["y"], DATA["median"][ds]["y"])
    assert np.array_equal(DATA["morph"][ds]["pid"], DATA["median"][ds]["pid"])

# verify against the authors' released DS1 order and counts
_auth = pd.read_csv(os.path.join(REPO_DIR, "splits/patient_ids_train.csv"))
CHECKS["ds1_matches_authors_order"] = bool(np.array_equal(_auth.patient_id.astype(str).values, DATA["morph"]["DS1"]["pid"])
                                           and np.array_equal(_auth.label.values, DATA["morph"]["DS1"]["y"]))
CHECKS["class_counts"] = {ds: np.bincount(DATA["morph"][ds]["y"], minlength=3).tolist() for ds in ("DS1", "DS2")}
CHECKS["expected_counts"] = {"DS1": [45827, 943, 3787], "DS2": [44216, 1837, 3219]}
print("DS1 order identical to authors' file:", CHECKS["ds1_matches_authors_order"], "| counts:", CHECKS["class_counts"])

# folds: the authors' exact indices, and a check that they regenerate
_fd = json.load(open(os.path.join(REPO_DIR, "splits/fold_definitions.json")))
FOLDS = [{"rep": f["rep"], "fold": f["fold"], "train_idx": np.array(f["train_idx"]), "val_idx": np.array(f["val_idx"])}
         for f in _fd["folds"]]
_ok, _k = True, 0
_y1, _p1 = DATA["morph"]["DS1"]["y"], DATA["morph"]["DS1"]["pid"]
for rep, sd in enumerate([11, 23, 37], 1):
    for fold, (tr, va) in enumerate(StratifiedGroupKFold(4, shuffle=True, random_state=sd).split(np.zeros(len(_y1)), _y1, _p1), 1):
        f = FOLDS[_k]; _k += 1
        _ok &= (f["rep"], f["fold"]) == (rep, fold) and np.array_equal(np.sort(f["train_idx"]), tr) and np.array_equal(np.sort(f["val_idx"]), va)
CHECKS["folds_regenerate_exactly"] = bool(_ok)
print("authors' fold indices regenerate exactly:", _ok)

# smoke test: keep a few beats per record and class, remap folds
if SMOKE_TEST:
    def _keep(D):
        keep = []
        for p in np.unique(D["pid"]):
            for c, k in SMOKE_PER_CLASS.items():
                keep += list(np.where((D["pid"] == p) & (D["y"] == c))[0][:k])
        return np.sort(np.array(keep))
    _k1, _k2 = _keep(DATA["morph"]["DS1"]), _keep(DATA["morph"]["DS2"])
    for f in DATA:
        for ds, kk in (("DS1", _k1), ("DS2", _k2)):
            DATA[f][ds] = {k: v[kk] for k, v in DATA[f][ds].items()}
    _remap_arr = -np.ones(int(_k1.max()) + 1, int); _remap_arr[_k1] = np.arange(len(_k1))
    def _remap(idx):
        r = _remap_arr[idx[idx <= _k1.max()]]
        return r[r >= 0]
    FOLDS = [{"rep": f["rep"], "fold": f["fold"], "train_idx": _remap(f["train_idx"]), "val_idx": _remap(f["val_idx"])}
             for f in FOLDS]
    print("SMOKE subset:", {ds: np.bincount(DATA['morph'][ds]['y'], minlength=3).tolist() for ds in ("DS1", "DS2")})

CV_SPLITS = [(f["rep"], f["fold"]) for f in FOLDS]
if SMOKE_TEST:
    CV_SPLITS = SMOKE_CV_SPLITS
FOLD_BY = {(f["rep"], f["fold"]): f for f in FOLDS}

# ---------------- normalization (two paths, as the authors have them) ----------------
def normalize_cv(X, tr_idx, norm):
    """CV path (script 03 get_fold_data + float32 cast before encoding)."""
    if norm == "standard":
        return np.asarray(A_strat.minmax_per_beat(X), np.float32), {}
    mn, mx = A_strat.robust_minmax_fit(X[tr_idx], low=0.1, high=99.9)
    return np.asarray(A_strat.robust_minmax_apply(X, mn, mx), np.float32), {"p_low": mn, "p_high": mx}

def normalize_final(X1, X2, norm):
    """Final DS1 -> DS2 path (script 01): float64, global DS1 percentiles for robust."""
    if norm == "standard":
        return A_strat.Rowwise_Normalization(X1), A_strat.Rowwise_Normalization(X2), {}
    lo, hi = np.percentile(X1, [0.1, 99.9])
    return A_strat.minmax_apply(X1, lo, hi), A_strat.minmax_apply(X2, lo, hi), {"p_low": float(lo), "p_high": float(hi)}


## 6. Spike encoders and measured A_in

In [ ]:
# Encoders (the authors' functions, device passed explicitly), RR spike channels, A_in measurement.
PAPER_AIN = {"multi": 0.05892, "rate": 20.8245, "latency": 0.9473}   # values hard-coded by the authors

def multithreshold_t(Xb, num_levels=200):
    """GPU version of the authors' MultiThreshold (detect_edges='both', include_endpoints=True,
    separate_polarity=False). Same thresholds (np.linspace in the data's dtype) and the same comparisons:
    rise = x_t >= thr & x_{t-1} < thr, fall = x_t < thr & x_{t-1} >= thr, spike = rise | fall."""
    X = np.ascontiguousarray(np.asarray(Xb))
    thr = torch.from_numpy(np.linspace(0.0, 1.0, num_levels, dtype=X.dtype)).to(DEVICE)
    x = torch.from_numpy(X).to(DEVICE)
    out = torch.zeros((X.shape[0], X.shape[1], num_levels), dtype=torch.uint8, device=DEVICE)
    above = x[:, :, None] >= thr                     # (B, T, K)
    out[:, 1:] = (above[:, 1:] != above[:, :-1]).to(torch.uint8)
    return out

def roundrobin_t(counts, bin_pop=10):
    """GPU version of the authors' counts_to_binary_population_roundrobin. Their loop keeps a pointer p per
    (beat, channel), sets sub-channels (p + j) % bin_pop for j < min(k, bin_pop), then p = (p + k) % bin_pop.
    So p_t = (counts before t) % bin_pop, and sub-channel b fires iff (b - p_t) % bin_pop < min(k_t, bin_pop)."""
    c = torch.as_tensor(np.asarray(counts).astype(np.int64), device=DEVICE)          # (B, L, C)
    p = torch.remainder(torch.cumsum(c, dim=1) - c, bin_pop)
    keff = torch.clamp(c, max=bin_pop)
    b = torch.arange(bin_pop, device=DEVICE)
    rel = torch.remainder(b[None, None, None, :] - p[..., None], bin_pop)
    out = (rel < keff[..., None]).to(torch.uint8)
    return out.reshape(c.shape[0], c.shape[1], c.shape[2] * bin_pop)

def encode_t(Xb, enc, num_levels=200):
    """(B, 577) normalized beats -> (B, T_in, C) uint8 spikes as a tensor on DEVICE."""
    if enc == "multi":
        return multithreshold_t(Xb, num_levels)
    if enc == "rate":
        s = A_enc.TemporalRateCoding(Xb, None, gain=1.0, offset=0.0, pop_size=64, batch_size=1024, device=DEVICE,
                                     out_dir=None, spike_filename=None, labels_filename=None, save_to_disk=False)
        return torch.from_numpy(np.asarray(s, dtype=np.uint8)).to(DEVICE)
    if enc == "latency":
        cnt = A_enc.BoundedDelayLatencyCoding(Xb, Dmax=64, pop_size=1, batch_size=1024, device=DEVICE)
        return roundrobin_t(cnt, bin_pop=10)
    raise ValueError(enc)

def encode(Xb, enc, num_levels=200):
    """Same as encode_t, returned as a NumPy array."""
    return encode_t(Xb, enc, num_levels).cpu().numpy()

N_IN = {"multi": 200, "rate": 64, "latency": 10}

# RR timing as extra input channels: thermometer code of 3 ratio features, 10 levels each.
RR_LIQ_FEATS = [RR_NAMES.index(n) for n in ("pre_over_local", "post_over_local", "pre_over_rec")]
RR_LEVELS, RR_PERIOD = 10, 5            # an active channel spikes every 5th step

def rr_liq_feats(rr_rows):
    """Columns turned into RR channels: the 3 ratios; with the 9-column array of rr_inputs(..., True) also the
    9th column, pre-RR / recording mean RR (40 channels instead of 30)."""
    return RR_LIQ_FEATS + ([len(RR_NAMES)] if rr_rows.shape[1] > len(RR_NAMES) else [])

def rr_thresholds(rr_train):
    q = np.linspace(0.05, 0.95, RR_LEVELS)
    return np.stack([np.quantile(rr_train[:, j], q) for j in rr_liq_feats(rr_train)])     # (3 or 4, 10)

def rr_channels(rr_rows, thr, T):
    active = (rr_rows[:, rr_liq_feats(rr_rows)][:, :, None] >= thr[None]).reshape(len(rr_rows), -1)   # (B, 30 or 40)
    pulse = (np.arange(T) % RR_PERIOD == 0).astype(np.uint8)
    return (pulse[None, :, None] * active[:, None, :]).astype(np.uint8)                   # (B, T, 30)

def rr_window_counts(rr_rows, thr, T):
    """rr_channels(rr_rows, thr, T) summed over the 32 windows of 20 steps (steps >= T are silent), (B, 32, 30) uint8."""
    active = (rr_rows[:, rr_liq_feats(rr_rows)][:, :, None] >= thr[None]).reshape(len(rr_rows), -1)
    steps = np.arange(NWIN * WIN)
    per_window = ((steps % RR_PERIOD == 0) & (steps < T)).reshape(NWIN, WIN).sum(1)
    return (per_window[None, :, None] * active[:, None, :]).astype(np.uint8)

def measure_ain(Xn, rows, enc, num_levels=200, rr=None, rr_thr=None, n=5000):
    """Average number of active input spikes per time step (summed over channels), over a random
    subset of training beats. Same quantity as the authors' estimate_K_spikes_per_timestep."""
    rows = np.asarray(rows)
    torch.manual_seed(12345)           # the Rate encoder is stochastic: make A_in independent of job order
    pick = np.sort(np.random.default_rng(0).choice(rows, min(n, len(rows)), replace=False))
    tot, cnt = 0.0, 0
    for i in range(0, len(pick), 1000):
        r = pick[i:i + 1000]
        S = encode(Xn[r], enc, num_levels)
        if rr is not None:
            S = np.concatenate([S, rr_channels(rr[r], rr_thr, S.shape[1])], axis=2)
        tot += float(S.sum(dtype=np.int64)); cnt += S.shape[0] * S.shape[1]
    return tot / cnt


## 7. The replication's liquid (the authors' class; used for X in E1, 1f and the ELM controls)

In [ ]:
# The liquid: built with the authors' class; modes patch the two bugs. GPU simulator with diagnostics.
LIQ = dict(P_in=0.1, betaE=0.95, betaI=0.8, Uthr=1.0, rho=0.9, reservoir_dim=(7, 7, 7), excit_ratio=0.8,
           dt=1.0, num_sim_steps=650, C_vals={"exc_exc": 0.3, "exc_inh": 0.2, "inh_exc": 0.4, "inh_inh": 0.1},
           lambda_val=0.8, scl_v=1, mu_rsv=[0.020, 0.025, 0.050, 0.040], g_rec=1, tau_syn=0.01)
I_TARGET = LIQ["rho"] * (1 - LIQ["betaE"]) * LIQ["Uthr"]
MODES = {   # gamma_fix: Gamma(shape, scale) as intended; ain: "paper" = hard-coded value, "measured" = Eq. as described
    "published": {"gamma_fix": False, "ain": "paper"},
    "fixed":     {"gamma_fix": True,  "ain": "measured"},
    "fix_gamma": {"gamma_fix": True,  "ain": "paper"},
    "fix_ain":   {"gamma_fix": False, "ain": "measured"},
}
_TorchGamma = torch.distributions.Gamma

def build_liquid(input_size, mu_in, seed, gamma_fix, lambda_val=0.8, recurrence=True, g_rec=None):
    """seed_everything(seed) then the authors' constructor, exactly as in their scripts."""
    seed_everything(seed, deterministic=False)
    if gamma_fix:
        torch.distributions.Gamma = lambda conc, scale: _TorchGamma(conc, 1.0 / scale)   # (shape, scale) as intended
    try:
        model = LSM3D(input_size, LIQ["P_in"], mu_in, LIQ["betaE"], LIQ["betaI"], LIQ["reservoir_dim"],
                      LIQ["excit_ratio"], LIQ["dt"], LIQ["num_sim_steps"], LIQ["C_vals"], lambda_val, LIQ["scl_v"],
                      np.array(LIQ["mu_rsv"]), LIQ["g_rec"] if g_rec is None else g_rec, LIQ["tau_syn"], DEVICE).to(DEVICE)
    finally:
        torch.distributions.Gamma = _TorchGamma
    if not recurrence:
        model.rsv_weights = torch.zeros_like(model.rsv_weights)
    return model

def structure_stats(model):
    W = model.rsv_weights.detach().cpu().numpy(); Win = model.input_rsv_weights.detach().cpu().numpy()
    E = model.is_excit.cpu().numpy()
    out = {"n_recurrent_synapses": int((W != 0).sum()), "mean_recurrent_in_degree": float((W != 0).sum(0).mean()),
           "neurons_without_recurrent_input": int(((W != 0).sum(0) == 0).sum()),
           "n_input_synapses": int((Win != 0).sum()), "mean_input_in_degree": float((Win != 0).sum(0).mean()),
           "mean_nonzero_input_weight": float(Win[Win != 0].mean()) if (Win != 0).any() else 0.0,
           "threshold": LIQ["Uthr"], "n_excitatory": int(E.sum())}
    for name, pre, post in (("EE", E, E), ("EI", E, ~E), ("IE", ~E, E), ("II", ~E, ~E)):
        blk = W[np.ix_(pre, post)]
        nz = blk[blk != 0]
        out[f"{name}_synapses"] = int(nz.size)
        out[f"{name}_mean_abs_weight"] = float(np.abs(nz).mean()) if nz.size else 0.0
    return out

def save_reservoir(model, meta):
    rid = stable_hash(meta, 12)
    path = os.path.join(WORK_DIR, "reservoirs", f"{rid}.pt")
    if not os.path.exists(path):
        Win = model.input_rsv_weights.detach().cpu(); W = model.rsv_weights.detach().cpu()
        iw, ir = Win.nonzero().T, W.nonzero().T
        torch.save({"meta": meta, "input_size": Win.shape[0], "n_neurons": W.shape[0],
                    "win_index": iw.to(torch.int16), "win_value": Win[iw[0], iw[1]].float(),
                    "wrec_index": ir.to(torch.int16), "wrec_value": W[ir[0], ir[1]].float(),
                    "is_excit": model.is_excit.cpu(), "beta": torch.where(model.is_excit.cpu(), LIQ["betaE"], LIQ["betaI"]).float(),
                    "structure": structure_stats(model), "torch": torch.__version__, "device": str(DEVICE)}, path)
        disk_add(os.path.getsize(path))
    return rid

WIN, NWIN, N_NEUR = 20, 32, 343          # fr_windows(win=20, stride=20) over 650 steps -> 32 windows

def simulate(model, Xn, enc, batch, num_levels=200, rr=None, rr_thr=None, y=None, enc_seed=0):
    """Encode + run the liquid exactly as the authors' loop does. Returns spike counts per (window, neuron)
    as uint8 (N, 32*343) in the authors' feature order (window-major), plus liquid diagnostics."""
    torch.manual_seed(enc_seed)
    N = len(Xn)
    out = np.empty((N, NWIN * N_NEUR), np.uint8)
    Win, Wr = model.input_rsv_weights, model.rsv_weights
    neur = torch.zeros(N_NEUR, dtype=torch.float64, device=DEVICE)
    per_beat = np.empty(N, np.float32)
    acc = {"in_spikes": 0.0, "in_steps": 0, "Iin_sum": 0.0, "Iin_steps": 0, "Irec_abs_sum": 0.0}
    t_enc = t_sim = 0.0
    with torch.no_grad():
        for s in range(0, N, batch):
            t0 = time.time()
            x = encode_t(Xn[s:s + batch], enc, num_levels)
            if rr is not None:
                x = torch.cat([x, torch.from_numpy(rr_channels(rr[s:s + batch], rr_thr, x.shape[1])).to(DEVICE)], dim=2)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t_enc += time.time() - t0; t0 = time.time()
            B, Tin, C = x.shape
            spk = torch.zeros(B, N_NEUR, device=DEVICE); mem = torch.zeros(B, N_NEUR, device=DEVICE)
            zero_in = torch.zeros(B, C, device=DEVICE)
            cnt = torch.zeros(B, NWIN, N_NEUR, dtype=torch.int16, device=DEVICE)
            Iin_acc = torch.zeros((), dtype=torch.float64, device=DEVICE); Irec_acc = torch.zeros((), dtype=torch.float64, device=DEVICE)
            for t in range(LIQ["num_sim_steps"]):
                x_t = x[:, t, :].float() if t < Tin else zero_in
                I_in = x_t @ Win
                I_rec = spk @ Wr
                I = (I_in + I_rec) * model.dt
                spk, mem = model.lif(I, mem)
                if t < NWIN * WIN:
                    cnt[:, t // WIN] += spk.to(torch.int16)
                if t < Tin:
                    Iin_acc += I_in.mean(dtype=torch.float64)
                Irec_acc += I_rec.abs().mean(dtype=torch.float64)
            c = cnt.reshape(B, -1)
            out[s:s + B] = c.to(torch.uint8).cpu().numpy()
            neur += cnt.sum((0, 1), dtype=torch.float64)
            per_beat[s:s + B] = c.sum(1).float().cpu().numpy()
            acc["in_spikes"] += float(x.sum(dtype=torch.int64)); acc["in_steps"] += B * Tin
            acc["Iin_sum"] += float(Iin_acc) * B; acc["Iin_steps"] += Tin * B
            acc["Irec_abs_sum"] += float(Irec_acc) * B
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t_sim += time.time() - t0
            del x, cnt, spk, mem
    neur = neur.cpu().numpy()
    steps = NWIN * WIN
    E = model.is_excit.cpu().numpy()
    stats = {"liquid_firing_prob_per_step": float(neur.sum() / (N * steps * N_NEUR)),
             "excitatory_firing_prob": float(neur[E].sum() / (N * steps * E.sum())),
             "inhibitory_firing_prob": float(neur[~E].sum() / (N * steps * (~E).sum())),
             "silent_neurons": int((neur == 0).sum()), "silent_fraction": float((neur == 0).mean()),
             "liquid_spikes_per_beat_mean": float(per_beat.mean()),
             "input_spikes_per_step": acc["in_spikes"] / max(acc["in_steps"], 1),
             "mean_input_current_per_neuron_step": acc["Iin_sum"] / max(acc["Iin_steps"], 1),
             "input_current_over_target": (acc["Iin_sum"] / max(acc["Iin_steps"], 1)) / I_TARGET,
             "mean_abs_recurrent_current": acc["Irec_abs_sum"] / max(N * LIQ["num_sim_steps"], 1),
             "n_beats": N, "encode_seconds": t_enc, "simulate_seconds": t_sim}
    if y is not None:
        for c, nm in enumerate(CLASS_NAMES):
            if (y == c).any():
                stats[f"liquid_spikes_per_beat_{nm}"] = float(per_beat[y == c].mean())
    return out, stats

def liquid_raster(model, S, batch=256):
    """Full spike raster (B, 650, 343) with the same update as simulate(), on the device. Also returns the mean
    |recurrent current| and mean input current per neuron and step (input steps only)."""
    Win, Wr = model.input_rsv_weights, model.rsv_weights
    T = LIQ["num_sim_steps"]; out = np.zeros((len(S), T, N_NEUR), bool); irec = iin = 0.0; tin = 1
    with torch.no_grad():
        for s in range(0, len(S), batch):
            x = torch.from_numpy(np.ascontiguousarray(S[s:s + batch])).to(DEVICE).float()
            B, tin, Cin = x.shape
            spk = torch.zeros(B, N_NEUR, device=DEVICE); mem = torch.zeros(B, N_NEUR, device=DEVICE)
            zero_in = torch.zeros(B, Cin, device=DEVICE)
            r = torch.zeros(B, T, N_NEUR, dtype=torch.bool, device=DEVICE)
            for t in range(T):
                I_in = (x[:, t, :] if t < tin else zero_in) @ Win
                I_rec = spk @ Wr
                spk, mem = model.lif((I_in + I_rec) * model.dt, mem)
                r[:, t] = spk > 0
                irec += float(I_rec.abs().mean()) * B
                if t < tin:
                    iin += float(I_in.mean()) * B
            out[s:s + B] = r.cpu().numpy()
    return out, irec / (len(S) * T), iin / (len(S) * tin)

def counts_to_rates(counts_rows):
    """Authors' features: window mean of 0/1 spikes in float64, stored float32."""
    return (counts_rows.astype(np.float64) / WIN).astype(np.float32)


## 8. Non-liquid controls and neural-network baselines

In [ ]:
# Non-liquid feature sets and neural-network baselines.
def encoder_window_counts(Xn, enc, num_levels=200, batch=2048):
    """'No liquid' control: the encoder's own spikes counted in the same 32 windows of 20 steps."""
    C = num_levels if enc == "multi" else N_IN[enc]
    out = np.zeros((len(Xn), NWIN, C), np.uint8)
    for s in range(0, len(Xn), batch):
        S = encode(Xn[s:s + batch], enc, num_levels)
        T = min(S.shape[1], NWIN * WIN)
        Sp = np.zeros((S.shape[0], NWIN * WIN, C), np.uint8); Sp[:, :T] = S[:, :T]
        out[s:s + batch] = Sp.reshape(S.shape[0], NWIN, WIN, C).sum(2)
    return out

def huang_matrices(seed, n, dim):
    """Huang Eq. 1 uses F = X (A+)^-1, the pseudo-inverse of the random matrix A (dim x 200): returns pinv(A), 200 x dim."""
    A = np.random.default_rng(seed).standard_normal((n, dim, 200))
    return [np.linalg.pinv(a) for a in A]

def hybrid_rule(y_pred, ratio, rule, threshold):
    """Huang's SVEB rule on beats not predicted VEB. replace: SVEB iff ratio < threshold, else N.
    add: keep the model's SVEB calls and also call SVEB every beat with ratio < threshold."""
    y_pred = np.asarray(y_pred); low = np.asarray(ratio) < threshold
    sveb = low if rule == "replace" else (low | (y_pred == 1))
    return np.where(y_pred == 2, 2, np.where(sveb, 1, 0)).astype(np.int64)

def huang_fit_predict(spec, tr, ev, Dtr, Dev, cancel=None):
    """One RBF SVM per random matrix on [projection, pre-RR in s], majority vote (ties go to the lower class index:
    N, then SVEB, then VEB), then the SVEB rule on the beats not voted VEB. Our three classes, no class weights (libsvm)."""
    votes = np.zeros((len(ev), 3), np.int16); models = []
    for P in huang_matrices(spec["seed"], spec["n_matrices"], spec["dim"]):
        if cancel is not None and cancel.is_set():
            raise Cancelled()
        Ftr = np.hstack([Dtr["huang_beats"][tr] @ P, Dtr["rr"][tr, :1]])
        Fev = np.hstack([Dev["huang_beats"][ev] @ P, Dev["rr"][ev, :1]])
        clf = SVC(C=spec["C"], kernel="rbf", gamma=spec["gamma"], cache_size=HUANG_SVM_CACHE_MB).fit(Ftr, Dtr["y"][tr])
        votes[np.arange(len(ev)), clf.predict(Fev).astype(int)] += 1
        models.append(clf)
    vote = votes.argmax(1)
    return hybrid_rule(vote, Dev["rr_mean_ratio"][ev], "replace", spec["threshold"]), vote, votes, models

def shape_features(Xn):
    return np.asarray(Xn, np.float32)[:, ::8]              # 73 samples of the normalized beat

class SmallCNN(nn.Module):
    def __init__(self, n_rr=0):
        super().__init__()
        # GroupNorm, not BatchNorm: BatchNorm's running statistics made eval-mode predictions unstable
        self.f = nn.Sequential(nn.Conv1d(1, 16, 7, padding=3), nn.GroupNorm(4, 16), nn.ReLU(), nn.MaxPool1d(2),
                               nn.Conv1d(16, 32, 5, padding=2), nn.GroupNorm(4, 32), nn.ReLU(), nn.MaxPool1d(2),
                               nn.Conv1d(32, 64, 5, padding=2), nn.GroupNorm(4, 64), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.h = nn.Sequential(nn.Linear(64 + n_rr, 32), nn.ReLU(), nn.Linear(32, 3))
    def forward(self, x, rr=None):
        z = self.f(x[:, None, :]).squeeze(-1)
        return self.h(torch.cat([z, rr], 1) if rr is not None else z)

class SmallGRU(nn.Module):
    def __init__(self, n_rr=0):
        super().__init__()
        self.g = nn.GRU(1, 64, batch_first=True)
        self.h = nn.Sequential(nn.Linear(64 + n_rr, 32), nn.ReLU(), nn.Linear(32, 3))
    def forward(self, x, rr=None):
        _, hN = self.g(x[:, ::4, None])                     # 577 -> 145 steps
        z = hN[-1]
        return self.h(torch.cat([z, rr], 1) if rr is not None else z)

# ---- Tunable versions for the baseline search (the fixed models above stay as the untuned reference) ----
DEFAULT_ARCH = {"cnn": {"layers": 3, "width": 16, "kernels": [7, 5, 5], "pool": "avg", "head": 32, "dropout": 0.0, "lr": 1e-3, "wd": 1e-4},
                "gru": {"hidden": 64, "layers": 1, "bidir": False, "ds": 4, "readout": "last", "head": 32, "dropout": 0.0,
                        "lr": 1e-3, "wd": 1e-4}}

class TunableCNN(nn.Module):
    def __init__(self, a, n_rr=0):
        super().__init__()
        layers, cin = [], 1
        for i in range(a["layers"]):
            c, k = min(64, a["width"] * 2 ** i), a["kernels"][i]
            layers += [nn.Conv1d(cin, c, k, padding=k // 2), nn.GroupNorm(4, c), nn.ReLU()]
            if a["dropout"] > 0:
                layers.append(nn.Dropout(a["dropout"]))
            if i < a["layers"] - 1:
                layers.append(nn.MaxPool1d(2))
            cin = c
        self.f, self.pool = nn.Sequential(*layers), a["pool"]
        nz = cin * {"avg": 1, "max": 1, "avgmax": 2, "flat8": 8}[a["pool"]]
        self.h = nn.Sequential(nn.Linear(nz + n_rr, a["head"]), nn.ReLU(), nn.Dropout(a["dropout"]), nn.Linear(a["head"], 3))
    def forward(self, x, rr=None):
        z = self.f(x[:, None, :])
        z = {"avg": lambda: z.mean(-1), "max": lambda: z.amax(-1), "avgmax": lambda: torch.cat([z.mean(-1), z.amax(-1)], 1),
             "flat8": lambda: nn.functional.adaptive_avg_pool1d(z, 8).flatten(1)}[self.pool]()
        return self.h(torch.cat([z, rr], 1) if rr is not None else z)

class TunableGRU(nn.Module):
    def __init__(self, a, n_rr=0):
        super().__init__()
        self.a = a
        self.g = nn.GRU(1, a["hidden"], num_layers=a["layers"], batch_first=True, bidirectional=a["bidir"],
                        dropout=a["dropout"] if a["layers"] > 1 else 0.0)
        nz = a["hidden"] * (2 if a["bidir"] else 1)
        self.h = nn.Sequential(nn.Linear(nz + n_rr, a["head"]), nn.ReLU(), nn.Dropout(a["dropout"]), nn.Linear(a["head"], 3))
    def forward(self, x, rr=None):
        out, hN = self.g(x[:, ::self.a["ds"], None])
        if self.a["readout"] == "mean":
            z = out.mean(1)
        else:
            z = torch.cat([hN[-2], hN[-1]], 1) if self.a["bidir"] else hN[-1]
        return self.h(torch.cat([z, rr], 1) if rr is not None else z)

def nn_params(kind, arch, n_rr=0):
    m = (TunableCNN if kind == "cnn" else TunableGRU)(arch, n_rr)
    return sum(p.numel() for p in m.parameters())

def nn_cost(kind, arch):
    """Compute of one forward pass relative to the untuned model (used only to estimate training time)."""
    def flops(a):
        if kind == "cnn":
            L, cin, tot = 577, 1, 0
            for i in range(a["layers"]):
                c = min(64, a["width"] * 2 ** i); tot += L * cin * c * a["kernels"][i]; cin = c; L //= 2
            return tot
        steps, h = math.ceil(577 / a["ds"]), a["hidden"]
        return steps * (2 if a["bidir"] else 1) * sum(3 * h * ((1 if l == 0 else h * (2 if a["bidir"] else 1)) + h)
                                                       for l in range(a["layers"]))
    return flops(arch) / flops(DEFAULT_ARCH[kind])

NN_SPACE = {"cnn": {"layers": [2, 3, 4], "width": [8, 12, 16, 24, 32], "kernel": [5, 7, 9, 11], "pool": ["avg", "max", "avgmax", "flat8"],
                    "head": [32, 64], "dropout": [0.0, 0.1, 0.3], "lr": [3e-4, 1e-3, 3e-3], "wd": [0.0, 1e-4, 1e-3]},
            "gru": {"hidden": [32, 48, 64, 80, 96], "layers": [1, 2], "bidir": [False, True], "ds": [2, 4, 8], "readout": ["last", "mean"],
                    "head": [32, 64], "dropout": [0.0, 0.1, 0.3], "lr": [3e-4, 1e-3, 3e-3], "wd": [0.0, 1e-4, 1e-3]}}

def nn_search_space(kind):
    """NN_TUNE_TRIALS architectures for one model: trial 0 is the untuned model, the rest are random draws that stay
    under NN_PARAM_CAP parameters (counted with the RR inputs, so both input variants use the same list)."""
    rng = np.random.default_rng({"cnn": 101, "gru": 202}[kind])
    out, seen = [dict(DEFAULT_ARCH[kind])], {json.dumps(DEFAULT_ARCH[kind], sort_keys=True)}
    S = NN_SPACE[kind]
    for _ in range(5000):
        if len(out) >= NN_TUNE_TRIALS:
            break
        a = {k: v[rng.integers(len(v))] for k, v in S.items()}
        a = {k: (v.item() if hasattr(v, "item") else v) for k, v in a.items()}
        if kind == "cnn":
            a["kernels"] = [a.pop("kernel")] * a["layers"]
        key = json.dumps(a, sort_keys=True)
        if key in seen or nn_params(kind, a, len(RR_NAMES)) > NN_PARAM_CAP:
            continue
        seen.add(key); out.append(a)
    return out[:NN_TUNE_TRIALS]

def train_nn(kind, Xtr, ytr, Xev, rr_tr=None, rr_ev=None, seed=0, yev_for_curve=None, arch=None, epochs=None):
    """arch=None: the fixed untuned model (Adam 1e-3, wd 1e-4, NN_EPOCHS epochs). arch=dict: a tuned architecture."""
    torch.manual_seed(seed); np.random.seed(seed)
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False   # same result every session
    epochs = epochs or NN_EPOCHS
    use_rr = rr_tr is not None
    if use_rr:
        mu, sd = rr_tr.mean(0), rr_tr.std(0) + 1e-6
        rr_tr_s, rr_ev_s = (rr_tr - mu) / sd, (rr_ev - mu) / sd
    n_rr = rr_tr.shape[1] if use_rr else 0
    if arch is None:
        model = (SmallCNN if kind == "cnn" else SmallGRU)(n_rr=n_rr).to(DEVICE)
        lr, wd = 1e-3, 1e-4
    else:
        model = (TunableCNN if kind == "cnn" else TunableGRU)(arch, n_rr).to(DEVICE)
        lr, wd = arch["lr"], arch["wd"]
    cw = soft_class_weight(ytr)
    lossf = nn.CrossEntropyLoss(weight=torch.tensor([cw.get(c, 1.0) for c in range(3)], dtype=torch.float32, device=DEVICE))
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    # the training set lives on the GPU (same values as copying each batch, much less transfer)
    Xt = torch.tensor(np.asarray(Xtr, np.float32), device=DEVICE); yt = torch.tensor(ytr, dtype=torch.long, device=DEVICE)
    Rt = torch.tensor(rr_tr_s, dtype=torch.float32, device=DEVICE) if use_rr else None
    def predict(X, R):
        model.eval(); outs = []
        with torch.no_grad():
            for i in range(0, len(X), 2048):
                xb = torch.tensor(np.asarray(X[i:i + 2048], np.float32), device=DEVICE)
                rb = torch.tensor(R[i:i + 2048], dtype=torch.float32, device=DEVICE) if R is not None else None
                outs.append(model(xb, rb).float().cpu())
        model.train()
        return torch.cat(outs).numpy()
    curve = []
    g = torch.Generator().manual_seed(seed)
    sub = np.random.default_rng(0).choice(len(Xtr), min(4000, len(Xtr)), replace=False)
    cost = 1.0 if arch is None else nn_cost(kind, arch) ** 0.5
    for ep in range(epochs):
        t0 = time.time(); perm = torch.randperm(len(Xt), generator=g).to(DEVICE); tot = 0.0
        for i in range(0, len(Xt), 256):
            b = perm[i:i + 256]
            xb, yb = Xt[b], yt[b]
            rb = Rt[b] if use_rr else None
            opt.zero_grad(); l = lossf(model(xb, rb), yb); l.backward(); opt.step(); tot += float(l) * len(b)
        row = {"epoch": ep + 1, "train_loss": tot / len(Xt), "epoch_seconds": time.time() - t0,
               "train_sub_macro_f1": compute_metrics(ytr[sub], predict(Xtr[sub], rr_tr_s[sub] if use_rr else None).argmax(1))["macro_f1"]}
        if yev_for_curve is not None:
            row["eval_macro_f1"] = compute_metrics(yev_for_curve, predict(Xev, rr_ev_s if use_rr else None).argmax(1))["macro_f1"]
        curve.append(row)
        if arch is None:
            update_speed("nn_s_per_epoch_kbeat", row["epoch_seconds"] / (len(Xtr) / 1000), sub=kind)
        else:
            update_speed("nn_unit_s_per_epoch_kbeat", row["epoch_seconds"] / (len(Xtr) / 1000) / cost, sub=kind)
    logits = predict(Xev, rr_ev_s if use_rr else None)
    state = {"_torch_state": True, "kind": kind, "arch": arch, "epochs": epochs,
             "n_params": sum(p.numel() for p in model.parameters()),
             "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
             "rr_mean": mu if use_rr else None, "rr_std": sd if use_rr else None}
    return logits.argmax(1), logits.astype(np.float32), curve, state


## 9. The replication's plan (its groups are disabled here; its run ids identify the cached runs)

In [ ]:
# The experiment plan: every run as a spec; identical specs across groups are run once.
ENCS = ["multi", "rate", "latency"]
CONFIG_IDS = SMOKE_CONFIGS if SMOKE_TEST else sorted(CONFIGS)
SPLITS_CV = [f"cv-r{r}f{f}" for r, f in CV_SPLITS]
def parse_split(s):
    m = re.match(r"cv-r(\d+)f(\d+)", s)
    return (int(m.group(1)), int(m.group(2))) if m else None

def lsm_spec(mode, seed, filt, norm, enc, readout, split, gain=1, lambda_val=0.8, recurrence=True,
             rr_in_liquid=False, ain_override=None, num_levels=200, rr_set=None, g_rec=None):
    s = {"kind": "lsm", "mode": mode, "seed": seed, "filter": filt, "norm": norm, "encoder": enc,
         "num_levels": num_levels if enc == "multi" else None, "gain": gain, "lambda_val": lambda_val,
         "recurrence": recurrence, "rr_in_liquid": rr_in_liquid, "ain_override": ain_override,
         "readout": readout, "split": split}
    if rr_set is not None:          # only new runs carry this key, so every earlier run id is unchanged
        s["rr_set"] = rr_set
    if g_rec is not None:           # likewise: recurrent gain of the working liquid
        s["g_rec"] = float(g_rec)
    return s

def feat_spec(model, split, enc=None, seed=None):
    return {"kind": "feat", "model": model, "encoder": enc, "seed": seed, "filter": "morph", "norm": "robust",
            "readout": "lda" if model == "lda" else "svm", "split": split}

def nn_spec(model, use_rr, seed, split, arch=None, epochs=None, trial=None):
    s = {"kind": "nn", "model": model, "use_rr": use_rr, "seed": seed, "filter": "morph", "norm": "robust", "split": split}
    if model == "cnn":
        s["norm_layer"] = "groupnorm"    # new run ids: CNN runs from before the BatchNorm fix are not reused
    if arch is not None:      # tuned baselines; the untuned spec keeps its original keys (and run ids)
        s.update({"arch": arch, "epochs": epochs, "trial": trial})
    return s

def huang_spec(seed, gamma, split):
    return {"kind": "huang", "seed": seed, "gamma": float(gamma), "C": HUANG_C, "n_matrices": HUANG_MATRICES, "dim": HUANG_DIM,
            "threshold": HUANG_THRESHOLD, "filter": "none", "norm": "zscore", "split": split}

def config_of(spec):
    if spec.get("kind") != "lsm" or spec["gain"] != 1 or spec["lambda_val"] != 0.8 or not spec["recurrence"] \
            or spec["rr_in_liquid"] or spec["num_levels"] not in (None, 200) or spec["readout"] == "svm_rr" \
            or spec.get("ain_override") is not None or spec.get("g_rec") is not None:   # k-stale / working liquid: not one of the 24
        return None
    for cid, c in CONFIGS.items():
        if (c["filter"], c["norm"], c["encoder"], c["readout"]) == (spec["filter"], spec["norm"], spec["encoder"], spec["readout"]):
            return cid
    return None

def run_id_of(spec):
    h = stable_hash({**spec, "_v": CODE_VERSION, "_smoke": SMOKE_TEST})
    k = spec["kind"]
    if k == "lsm":
        tag = [spec["mode"], f"s{spec['seed']}", f"{spec['filter']}-{spec['norm']}-{spec['encoder']}"
               + (f"K{spec['num_levels']}" if spec["num_levels"] not in (None, 200) else ""), spec["readout"]]
        if spec["gain"] != 1: tag.append(f"g{spec['gain']}")
        if spec["lambda_val"] != 0.8: tag.append(f"lam{spec['lambda_val']}")
        if not spec["recurrence"]: tag.append("norec")
        if spec["rr_in_liquid"]: tag.append("rrliq")
        if spec.get("rr_set"): tag.append(f"{spec['rr_set']}rr")
        if spec.get("g_rec") is not None: tag.append(f"grec{spec['g_rec']:g}")
        if spec["ain_override"] is not None: tag.append(f"ain{spec['ain_override']}")
    elif k == "feat":
        tag = [spec["model"]] + ([spec["encoder"]] if spec["encoder"] else []) + ([f"s{spec['seed']}"] if spec["seed"] is not None else [])
    elif k == "nn":
        t = spec.get("trial")
        tag = [spec["model"] + ("+rr" if spec["use_rr"] else "") + ("" if t is None else "-tuned" if t == "best" else f"-t{t}"),
               f"s{spec['seed']}"] + ([f"e{spec['epochs']}"] if t == "best" else [])
        if spec.get("rr_set"):
            tag[0] += f"-{spec['rr_set']}rr"
    elif k == "huang":
        tag = [f"g{spec['gamma']:.4g}", f"s{spec['seed']}"]
    elif k == "hybrid":
        tag = [spec["rule"], spec["base"].split("__")[1]]
    else:
        tag = [k]
    return f"{'SMOKE__' if SMOKE_TEST else ''}{k}__{'_'.join(tag)}__{spec.get('split', 'na')}__{h}"

def _retune_combos():
    grid = list(itertools.product(RETUNE_GRID["gain"], RETUNE_GRID["lambda_val"], RETUNE_GRID["num_levels"]))
    pick = np.random.default_rng(0).permutation(len(grid))[:RETUNE_TRIALS]
    return [dict(zip(("gain", "lambda_val", "num_levels"), grid[i])) for i in sorted(pick)]

RETUNE_STAGE1_SPLITS = SPLITS_CV if SMOKE_TEST else [f"cv-r1f{f}" for f in range(1, 5)]

def retune_choice():
    """Best stage-1 combination by mean Macro-F1 on DS1 CV (rep 1), or None if stage 1 is unfinished."""
    rows = []
    for c in _retune_combos():
        vals = []
        for sp in RETUNE_STAGE1_SPLITS:
            rid = run_id_of(lsm_spec("fixed", SEEDS[0], "morph", "robust", "multi", "svm", sp, **c))
            if rid not in CACHE.done or CACHE.done[rid][2] == "-":
                return None, None
            vals.append(float(CACHE.done[rid][2]))
        rows.append({**c, "cv_macro_f1": float(np.mean(vals))})
    df = pd.DataFrame(rows).sort_values("cv_macro_f1", ascending=False)
    return df.iloc[0][["gain", "lambda_val", "num_levels"]].to_dict(), df

NN_TUNE_SPLITS = RETUNE_STAGE1_SPLITS
NN_ARCHS = {m: nn_search_space(m) for m in ("cnn", "gru")}

def _run_metrics(rid):
    p = os.path.join(WORK_DIR, "runs", rid, "metrics.json")
    return json.load(open(p)) if os.path.exists(p) else None

def nn_tune_choice(model, use_rr):
    """Best (architecture, epoch count) by mean held-out Macro-F1 over the rep-1 DS1 folds, or None if unfinished."""
    rows = []
    for t, a in enumerate(NN_ARCHS[model]):
        per_fold = []
        for sp in NN_TUNE_SPLITS:
            rid = run_id_of(nn_spec(model, use_rr, SEEDS[0], sp, arch=a, epochs=NN_TUNE_MAX_EPOCHS, trial=t))
            m = _run_metrics(rid) if rid in CACHE.done else None
            if m is None or "eval_macro_f1_by_epoch" not in m:
                return None, None
            per_fold.append(m["eval_macro_f1_by_epoch"])
        for ep in NN_TUNE_CHECKPOINTS:
            rows.append({"model": model, "use_rr": use_rr, "trial": t, "epochs": ep,
                         "cv_macro_f1": float(np.mean([f[str(ep)] for f in per_fold])),
                         "cv_macro_f1_sd": float(np.std([f[str(ep)] for f in per_fold], ddof=1)) if len(per_fold) > 1 else np.nan,
                         "n_params": nn_params(model, a, len(RR_NAMES) if use_rr else 0), "arch": json.dumps(a, sort_keys=True)})
    df = pd.DataFrame(rows).sort_values(["cv_macro_f1", "trial", "epochs"], ascending=[False, True, True]).reset_index(drop=True)
    b = df.iloc[0]
    return {"trial": int(b.trial), "arch": NN_ARCHS[model][int(b.trial)], "epochs": int(b.epochs), "cv_macro_f1": float(b.cv_macro_f1)}, df

def working_diag_spec():
    return {"kind": "wdiag", "seed": SEEDS[0], "encoder": "multi", "filter": "morph", "norm": "robust",
            "lambda_val": WORKING_LAMBDA, "gains": list(WORKING_GAINS), "n_beats": WORKING_DIAG_BEATS,
            "t_perturb": WORKING_T_PERTURB, "max_firing": WORKING_MAX_FIRING, "split": "na"}

def working_choice():
    """g_rec picked by the finished diagnostic (checks/working_liquid_diag.json, tied to its run id), else None."""
    rid = run_id_of(working_diag_spec())
    p = os.path.join(WORK_DIR, "checks", "working_liquid_diag.json")
    if rid not in CACHE.done or not os.path.exists(p):
        return None
    d = json.load(open(p))
    return d["chosen_g_rec"] if d.get("run_id") == rid else None

def hybrid_bases():
    """(label, spec) of the runs the hybrid rule is applied to: published config 11, Multi + RR in the liquid
    (published and fixed), tuned GRU / CNN + RR; every seed, the final test and all CV splits."""
    nn_best = {m: nn_tune_choice(m, True)[0] for m in ("gru", "cnn")}
    out = []
    for seed in SEEDS:
        for sp in ["final"] + SPLITS_CV:
            out.append(("LSM published, config 11", lsm_spec("published", seed, "morph", "robust", "multi", "svm", sp)))
            for mode in ("published", "fixed"):
                out.append((f"LSM {mode}, Multi + RR in liquid",
                            lsm_spec(mode, seed, "morph", "robust", "multi", "svm", sp, rr_in_liquid=True)))
            for m, best in nn_best.items():
                if best is not None:
                    out.append((f"{m.upper()} tuned + RR",
                                nn_spec(m, True, seed, sp, arch=best["arch"], epochs=best["epochs"], trial="best")))
    return out

def build_plan():
    P = []
    def add(group, spec):
        if GROUPS_ENABLED.get(group, False):
            P.append((group, spec))
    for group in GROUP_ORDER:
        part = group.rsplit("_", 1)[-1]
        splits = ["final"] if part == "final" else SPLITS_CV
        if group == "checks":
            add(group, {"kind": "check", "split": "na"})
        elif group == "brian2":
            add(group, {"kind": "brian2", "split": "na", "beats": BRIAN2_BEATS})
        elif group.startswith("main_"):
            for modes in (["published", "fixed"], ["fix_gamma", "fix_ain"]):
                for seed in SEEDS:
                    for mode in modes:
                        for cid in CONFIG_IDS:
                            c = CONFIGS[cid]
                            for sp in splits:
                                add(group, lsm_spec(mode, seed, c["filter"], c["norm"], c["encoder"], c["readout"], sp))
        elif group.startswith("gain_"):
            for seed in SEEDS:
                for enc in ENCS:
                    for g in GAINS:
                        for sp in splits:
                            add(group, lsm_spec("fixed", seed, "morph", "robust", enc, "svm", sp, gain=g))
        elif group.startswith("ablation_"):
            for seed in SEEDS:
                for enc in ENCS:
                    for sp in splits:
                        for mode in ("published", "fixed"):
                            add(group, lsm_spec(mode, seed, "morph", "robust", enc, "svm", sp, recurrence=False))
                        for lam in DENSE_LAMBDAS:
                            add(group, lsm_spec("fixed", seed, "morph", "robust", enc, "svm", sp, lambda_val=lam))
                        add(group, feat_spec("no_liquid", sp, enc=enc, seed=seed if enc == "rate" else None))
                        add(group, feat_spec("elm", sp, enc=enc, seed=seed))
        elif group.startswith("rr_"):
            for sp in splits:
                for m in ("rr_only", "rr_shape", "lda"):
                    add(group, feat_spec(m, sp))
            for seed in SEEDS:
                for enc in ENCS:
                    for mode in ("published", "fixed"):
                        for sp in splits:
                            add(group, lsm_spec(mode, seed, "morph", "robust", enc, "svm_rr", sp))
                            add(group, lsm_spec(mode, seed, "morph", "robust", enc, "svm", sp, rr_in_liquid=True))
        elif group.startswith("baseline_"):
            for sp in splits:
                add(group, feat_spec("raw_beat", sp))
            for seed in SEEDS:
                for m in ("cnn", "gru"):
                    for use_rr in (False, True):
                        for sp in splits:
                            add(group, nn_spec(m, use_rr, seed, sp))
        elif group == "nn_tune":
            for m in ("cnn", "gru"):
                for use_rr in (False, True):
                    for t, a in enumerate(NN_ARCHS[m]):
                        for sp in NN_TUNE_SPLITS:
                            add(group, nn_spec(m, use_rr, SEEDS[0], sp, arch=a, epochs=NN_TUNE_MAX_EPOCHS, trial=t))
                    best, _ = nn_tune_choice(m, use_rr)
                    if best is not None:            # stage 2 appears once all of this variant's trials are done
                        for seed in SEEDS:
                            for sp in ["final"] + SPLITS_CV:
                                add(group, nn_spec(m, use_rr, seed, sp, arch=best["arch"], epochs=best["epochs"], trial="best"))
        elif group == "retune":
            for c in _retune_combos():
                for sp in RETUNE_STAGE1_SPLITS:
                    add(group, lsm_spec("fixed", SEEDS[0], "morph", "robust", "multi", "svm", sp, **c))
            best, _ = retune_choice()
            if best is not None:
                best = {"gain": int(best["gain"]), "lambda_val": float(best["lambda_val"]), "num_levels": int(best["num_levels"])}
                for seed in SEEDS:
                    for sp in ["final"] + SPLITS_CV:
                        add(group, lsm_spec("fixed", seed, "morph", "robust", "multi", "svm", sp, **best))
        elif group.startswith("kstale_"):
            for cid in CONFIG_IDS:
                c = CONFIGS[cid]
                if c["encoder"] == "multi":
                    continue
                for sp in splits:
                    add(group, lsm_spec("published", SEEDS[0], c["filter"], c["norm"], c["encoder"], c["readout"], sp,
                                        ain_override=PAPER_AIN["multi"]))
        elif group.startswith("huang_"):
            for seed in SEEDS:
                for g in HUANG_GAMMAS:
                    for sp in splits:
                        add(group, huang_spec(seed, g, sp))
        elif group.startswith("meanrr_") and group != "meanrr_nn":
            for sp in splits:
                for m in ("rr_only_mrr", "rr_shape_mrr"):
                    add(group, feat_spec(m, sp))
        elif group.startswith("elmrr_"):
            for seed in SEEDS:
                for enc in ENCS:
                    for sp in splits:
                        add(group, feat_spec("elm_rr", sp, enc=enc, seed=seed))
        elif group.startswith("rrliqmean_"):
            for seed in SEEDS:
                for enc in ENCS:
                    for mode in ("published", "fixed"):
                        for sp in splits:
                            add(group, lsm_spec(mode, seed, "morph", "robust", enc, "svm", sp, rr_in_liquid=True, rr_set="mean"))
        elif group == "working_diag":
            add(group, working_diag_spec())
        elif group.startswith("working_"):
            g = working_choice()
            if g is not None:                   # appears once the diagnostic has picked g_rec
                for seed in SEEDS:
                    for sp in splits:
                        for rec in (True, False):
                            for ro in ("svm", "svm_late"):
                                add(group, lsm_spec("fixed", seed, "morph", "robust", "multi", ro, sp,
                                                    lambda_val=WORKING_LAMBDA, recurrence=rec, g_rec=g))
        elif group == "meanrr_nn":
            for m in ("cnn", "gru"):
                best, _ = nn_tune_choice(m, True)
                if best is not None:            # appears once the tuned + RR architecture is known
                    for seed in SEEDS:
                        for sp in ["final"] + SPLITS_CV:
                            s = nn_spec(m, True, seed, sp, arch=best["arch"], epochs=best["epochs"], trial="best")
                            s["rr_set"] = "mean"
                            add(group, s)
        elif group == "hybrid" and GROUPS_ENABLED.get(group, False):
            for label, bspec in hybrid_bases():
                brid = run_id_of(bspec)
                if brid in CACHE.done and os.path.exists(os.path.join(WORK_DIR, "runs", brid, "predictions.npz")):
                    for rule in HYBRID_RULES:
                        add(group, {"kind": "hybrid", "base": brid, "base_label": label, "rule": rule,
                                    "threshold": HUANG_THRESHOLD, "seed": bspec["seed"], "split": bspec["split"]})
    seen, plan = {}, []
    for group, spec in P:
        rid = run_id_of(spec)
        if rid in seen:
            seen[rid]["groups"].append(group)
            continue
        e = {"rid": rid, "group": group, "groups": [group], "spec": spec}
        seen[rid] = e
        plan.append(e)
    if SHARE != "BOTH":              # each run belongs to the share of its (first) group: the shares never overlap
        plan = [e for e in plan if e["group"] in SHARE_GROUPS[SHARE]]
    return plan

def group_on(g):
    return GROUPS_ENABLED.get(g, False) and (SHARE == "BOTH" or g in SHARE_GROUPS[SHARE])

def states_key(spec):
    """Runs with the same states_key share one simulation (or one feature computation)."""
    sp = spec["split"]
    if spec["kind"] == "lsm":
        measured = MODES[spec["mode"]]["ain"] == "measured" and spec["ain_override"] is None
        if sp == "final":
            nk = "final"
        elif spec["norm"] == "standard" and not measured and not spec["rr_in_liquid"]:
            nk = "cv-shared"
        else:
            nk = sp
        core = {k: v for k, v in spec.items() if k not in ("readout", "split")}
        return ("lsm", json.dumps(core, sort_keys=True), nk)
    if spec["kind"] == "feat":
        core = {k: v for k, v in spec.items() if k not in ("readout", "split")}
        return ("feat", json.dumps(core, sort_keys=True), sp)
    return (spec["kind"], run_id_of(spec), sp)

def make_jobs(plan):
    jobs, index = [], {}
    for e in plan:
        k = states_key(e["spec"])
        if k not in index:
            index[k] = len(jobs)
            jobs.append({"key": k, "runs": []})
        jobs[index[k]]["runs"].append(e)
    return jobs

_p = build_plan()
_counts = pd.Series([e["group"] for e in _p]).value_counts().reindex(GROUP_ORDER).fillna(0).astype(int)
print(f"plan: {len(_p)} unique runs in {len(make_jobs(_p))} jobs")
print(_counts.to_string())

if SHARE != "BOTH":
    print(f"SHARE = {SHARE!r}: this machine runs the groups {', '.join(SHARE_GROUPS[SHARE])}")
    if GROUPS_ENABLED.get("hybrid"):
        print("NOTE: the hybrid group runs only with SHARE = 'BOTH' (it reads predictions made on both machines)")


## 10. Stage 1: level 1, the Maass check, the ladder, the confirmation (no labels)

In [ ]:
# Stage 1 (plan, Stages): level 1 checks, the Maass check (testing the test), the X to Y ladder (E0), the
# confirmation and the speed test. No labels are used anywhere in this cell. Each step saves its result in
# checks/stage1/, so a restarted session continues where it stopped. The last step writes the calibration file.
import importlib, urllib.request
from lsm2 import config as LC, liquid as LL, sim as LS, measure as LMe, level1 as L1, level2 as L2, maass_check as LMC
from lsm2 import ladder as LLd, calib as LCa, data as LDa, gain as LGa

SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]          # the replication's chart colours
LC.CODE["x_update"] = X_UPDATE
S1_DIR = os.path.join(WORK_DIR, "checks", "stage1")
os.makedirs(S1_DIR, exist_ok=True)
CAL_PATH = os.path.join(WORK_DIR, "checks", "redesign_calibration.json")
MAASS_SCRIPT_SHA256 = "81f50fec6001c76b16ba548f2e03e915de9a9799c554ca33aa0598e8bb47ac4e"   # the copy the code was tested on

if SMOKE_TEST:     # a logic test of every step on a few beats and a coarse grid; the numbers mean nothing
    LC.N_CAL_BEATS, LC.GEN_COPIES = 24, 6
    LC.CODE.update(twin_beats=4, memory_sequences=16)
    LC.G_COARSE, LC.G_LOW, LC.G_FINE_K = (1, 10, 100), (0.3,), 1
    LC.N_SUB_REFERENCE, LC.E1D_N_TRY_MAX = 4, 4
    LC.SEEDS_SEARCH, LC.SEEDS_CONFIRM, LC.SEEDS_RETRY = (1, 2), (101, 202), (404, 505)

def s1_gate(ok, msg):
    """A stop point of the plan. In the smoke test the numbers mean nothing, so it only warns."""
    if ok:
        return
    if SMOKE_TEST:
        log("SMOKE TEST: would stop here: " + msg)
        return
    raise RuntimeError(msg)

def s1_path(step):
    return os.path.join(S1_DIR, f"{step}.json")

def s1_done(step):
    return os.path.exists(s1_path(step))

def s1_save(step, obj):
    jdump(obj, s1_path(step))

def s1_load(step):
    return json.load(open(s1_path(step)))

# ---------------------------------------------------------------- label-free inputs: the two halves of DS1
S1_SEARCH_RECS, S1_CONFIRM_RECS = LDa.split_records(DS_RECORDS["DS1"])
def half_beats(recs, filt="morph", norm="robust"):
    """One half of DS1 through a front end's filter and normalisation (robust percentiles fitted on that half)."""
    D = DATA[filt]["DS1"]
    rows = np.nonzero(np.isin(D["pid"], recs))[0]
    if norm == "standard":
        Xn = np.asarray(A_strat.minmax_per_beat(D["X"][rows]), np.float32)
    else:
        mn, mx = A_strat.robust_minmax_fit(D["X"][rows], low=0.1, high=99.9)
        Xn = np.asarray(A_strat.robust_minmax_apply(D["X"][rows], mn, mx), np.float32)
    return Xn, rows

def enc_np(X, enc="multi"):
    out = []
    for s in range(0, len(X), 2000):
        out.append(encode(X[s:s + 2000], enc))
    return np.concatenate(out)

S1_CTX = LMe.Ctx(DEVICE, torch.float32, NEW_SIM_BATCH, log)
S1_CTX64 = LMe.Ctx(DEVICE, torch.float64, NEW_SIM_BATCH, log)
_beats_s, _rows_s = half_beats(S1_SEARCH_RECS)
_beats_c, _rows_c = half_beats(S1_CONFIRM_RECS)
# the package's encoder must be the authors' (used inside the package for the surrogate only)
assert np.array_equal(LDa.multithreshold(_beats_s[:50]), encode(_beats_s[:50], "multi")), "encoder mismatch"
SEARCH = LDa.make_search_data("search", _beats_s, enc_np, seed=1)
CONFIRM = LDa.make_search_data("confirm", _beats_c, enc_np, seed=2)
log(f"stage 1 inputs: search half {len(S1_SEARCH_RECS)} records ({len(_beats_s)} beats), confirmation half "
    f"{len(S1_CONFIRM_RECS)} records ({len(_beats_c)} beats); A_in {SEARCH.ain:.4f} / {CONFIRM.ain:.4f}; "
    f"surrogate/ECG input spikes {SEARCH.meta['surrogate_vs_ecg']:.3f} / {CONFIRM.meta['surrogate_vs_ecg']:.3f}")

def x_liquid(seed, n_sub, n_ch=200):
    return LL.build_candidate(frozenset(), seed, n_sub, n_ch)

def y_liquid(seed, n_sub, n_ch=200):
    return LL.build_candidate(LC.ALL_FACTORS, seed, n_sub, n_ch)

def x_scale(liq):
    return LL.paper_rule_scale(SEARCH.ain, liq)

def maass_script_path():
    if MAASS_SCRIPT_PATH:
        p = os.path.expanduser(MAASS_SCRIPT_PATH)
    else:
        p = os.path.join(SCRATCH_DIR, "maass2002.py")
        if not os.path.exists(p):
            urllib.request.urlretrieve(MAASS_SCRIPT_URL, p)
    h = hashlib.sha256(open(p, "rb").read()).hexdigest()
    if h != MAASS_SCRIPT_SHA256:
        log(f"WARNING: (d)'s script differs from the tested copy (sha256 {h[:12]} vs {MAASS_SCRIPT_SHA256[:12]})")
    return p

# ---------------------------------------------------------------- step 1: level 1
def stage1_level1():
    out = {}
    s0 = LC.SEEDS_SEARCH[0]
    beats10 = SEARCH.sets["sep"][:10 if not SMOKE_TEST else 2]
    try:
        import pytest
        tdir = os.path.join(SCRATCH_DIR, "lsm2_tests")
        r = subprocess.run([sys.executable, "-m", "pytest", "-q", tdir], capture_output=True, text=True,
                           env={**os.environ, "PYTHONPATH": SCRATCH_DIR, "MAASS_SCRIPT": maass_script_path(), "AUTHORS_REPO_DIR": REPO_DIR})
        out["1a"] = {"pass": r.returncode == 0, "tail": r.stdout[-2000:]}
    except ImportError:
        out["1a"] = {"pass": None, "note": "pytest not installed: level 1a was run by the build, not on this machine"}
    log(f"  1a: {out['1a']}")
    lx, ly = x_liquid(s0, 1), y_liquid(s0, LC.N_SUB_REFERENCE)
    gx, inx, gy, iny = 1.0, x_scale(lx), ly.g_maass, ly.in_scale_fixed
    out["1b"] = {"X": L1.level1b(lx, beats10, gx, inx, s0, log=log), "Y": L1.level1b(ly, beats10, gy, iny, s0, log=log)}
    out["1c"] = {"X": L1.level1c(lx, beats10, gx, inx, s0, DEVICE, log=log), "Y": L1.level1c(ly, beats10, gy, iny, s0, DEVICE, log=log)}
    out["1d"] = L1.level1d(S1_CTX, x_liquid, y_liquid, lambda l: 1.0, x_scale, lambda l: l.g_maass,
                           lambda l: l.in_scale_fixed, SEARCH.sets, LC.SEEDS_SEARCH, log=log)
    out["1e"] = L1.level1e(S1_CTX, maass_script_path(), seeds=list(range(1, LC.E1E_RUNS + 1)) if not SMOKE_TEST else [1, 2],
                           n_pairs=200 if not SMOKE_TEST else 3, processes=LEVEL1E_PROCESSES, log=log, d_replacement=D_REPLACEMENT)
    out["1f"] = stage1_level1f()
    gate = {k: (out[k]["X"]["pass"] and out[k]["Y"]["pass"]) if k in ("1b", "1c") else out[k]["pass"] for k in ("1b", "1c", "1d", "1e")}
    out["gate"] = gate
    out["pass"] = all(gate.values()) and out["1a"]["pass"] is not False
    return out

def stage1_level1f():
    """Cached X runs vs the new simulator in X mode: identical window counts on the saved DS2 state vectors of
    config 11 (the replication saved them for the fixed liquid, seeds 2026 / 69 / 420)."""
    D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
    X1n, X2n, _ = normalize_final(D1["X"], D2["X"], "robust")
    ain = measure_ain(X1n, np.arange(len(X1n)), "multi")
    rows = []
    for seed in SEEDS:
        spec = lsm_spec("fixed", seed, "morph", "robust", "multi", "svm", "final")
        p = os.path.join(WORK_DIR, "states", f"{run_id_of(spec)}__DS2_window_spike_counts.npz")
        if not os.path.exists(p):
            rows.append({"seed": seed, "available": False}); continue
        saved = np.load(p)["counts"]
        model = build_liquid(200, I_TARGET / (ain * LIQ["P_in"]), seed, True)
        liq = LL.from_authors_model(model)
        n = len(X2n) if not SMOKE_TEST else 64
        sm = LS.Simulator(liq, device=DEVICE, dtype=torch.float32, g=1.0, in_scale=1.0)
        new = np.zeros((n, NWIN * N_NEUR), np.uint8)
        for s in range(0, n, NEW_SIM_BATCH):
            x = encode_t(X2n[s:s + NEW_SIM_BATCH], "multi")
            o = sm.run(x, np.zeros((len(x), liq.N)), LIQ["num_sim_steps"], rec={"windows": (WIN, NWIN)})
            new[s:s + len(x)] = o["windows"].reshape(len(x), -1).to(torch.uint8).cpu().numpy()
        diff = int((new != saved[:n]).sum())
        rows.append({"seed": seed, "available": True, "cells_differing": diff, "identical": diff == 0, "beats": n})
        log(f"  1f seed {seed}: {diff} window counts differ")
    ok = bool(rows) and all(r.get("identical") for r in rows)
    return {"pass": ok, "rows": rows, "note": "1f never stops the plan: it decides whether cached runs are reused"}

# ---------------------------------------------------------------- step 2: the Maass check
def stage1_maass_check():
    data = LDa.make_maass_data(seed=1)
    return LMC.maass_check(S1_CTX, data, seeds=LC.SEEDS_SEARCH, log=log)

# ---------------------------------------------------------------- step 3: the ladder and the confirmation
def stage1_ladder(n_fine, maass_target):
    cache_dir = os.path.join(S1_DIR, "candidates"); os.makedirs(cache_dir, exist_ok=True)
    cache = {}
    for fn in glob.glob(os.path.join(cache_dir, "*.json")):
        d = json.load(open(fn)); cache[d["_key"]] = d["record"]
    def on_record(k, rec):
        jdump({"_key": k, "record": rec}, os.path.join(cache_dir, re.sub(r"[^A-Za-z0-9.+_-]", "_", k) + ".json"))
    lad = LLd.Ladder(S1_CTX, SEARCH, CONFIRM, n_fine, maass_target, seeds=LC.SEEDS_SEARCH, log=log,
                     on_record=on_record, cache=cache)
    out = lad.run()
    out["records"] = lad.records
    if out.get("final") is not None:          # 1b and 1c again on the final liquid's exact combination of factors
        f = frozenset(out["final"]); s0 = LC.SEEDS_SEARCH[0]
        lf = LL.build_candidate(f, s0, out["n_sub"], 200)
        g = lf.g_maass if 10 in f else out["g"]
        beats = SEARCH.sets["sep"][:10 if not SMOKE_TEST else 2]
        out["final_1b"] = L1.level1b(lf, beats, g, out["in_scale"], s0, log=log)
        out["final_1c"] = L1.level1c(lf, beats, g, out["in_scale"], s0, DEVICE, log=log)
    return out

def ladder_figures(records):
    for k, rec in records.items():
        if "sweep" not in rec:
            continue
        t = rec["sweep"]["table"]
        gs = sorted(float(g) for g in t if float(g) > 0)
        fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
        ab = [t[repr(g)]["abs_g"] for g in gs]
        for a, key, lab in zip(ax, ("rate_hz", "gap", "twin_ratio"), ("firing (Hz)", "separation - generalisation", "twin late/early")):
            a.plot(ab, [t[repr(g)].get(key, np.nan) for g in gs], "o-", color=SERIES[0])
            a.set_xscale("log"); a.set_xlabel("mean isolated EE push (thresholds)"); a.set_ylabel(lab)
            a.axvline(LC.MAASS_EE_PUSH, color="#999", ls="--", lw=1)
            if rec.get("g_rule") is not None:
                a.axvline(t[repr(float(rec["g_rule"]))]["abs_g"], color=SERIES[1], lw=1.5)
        ax[0].axhspan(*LC.BAND_HZ, color="#ddd", zorder=0)
        fig.suptitle(f"candidate {rec['key']} (n = {rec['n_sub']}): chosen g in orange, Maass's scale dashed")
        fig.tight_layout(); fig.savefig(os.path.join(WORK_DIR, "figures", f"ladder_{re.sub(r'[^A-Za-z0-9+]', '_', k)}.png"), dpi=110)
        plt.close(fig)

def ladder_table(records):
    rows = []
    for k, r in records.items():
        rows.append({"record": k, "candidate": r["key"], "n_sub": r["n_sub"], "data": r["data"], "pass": r["pass"],
                     "first_failed": r["first_failed_level"], "g_rule": r["g_rule"],
                     "abs_g_used": np.mean([v for v in r["abs_g_used"].values() if v is not None]) if any(v is not None for v in r["abs_g_used"].values()) else None,
                     "band_limited": r.get("band_limited"), "input_rule": r["input"]["rule"], "in_scale": r["in_scale"],
                     "level2": r["level2"]["pass"], "level3": r["level3"]["pass"], "level4": r["level4"]["pass"],
                     "rate_hz": np.mean([x["rate_hz"] for x in r["level3"].get("rows", [])]) if r["level3"].get("rows") else None,
                     "memory_gain_100ms": np.mean(list(r["level4"]["gain_100ms"].values())) if "gain_100ms" in r["level4"] else None})
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(WORK_DIR, "tables", "ladder_candidates.csv"), index=False)
    return df

# ---------------------------------------------------------------- input sizes for every front end (stage 2)
def front_end_scales(final, n_sub, maass_target, config11_scale):
    """Input size per front end, measured once on the search half (plan, Input and calibration)."""
    out = {}
    f = frozenset(final)
    for filt in ("morph", "median"):
        for norm in ("standard", "robust"):
            for enc in ("multi", "rate", "latency"):
                key = f"{filt}-{norm}-{enc}"
                if (filt, norm, enc) == ("morph", "robust", "multi"):
                    out[key] = {"scale": config11_scale, "rule": "frozen by the ladder (config 11)"}; continue
                Xn, _ = half_beats(S1_SEARCH_RECS, filt, norm)
                n_ch = 200 if enc == "multi" else N_IN[enc]
                liqs = [LL.build_candidate(f, s, n_sub, n_ch) for s in LC.SEEDS_SEARCH]
                for s, l in zip(LC.SEEDS_SEARCH, liqs):
                    l.meta["seed"] = s
                torch.manual_seed(12345)                      # the Rate encoder is stochastic
                pick = np.sort(np.random.default_rng(0).choice(len(Xn), min(LC.N_CAL_BEATS, len(Xn)), replace=False))
                cal = enc_np(Xn[pick], enc)
                ain = measure_ain(Xn, np.arange(len(Xn)), enc)
                sc, how = LCa.input_scale(S1_CTX, liqs, cal, ain, maass_target, log=log)
                out[key] = {"scale": sc, **how}
    return out

def front_end_levels(cal):
    """E2: each front end's four-level result with the final liquid at config 11's g and its own input size
    (reported, not used to re-tune; plan, Experiments)."""
    f = frozenset(cal["final"]); out = {}
    lad = LLd.Ladder(S1_CTX, SEARCH, CONFIRM, cal["n_sub"], cal["factor5_target_hz"], seeds=LC.SEEDS_SEARCH, log=log)
    lad.n_sub_for = lambda factors, y_grid=None: cal["n_sub"]
    for filt in ("morph", "median"):
        for norm in ("standard", "robust"):
            for enc in ("multi", "rate", "latency"):
                key = f"{filt}-{norm}-{enc}"
                Xn, _ = half_beats(S1_SEARCH_RECS, filt, norm)
                torch.manual_seed(12345)
                data = SEARCH if key == "morph-robust-multi" else LDa.make_search_data(key, Xn, lambda X, e=enc: enc_np(X, e), seed=1)
                r = lad.evaluate(f, data=data, frozen={"g": cal["g"], "in_scale": cal["input_scales"][key]["scale"]},
                                 n_channels=200 if enc == "multi" else N_IN[enc], tag=f"frontend|{key}")
                out[key] = {"pass": r["pass"], "first_failed_level": r["first_failed_level"],
                            "rate_hz": [x["rate_hz"] for x in r["level3"].get("rows", [])],
                            "memory_gain_100ms": r["level4"].get("gain_100ms")}
                log(f"  front end {key}: four levels {'pass' if r['pass'] else 'fail at ' + str(r['first_failed_level'])} (reported only)")
    return out

# ---------------------------------------------------------------- step 4: speed
def stage1_speed(cal):
    f = frozenset(cal["final"]); liq = LL.build_candidate(f, SEEDS[0], cal["n_sub"], 200)
    g = liq.g_maass if 10 in f else cal["g"]
    n = 2 * NEW_SIM_BATCH if not SMOKE_TEST else 32
    x = SEARCH.sets["sep"][:n] if n <= len(SEARCH.sets["sep"]) else enc_np(_beats_s[:n])
    t0 = time.time()
    LMe.simulate(S1_CTX, liq, g, cal["input_scales"]["morph-robust-multi"]["scale"], x, np.zeros((len(x), liq.N)) + liq.start_lo,
                 LIQ["num_sim_steps"], {"windows": (WIN, NWIN), "states_at": list(LC.SNAPSHOT_SAMPLES)})
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    bps = len(x) / (time.time() - t0)
    return {"beats_per_s": bps, "device": ENV.get("gpu", str(DEVICE)), "machine": MACHINE_ID,
            "note": "use this as this machine's weight in UNIT_SHARES"}

# ---------------------------------------------------------------- run stage 1
STAGE1 = {}
_1c_path = os.path.join(WORK_DIR, "checks", f"level1c_{out_tag('card', MACHINE_ID)}.json")
if os.path.exists(CAL_PATH) and not os.path.exists(_1c_path):
    # every machine runs level 1c on its own card before its stage-2 units (plan, Stages)
    cal = json.load(open(CAL_PATH)); f = frozenset(cal["final"]); s0 = LC.SEEDS_SEARCH[0]
    res = {}
    for name, lq, g_, sc_ in (("X", x_liquid(s0, 1), 1.0, None), ("Y", y_liquid(s0, LC.N_SUB_REFERENCE), None, None),
                              ("final", LL.build_candidate(f, s0, cal["n_sub"], 200), cal["g"], cal["in_scale"])):
        g_ = lq.g_maass if (g_ is None or (name == "final" and 10 in f)) else g_
        sc_ = sc_ if sc_ is not None else (lq.in_scale_fixed or x_scale(lq))
        res[name] = L1.level1c(lq, SEARCH.sets["sep"][:10], g_, sc_, s0, DEVICE, log=log)
    res["pass"] = all(v["pass"] for v in res.values())
    jdump(res, _1c_path)
    if not res["pass"]:
        raise RuntimeError("level 1c failed on this card: stage 2 does not start here")
if RUN_STAGE1 and not os.path.exists(CAL_PATH):
    if "level1" in STAGE1_STEPS:
        if not s1_done("level1"):
            log("stage 1, step 1: level 1 checks"); s1_save("level1", stage1_level1())
        STAGE1["level1"] = s1_load("level1")
        s1_gate(STAGE1["level1"]["pass"], f"level 1 failed {STAGE1['level1']['gate']}: fix the bug before anything else "
                f"runs (details in {s1_path('level1')}; delete it to rerun)")
    if "maass_check" in STAGE1_STEPS:
        if not s1_done("maass_check"):
            log("stage 1, step 2: testing the test on Maass's liquid"); s1_save("maass_check", stage1_maass_check())
        STAGE1["maass_check"] = s1_load("maass_check")
        s1_gate(STAGE1["maass_check"]["pass"], "the Maass check failed: the test must be redesigned (plan) and rechecked "
                f"on seeds {LC.SEEDS_TEST_REDESIGN}. Details in {s1_path('maass_check')}")
    if "ladder" in STAGE1_STEPS:
        if not s1_done("ladder"):
            log("stage 1, step 3: the X to Y ladder and the confirmation")
            s1_save("ladder", stage1_ladder(STAGE1["level1"]["1d"]["n"], STAGE1["maass_check"]["factor5_target_hz"]))
        STAGE1["ladder"] = s1_load("ladder")
        ladder_figures(STAGE1["ladder"]["records"]); print(ladder_table(STAGE1["ladder"]["records"]).to_string(index=False))
        if "final_1b" in STAGE1["ladder"]:                     # 1b and 1c on the final liquid's own code paths
            s1_gate(STAGE1["ladder"]["final_1b"]["pass"] and STAGE1["ladder"]["final_1c"]["pass"],
                    "level 1b or 1c failed on the final liquid's combination of factors: fix the bug before E1 to E3")
        if STAGE1["ladder"].get("final") is None and SMOKE_TEST:
            log("SMOKE TEST: no candidate passed; using Y as the final liquid to exercise stage 2")
            yk = [k for k in STAGE1["ladder"]["records"] if k.startswith("+".join(map(str, range(1, 12))) + "|")][0]
            yr = STAGE1["ladder"]["records"][yk]
            STAGE1["ladder"].update(final=list(range(1, 12)), final_key=yr["key"], n_sub=yr["n_sub"], g=yr["g_rule"] or 1.0,
                                    maass_scale=True, in_scale=yr["in_scale"], attempts=0)
        if STAGE1["ladder"].get("final") is None:
            raise RuntimeError(f"the ladder stopped: {STAGE1['ladder']['stop']}. Stop before E1 to E3 and report (decision 12).")
    if "speed" in STAGE1_STEPS:
        L = STAGE1["ladder"]
        cal = {"final": L["final"], "final_key": L["final_key"], "n_sub": L["n_sub"], "g": L["g"],
               "maass_scale": L["maass_scale"], "in_scale": L["in_scale"], "attempts": L["attempts"],
               "factor5_target_hz": STAGE1["maass_check"]["factor5_target_hz"],
               "level1f_pass": STAGE1["level1"]["1f"]["pass"], "d_replacement": D_REPLACEMENT, "x_update": X_UPDATE,
               "search_records": S1_SEARCH_RECS, "confirm_records": S1_CONFIRM_RECS,
               "made_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"), "machine": MACHINE_ID,
               "notebook_build": NOTEBOOK_BUILD, "smoke_test": SMOKE_TEST}
        cal["input_scales"] = front_end_scales(cal["final"], cal["n_sub"], cal["factor5_target_hz"], cal["in_scale"])
        if not s1_done("front_end_levels"):
            s1_save("front_end_levels", front_end_levels(cal))
        cal["front_end_levels"] = s1_load("front_end_levels")
        cal["speed"] = stage1_speed(cal)
        cal["hash"] = stable_hash({k: cal[k] for k in ("final", "n_sub", "g", "maass_scale", "input_scales", "d_replacement", "x_update")}, 10)
        jdump(cal, CAL_PATH)
        log(f"stage 1 done: final liquid {cal['final_key']} (n = {cal['n_sub']}, g {cal['g']}, Maass scale {cal['maass_scale']}); "
            f"calibration written to {CAL_PATH}; {cal['speed']['beats_per_s']:.0f} beats/s here")
CAL = json.load(open(CAL_PATH)) if os.path.exists(CAL_PATH) else None
if CAL is not None and CAL.get("x_update", "equations") != X_UPDATE:
    raise RuntimeError(f"X_UPDATE = {X_UPDATE!r} but the calibration file was made with {CAL.get('x_update')!r}")
if CAL is None:
    print("no calibration file yet: stage 2 has nothing to run")


## 11. Executors of the replication (incl. the hybrid rule)

In [ ]:
# Executors: one function per job kind. Each saves its runs and updates cache.txt as it goes.
# With PIPELINE=True the GPU-side work (simulation, encoding, NN training) runs in the main thread, one job at a time,
# and every classifier fit becomes a task for the FIT_POOL: one fit per CPU core, in parallel with the next simulation.
def _avail_ram_gb():
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemAvailable"):
                return int(line.split()[1]) / 1e6
    except Exception:
        pass
    return 8.0

def _proc_ram_gb():
    """(current, peak) resident memory of this process."""
    out = {"VmRSS": 0.0, "VmHWM": 0.0}
    try:
        for line in open("/proc/self/status"):
            k = line.split(":")[0]
            if k in out:
                out[k] = int(line.split()[1]) / 1e6
    except Exception:
        pass
    return out["VmRSS"], out["VmHWM"]

def fit_n_jobs(threads):
    """Cores per classifier fit: spare cores go to the fit's 3 one-vs-rest problems (results unchanged)."""
    return max(1, min(3, READOUT_JOBS // max(threads, 1)))

def fit_ram_gb(nfeat, ntr=4096):
    """Working memory of one fit on the batch-by-batch path (batches, their float64 temporaries, the curve subset)."""
    return min(4096, ntr) * nfeat * 4 * 6 / 1e9 + min(4000, ntr) * nfeat * 4 / 1e9 + 0.3

def _readout_plan(n_runs, ntr, nev, nfeat):
    """Older runner: threads and whether to pre-scale: as many parallel fits as cores allow; pre-scale only if RAM fits."""
    avail = _avail_ram_gb() - 2.5
    per_batch = 4096 * nfeat * 4 * 6 / 1e9 + 0.3
    per_pre = ntr * nfeat * 4 * 1.15 / 1e9 + per_batch
    available = max(1, min(READOUT_JOBS, int(avail // max(per_batch, 0.05))))
    update_speed("fit_threads", float(available))      # parallel fits the machine allows (for the projection)
    threads = min(available, n_runs)
    return threads, threads * per_pre < avail

def _split_rows(split, n1):
    """train rows, eval rows (into DS1 or DS2), readout rng seed (authors' formulas)."""
    if split == "final":
        return np.arange(n1), None, 2026
    rep, fold = parse_split(split)
    f = FOLD_BY[(rep, fold)]
    return f["train_idx"], f["val_idx"], 2026 + rep * 10 + fold

def _finish_run(e, metrics, eval_idx, eval_pid, y_true, y_pred, scores, model_obj, curve, liq, timing, extra=None):
    save_run(e["rid"], e["spec"], e["group"], metrics, eval_idx, eval_pid, y_true, y_pred, scores, model_obj,
             curve, liq, timing, {"groups": e["groups"], "config_id": config_of(e["spec"]), "share": SHARE,
                                  "machine_id": MACHINE_ID, "notebook_build": NOTEBOOK_BUILD, **(extra or {})})
    CACHE.mark_done(e["rid"], timing.get("total_seconds", 0.0), metrics, e["group"])
    log(f"  done {e['rid'][:90]:<90} mF1={metrics['macro_f1']:.4f} acc={metrics['accuracy']:.4f} "
        f"Se S/V={metrics['recall_SVEB']:.3f}/{metrics['recall_VEB']:.3f}")

def _record_failure(rid, tb):
    CACHE.mark_failed(rid, tb.strip().splitlines()[-1])
    os.makedirs(os.path.join(WORK_DIR, "logs", "failures"), exist_ok=True)
    atomic_write_text(os.path.join(WORK_DIR, "logs", "failures", rid + ".txt"), tb)
    log("  FAILED", rid, tb.strip().splitlines()[-1])

# ---------------- pipeline: RAM accounting and the fit-worker pool ----------------
class RamBudget:
    """Bookkeeping of RAM promised to jobs (their spike counts) and fits (their working memory)."""
    def __init__(self, total_gb):
        self.total, self.used, self.cv = total_gb, 0.0, threading.Condition()
    def reserve(self, gb, headroom=0.0, block=True):
        """Waits until gb (+ headroom kept free for fits) fits in the budget; something bigger than the whole budget
        waits until nothing else holds RAM."""
        t0, warned = time.time(), 0
        with self.cv:
            while self.used > 1e-6 and self.used + gb + headroom > self.total:
                if not block:
                    return False
                self.cv.wait(2.0)
                if time.time() - t0 > 300 * (warned + 1):
                    warned += 1
                    log(f"  waiting for RAM for {(time.time() - t0) / 60:.0f} min: need {gb:.2f}+{headroom:.2f} GB, "
                        f"{self.used:.2f} of {self.total:.2f} GB promised, {_avail_ram_gb():.1f} GB actually available")
            self.used += gb
            return True
    def release(self, gb):
        with self.cv:
            self.used -= gb
            if self.used < 1e-6:
                self.used = 0.0              # no floating-point leftovers
            self.cv.notify_all()

class SharedPrep:
    """Scaler (and, if RAM allows, the scaled training matrix) shared by the fits of one job on the same rows."""
    def __init__(self):
        self.lock, self.items, self.extra_gb = threading.Lock(), {}, 0.0
    def get(self, key, Xtr_fn, ntr, nfeat, ram):
        with self.lock:
            if key not in self.items:
                xs_gb = ntr * nfeat * 4 * 1.05 / 1e9
                pre = ram.reserve(xs_gb, block=False)
                if pre:
                    self.extra_gb += xs_gb
                self.items[key] = prepare_train(Xtr_fn, ntr, pre)
            return self.items[key]

class FitPool:
    def __init__(self, n_workers, ram):
        self.n, self.ram = n_workers, ram
        self.ex = ThreadPoolExecutor(n_workers, thread_name_prefix="fit")
        self.lock, self.cancel = threading.Lock(), threading.Event()
        self.queued = self.running = 0
        self.backlog_s = 0.0                 # estimated seconds of fits not yet finished
        self.jobs, self.next_id, self.futs = {}, 0, set()
        self.n_done = self.n_failed = self.n_cancelled = 0
    def submit_job(self, tasks, job_ram_gb, shared):
        with self.lock:
            jid = self.next_id; self.next_id += 1
            self.jobs[jid] = {"left": len(tasks), "ram": job_ram_gb, "shared": shared}
            for t in tasks:
                self.queued += 1; self.backlog_s += t["est_s"]
                fu = self.ex.submit(self._run, jid, t)
                self.futs.add(fu); fu.add_done_callback(self.futs.discard)
    def _job_task_done(self, jid):
        with self.lock:
            j = self.jobs[jid]; j["left"] -= 1
            if j["left"] == 0:
                self.ram.release(j["ram"] + j["shared"].extra_gb)
                del self.jobs[jid]            # drops the job's spike counts and shared matrices
    def _run(self, jid, t):
        with self.lock:
            self.queued -= 1
        reserved = started = False
        try:
            if self.cancel.is_set():
                raise Cancelled()
            self.ram.reserve(t["ram_gb"]); reserved = True
            with self.lock:
                self.running += 1; busy = self.running
            started = True
            t0 = time.time()
            res = t["fit"](self.cancel, self.jobs[jid]["shared"])
            dt = time.time() - t0
            with self.lock:
                busy = min(busy, self.running); self.running -= 1
            started = False
            if t.get("gcells") and busy >= max(1, 0.75 * self.n):   # speed under full load (SGD fits only)
                update_speed("fit_s_per_Gcell", dt / t["gcells"], sub="pipe", robust=True)
            t["record"](res, dt, busy)
            with self.lock:
                self.n_done += 1
        except Cancelled:
            with self.lock:
                self.n_cancelled += 1
            log(f"  cancelled (time is up; stays pending): {t['rid'][:90]}")
        except Exception:
            with self.lock:
                self.n_failed += 1
            _record_failure(t["rid"], traceback.format_exc())
        finally:
            with self.lock:
                if started:
                    self.running -= 1
                self.backlog_s -= t["est_s"]
            if reserved:
                self.ram.release(t["ram_gb"])
            self._job_task_done(jid)
    def outstanding(self):
        with self.lock:
            return self.queued + self.running
    def wait_any(self, timeout=5.0):
        futs = list(self.futs)
        if futs:
            concurrent.futures.wait(futs, timeout=timeout, return_when=concurrent.futures.FIRST_COMPLETED)

FIT_POOL = None          # created by the runner when PIPELINE=True
_JOB_CTX = {}            # the job the main thread is executing: its RAM reservation is handed to the pool

def _run_readouts(runs, feat_tr, feat_ev, ytr_all, D_eval, final, liq, base_timing, ntr_full):
    """feat_tr(rows)->float32 train features; feat_ev(rows)->float32 eval features (DS2 rows if final, else DS1)."""
    D_TR_RR = DATA["morph"]["DS1"]["rr"]
    D_EV_RR = D_eval["rr"]
    nfeat0 = len(feat_tr(np.arange(1)).ravel())
    LATE = slice((NWIN - 1) * N_NEUR, NWIN * N_NEUR)
    late_of = lambda e: e["spec"].get("readout") == "svm_late"
    nfeat_of = lambda e: (N_NEUR if late_of(e) else nfeat0) + (len(RR_NAMES) if e["spec"].get("readout") == "svm_rr" else 0)
    def parts(e):
        tr, ev, rng_seed = _split_rows(e["spec"]["split"], ntr_full)
        if final:
            ev = np.arange(len(D_eval["y"]))
        add_rr = e["spec"].get("readout") == "svm_rr"
        if late_of(e):
            ftr = lambda r: feat_tr(tr[r])[:, LATE]; fev = lambda r: feat_ev(ev[r])[:, LATE]
        else:
            ftr = (lambda r: np.hstack([feat_tr(tr[r]), D_TR_RR[tr[r]]])) if add_rr else (lambda r: feat_tr(tr[r]))
            fev = (lambda r: np.hstack([feat_ev(ev[r]), D_EV_RR[ev[r]]])) if add_rr else (lambda r: feat_ev(ev[r]))
        return tr, ev, rng_seed, add_rr, ftr, fev
    def one(e, nj, prescale=False, cancel=None, shared=None):
        tr, ev, rng_seed, add_rr, ftr, fev = parts(e)
        yev = D_eval["y"][ev]
        loss = "log_loss" if e["spec"].get("readout") == "lr" else "hinge"
        nfeat = nfeat_of(e)
        prep = shared.get((e["spec"]["split"], add_rr, late_of(e)), ftr, len(tr), nfeat, FIT_POOL.ram) if shared is not None else None
        scaler, clf, yp, sc, curve = fit_readout(ftr, len(tr), ytr_all[tr], fev, len(ev), loss, rng_seed,
                                                 yev_for_curve=None if final else yev, prescale=prescale, n_jobs=nj,
                                                 prepared=prep, cancel=cancel)
        pre = prep[1] is not None if prep is not None else prescale
        return e, compute_metrics(yev, yp), ev, D_eval["pid"][ev], yev, yp, sc, {"scaler": scaler, "classifier": clf}, curve, pre
    def record(res, dt, threads, nj):
        e, m, ev, pid, yev, yp, sc, mobj, curve, pre = res
        _finish_run(e, m, ev, pid, yev, yp, sc, mobj, curve, liq,
                    {**base_timing, "readout_seconds": dt, "total_seconds": base_timing.get("shared_seconds", 0) + dt,
                     "readout_threads": threads, "readout_cores_per_fit": nj, "prescaled": pre,
                     "runner": "pipeline" if FIT_POOL is not None else "sequential"})
    if FIT_POOL is not None:
        tasks = []
        for e in runs:
            tr = _split_rows(e["spec"]["split"], ntr_full)[0]
            nfeat = nfeat_of(e)
            g = len(tr) * nfeat / 1e9
            tasks.append({"rid": e["rid"], "gcells": g, "est_s": g * SPEED["fit_s_per_Gcell"]["pipe"], "ram_gb": fit_ram_gb(nfeat, len(tr)),
                          "fit": (lambda cancel, shared, e=e: one(e, 1, cancel=cancel, shared=shared)),
                          "record": (lambda res, dt, busy: record(res, dt, busy, 1))})
        FIT_POOL.submit_job(tasks, _JOB_CTX.get("ram_gb", 0.0), SharedPrep())
        _JOB_CTX["submitted"] = True
        return
    # older runner: this job's fits in parallel threads, sharing the cores
    n_ev = len(D_eval["y"]) if final else int(np.mean([len(_split_rows(e["spec"]["split"], ntr_full)[1]) for e in runs]))
    n_tr = ntr_full if final else int(ntr_full * 0.76)
    nfeat = nfeat0 + (len(RR_NAMES) if any(e["spec"].get("readout") == "svm_rr" for e in runs) else 0)
    threads, prescale = _readout_plan(len(runs), n_tr, n_ev, nfeat)
    nj = fit_n_jobs(threads)
    def timed(e):
        t0 = time.time(); r = one(e, nj, prescale); dt = time.time() - t0
        tr = _split_rows(e["spec"]["split"], ntr_full)[0]
        update_speed("fit_s_per_Gcell", dt / (len(tr) * nfeat / 1e9), sub=str(nj))
        return r, dt
    with ThreadPoolExecutor(threads) as ex:
        futs = [ex.submit(timed, e) for e in runs]
        for fu, e in zip(futs, runs):
            try:
                res, dt = fu.result()
                record(res, dt, threads, nj)
            except Exception:
                _record_failure(e["rid"], traceback.format_exc())

def _maybe_save_states(runs, counts_test, D2):
    for e in runs:
        s = e["spec"]
        if s["split"] != "final":
            continue
        cid = config_of(s)
        for want in SAVE_STATES:
            if cid == want["config_id"] and s["mode"] == want["mode"] and s["seed"] == want["seed"]:
                p = os.path.join(WORK_DIR, "states", f"{e['rid']}__DS2_window_spike_counts.npz")
                if not os.path.exists(p):
                    np.savez_compressed(p, counts=counts_test, y=D2["y"], patient_id=D2["pid"].astype("U4"),
                                        layout="(beat, window*343 + neuron) spike counts per 20-step window; rate = count/20")
                    disk_add(os.path.getsize(p))
                    log(f"  saved state vectors -> {os.path.basename(p)} ({os.path.getsize(p)/1e6:.0f} MB)")

def execute_lsm(job, runs):
    s0 = runs[0]["spec"]; nk = job["key"][2]; final = nk == "final"
    enc, mode, K = s0["encoder"], s0["mode"], s0["num_levels"] or 200
    D1, D2 = DATA[s0["filter"]]["DS1"], DATA[s0["filter"]]["DS2"]
    t0 = time.time()
    if final:
        X1n, X2n, normp = normalize_final(D1["X"], D2["X"], s0["norm"])
        tr_rows = np.arange(len(D1["y"]))
    elif nk == "cv-shared":
        X1n, normp = normalize_cv(D1["X"], None, "standard"); X2n = None
        tr_rows = np.arange(len(D1["y"]))
    else:
        f = FOLD_BY[parse_split(nk)]
        X1n, normp = normalize_cv(D1["X"], f["train_idx"], s0["norm"]); X2n = None
        tr_rows = f["train_idx"]
    rr1 = rr2 = None
    if s0["rr_in_liquid"]:           # 8 RR columns (30 channels), or 9 with the mean-RR ratio (40 channels)
        mean_set = s0.get("rr_set") == "mean"
        rr1 = rr_inputs(D1, np.arange(len(D1["y"])), mean_set); rr2 = rr_inputs(D2, np.arange(len(D2["y"])), mean_set)
    rr_thr = rr_thresholds(rr1[tr_rows]) if rr1 is not None else None
    if s0["ain_override"] is not None:
        ain, src = float(s0["ain_override"]), "override"
    elif MODES[mode]["ain"] == "paper":
        ain, src = PAPER_AIN[enc], "paper (hard-coded)"
    else:
        ain = measure_ain(X1n, tr_rows, enc, K, rr=rr1, rr_thr=rr_thr)
        src = "measured on training rows"
    mu_in = I_TARGET / (ain * LIQ["P_in"]) * s0["gain"]
    n_in = (K if enc == "multi" else N_IN[enc]) + (len(rr_thr) * RR_LEVELS if rr_thr is not None else 0)
    model = build_liquid(n_in, mu_in, s0["seed"], MODES[mode]["gamma_fix"], s0["lambda_val"], s0["recurrence"], s0.get("g_rec"))
    res_meta = {"mode": mode, "seed": s0["seed"], "input_size": n_in, "mu_in": repr(mu_in),
                "lambda_val": s0["lambda_val"], "recurrence": s0["recurrence"],
                "gamma_fix": MODES[mode]["gamma_fix"], "torch": torch.__version__, "device": DEVICE.type}
    if s0.get("g_rec") is not None:
        res_meta["g_rec"] = s0["g_rec"]
    res_id = save_reservoir(model, res_meta)
    enc_seed = int(stable_hash(job["key"], 6), 16)
    batch = SIM_BATCH_FINAL if final else SIM_BATCH_CV
    C1, st1 = simulate(model, X1n, enc, batch, K, rr=rr1, rr_thr=rr_thr, y=D1["y"], enc_seed=enc_seed)
    st2 = None
    if final:
        C2, st2 = simulate(model, X2n, enc, batch, K, rr=rr2, rr_thr=rr_thr, y=D2["y"], enc_seed=enc_seed + 1)
    n_sim = st1["n_beats"] + (st2["n_beats"] if st2 else 0)
    sim_s = st1["encode_seconds"] + st1["simulate_seconds"] + ((st2["encode_seconds"] + st2["simulate_seconds"]) if st2 else 0)
    update_speed("sim_beats_per_s", n_sim / max(sim_s, 1e-6), sub=enc)
    enc_s = st1["encode_seconds"] + (st2["encode_seconds"] if st2 else 0)
    update_speed("encode_beats_per_s", n_sim / max(enc_s, 1e-6), sub=enc)
    liq = {"A_in_used": ain, "A_in_source": src, "A_in_paper": PAPER_AIN[enc], "mu_in": mu_in, "I_target": I_TARGET,
           "normalization": normp, "reservoir_id": res_id, "structure": structure_stats(model),
           "DS1_simulation": st1, "DS2_simulation": st2}
    shared = time.time() - t0
    update_speed("lsm_job_overhead_s", max(0.0, shared - sim_s) + 1.0)   # + saving / bookkeeping between jobs
    log(f"  liquid: A_in={ain:.4f} ({src}) mu_in={mu_in:.4f} firing={st1['liquid_firing_prob_per_step']:.4f} "
        f"drive/target={st1['input_current_over_target']:.2f} silent={st1['silent_neurons']} sim={sim_s:.0f}s")
    if final:
        _maybe_save_states(runs, C2, D2)
    feat_tr = lambda rows: counts_to_rates(C1[rows])
    feat_ev = (lambda rows: counts_to_rates(C2[rows])) if final else feat_tr
    _run_readouts(runs, feat_tr, feat_ev, D1["y"], D2 if final else D1, final, liq,
                  {"shared_seconds": shared, "simulation_seconds": sim_s, "beats_simulated": n_sim}, len(D1["y"]))
    # (the spike counts stay alive through the fit closures until the last fit of this job finishes)

def _front_end(split):
    D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
    if split == "final":
        X1n, X2n, normp = normalize_final(D1["X"], D2["X"], "robust")
        return D1, D2, np.asarray(X1n, np.float32), np.asarray(X2n, np.float32), np.arange(len(D1["y"])), normp
    f = FOLD_BY[parse_split(split)]
    X1n, normp = normalize_cv(D1["X"], f["train_idx"], "robust")
    return D1, D2, X1n, None, f["train_idx"], normp

def rr_inputs(D, rows, mean_ratio=False):
    """The 8 RR features, plus pre-RR / recording mean RR as a 9th when mean_ratio."""
    return np.hstack([D["rr"][rows], D["rr_mean_ratio"][rows, None]]) if mean_ratio else D["rr"][rows]

def execute_feat(job, runs):
    s0 = runs[0]["spec"]; split = job["key"][2]; final = split == "final"
    t0 = time.time()
    D1, D2, X1n, X2n, tr_rows, normp = _front_end(split)
    model_name, enc = s0["model"], s0["encoder"]
    info = {"normalization": normp, "feature_model": model_name}
    if model_name in ("no_liquid", "elm", "elm_rr"):
        if s0["seed"] is not None:
            torch.manual_seed(int(stable_hash(job["key"], 6), 16))
        c1 = encoder_window_counts(X1n, enc); c2 = encoder_window_counts(X2n, enc) if final else None
        rr_thr = None
        if model_name == "elm_rr":       # the 30 RR channels of RR-in-liquid, counted per window, next to the encoder's
            rr_thr = rr_thresholds(D1["rr"][tr_rows])
            T_in = encode(X1n[:1], enc).shape[1]    # encoder output length, as the liquid sees it (after the counts above)
            c1 = np.concatenate([c1, rr_window_counts(D1["rr"], rr_thr, T_in)], axis=2)
            c2 = np.concatenate([c2, rr_window_counts(D2["rr"], rr_thr, T_in)], axis=2) if final else None
        if model_name == "no_liquid":
            rates = lambda c: (c.reshape(len(c), -1).astype(np.float64) / WIN).astype(np.float32)
            F1 = lambda r: rates(c1[r]); F2 = (lambda r: rates(c2[r])) if final else None
        else:
            # elm_rr: same A_in, mu_in, seed and input size as the fixed RR-in-liquid liquid, so the same input weights
            ain = measure_ain(X1n, tr_rows, enc, rr=D1["rr"] if rr_thr is not None else None, rr_thr=rr_thr)
            mu_in = I_TARGET / (ain * LIQ["P_in"])
            Wf = build_liquid(c1.shape[2], mu_in, s0["seed"], True).input_rsv_weights.detach().cpu().numpy().astype(np.float32)
            proj = lambda c: np.einsum("nwc,ck->nwk", c.astype(np.float32), Wf).reshape(len(c), -1)
            tot = np.zeros(NWIN * N_NEUR, np.float64)
            for i in range(0, len(tr_rows), 4096):
                tot += proj(c1[tr_rows[i:i + 4096]]).sum(0)
            mean = (tot / len(tr_rows)).astype(np.float32)
            F1 = lambda r: np.maximum(proj(c1[r]) - mean, 0)
            F2 = (lambda r: np.maximum(proj(c2[r]) - mean, 0)) if final else None
            info.update({"A_in_measured": ain, "mu_in": mu_in, "elm": "window input counts @ fixed-liquid W_in, centered on train mean, ReLU"})
    elif model_name == "raw_beat":
        F1 = lambda r: X1n[r]; F2 = (lambda r: X2n[r]) if final else None
    elif model_name == "rr_only":
        F1 = lambda r: D1["rr"][r]; F2 = (lambda r: D2["rr"][r]) if final else None
    elif model_name == "rr_only_mrr":
        F1 = lambda r: rr_inputs(D1, r, True); F2 = (lambda r: rr_inputs(D2, r, True)) if final else None
    elif model_name == "rr_shape_mrr":
        F1 = lambda r: np.hstack([shape_features(X1n[r]), rr_inputs(D1, r, True)])
        F2 = (lambda r: np.hstack([shape_features(X2n[r]), rr_inputs(D2, r, True)])) if final else None
    else:  # rr_shape and lda
        F1 = lambda r: np.hstack([shape_features(X1n[r]), D1["rr"][r]])
        F2 = (lambda r: np.hstack([shape_features(X2n[r]), D2["rr"][r]])) if final else None
    shared = time.time() - t0
    if model_name == "lda":
        for e in runs:
            t1 = time.time()
            tr, ev, _ = _split_rows(e["spec"]["split"], len(D1["y"]))
            ev = np.arange(len(D2["y"])) if final else ev
            Fev, Dev = (F2, D2) if final else (F1, D1)
            sc = StandardScaler().fit(F1(tr))
            lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(sc.transform(F1(tr)), D1["y"][tr])
            yp = lda.predict(sc.transform(Fev(ev))); scores = lda.decision_function(sc.transform(Fev(ev)))
            m = compute_metrics(Dev["y"][ev], yp)
            _finish_run(e, m, ev, Dev["pid"][ev], Dev["y"][ev], yp, scores, {"scaler": sc, "classifier": lda}, None, info,
                        {"shared_seconds": shared, "total_seconds": shared + time.time() - t1})
        return
    _run_readouts(runs, F1, F2 if final else F1, D1["y"],
                  D2 if final else D1, final, info, {"shared_seconds": shared}, len(D1["y"]))

def execute_nn(job, runs):
    for e in runs:
        s = e["spec"]; t0 = time.time()
        D1, D2, X1n, X2n, tr_rows, normp = _front_end(s["split"])
        tr, ev, _ = _split_rows(s["split"], len(D1["y"]))
        final = s["split"] == "final"
        Xev, Dev = (X2n, D2) if final else (X1n, D1)
        ev = np.arange(len(D2["y"])) if final else ev
        yp, logits, curve, state = train_nn(s["model"], X1n[tr], D1["y"][tr], Xev[ev],
                                            rr_tr=rr_inputs(D1, tr, s.get("rr_set") == "mean") if s["use_rr"] else None,
                                            rr_ev=rr_inputs(Dev, ev, s.get("rr_set") == "mean") if s["use_rr"] else None, seed=s["seed"],
                                            yev_for_curve=None if final else Dev["y"][ev],
                                            arch=s.get("arch"), epochs=s.get("epochs"))
        m = compute_metrics(Dev["y"][ev], yp)
        if not final:            # held-out Macro-F1 after every epoch: the tuning search picks its epoch count from these
            m["eval_macro_f1_by_epoch"] = {str(r["epoch"]): r["eval_macro_f1"] for r in curve}
        m["n_params"] = state["n_params"]
        _finish_run(e, m, ev, Dev["pid"][ev], Dev["y"][ev], yp, logits, state, curve, {"normalization": normp},
                    {"total_seconds": time.time() - t0, "epochs": s.get("epochs") or NN_EPOCHS})

def execute_check(job, runs):
    e = runs[0]; t0 = time.time()
    g = {}
    for mu in LIQ["mu_rsv"]:
        s_code = _TorchGamma(1.0, mu).sample((200000,)).mean().item()
        s_fix = _TorchGamma(1.0, 1.0 / mu).sample((200000,)).mean().item()
        g[f"intended_mean_{mu}"] = {"authors_code_Gamma(k,theta)": s_code, "fixed_Gamma(k,1/theta)": s_fix, "1/mu": 1 / mu}
    CHECKS["gamma_parameterisation"] = g
    rows = []
    for filt in ("morph", "median"):
        for norm in ("standard", "robust"):
            X1n, _, _ = normalize_final(DATA[filt]["DS1"]["X"], DATA[filt]["DS2"]["X"], norm)
            for enc in ENCS:
                a = measure_ain(X1n, np.arange(len(X1n)), enc)
                rows.append({"filter": filt, "norm": norm, "encoder": enc, "A_in_measured": a, "A_in_paper": PAPER_AIN[enc],
                             "measured_over_paper": a / PAPER_AIN[enc], "mu_in_published": I_TARGET / (PAPER_AIN[enc] * LIQ["P_in"]),
                             "mu_in_fixed": I_TARGET / (a * LIQ["P_in"])})
    ain_df = pd.DataFrame(rows)
    for d in ("tables", "checks"):       # checks/ is carried to later sessions and merges; tables/ is rebuilt
        ain_df.to_csv(os.path.join(WORK_DIR, d, "ain_measured.csv"), index=False)
    struct = []
    a11 = ain_df.query("filter=='morph' and norm=='robust' and encoder=='multi'").A_in_measured.iloc[0]
    for mode, mcfg in MODES.items():
        a = PAPER_AIN["multi"] if mcfg["ain"] == "paper" else a11
        mdl = build_liquid(200, I_TARGET / (a * LIQ["P_in"]), SEEDS[0], mcfg["gamma_fix"])
        struct.append({"mode": mode, "seed": SEEDS[0], "A_in": a, **structure_stats(mdl)})
    for d in ("tables", "checks"):
        pd.DataFrame(struct).to_csv(os.path.join(WORK_DIR, d, "liquid_structure.csv"), index=False)
    CHECKS["A_in_table"] = rows; CHECKS["liquid_structure"] = struct
    jdump({"checks": CHECKS, "env": ENV, "install": INSTALL_REPORT}, os.path.join(WORK_DIR, "checks", "checks.json"))
    print(ain_df.round(4).to_string(index=False))
    print(pd.DataFrame(struct)[["mode", "n_recurrent_synapses", "EE_mean_abs_weight", "IE_mean_abs_weight", "mean_nonzero_input_weight"]].to_string(index=False))
    CACHE.mark_done(e["rid"], time.time() - t0, None, e["group"])

def _torch_raster(model, S, dtype, use_lif):
    """Full spike raster (B, 650, 343). use_lif=True: the authors' snn.Leaky; False: the update rule written out."""
    Win = model.input_rsv_weights.to(dtype).cpu(); Wr = model.rsv_weights.to(dtype).cpu()
    beta = torch.where(model.is_excit.cpu(), LIQ["betaE"], LIQ["betaI"]).to(dtype)
    x = torch.from_numpy(S).to(dtype)
    B, Tin, C = x.shape
    spk = torch.zeros(B, N_NEUR, dtype=dtype); mem = torch.zeros(B, N_NEUR, dtype=dtype)
    lif = snntorch.Leaky(beta=beta, threshold=1.0, reset_mechanism="subtract", reset_delay=False) if use_lif else None
    out = torch.zeros(B, LIQ["num_sim_steps"], N_NEUR, dtype=torch.bool)
    for t in range(LIQ["num_sim_steps"]):
        I = ((x[:, t] if t < Tin else torch.zeros(B, C, dtype=dtype)) @ Win + spk @ Wr) * LIQ["dt"]
        if use_lif:
            spk, mem = lif(I, mem)
        else:
            rst = (mem > 1.0).to(dtype)
            u = beta.clamp(0, 1) * mem + I - rst * 1.0
            spk = (u > 1.0).to(dtype)
            mem = u - (spk - rst) * 1.0
        out[:, t] = spk > 0.5
    return out.numpy()

def _brian2_raster(model, S1):
    import brian2 as b2
    b2.prefs.codegen.target = "numpy"
    b2.BrianLogger.log_level_error()
    b2.start_scope()
    b2.defaultclock.dt = 1 * b2.ms
    Win = model.input_rsv_weights.double().cpu().numpy(); W = model.rsv_weights.double().cpu().numpy()
    G = b2.NeuronGroup(N_NEUR, "mem : 1\nspk : 1\nIacc : 1\nbeta : 1 (constant)", threshold="spk > 0.5",
                       reset="", namespace={})
    G.beta = np.where(model.is_excit.cpu().numpy(), LIQ["betaE"], LIQ["betaI"])
    # Brian2 step k = model step k-1: input spikes of model step t are sent at t ms and summed into Iacc,
    # which the update at step t+1 consumes; recurrent spikes arrive one model step later, as in the authors' loop.
    G.run_regularly("rst = int(mem > 1.0)\n"
                    "u = beta*mem + Iacc*1.0 - rst*1.0\n"
                    "spk = int(u > 1.0)\n"
                    "mem = u - (spk - rst)*1.0\n"
                    "Iacc = 0", when="start")
    tt, cc = np.nonzero(S1)
    inp = b2.SpikeGeneratorGroup(S1.shape[1], cc, tt * b2.ms)
    pi, pj = np.nonzero(Win)
    Sin = b2.Synapses(inp, G, "w : 1", on_pre="Iacc_post += w"); Sin.connect(i=pi, j=pj); Sin.w = Win[pi, pj]
    objs = [G, inp, Sin]
    ri, rj = np.nonzero(W)
    if len(ri):
        Srec = b2.Synapses(G, G, "w : 1", on_pre="Iacc_post += w"); Srec.connect(i=ri, j=rj); Srec.w = W[ri, rj]
        objs.append(Srec)
    mon = b2.SpikeMonitor(G); objs.append(mon)
    net = b2.Network(*objs)
    net.run((LIQ["num_sim_steps"] + 1) * b2.ms)
    r = np.zeros((LIQ["num_sim_steps"], N_NEUR), bool)
    k = np.round(np.asarray(mon.t / b2.ms)).astype(int) - 1
    ok = (k >= 0) & (k < LIQ["num_sim_steps"])
    r[k[ok], np.asarray(mon.i)[ok]] = True
    return r

def execute_brian2(job, runs):
    e = runs[0]; t0 = time.time()
    D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
    X1n, X2n, _ = normalize_final(D1["X"], D2["X"], "robust")
    ain = measure_ain(X1n, np.arange(len(X1n)), "multi")
    res = {"A_in": ain, "liquids": {}}
    for mode in ("fixed", "published"):
        a = ain if MODES[mode]["ain"] == "measured" else PAPER_AIN["multi"]
        model = build_liquid(200, I_TARGET / (a * LIQ["P_in"]), SEEDS[0], MODES[mode]["gamma_fix"])
        pick = np.concatenate([np.where(D2["y"] == c)[0][:k] for c, k in
                               ((0, BRIAN2_BEATS - 2 * (BRIAN2_BEATS // 3)), (1, BRIAN2_BEATS // 3), (2, BRIAN2_BEATS // 3))])
        S = encode(X2n[pick], "multi")
        lif32 = _torch_raster(model, S, torch.float32, True)
        eq32 = _torch_raster(model, S, torch.float32, False)
        eq64 = _torch_raster(model, S, torch.float64, False)
        rows = []
        for b in range(len(pick)):
            br = _brian2_raster(model, S[b])
            rows.append({"beat_index_DS2": int(pick[b]), "class": CLASS_NAMES[D2["y"][pick[b]]],
                         "spikes_brian2": int(br.sum()), "spikes_torch64_equations": int(eq64[b].sum()),
                         "spikes_authors_lif_float32": int(lif32[b].sum()),
                         "brian2_vs_torch64_mismatches": int((br != eq64[b]).sum()),
                         "brian2_vs_authors_float32_mismatches": int((br != lif32[b]).sum()),
                         "equations_vs_snntorch_float32_mismatches": int((eq32[b] != lif32[b]).sum())})
        df = pd.DataFrame(rows)
        res["liquids"][mode] = {"per_beat": rows, "total_cells": int(len(pick) * LIQ["num_sim_steps"] * N_NEUR),
                                "brian2_vs_torch64_mismatches": int(df.brian2_vs_torch64_mismatches.sum()),
                                "brian2_vs_authors_float32_mismatches": int(df.brian2_vs_authors_float32_mismatches.sum()),
                                "equations_vs_snntorch_float32_mismatches": int(df.equations_vs_snntorch_float32_mismatches.sum())}
        print(mode); print(df.to_string(index=False))
    jdump(res, os.path.join(WORK_DIR, "checks", "brian2_check.json"))
    CACHE.mark_done(e["rid"], time.time() - t0, None, e["group"])

def execute_huang(job, runs):
    """Huang et al. 2014 on one split: the 15 SVM fits run as one task in the fit pool (one core)."""
    e = runs[0]; s = e["spec"]; t0 = time.time()
    D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
    final = s["split"] == "final"
    tr, ev, _ = _split_rows(s["split"], len(D1["y"]))
    Dev = D2 if final else D1
    if final:
        ev = np.arange(len(D2["y"]))
    def fit(cancel=None, shared=None):
        return huang_fit_predict(s, tr, ev, D1, Dev, cancel)
    def record(res, dt):
        y_pred, vote, votes, models = res
        yev = Dev["y"][ev]
        m = compute_metrics(yev, y_pred)
        vm = compute_metrics(yev, vote)          # the SVM ensemble alone (before the SVEB rule)
        m.update({f"vote_{k}": vm[k] for k in ("macro_f1", "recall_N", "precision_N", "recall_VEB", "precision_VEB")})
        m["vote_ave_N_VEB"] = float(np.mean([vm["recall_N"], vm["precision_N"], vm["recall_VEB"], vm["precision_VEB"]]))  # Huang's Ave
        m["n_support_vectors"] = int(sum(c.n_support_.sum() for c in models))
        update_speed("huang_s_per_run", dt, robust=True)
        _finish_run(e, m, ev, Dev["pid"][ev], yev, y_pred, (votes / len(models)).astype(np.float32),
                    {"svms": models, "matrix_seed": s["seed"], "pinv": True}, None,
                    {"feature": "200 raw samples z-scored @ pinv(A) (50) + pre-RR (s)", "sveb_rule": "pre-RR / record mean RR"},
                    {"total_seconds": time.time() - t0 + dt, "fit_seconds": dt})
    if FIT_POOL is not None:
        FIT_POOL.submit_job([{"rid": e["rid"], "gcells": None, "est_s": SPEED["huang_s_per_run"], "ram_gb": 1.5,
                              "fit": fit, "record": lambda res, dt, busy: record(res, dt)}],
                            _JOB_CTX.get("ram_gb", 0.0), SharedPrep())
        _JOB_CTX["submitted"] = True
        return
    t1 = time.time(); res = fit(); record(res, time.time() - t1)

def execute_hybrid(job, runs):
    """Huang's SVEB rule on a finished run's saved per-beat predictions (no training)."""
    for e in runs:
        s = e["spec"]; t0 = time.time()
        z = np.load(os.path.join(WORK_DIR, "runs", s["base"], "predictions.npz"))
        D = DATA["morph"]["DS2" if s["split"] == "final" else "DS1"]
        ev = z["eval_index"].astype(int)
        if not np.array_equal(z["y_true"].astype(int), D["y"][ev]):
            raise ValueError(f"saved predictions of {s['base']} do not match this session's beat order")
        y_pred = hybrid_rule(z["y_pred"].astype(int), D["rr_mean_ratio"][ev], s["rule"], s["threshold"])
        _finish_run(e, compute_metrics(D["y"][ev], y_pred), ev, D["pid"][ev], D["y"][ev], y_pred, None, None, None,
                    {"base_run": s["base"], "base_label": s["base_label"], "rule": s["rule"], "threshold": s["threshold"]},
                    {"total_seconds": time.time() - t0})

def working_settings(spec):
    """Candidates (lambda = 2, each g_rec) plus three references that are never chosen."""
    rows = [{"label": f"lambda {spec['lambda_val']:g}, g_rec {g:g}", "lambda_val": spec["lambda_val"], "g_rec": float(g),
             "mode": "fixed", "recurrence": True, "candidate": True} for g in spec["gains"]]
    rows += [{"label": f"lambda {spec['lambda_val']:g}, recurrence off", "lambda_val": spec["lambda_val"], "g_rec": 1.0,
              "mode": "fixed", "recurrence": False, "candidate": False},
             {"label": "paper as described (lambda 0.8, g_rec 1)", "lambda_val": 0.8, "g_rec": 1.0, "mode": "fixed",
              "recurrence": True, "candidate": False},
             {"label": "paper as published (both bugs)", "lambda_val": 0.8, "g_rec": 1.0, "mode": "published",
              "recurrence": True, "candidate": False}]
    return rows

def choose_g_rec(rows, max_firing):
    """Edge of chaos: among non-saturated candidates, the one whose perturbation neither dies nor grows
    (damage ratio closest to 1 on a log scale); if none spreads at all, the strongest non-saturated one."""
    ok = [r for r in rows if r["candidate"] and r["firing_per_step"] <= max_firing]
    if not ok:
        return min(r["g_rec"] for r in rows if r["candidate"])
    spread = [r for r in ok if r["damage_ratio"] > 0]
    if spread:
        return min(spread, key=lambda r: abs(np.log(r["damage_ratio"])))["g_rec"]
    return max(r["g_rec"] for r in ok)

def execute_wdiag(job, runs):
    """Label-free diagnostic: for each setting, flip one input spike at step t_perturb and count how many neurons'
    spikes differ afterwards (mean over beats). damage ratio = mean difference 41-80 steps later / 1-10 steps later:
    about 0 = ordered (the liquid forgets at once), about 1 = edge of chaos, >> 1 = chaotic."""
    e = runs[0]; s = e["spec"]; t0 = time.time()
    D1 = DATA[s["filter"]]["DS1"]
    X1n, _, _ = normalize_final(D1["X"], D1["X"][:1], s["norm"])
    ain = measure_ain(X1n, np.arange(len(X1n)), s["encoder"])
    rows_b = np.sort(np.random.default_rng(s["seed"]).choice(len(X1n), min(s["n_beats"], len(X1n)), replace=False))
    S = encode(X1n[rows_b], s["encoder"]); tp = s["t_perturb"]
    out = []
    for st in working_settings(s):
        a = ain if MODES[st["mode"]]["ain"] == "measured" else PAPER_AIN[s["encoder"]]
        model = build_liquid(S.shape[2], I_TARGET / (a * LIQ["P_in"]), s["seed"], MODES[st["mode"]]["gamma_fix"],
                             st["lambda_val"], st["recurrence"], st["g_rec"])
        ch = int((model.input_rsv_weights != 0).sum(1).argmax())     # the input channel with the most synapses
        S2 = S.copy(); S2[:, tp, ch] = 1 - S2[:, tp, ch]
        R1, irec, iin = liquid_raster(model, S)
        R2, _, _ = liquid_raster(model, S2)
        d = (R1 != R2).sum(2).mean(0)
        early, late = float(d[tp + 1:tp + 11].mean()), float(d[tp + 41:tp + 81].mean())
        ss = structure_stats(model)
        out.append({**st, "firing_per_step": float(R1[:, :NWIN * WIN].mean()),
                    "last_window_firing_per_step": float(R1[:, (NWIN - 1) * WIN:NWIN * WIN].mean()),
                    "mean_abs_recurrent_current": irec, "mean_input_current": iin,
                    "recurrent_over_input_current": irec / max(iin, 1e-12),
                    "perturbation_early": early, "perturbation_late": late,
                    "damage_ratio": late / early if early > 0 else 0.0,
                    "n_recurrent_synapses": ss["n_recurrent_synapses"], "mean_recurrent_in_degree": ss["mean_recurrent_in_degree"],
                    "perturbation_curve_from_t_perturb": [round(float(v), 4) for v in d[tp:tp + 121]]})
        log(f"  working-liquid diagnostic: {st['label']}: firing {out[-1]['firing_per_step']:.3f}/step, "
            f"recurrent/input current {out[-1]['recurrent_over_input_current']:.3f}, damage ratio {out[-1]['damage_ratio']:.3f}")
    g = choose_g_rec(out, s["max_firing"])
    res = {"run_id": e["rid"], "session": SESSION_TAG, "A_in_measured": ain, "beats_DS1": rows_b.tolist(), "t_perturb": tp,
           "perturbation": "one input spike flipped (channel with the most input synapses)", "settings": out,
           "max_firing": s["max_firing"], "chosen_g_rec": g,
           "rule": "non-saturated candidate with damage ratio closest to 1 (log scale); none spreading -> strongest non-saturated"}
    jdump(res, os.path.join(WORK_DIR, "checks", "working_liquid_diag.json"))
    log(f"  working-liquid diagnostic: chosen g_rec = {g:g} (lambda {s['lambda_val']:g})")
    CACHE.mark_done(e["rid"], time.time() - t0, None, e["group"])

EXECUTORS = {"lsm": execute_lsm, "feat": execute_feat, "nn": execute_nn, "check": execute_check, "brian2": execute_brian2,
             "huang": execute_huang, "hybrid": execute_hybrid, "wdiag": execute_wdiag}

FEAT_DIM = {"no_liquid": lambda enc: NWIN * (200 if enc == "multi" else N_IN[enc]), "elm": lambda enc: NWIN * N_NEUR,
            "raw_beat": lambda enc: 577, "rr_only": lambda enc: len(RR_NAMES), "rr_shape": lambda enc: 73 + len(RR_NAMES),
            "lda": lambda enc: 0, "elm_rr": lambda enc: NWIN * N_NEUR, "rr_only_mrr": lambda enc: len(RR_NAMES) + 1,
            "rr_shape_mrr": lambda enc: 73 + len(RR_NAMES) + 1}

def _sizes(spec):
    n1 = len(DATA["morph"]["DS1"]["y"]); n2 = len(DATA["morph"]["DS2"]["y"])
    final = spec.get("split") == "final"
    return n1, n2, final, (n1 if final else int(0.75 * n1)), n1 + (n2 if final else 0)

def _nfeat(spec):
    if spec["kind"] == "lsm":
        if spec.get("readout") == "svm_late":
            return N_NEUR
        return NWIN * N_NEUR + (len(RR_NAMES) if spec.get("readout") == "svm_rr" else 0)
    return FEAT_DIM[spec["model"]](spec["encoder"])

def estimate_parts(job, runs):
    """(seconds of main-thread / GPU work, [seconds of each classifier fit], fit speed key) for one job."""
    s = runs[0]["spec"]; k = s["kind"]
    n1, n2, final, ntr, beats = _sizes(s)
    if k == "lsm":
        main = beats / SPEED["sim_beats_per_s"][s["encoder"]] + SPEED["lsm_job_overhead_s"]
    elif k == "feat":
        main = (beats / SPEED["encode_beats_per_s"][s["encoder"]] if s["encoder"] else 0) + 10
        if s["model"] == "lda":
            return main + 5 * len(runs), []
    elif k == "nn":
        t = 0.0
        for r in runs:
            sp = r["spec"]
            if sp.get("arch"):
                t += sp["epochs"] * ntr / 1000 * SPEED["nn_unit_s_per_epoch_kbeat"][sp["model"]] * nn_cost(sp["model"], sp["arch"]) ** 0.5
            else:
                t += NN_EPOCHS * ntr / 1000 * SPEED["nn_s_per_epoch_kbeat"][sp["model"]]
        return t + 20 * len(runs), []
    elif k == "check":
        return 120, []
    elif k == "huang":
        if PIPELINE:          # one pool task per run, expressed in the pool's units so the guards and projection see it
            return 15.0 * len(runs), [SPEED["huang_s_per_run"] / SPEED["fit_s_per_Gcell"]["pipe"]] * len(runs)
        return (15.0 + SPEED["huang_s_per_run"]) * len(runs), []
    elif k == "hybrid":
        return 2.0 * len(runs), []
    elif k == "wdiag":
        return 20.0 * (len(WORKING_GAINS) + 3) + 60, []
    else:
        return 30 * BRIAN2_BEATS + 30, []
    return main, [ntr * _nfeat(r["spec"]) / 1e9 for r in runs]      # fits returned in Gcells

def _seq_fit_seconds(gcells):
    if not gcells:
        return 0.0
    threads = max(1, min(int(round(SPEED["fit_threads"])), len(gcells)))
    return math.ceil(len(gcells) / threads) * max(gcells) * SPEED["fit_s_per_Gcell"][str(fit_n_jobs(threads))]

def estimate_seconds(job, runs):
    """Older runner: projected wall time of one job (main-thread work + its fits sharing the cores)."""
    main, gcells = estimate_parts(job, runs)
    return main + _seq_fit_seconds(gcells)

def fit_seconds_pipe(gcells):
    return [g * SPEED["fit_s_per_Gcell"]["pipe"] for g in gcells]

def job_ram_gb(job, runs):
    """RAM a job holds until all its fits finish (spike counts or features, plus its normalised beats)."""
    s = runs[0]["spec"]; n1, n2, final, ntr, beats = _sizes(s)
    if s["kind"] == "lsm":
        return beats * NWIN * N_NEUR / 1e9 + 0.6
    if s["kind"] == "feat":
        return beats * min(_nfeat(s), 12000) * (1 if s["model"] in ("no_liquid", "elm") else 4) / 1e9 + 0.6
    return 1.0

CORE_SEARCH_PATH = os.path.join(WORK_DIR, "checks", "core_search.json")

def _core_search_saved():
    """The core search's choice from an earlier benchmark on this machine (local only), or None."""
    if MACHINE != "local" or FIT_WORKERS != "auto" or not os.path.exists(CORE_SEARCH_PATH):
        return None
    try:
        cs = json.load(open(CORE_SEARCH_PATH))
        if cs.get("cpu_count") == N_CPU and cs.get("gpu") == ENV.get("gpu"):
            return cs
    except Exception:
        pass
    return None

def pipeline_workers():
    """Parallel fits: every core (or FIT_WORKERS, or the local core search's choice), capped by RAM so each worker can
    hold a fit and a job's counts."""
    budget = _avail_ram_gb() - RAM_RESERVE_GB
    by_ram = max(1, int(budget // (fit_ram_gb(NWIN * N_NEUR, len(DATA["morph"]["DS1"]["y"])) + 1.1)))
    saved = _core_search_saved()
    want = int(saved["best_workers"]) if saved else (N_CPU if FIT_WORKERS == "auto" else int(FIT_WORKERS))
    return max(1, min(want, by_ram)), budget, by_ram

def simulate_pipeline(items, W):
    """items: [(main_seconds, [fit seconds])] in run order. As in the runner: the main thread may start a job with fits
    only while fewer than W fits wait for a worker; while it would wait, it runs pending jobs without fits (CNN/GRU)
    instead. Returns total seconds."""
    free = [0.0] * W; heapq.heapify(free)
    starts, tp, end = [], 0.0, 0.0
    fillers = collections.deque(m for m, f in items if not f)
    for main, fits in items:
        if not fits:
            continue                                   # jobs without fits are taken from `fillers`
        while True:
            waiting = sorted(x for x in starts if x > tp)
            if len(waiting) < W:
                break
            if fillers:
                tp += fillers.popleft()                # GPU busy with a CNN/GRU job meanwhile
            else:
                tp = waiting[len(waiting) - W]
        starts = [x for x in starts if x > tp]
        tp += main
        for d in fits:
            w = heapq.heappop(free); st = max(w, tp); heapq.heappush(free, st + d)
            starts.append(st); end = max(end, st + d)
    tp += sum(fillers)                                 # whatever is left after the last fits were handed out
    return max(tp, end)

def project_time(jobs):
    """Remaining time for every pending job (plus the second stages of the re-tunings if not yet planned), by group."""
    rows, items = [], []
    def add(group, nruns, main, gcells):
        fits = fit_seconds_pipe(gcells) if PIPELINE else []
        secs = main + (sum(fits) / PIPE_W if PIPELINE else _seq_fit_seconds(gcells))
        rows.append({"group": group, "runs": nruns, "jobs": 1, "seconds": secs})
        items.append((main, fits))
    for j in jobs:
        pr = [r for r in j["runs"] if not CACHE.is_done(r["rid"])]
        if pr:
            main, gcells = estimate_parts(j, pr)
            add(pr[0]["group"], len(pr), main, gcells)
    if group_on("retune") and retune_choice()[0] is None:      # stage 2 appears once stage 1 is done
        for sp in ["final"] + SPLITS_CV:
            spec = lsm_spec("fixed", SEEDS[0], "morph", "robust", "multi", "svm", sp)
            fake = [{"spec": spec, "rid": "", "group": "retune"}]
            for _ in SEEDS:
                main, gcells = estimate_parts({"key": None}, fake)
                add("retune", 1, main, gcells)
    if group_on("nn_tune"):
        for m in ("cnn", "gru"):
            for use_rr in (False, True):
                if nn_tune_choice(m, use_rr)[0] is None:
                    a = DEFAULT_ARCH[m]
                    for seed in SEEDS:
                        for sp in ["final"] + SPLITS_CV:
                            fake = [{"spec": nn_spec(m, use_rr, seed, sp, arch=a, epochs=NN_TUNE_CHECKPOINTS[len(NN_TUNE_CHECKPOINTS) // 2], trial="best"),
                                     "rid": "", "group": "nn_tune"}]
                            add("nn_tune", 1, *estimate_parts({"key": None}, fake))
    df = pd.DataFrame(rows, columns=["group", "runs", "jobs", "seconds"])
    if df.empty:
        return df
    df = df.groupby("group", sort=False).sum().reindex([g for g in GROUP_ORDER if g in set(df.group)]).reset_index()
    if PIPELINE:            # the schedule overlaps GPU and CPU work: scale the group shares to the simulated total
        total = simulate_pipeline(items, PIPE_W)
        df["seconds"] = df.seconds * total / max(df.seconds.sum(), 1e-9)
    df["hours"] = df.seconds / 3600
    return df

def print_projection(jobs, title):
    df = project_time(jobs)
    if df.empty:
        print(title + ": nothing left to run."); return df
    total = df.hours.sum()
    per_session = SESSION_BUDGET_HOURS - ANALYSIS_RESERVE_MIN / 60 - 0.2       # minus setup and analysis
    keys = {"sim_beats_per_s", "encode_beats_per_s", "nn_s_per_epoch_kbeat", "nn_unit_s_per_epoch_kbeat"}
    fit_key = "fit_s_per_Gcell:pipe" if PIPELINE else None
    guessed = sorted({f"{k}:{e}" for k in keys for e in SPEED[k]} - SPEED_MEASURED)
    if PIPELINE and fit_key not in SPEED_MEASURED:
        guessed.append(fit_key)
    if not PIPELINE:
        guessed += sorted({f"fit_s_per_Gcell:{e}" for e in ("1", "2", "3")} | {"fit_threads"} - SPEED_MEASURED)
    runner = (f"pipeline: {PIPE_W} parallel fits (CPU cores {N_CPU}, RAM allows {PIPE_BY_RAM}), "
              f"{SPEED['fit_s_per_Gcell']['pipe']:.0f} s per Gcell per fit under load" if PIPELINE else "older one-job-at-a-time runner")
    sess = (f"about {math.ceil(total / per_session)} session(s) of {SESSION_BUDGET_HOURS} h" if MACHINE == "kaggle"
            else f"one continuous run (safety stop at {SESSION_BUDGET_HOURS} h)")
    lines = [f"\n===== {title} =====", df.assign(hours=df.hours.round(2)).drop(columns="seconds").to_string(index=False),
             f"TOTAL remaining: {total:.1f} h  ->  {sess}" + (f", about {total / 30:.1f} week(s) of a ~30 h/week GPU quota" if MACHINE == "kaggle" else ""),
             f"runner: {runner}",
             f"machine: {MACHINE}; device: {ENV.get('gpu', DEVICE.type)}; CPU: {ENV.get('cpu_model')}",
             f"measured speeds: {sorted(SPEED_MEASURED)}",
             ("still default guesses (less reliable): " + ", ".join(guessed)) if guessed else "all speeds measured"]
    print("\n".join(lines))
    df.to_csv(os.path.join(WORK_DIR, "tables", "time_projection.csv"), index=False)
    atomic_write_text(os.path.join(WORK_DIR, "tables", "time_projection.txt"), "\n".join(lines) + "\n")
    return df

PIPE_W, PIPE_BUDGET_GB, PIPE_BY_RAM = pipeline_workers()
if PIPELINE:
    print(f"pipeline: {PIPE_W} parallel fits (CPU cores: {N_CPU}, FIT_WORKERS={FIT_WORKERS!r}, RAM allows {PIPE_BY_RAM} "
          f"with {PIPE_BUDGET_GB:.1f} GB available after the {RAM_RESERVE_GB} GB reserve)"
          + (f"; {_core_search_saved()['best_workers']} chosen by the core search of session {_core_search_saved().get('session')}"
             if _core_search_saved() else ""))


## 12. Stage 2 plan: E1 to E3

In [ ]:
# Stage 2 plan (plan, Experiments and controls): every new classification run as a spec. Run ids include the
# calibration file's hash, so a different final liquid can never reuse these runs.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SNAP_READOUTS = {"snap288": 0, "snap342": 1, "snap396": 2}         # R, R + 150 ms, R + 300 ms
FRONT_ENDS = [(f, n, e) for f in ("morph", "median") for n in ("standard", "robust") for e in ("multi", "rate", "latency")]
CFG11 = ("morph", "robust", "multi")

def lsm2_spec(liquid, fe, readout, split, seed, variant=None, loss="svm"):
    """liquid: "final", "final_off" (g = 0), or "X" (the paper's liquid at g = 1, (a)'s construction).
    readout: "film" (32 windows x neurons), "film_rr" (film + RR features), or a snapshot.
    variant: None, "ablated" (memory ablation), "rrliq" (30 RR channels in the liquid), "rrliq_mean" (40)."""
    s = {"kind": "lsm2", "liquid": liquid, "filter": fe[0], "norm": fe[1], "encoder": fe[2], "readout": readout,
         "loss": loss, "split": split, "seed": seed, "variant": variant,
         "cal": CAL["hash"] if (CAL is not None and liquid != "X") else None}
    return s

def feat2_spec(model, readout, split, seed):
    """Encoder-only controls at the snapshot moments: "noliq30" (30 ms filter), "fade" (30 / 100 / 300 ms),
    "selm" (random expansion of the 30 ms-filtered input; seed = the expansion's draw)."""
    return {"kind": "feat2", "model": model, "readout": readout, "split": split, "seed": seed,
            "filter": "morph", "norm": "robust", "encoder": "multi"}

_old_run_id_of = run_id_of
def run_id_of(spec):
    k = spec["kind"]
    if k not in ("lsm2", "feat2"):
        return _old_run_id_of(spec)
    h = stable_hash({**spec, "_v": CODE_VERSION, "_smoke": SMOKE_TEST})
    if k == "lsm2":
        tag = [spec["liquid"], f"s{spec['seed']}", f"{spec['filter']}-{spec['norm']}-{spec['encoder']}",
               spec["readout"] + ("" if spec["loss"] == "svm" else f"-{spec['loss']}")] + ([spec["variant"]] if spec["variant"] else [])
    else:
        tag = [spec["model"], spec["readout"], f"s{spec['seed']}"]
    return f"{'SMOKE__' if SMOKE_TEST else ''}{k}__{'_'.join(tag)}__{spec['split']}__{h}"

def unit_of(spec):
    """The smallest piece that shares one simulation (plan, Stages): (front end, seed, split) for a robust front end,
    (front end, seed, "all") for a standard one."""
    fe = (spec.get("filter"), spec.get("norm"), spec.get("encoder"))
    return (fe, spec.get("seed"), spec["split"] if spec.get("norm") == "robust" else "all")

def unit_assignment(units):
    """Deterministic, the same on every machine: units in a stable order, each given to the machine whose load
    divided by its weight is lowest."""
    names = sorted(UNIT_SHARES)
    load = {m: 0.0 for m in names}
    out = {}
    for u in sorted(units, key=lambda u: stable_hash(u, 12)):
        m = min(names, key=lambda m: ((load[m] + 1) / UNIT_SHARES[m], m))
        out[u] = m; load[m] += 1
    return out

def e3_partner_needed():
    return CAL is not None and not CAL.get("level1f_pass", False)

def build_plan2():
    P = []
    if CAL is None:
        return P
    splits = ["final"] + SPLITS_CV
    seeds = SEEDS
    def add(group, spec):
        if GROUPS_ENABLED.get(group, False):
            P.append((group, spec))
    for seed in seeds:
        for sp in splits:
            for liq in ("final", "final_off", "X"):
                for ro in ["film"] + list(SNAP_READOUTS):
                    add("e1", lsm2_spec(liq, CFG11, ro, sp, seed))
            for liq in ("final", "final_off"):
                for ro in SNAP_READOUTS:
                    add("e1_ablation", lsm2_spec(liq, CFG11, ro, sp, seed, variant="ablated"))
            for model in ("noliq30", "fade", "selm"):
                for ro in SNAP_READOUTS:
                    add("e1_controls", feat2_spec(model, ro, sp, seed))
            for fe in FRONT_ENDS:
                for loss in ("svm", "lr"):
                    if CONFIGS_BY_FE.get((fe, loss)) is not None:
                        add("e2", lsm2_spec("final", fe, "film", sp, seed, loss=loss))
            add("e3", lsm2_spec("final", CFG11, "film_rr", sp, seed))
            add("e3", lsm2_spec("final", CFG11, "film", sp, seed, variant="rrliq"))
            add("e3", lsm2_spec("final", CFG11, "film", sp, seed, variant="rrliq_mean"))
            if e3_partner_needed():
                add("e3_partner", lsm2_spec("X", CFG11, "film", sp, seed, variant="rrliq"))
    # hybrid rule on the base, RR-readout and RR-liquid runs (needs every machine's predictions: after the merge)
    if GROUPS_ENABLED.get("e3_hybrid") and (MERGE_MODE or UNIT_SHARES == {THIS_MACHINE: UNIT_SHARES.get(THIS_MACHINE)}):
        for seed in seeds:
            for sp in splits:
                for label, bspec in (("final liquid, film", lsm2_spec("final", CFG11, "film", sp, seed)),
                                     ("final liquid, film + RR features", lsm2_spec("final", CFG11, "film_rr", sp, seed)),
                                     ("final liquid, RR in liquid", lsm2_spec("final", CFG11, "film", sp, seed, variant="rrliq"))):
                    brid = run_id_of(bspec)
                    if brid in CACHE.done and os.path.exists(os.path.join(WORK_DIR, "runs", brid, "predictions.npz")):
                        for rule in HYBRID_RULES:
                            P.append(("e3_hybrid", {"kind": "hybrid", "base": brid, "base_label": label, "rule": rule,
                                                    "threshold": HUANG_THRESHOLD, "seed": seed, "split": sp}))
    return P

CONFIGS_BY_FE = {((c["filter"], c["norm"], c["encoder"]), c["readout"]): cid for cid, c in CONFIGS.items()}

_old_build_plan = build_plan
def build_plan():
    plan = _old_build_plan()                       # the replication's groups (disabled here: nothing pending)
    seen = {e["rid"] for e in plan}
    new = []
    for group, spec in build_plan2():
        rid = run_id_of(spec)
        if rid in seen:
            continue
        seen.add(rid)
        new.append({"rid": rid, "group": group, "groups": [group], "spec": spec})
    units = {unit_of(e["spec"]) for e in new if e["spec"]["kind"] in ("lsm2", "feat2")}
    assign = unit_assignment(units)
    mine = [e for e in new if e["spec"]["kind"] not in ("lsm2", "feat2") or assign[unit_of(e["spec"])] == THIS_MACHINE
            or MERGE_MODE]
    return plan + mine

_old_states_key = states_key
def states_key(spec):
    """Runs with the same key share one simulation: the liquid, front end, seed, input variant and data split
    (standard-normalisation CV splits share one CV simulation)."""
    if spec["kind"] == "lsm2":
        sp = spec["split"]
        nk = "final" if sp == "final" else ("cv-shared" if spec["norm"] == "standard" else sp)
        core = {k: spec[k] for k in ("liquid", "filter", "norm", "encoder", "seed", "variant", "cal")}
        return ("lsm2", json.dumps(core, sort_keys=True), nk)
    if spec["kind"] == "feat2":
        core = {k: spec[k] for k in ("model", "seed")}
        return ("feat2", json.dumps(core, sort_keys=True), spec["split"])
    return _old_states_key(spec)

_p2 = build_plan()
_c2 = pd.Series([e["group"] for e in _p2]).value_counts()
print(f"stage-2 plan on this machine ({THIS_MACHINE}): {len(_p2)} runs in {len(make_jobs(_p2))} jobs")
print(_c2.to_string() if len(_c2) else "(nothing: no calibration file yet, or all done)")


## 13. Stage 2 executors: the new liquid and the snapshot controls

In [ ]:
# Stage-2 executors: the new liquid simulations (kind "lsm2") and the encoder-only snapshot controls ("feat2").
# Same readout, splits, metrics and saving as the replication (fit_readout, _finish_run); only the features differ.
SNAP_T = list(LC.SNAPSHOT_SAMPLES)
ABLATE_SAMPLES = int(round(50.0 / LC.SAMPLE_MS))            # input before t - 50 ms is replaced (18 samples)
START_STREAM = {"DS1": 101, "DS2": 102}                      # per-beat starting voltages from (seed, stream, beat index)

def _norm_for(nk, filt, norm):
    D1, D2 = DATA[filt]["DS1"], DATA[filt]["DS2"]
    if nk == "final":
        X1n, X2n, normp = normalize_final(D1["X"], D2["X"], norm)
        return np.asarray(X1n, np.float32), np.asarray(X2n, np.float32), np.arange(len(D1["y"])), normp
    if nk == "cv-shared":
        X1n, normp = normalize_cv(D1["X"], None, "standard")
        return X1n, None, np.arange(len(D1["y"])), normp
    f = FOLD_BY[parse_split(nk)]
    X1n, normp = normalize_cv(D1["X"], f["train_idx"], norm)
    return X1n, None, f["train_idx"], normp

def _cal_scale(filt, norm, enc):
    return float(CAL["input_scales"][f"{filt}-{norm}-{enc}"]["scale"])

def _ablate(Xn, t_snap, mean_beat):
    """Input before t - 50 ms replaced by the R-aligned mean beat, shifted to meet the real beat at t - 50 ms."""
    cut = t_snap - ABLATE_SAMPLES
    out = Xn.copy()
    out[:, :cut] = mean_beat[None, :cut] + (Xn[:, cut] - mean_beat[cut])[:, None]
    return np.clip(out, 0.0, 1.0)

def _simulate_set(liq, g, in_scale, Xn, enc, rr_rows, rr_thr, seed, ds, rec, recurrence=True):
    """Encode (+ RR channels) and run the new simulator in fixed-size batches; numpy outputs."""
    sm = LS.Simulator(liq, device=DEVICE, dtype=torch.float32, g=g, in_scale=in_scale, recurrence=recurrence)
    outs = {}
    t_enc = t_sim = 0.0
    for s in range(0, len(Xn), NEW_SIM_BATCH):
        t0 = time.time()
        x = encode_t(Xn[s:s + NEW_SIM_BATCH], enc)
        if rr_rows is not None:
            x = torch.cat([x, torch.from_numpy(rr_channels(rr_rows[s:s + NEW_SIM_BATCH], rr_thr, x.shape[1])).to(DEVICE)], 2)
        t_enc += time.time() - t0; t0 = time.time()
        ids = np.arange(s, s + len(x))
        u0 = LS.start_voltages(liq, seed, ids, START_STREAM[ds])
        o = sm.run(x, u0, LIQ["num_sim_steps"], rec=rec)
        for k, v in o.items():
            if torch.is_tensor(v) and k != "u_end":
                outs.setdefault(k, []).append(v.cpu().numpy() if k != "windows" else v.to(torch.uint8).cpu().numpy())
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t_sim += time.time() - t0
    return {k: np.concatenate(v) for k, v in outs.items()}, t_enc, t_sim

def _x_liquid(n_in, X1n, tr_rows, enc, rr1, rr_thr, seed):
    """The paper's liquid X at g = 1, built exactly as the replication's fixed runs (its class, A_in measured on
    the training rows), simulated in the new code."""
    ain = measure_ain(X1n, tr_rows, enc, 200, rr=rr1, rr_thr=rr_thr)
    model = build_liquid(n_in, I_TARGET / (ain * LIQ["P_in"]), seed, True)
    return LL.from_authors_model(model), {"A_in_used": ain, "construction": "authors' class, fixed mode"}

def execute_lsm2(job, runs):
    s0 = runs[0]["spec"]; nk = job["key"][2]; final = nk == "final"
    filt, norm, enc, seed, liqname, variant = s0["filter"], s0["norm"], s0["encoder"], s0["seed"], s0["liquid"], s0["variant"]
    D1, D2 = DATA[filt]["DS1"], DATA[filt]["DS2"]
    t0 = time.time()
    torch.manual_seed(int(stable_hash(job["key"], 6), 16))   # the Rate encoder is stochastic: fixed per simulation
    X1n, X2n, tr_rows, normp = _norm_for(nk, filt, norm)
    rr1 = rr2 = rr_thr = None
    if variant in ("rrliq", "rrliq_mean"):
        mean_set = variant == "rrliq_mean"
        rr1 = rr_inputs(D1, np.arange(len(D1["y"])), mean_set); rr2 = rr_inputs(D2, np.arange(len(D2["y"])), mean_set)
        rr_thr = rr_thresholds(rr1[tr_rows])
    n_in = (200 if enc == "multi" else N_IN[enc]) + (len(rr_thr) * RR_LEVELS if rr_thr is not None else 0)
    if liqname == "X":
        liq, info = _x_liquid(n_in, X1n, tr_rows, enc, rr1, rr_thr, seed)
        g, scale = 1.0, 1.0
    else:
        liq = LL.build_candidate(frozenset(CAL["final"]), seed, CAL["n_sub"], n_in)
        g = (liq.g_maass if CAL["maass_scale"] else float(CAL["g"])) if liqname == "final" else 0.0
        scale = _cal_scale(filt, norm, enc)
        info = {"final_liquid": CAL["final_key"], "n_sub": CAL["n_sub"], "g": g, "abs_g": liq.ee_mean_push(g),
                "input_scale": scale, "calibration": CAL["hash"]}
    rec = {"windows": (WIN, NWIN), "states_at": SNAP_T, "neuron_counts": (0, LC.BEAT_SAMPLES)}
    sets = [("DS1", X1n, rr1)] + ([("DS2", X2n, rr2)] if final else [])
    F, t_enc, t_sim, stats = {}, 0.0, 0.0, {}
    if variant == "ablated":
        mean_beat = X1n[tr_rows].mean(0)                     # R-aligned mean beat of the training rows (labels unused)
        for ds, Xn, rr in sets:
            snaps = np.zeros((len(Xn), len(SNAP_T), liq.N), np.float32)
            for i, t in enumerate(SNAP_T):
                o, a, b = _simulate_set(liq, g, scale, _ablate(Xn, t, mean_beat), enc, None, None, seed, ds, {"states_at": [t]},
                                        recurrence=liqname != "final_off")
                snaps[:, i] = o["states_at"][:, 0]; t_enc += a; t_sim += b
            F[ds] = {"snap": snaps}
    else:
        for ds, Xn, rr in sets:
            o, a, b = _simulate_set(liq, g, scale, Xn, enc, rr, rr_thr, seed, ds, rec, recurrence=liqname != "final_off")
            t_enc += a; t_sim += b
            F[ds] = {"film": o["windows"], "snap": o["states_at"]}
            dur = LC.BEAT_SAMPLES * LC.SAMPLE_MS / 1000
            stats[f"{ds}_firing_hz"] = float(o["neuron_counts"].mean() / dur)
            stats[f"{ds}_silent_neurons"] = int((o["neuron_counts"].sum(0) == 0).sum())
    sim_s = t_enc + t_sim
    update_speed("sim_beats_per_s", sum(len(x) for _, x, _ in sets) / max(sim_s, 1e-6), sub=f"lsm2-{enc}")
    liq_info = {**info, "normalization": normp, "liquid": liqname, "variant": variant, "N": liq.N, **stats}
    log(f"  lsm2 {liqname} {filt}-{norm}-{enc} s{seed} {nk} {variant or ''}: sim {sim_s:.0f}s "
        + " ".join(f"{k}={v:.2f}" for k, v in stats.items() if k.endswith("hz")))
    def film_fn(ds):
        C_ = F[ds]["film"]
        return lambda rows: (C_[rows].reshape(len(rows), -1).astype(np.float64) / WIN).astype(np.float32)
    def snap_fn(ds, i):
        S_ = F[ds]["snap"]
        return lambda rows: np.ascontiguousarray(S_[rows, i], np.float32)
    groups = []
    for e in runs:
        ro = e["spec"]["readout"]
        if ro in ("film", "film_rr"):
            groups.append((e, film_fn("DS1"), film_fn("DS2") if final else film_fn("DS1")))
        else:
            i = SNAP_READOUTS[ro]
            groups.append((e, snap_fn("DS1", i), snap_fn("DS2", i) if final else snap_fn("DS1", i)))
    _run_readouts2(groups, D1["y"], D2 if final else D1, final, liq_info,
                   {"shared_seconds": time.time() - t0, "simulation_seconds": sim_s, "beats_simulated": sum(len(x) for _, x, _ in sets)},
                   len(D1["y"]))

def _filtered_input(Xn, enc, taus_ms, t_read):
    """Encoder spikes through decaying filters (one per tau), read at the snapshot samples: (n, len(t_read), C*len(taus))."""
    out = np.zeros((len(Xn), len(t_read), (200 if enc == "multi" else N_IN[enc]) * len(taus_ms)), np.float32)
    dec = torch.tensor([math.exp(-LC.SAMPLE_MS / t) for t in taus_ms], device=DEVICE, dtype=torch.float32)
    for s in range(0, len(Xn), NEW_SIM_BATCH):
        x = encode_t(Xn[s:s + NEW_SIM_BATCH], enc).float()
        z = torch.zeros(x.shape[0], len(taus_ms), x.shape[2], device=DEVICE)
        for t in range(max(t_read) + 1):
            if t < x.shape[1]:
                z = z * dec[None, :, None] + x[:, t][:, None, :]
            else:
                z = z * dec[None, :, None]
            if t in t_read:
                out[s:s + len(x), t_read.index(t)] = z.permute(0, 2, 1).reshape(len(x), -1).cpu().numpy()
    return out

def execute_feat2(job, runs):
    s0 = runs[0]["spec"]; split = job["key"][2]; final = split == "final"
    t0 = time.time()
    D1, D2, X1n, X2n, tr_rows, normp = _front_end(split)
    model = s0["model"]
    taus = {"noliq30": [30.0], "fade": [30.0, 100.0, 300.0], "selm": [30.0]}[model]
    Z1 = _filtered_input(X1n, "multi", taus, SNAP_T); Z2 = _filtered_input(X2n, "multi", taus, SNAP_T) if final else None
    info = {"normalization": normp, "feature_model": model, "taus_ms": taus}
    if model == "selm":
        # a random nonlinear expansion with no memory: the 30 ms-filtered input through the fixed liquid's input
        # weights (same construction and seed as the replication's ELM), centred on the training mean, ReLU
        ain = measure_ain(X1n, tr_rows, "multi")
        Wf = build_liquid(200, I_TARGET / (ain * LIQ["P_in"]), s0["seed"], True).input_rsv_weights.detach().cpu().numpy().astype(np.float32)
        P1 = np.einsum("ntc,ck->ntk", Z1, Wf); P2 = np.einsum("ntc,ck->ntk", Z2, Wf) if final else None
        mu = P1[tr_rows].mean(0)
        Z1 = np.maximum(P1 - mu, 0); Z2 = np.maximum(P2 - mu, 0) if final else None
        info.update({"A_in_measured": ain, "elm": "30 ms-filtered input @ fixed-liquid W_in, centred on train mean, ReLU"})
    groups = []
    for e in runs:
        i = SNAP_READOUTS[e["spec"]["readout"]]
        f1 = (lambda Z, i: (lambda rows: np.ascontiguousarray(Z[rows, i])))(Z1, i)
        f2 = (lambda Z, i: (lambda rows: np.ascontiguousarray(Z[rows, i])))(Z2, i) if final else f1
        groups.append((e, f1, f2))
    _run_readouts2(groups, D1["y"], D2 if final else D1, final, info, {"shared_seconds": time.time() - t0}, len(D1["y"]))

def _run_readouts2(groups, ytr_all, D_eval, final, liq, base_timing, ntr_full):
    """The replication's readout for runs with their own feature functions: groups = [(run, feat_tr, feat_ev)].
    The loss is the spec's ("svm" = hinge, "lr" = log loss); "film_rr" appends the 8 RR features as in svm_rr."""
    D_TR_RR, D_EV_RR = DATA["morph"]["DS1"]["rr"], D_eval["rr"]
    def parts(e, feat_tr, feat_ev):
        tr, ev, rng_seed = _split_rows(e["spec"]["split"], ntr_full)
        if final:
            ev = np.arange(len(D_eval["y"]))
        add_rr = e["spec"]["readout"] == "film_rr"
        ftr = (lambda r: np.hstack([feat_tr(tr[r]), D_TR_RR[tr[r]]])) if add_rr else (lambda r: feat_tr(tr[r]))
        fev = (lambda r: np.hstack([feat_ev(ev[r]), D_EV_RR[ev[r]]])) if add_rr else (lambda r: feat_ev(ev[r]))
        return tr, ev, rng_seed, add_rr, ftr, fev
    def one(e, feat_tr, feat_ev, nj, cancel=None, shared=None):
        tr, ev, rng_seed, add_rr, ftr, fev = parts(e, feat_tr, feat_ev)
        yev = D_eval["y"][ev]
        loss = "log_loss" if e["spec"].get("loss") == "lr" else "hinge"
        nfeat = len(ftr(np.arange(1)).ravel())
        key = (e["spec"]["split"], e["spec"]["readout"], add_rr)
        prep = shared.get(key, ftr, len(tr), nfeat, FIT_POOL.ram) if shared is not None else None
        scaler, clf, yp, sc, curve = fit_readout(ftr, len(tr), ytr_all[tr], fev, len(ev), loss, rng_seed,
                                                 yev_for_curve=None if final else yev, n_jobs=nj, prepared=prep, cancel=cancel)
        return e, compute_metrics(yev, yp), ev, D_eval["pid"][ev], yev, yp, sc, {"scaler": scaler, "classifier": clf}, curve
    def record(res, dt, threads):
        e, m, ev, pid, yev, yp, sc, mobj, curve = res
        _finish_run(e, m, ev, pid, yev, yp, sc, mobj, curve, liq,
                    {**base_timing, "readout_seconds": dt, "total_seconds": base_timing.get("shared_seconds", 0) + dt,
                     "readout_threads": threads, "runner": "pipeline" if FIT_POOL is not None else "sequential"})
    if FIT_POOL is not None:
        tasks = []
        for e, ft, fe in groups:
            tr = _split_rows(e["spec"]["split"], ntr_full)[0]
            nfeat = len(ft(np.arange(1)).ravel())
            gc_ = len(tr) * nfeat / 1e9
            tasks.append({"rid": e["rid"], "gcells": gc_, "est_s": gc_ * SPEED["fit_s_per_Gcell"]["pipe"],
                          "ram_gb": fit_ram_gb(nfeat, len(tr)),
                          "fit": (lambda cancel, shared, e=e, ft=ft, fe=fe: one(e, ft, fe, 1, cancel=cancel, shared=shared)),
                          "record": (lambda res, dt, busy: record(res, dt, busy))})
        FIT_POOL.submit_job(tasks, _JOB_CTX.get("ram_gb", 0.0), SharedPrep())
        _JOB_CTX["submitted"] = True
        return
    for e, ft, fe in groups:
        try:
            t1 = time.time(); r = one(e, ft, fe, READOUT_JOBS); record(r, time.time() - t1, 1)
        except Exception:
            _record_failure(e["rid"], traceback.format_exc())

EXECUTORS.update({"lsm2": execute_lsm2, "feat2": execute_feat2})

# cost and RAM estimates for the runner's guards and projection
_old_estimate_parts, _old_job_ram_gb = estimate_parts, job_ram_gb
def _lsm2_n_neur():
    return int(np.prod(LC.MAASS["dims"] if 11 in CAL["final"] else LC.X["dims"])) if CAL else N_NEUR

def estimate_parts(job, runs):
    s = runs[0]["spec"]
    if s["kind"] not in ("lsm2", "feat2"):
        return _old_estimate_parts(job, runs)
    n1, n2, final, ntr, beats = _sizes(s)
    N = _lsm2_n_neur()
    gc_ = []
    for r in runs:
        ro = r["spec"]["readout"]
        nf = (NWIN * N + (len(RR_NAMES) if ro == "film_rr" else 0)) if ro in ("film", "film_rr") else \
             ({"noliq30": 200, "fade": 600, "selm": N_NEUR}.get(r["spec"].get("model"), N))
        gc_.append(ntr * nf / 1e9)
    if s["kind"] == "feat2":
        return beats / SPEED["encode_beats_per_s"]["multi"] + 20, gc_
    mult = 3 if s.get("variant") == "ablated" else 1
    bps = SPEED["sim_beats_per_s"].get(f"lsm2-{s['encoder']}", SPEED["sim_beats_per_s"][s["encoder"]] / 4)
    return mult * beats / bps + SPEED["lsm_job_overhead_s"], gc_

def job_ram_gb(job, runs):
    s = runs[0]["spec"]
    if s["kind"] not in ("lsm2", "feat2"):
        return _old_job_ram_gb(job, runs)
    n1, n2, final, ntr, beats = _sizes(s)
    N = _lsm2_n_neur()
    return beats * (NWIN * N + 3 * N * 4) / 1e9 + 0.8


## 14. Run everything that is pending

In [ ]:
# The runner: executes every pending job in priority order until done, out of time, or out of disk.
# PIPELINE=True: this thread does the GPU-side work one job at a time (simulation, NN training, checks); each finished
# job's classifier fits go to FIT_POOL (one fit per core). While the pool already has a full queue, the GPU is given
# a pending neural-network job instead of waiting. Results do not depend on the order: every run has its own seeds.
def _pending(job, skip):
    return [r for r in job["runs"] if not CACHE.is_done(r["rid"]) and r["rid"] not in skip
            and (RETRY_FAILED or r["rid"] not in CACHE.failed)]

def refresh_plan():
    plan = build_plan()
    with LOCK:
        CACHE.pending = [(e["rid"], e["group"]) for e in plan if not CACHE.is_done(e["rid"])]
        CACHE.write()
    return plan, make_jobs(plan)

def benchmark_selection(jobs):
    """Representative full-size jobs: one final-test and one single-fold CV liquid job per encoder, one CNN and one GRU
    job (untuned and one tuning trial each), the cheap checks; with the pipeline, enough further jobs in plan order to
    keep every fit worker busy for a while (so the speed under full load is measured). All are ordinary runs."""
    want = [("lsm", True, e) for e in ENCS] + [("lsm", False, e) for e in ENCS] + [("nn", True, "cnn"), ("nn", True, "gru")]
    keys = {j["key"] for j in jobs if j["key"][0] in ("check", "brian2")}
    for kind, final, x in want:
        for j in jobs:
            s = j["runs"][0]["spec"]
            if s["kind"] != kind or not _pending(j, set()) or (s["split"] == "final") != final or s.get("arch"):
                continue
            if kind == "lsm" and (s["encoder"] != x or (not final and j["key"][2] == "cv-shared")):
                continue
            if kind == "nn" and s["model"] != x:
                continue
            keys.add(j["key"]); break
    for final in (True, False):        # one Huang job of each size: measures the SVM ensemble's speed
        j = next((j for j in jobs if j["runs"][0]["spec"]["kind"] == "huang" and (j["runs"][0]["spec"]["split"] == "final") == final
                  and _pending(j, set())), None)
        if j is not None:
            keys.add(j["key"])
    for m in ("cnn", "gru"):          # one tuning trial each: measures the speed of tuned architectures
        j = next((j for j in jobs if j["runs"][0]["spec"].get("arch") and j["runs"][0]["spec"]["model"] == m
                  and j["runs"][0]["spec"].get("trial") not in (None, 0, "best") and _pending(j, set())), None)
        if j is not None:
            keys.add(j["key"])
    if PIPELINE:
        n_fits = sum(len(estimate_parts(j, _pending(j, set()))[1]) for j in jobs if j["key"] in keys and _pending(j, set()))
        for j in jobs:
            if n_fits >= 3 * PIPE_W:
                break
            if j["key"] in keys or j["key"][0] not in ("lsm", "feat") or not _pending(j, set()):
                continue
            keys.add(j["key"]); n_fits += len(estimate_parts(j, _pending(j, set()))[1])
    else:
        single = next((j for j in jobs if j["key"][0] == "lsm" and len(_pending(j, set())) == 1 and j["key"][2] != "final"), None)
        if single is not None:
            keys.add(single["key"])        # a one-fit job: measures the fit speed when it gets 3 cores
    return keys

RAM_LOG = os.path.join(WORK_DIR, "logs", f"resources_{SESSION_TAG}.csv")
def log_resources(event):
    rss, peak = _proc_ram_gb()
    row = {"elapsed_h": round(elapsed_h(), 4), "event": event, "rss_gb": round(rss, 2), "peak_rss_gb": round(peak, 2),
           "available_gb": round(_avail_ram_gb(), 2), "reserved_gb": round(RAM.used, 2) if PIPELINE else None,
           "fits_waiting": FIT_POOL.queued if PIPELINE else None, "fits_running": FIT_POOL.running if PIPELINE else None,
           "jobs_holding_ram": len(FIT_POOL.jobs) if PIPELINE else None,
           "gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == "cuda" else None}
    new = not os.path.exists(RAM_LOG)
    with LOCK, open(RAM_LOG, "a") as f:
        if new:
            f.write(",".join(row) + "\n")
        f.write(",".join("" if v is None else str(v) for v in row.values()) + "\n")
    return row

def _guard(job, runs):
    """None if the job may start, else the reason to stop."""
    main, gcells = estimate_parts(job, runs)
    if PIPELINE:
        fits = fit_seconds_pipe(gcells)
        need = main + (FIT_POOL.backlog_s + sum(fits)) / FIT_POOL.n + (max(fits) if fits else 0)
    else:
        need = estimate_seconds(job, runs)
    if need > seconds_left():
        return f"time budget: next job needs ~{need/60:.1f} min (incl. queued fits), {max(seconds_left(), 0)/60:.1f} min left"
    if work_dir_gb() > DISK_LIMIT_GB:
        return f"disk: {WORK_DIR} is {work_dir_gb(force=True):.1f} GB (limit {DISK_LIMIT_GB})"
    return None

def run_job(job, runs, n):
    s0 = runs[0]["spec"]
    main, gcells = estimate_parts(job, runs)
    log(f"job {n}: {runs[0]['group']} | {s0['kind']} | {len(runs)} run(s) | e.g. {runs[0]['rid'][:80]} | "
        f"main {main/60:.1f} min, {len(gcells)} fit(s)")
    _JOB_CTX.clear()
    _JOB_CTX.update(ram_gb=job_ram_gb(job, runs) if PIPELINE else 0.0, submitted=False)
    if PIPELINE:
        RAM.reserve(_JOB_CTX["ram_gb"], headroom=2 * fit_ram_gb(NWIN * N_NEUR, len(DATA["morph"]["DS1"]["y"])))
    try:
        EXECUTORS[s0["kind"]](job, runs)
    except Exception:
        tb = traceback.format_exc()
        for r in runs:
            if not CACHE.is_done(r["rid"]) and not _JOB_CTX.get("submitted"):
                _record_failure(r["rid"], tb)
        log("  JOB FAILED:", tb.strip().splitlines()[-1])
    finally:
        if PIPELINE and not _JOB_CTX.get("submitted"):
            RAM.release(_JOB_CTX["ram_gb"])
    for r in runs:
        tried_this_session.add(r["rid"])       # never retry within the same session
    save_speeds()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def run_self_test():
    """SELF_TEST = True: check the machine and every stage of the code on a few beats, run no experiment."""
    res, t0 = {}, time.time()
    def item(name, ok, detail):
        res[name] = {"ok": bool(ok), "detail": detail}
        tag = "PASS" if ok else ("NOTE" if name.startswith("torch 2.4.0") else "FAIL")
        print(f"  [{tag}] {name:<28} {detail}")
    print("\n===== SELF-TEST (runs at the start of every session) =====")
    item("packages", True, ", ".join(f"{k} {ENV.get(k)}" for k in ("python", "torch", "numpy", "sklearn", "snntorch", "wfdb", "brian2")))
    item("torch 2.4.0 (authors')", TORCH_PINNED, torch.__version__ + ("" if TORCH_PINNED else " (fine; only matters for the exact-replication attempt)"))
    item("brian2", not str(ENV.get("brian2", "")).startswith("unavailable"), str(ENV.get("brian2")))
    if DEVICE.type == "cuda":
        x = torch.randn(4096, N_NEUR, device=DEVICE); w = torch.randn(N_NEUR, N_NEUR, device=DEVICE)
        torch.cuda.synchronize(); t = time.time()
        for _ in range(LIQ["num_sim_steps"]):
            y = x @ w
        torch.cuda.synchronize()
        item("GPU", True, f"{ENV.get('gpu')}, {ENV.get('gpu_mem_gb')} GB, CUDA {ENV.get('cuda')}; 650 liquid-sized steps in {time.time()-t:.2f} s")
        try:
            c = nn.Conv1d(1, 8, 5).to(DEVICE); c(torch.randn(16, 1, 577, device=DEVICE)).sum().backward()
            item("cuDNN", True, "convolution forward + backward ok")
        except Exception as ex:
            item("cuDNN", False, repr(ex))
    else:
        item("GPU", False, "PyTorch sees no GPU: everything would run on the CPU (very slow)")
    item("authors' code", ENV.get("authors_commit") == AUTHORS_COMMIT, f"commit {ENV.get('authors_commit')}")
    cs = CHECKS.get("checksums", {})
    item("MIT-BIH checksums", not cs.get("mismatches"), f"{cs.get('files_checked')} files checked, {len(cs.get('mismatches', []))} mismatches, "
         f"missing (unused) {len(cs.get('missing', []))}")
    item("DS1 = authors' order", CHECKS.get("ds1_matches_authors_order"), f"class counts {CHECKS.get('class_counts')}")
    item("authors' folds", CHECKS.get("folds_regenerate_exactly"), "fold indices regenerate exactly")
    try:
        D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
        X1n, X2n, _ = normalize_final(D1["X"][:64], D2["X"][:64], "robust")
        for mode in ("published", "fixed"):
            a = PAPER_AIN["multi"] if MODES[mode]["ain"] == "paper" else measure_ain(X1n, np.arange(64), "multi")
            mdl = build_liquid(200, I_TARGET / (a * LIQ["P_in"]), SEEDS[0], MODES[mode]["gamma_fix"])
            C, st = simulate(mdl, X1n, "multi", 64, 200, y=D1["y"][:64], enc_seed=1)
            F = counts_to_rates(C)
            yy = np.arange(64) % 3
            fit_readout(lambda r: F[r], 64, yy, lambda r: F[r], 64, "hinge", 1)
            item(f"liquid + readout ({mode})", True, f"64 beats: firing {st['liquid_firing_prob_per_step']:.4f}/step, "
                 f"drive/target {st['input_current_over_target']:.2f}")
    except Exception as ex:
        item("liquid + readout", False, f"{ex.__class__.__name__}: {ex}")
    try:   # the GPU simulator behind every result vs the float64 equations that Brian2 matched (same beats as the Brian2 check)
        D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
        X1n, X2n, _ = normalize_final(D1["X"], D2["X"], "robust")
        a_meas = measure_ain(X1n, np.arange(len(X1n)), "multi")
        pick = np.concatenate([np.where(D2["y"] == c)[0][:k] for c, k in
                               ((0, BRIAN2_BEATS - 2 * (BRIAN2_BEATS // 3)), (1, BRIAN2_BEATS // 3), (2, BRIAN2_BEATS // 3))])
        S = encode(X2n[pick], "multi")
        gv = {"session": SESSION_TAG, "device": str(DEVICE), "gpu": ENV.get("gpu"), "beats_DS2": pick.tolist(), "liquids": {}}
        for mode in ("fixed", "published"):
            a = a_meas if MODES[mode]["ain"] == "measured" else PAPER_AIN["multi"]
            mdl = build_liquid(200, I_TARGET / (a * LIQ["P_in"]), SEEDS[0], MODES[mode]["gamma_fix"])
            ref = _torch_raster(mdl, S, torch.float64, False)            # (beats, 650, 343): what Brian2 matched exactly
            ref_c = ref[:, :NWIN * WIN].reshape(len(pick), NWIN, WIN, N_NEUR).sum(2).reshape(len(pick), -1)
            sim_c, _ = simulate(mdl, X2n[pick], "multi", len(pick), 200, enc_seed=0)   # window counts, as in every run
            sim_c = sim_c.astype(np.int64)
            gv["liquids"][mode] = {"window_count_cells": int(ref_c.size), "mismatches": int((sim_c != ref_c).sum()),
                                   "spikes_reference": int(ref_c.sum()), "spikes_simulate": int(sim_c.sum())}
        jdump(gv, os.path.join(WORK_DIR, "checks", "gpu_vs_reference.json"))
        item("GPU simulator vs reference", sum(v["mismatches"] for v in gv["liquids"].values()) == 0,
             "; ".join(f"{m}: {v['mismatches']} of {v['window_count_cells']:,} window counts differ" for m, v in gv["liquids"].items())
             + f" (simulate() on {DEVICE} vs the float64 equations Brian2 matched, {len(pick)} DS2 beats)")
    except Exception as ex:
        item("GPU simulator vs reference", False, f"{ex.__class__.__name__}: {ex}")
    try:   # extensions: Huang ensemble on a few beats, the hybrid rule, the RR window counts used by ELM + RR
        D1 = DATA["morph"]["DS1"]
        rows = np.sort(np.concatenate([np.where(D1["y"] == c)[0][:k] for c, k in ((0, 120), (1, 40), (2, 40))]))
        hs = huang_spec(SEEDS[0], HUANG_GAMMAS[0], "final"); hs["n_matrices"] = 2
        yp, vote, votes, _ = huang_fit_predict(hs, rows[::2], rows[1::2], D1, D1)
        ok_rule = (hybrid_rule([0, 1, 2, 0], [0.5, 0.9, 0.5, 0.9], "replace", 0.8).tolist() == [1, 0, 2, 0]
                   and hybrid_rule([0, 1, 2, 0], [0.5, 0.9, 0.5, 0.9], "add", 0.8).tolist() == [1, 1, 2, 0])
        thr = rr_thresholds(D1["rr"]); T_in = D1["X"].shape[1]
        ch = rr_channels(D1["rr"][:8], thr, T_in)
        pad = np.zeros((8, NWIN * WIN, ch.shape[2]), np.int64); pad[:, :min(T_in, NWIN * WIN)] = ch[:, :NWIN * WIN]
        ok_rrw = np.array_equal(rr_window_counts(D1["rr"][:8], thr, T_in), pad.reshape(8, NWIN, WIN, -1).sum(2))
        rr9 = rr_inputs(D1, np.arange(len(D1["y"])), True)
        ok_rrw = ok_rrw and rr_channels(rr9[:8], rr_thresholds(rr9), T_in).shape[2] == 4 * RR_LEVELS
        item("extensions (Huang, hybrid, ELM+RR)", ok_rule and ok_rrw and len(yp) == len(rows[1::2]),
             f"Huang 2-matrix ensemble on {len(rows)} beats: Macro-F1 {compute_metrics(D1['y'][rows[1::2]], yp)['macro_f1']:.3f}; "
             f"hybrid rule {'ok' if ok_rule else 'WRONG'}; RR window counts {'ok' if ok_rrw else 'WRONG'}; "
             f"Huang beats {D1['huang_beats'].shape}, mean-RR ratio median {np.median(D1['rr_mean_ratio']):.3f}")
    except Exception as ex:
        item("extensions (Huang, hybrid, ELM+RR)", False, f"{ex.__class__.__name__}: {ex}")
    try:   # working liquid: g_rec multiplies only the recurrent weights; the full raster matches simulate()
        D1 = DATA["morph"]["DS1"]
        Xw, _, _ = normalize_final(D1["X"][:64], D1["X"][:1], "robust")
        m1 = build_liquid(200, 0.3, SEEDS[0], True, WORKING_LAMBDA, True)
        m3 = build_liquid(200, 0.3, SEEDS[0], True, WORKING_LAMBDA, True, 3.0)
        ok_g = bool(torch.allclose(m3.rsv_weights, 3.0 * m1.rsv_weights) and torch.equal(m3.input_rsv_weights, m1.input_rsv_weights))
        Rw, _, _ = liquid_raster(m3, encode(Xw[:4], "multi"))
        cw, _ = simulate(m3, Xw[:4], "multi", 4, 200, enc_seed=0)
        ok_r = np.array_equal(Rw[:, :NWIN * WIN].reshape(4, NWIN, WIN, N_NEUR).sum(2).reshape(4, -1), cw.astype(np.int64))
        item("working liquid (g_rec, raster)", ok_g and ok_r,
             f"g_rec scales recurrent weights only: {ok_g}; raster == simulate(): {ok_r}; "
             f"lambda {WORKING_LAMBDA:g}: {structure_stats(m1)['n_recurrent_synapses']} recurrent synapses")
    except Exception as ex:
        item("working liquid (g_rec, raster)", False, f"{ex.__class__.__name__}: {ex}")
    free = shutil.disk_usage(WORK_DIR).free / 1e9
    item("disk", free > 15, f"{free:.0f} GB free at {WORK_DIR} (all outputs together grow to ~12 GB)")
    item("RAM / cores", True, f"{ENV.get('ram_gb')} GB RAM, {N_CPU} cores -> {PIPE_W} parallel fits" if PIPELINE else f"{N_CPU} cores")
    item("log file", os.path.exists(LOG_PATH), f"{LOG_PATH} (tail -f {os.path.join(WORK_DIR, 'logs', 'latest.log')})")
    n_fail = sum(not v["ok"] for k, v in res.items() if k not in ("torch 2.4.0 (authors')",))
    print(f"===== SELF-TEST: {'all checks passed' if n_fail == 0 else str(n_fail) + ' check(s) FAILED'} ({time.time()-t0:.0f} s) =====\n")
    jdump({"passed": n_fail == 0, "checks": res, "env": ENV, "session": SESSION_TAG}, os.path.join(WORK_DIR, "checks", "self_test.json"))
    return n_fail

# ---------------- core search (local machine, benchmark only) ----------------
CORE_SEARCH_EPOCHS, CORE_SEARCH_ROWS = 3, 16000

def _core_counts():
    cap = max(1, min(N_CPU, PIPE_BY_RAM))
    return sorted({k for k in (1, 2, 3, 4, 6, 8, 10, 12, 14, 16, 20, 24, 28, 32, 40, 48, 56, 64) if k <= cap} | {cap})

def _score_workers(table, sim_uncontended, fit_anchor=None):
    """Projected total hours for each worker count: fit speed and simulation slowdown as measured by the search
    (scaled to the benchmark's own measurements when fit_anchor = (K, measured s/Gcell at K) is given)."""
    global PIPE_W
    keep = (PIPE_W, dict(SPEED["sim_beats_per_s"]), SPEED["fit_s_per_Gcell"]["pipe"])
    ref = {r["workers"]: r for r in table}
    out = []
    try:
        for r in table:
            PIPE_W = r["workers"]
            SPEED["sim_beats_per_s"] = {e: v * r["sim_speed_factor"] for e, v in sim_uncontended.items()}
            SPEED["fit_s_per_Gcell"]["pipe"] = (r["fit_s_per_Gcell"] if fit_anchor is None else
                                                fit_anchor[1] * r["fit_s_per_Gcell"] / ref[fit_anchor[0]]["fit_s_per_Gcell"])
            df = project_time(JOBS)
            out.append({**r, "projected_hours": float(df.hours.sum()) if len(df) else 0.0})
    finally:
        PIPE_W, SPEED["sim_beats_per_s"], SPEED["fit_s_per_Gcell"]["pipe"] = keep[0], keep[1], keep[2]
    best = min(out, key=lambda r: r["projected_hours"])
    # fewer workers if within 1 % (less RAM, same speed)
    best = min((r for r in out if r["projected_hours"] <= best["projected_hours"] * 1.01), key=lambda r: r["workers"])
    return out, best

def core_search():
    """Runs K classifier fits at once for several K while the GPU keeps simulating, as in the real pipeline, and
    measures (a) how long a fit takes at each K and (b) how much the simulation slows down. The projected total time of
    the whole plan then decides K. The worker count never changes a result, only the speed."""
    t_all = time.time()
    D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
    X1n, _, _ = normalize_final(D1["X"], D2["X"], "robust")
    rows = np.sort(np.random.default_rng(0).choice(len(D1["y"]), min(CORE_SEARCH_ROWS, len(D1["y"])), replace=False))
    a = measure_ain(X1n, np.arange(len(X1n)), "multi")
    mdl = build_liquid(200, I_TARGET / (a * LIQ["P_in"]), SEEDS[0], True)
    C, _ = simulate(mdl, X1n[rows], "multi", SIM_BATCH_CV, 200, y=D1["y"][rows], enc_seed=7)
    F = lambda r: counts_to_rates(C[r]); y = D1["y"][rows]; n = len(rows)
    gcells = n * C.shape[1] / 1e9
    prep = prepare_train(F, n, True)
    Xsim = X1n[rows[:min(SIM_BATCH_CV, n)]]
    def sim_until(stop_fn, min_s=0.0):
        beats, t0 = 0, time.time()
        while not stop_fn() or time.time() - t0 < min_s:
            simulate(mdl, Xsim, "multi", SIM_BATCH_CV, 200, enc_seed=7); beats += len(Xsim)
        return beats / (time.time() - t0)
    sim_alone = sim_until(lambda: True, min_s=10.0)
    log(f"core search: {n} beats x {C.shape[1]} features, {CORE_SEARCH_EPOCHS} epochs per fit; simulation alone {sim_alone:.0f} beats/s; "
        f"trying {_core_counts()} parallel fits (RAM allows {PIPE_BY_RAM}, CPU cores {N_CPU})")
    table = []
    for K in _core_counts():
        durs, lock = [], threading.Lock()
        def one():
            t = time.time()
            fit_readout(F, n, y, F, 1, "hinge", 1, prepared=prep, n_jobs=1, epochs=CORE_SEARCH_EPOCHS)
            with lock:
                durs.append(time.time() - t)
        th = [threading.Thread(target=one) for _ in range(K)]
        t0 = time.time()
        for t in th:
            t.start()
        sim_bps = sim_until(lambda: not any(t.is_alive() for t in th))
        for t in th:
            t.join()
        wall = time.time() - t0
        per_fit = float(np.mean(durs))
        r = {"workers": K, "seconds_per_fit": round(per_fit, 2), "wall_seconds": round(wall, 2),
             "fit_s_per_Gcell": per_fit / gcells * READOUT_EPOCHS / CORE_SEARCH_EPOCHS,
             "fits_per_hour": 0.0,
             "sim_beats_per_s": round(sim_bps), "sim_speed_factor": sim_bps / sim_alone}
        r["fits_per_hour"] = round(K * 3600 / (r["fit_s_per_Gcell"] * len(D1["y"]) * C.shape[1] / 1e9), 1)   # full-size final-test fits
        table.append(r)
        log(f"  {K:3d} parallel fits: {per_fit:6.1f} s per fit, {r['fit_s_per_Gcell']:5.0f} s/Gcell, "
            f"~{r['fits_per_hour']:5.0f} full-size fits/hour; simulation at {r['sim_speed_factor']:.0%} of its speed alone")
        gc.collect()
    del C, prep
    sim_unc = {e: (v if f"sim_beats_per_s:{e}" in SPEED_MEASURED else sim_alone) for e, v in SPEED["sim_beats_per_s"].items()}
    scored, best = _score_workers(table, sim_unc)
    for r in scored:
        log(f"  {r['workers']:3d} workers -> projected total {r['projected_hours']:6.1f} h" + ("   <- chosen" if r is best else ""))
    cs = {"session": SESSION_TAG, "cpu_count": N_CPU, "gpu": ENV.get("gpu"), "cpu_model": ENV.get("cpu_model"), "ram_gb": ENV.get("ram_gb"),
          "best_workers": best["workers"], "table": scored, "sim_uncontended_beats_per_s": sim_unc, "rescored_after_benchmark": False,
          "seconds": round(time.time() - t_all)}
    jdump(cs, CORE_SEARCH_PATH)
    pd.DataFrame(scored).to_csv(os.path.join(WORK_DIR, "tables", "core_search.csv"), index=False)
    log(f"core search done in {(time.time() - t_all) / 60:.1f} min: {best['workers']} parallel fits (saved for later sessions in {CORE_SEARCH_PATH})")
    return cs

def core_search_rescore(cs):
    """After the benchmark: redo the choice with the speeds the benchmark measured at the chosen worker count."""
    K = cs["best_workers"]
    if "fit_s_per_Gcell:pipe" not in SPEED_MEASURED:
        return
    r_k = next(r for r in cs["table"] if r["workers"] == K)
    sim_unc = {e: v / max(r_k["sim_speed_factor"], 1e-6) for e, v in SPEED["sim_beats_per_s"].items()}
    scored, best = _score_workers(cs["table"], sim_unc, fit_anchor=(K, SPEED["fit_s_per_Gcell"]["pipe"]))
    log("core search, re-scored with the benchmark's measured speeds:")
    for r in scored:
        log(f"  {r['workers']:3d} workers -> projected total {r['projected_hours']:6.1f} h" + ("   <- best" if r is best else ""))
    cs.update({"table": scored, "best_workers": best["workers"], "rescored_after_benchmark": True, "sim_uncontended_beats_per_s": sim_unc})
    jdump(cs, CORE_SEARCH_PATH)
    pd.DataFrame(scored).to_csv(os.path.join(WORK_DIR, "tables", "core_search.csv"), index=False)
    if best["workers"] != K:
        log(f"  -> later sessions will use {best['workers']} parallel fits (this benchmark used {K})")

PLAN, JOBS = refresh_plan()
tried_this_session, STOP_REASON, n_jobs_run = set(), None, 0
SELF_TEST_FAILS = 0 if MERGE_MODE else run_self_test()   # every session (not merge mode); changes no result
CORE_SEARCH_RESULT = None
if (CORE_SEARCH and MACHINE == "local" and BENCHMARK and PIPELINE and FIT_WORKERS == "auto" and not SELF_TEST and not MERGE_MODE):
    CORE_SEARCH_RESULT = core_search()
    PIPE_W = CORE_SEARCH_RESULT["best_workers"]
RAM = RamBudget(PIPE_BUDGET_GB) if PIPELINE else None
FIT_POOL = FitPool(PIPE_W, RAM) if PIPELINE else None
BENCH_KEYS = benchmark_selection(JOBS) if BENCHMARK else None
if MERGE_MODE:
    BENCH_KEYS = set()                        # merge mode: run nothing, only combine and analyse
    log(f"MERGE mode: combined {len(MERGE_DIRS)} output folders into {WORK_DIR}; nothing is run")
elif SELF_TEST:
    BENCH_KEYS = set()                        # SELF_TEST = True: stop after the checks
    if SMOKE_TEST or BENCHMARK:
        log("NOTE: SELF_TEST = True stops after the checks, so the smoke test / benchmark did not run. "
            "The checks run at the start of every session anyway: set SELF_TEST = False to run them.")
elif SELF_TEST_FAILS:
    log(f"WARNING: {SELF_TEST_FAILS} self-test check(s) failed (see above); continuing")
if BENCHMARK:
    log(f"BENCHMARK mode: {len(BENCH_KEYS)} jobs selected (checks + representative full-size jobs)")
log(f"session start: {len(PLAN)} planned runs, {len(CACHE.done)} already done, {len(CACHE.pending)} pending; "
    f"budget {SESSION_BUDGET_HOURS} h, disk {work_dir_gb(force=True):.2f} GB; "
    + (f"pipeline with {PIPE_W} fit workers, RAM budget {PIPE_BUDGET_GB:.1f} GB" if PIPELINE else "one job at a time"))
log_resources("start")
_done_at_refresh, _last_status = len(CACHE.done), time.time()

def _next_job(nn_only=False):
    for j in JOBS:
        if BENCH_KEYS is not None and j["key"] not in BENCH_KEYS:
            continue
        if nn_only and j["key"][0] != "nn":
            continue
        pr = _pending(j, tried_this_session)
        if pr:
            return j, pr
    return None

def _maybe_refresh(force=False):
    global PLAN, JOBS, _done_at_refresh
    if force or len(CACHE.done) != _done_at_refresh:
        PLAN, JOBS = refresh_plan()            # second stages of the re-tunings appear once their first stage is done
        _done_at_refresh = len(CACHE.done)

while True:
    if PIPELINE and time.time() - _last_status > 600:
        r = log_resources("status")
        log(f"  status: fits waiting {r['fits_waiting']}, running {r['fits_running']}, done {FIT_POOL.n_done}, "
            f"failed {FIT_POOL.n_failed}; RAM reserved {r['reserved_gb']} GB, available {r['available_gb']} GB, "
            f"process peak {r['peak_rss_gb']} GB")
        _last_status = time.time()
    if PIPELINE and FIT_POOL.queued >= FIT_POOL.n:
        todo = _next_job(nn_only=True)        # the fit queue is full: give the GPU a neural-network job meanwhile
        if todo is None:
            FIT_POOL.wait_any(5.0)
            continue
    else:
        todo = _next_job()
    if todo is None:
        if PIPELINE and FIT_POOL.outstanding() > 0:
            FIT_POOL.wait_any(10.0)
            _maybe_refresh()
            continue
        _maybe_refresh(force=True)
        if _next_job() is not None:
            continue
        STOP_REASON = ("merge mode, no experiment run" if MERGE_MODE else "self-test only, no experiment run" if SELF_TEST else
                       "benchmark jobs finished" if BENCHMARK else "all planned runs are done")
        break
    job, runs = todo
    reason = _guard(job, runs)
    if reason:
        STOP_REASON = reason
        break
    n_jobs_run += 1
    run_job(job, runs, n_jobs_run)
    if any(set(r["groups"]) & {"retune", "nn_tune"} for r in runs) or n_jobs_run % 25 == 0:
        _maybe_refresh()
    else:
        with LOCK:
            CACHE.pending = [(rid, g) for rid, g in CACHE.pending if not CACHE.is_done(rid)]
            CACHE.write()
    if n_jobs_run % 10 == 0:
        log_resources(f"job {n_jobs_run}")
    gc.collect()

if PIPELINE:
    log(f"no more jobs will start ({STOP_REASON}); waiting for {FIT_POOL.outstanding()} classifier fit(s) to finish")
    while FIT_POOL.outstanding() > 0:
        if seconds_left() < 0 and not FIT_POOL.cancel.is_set():
            log("  time is up: cancelling the remaining fits (they stay pending for the next session)")
            FIT_POOL.cancel.set()
        FIT_POOL.wait_any(10.0)
        if time.time() - _last_status > 600:
            r = log_resources("draining")
            log(f"  draining: {r['fits_waiting']} waiting, {r['fits_running']} running")
            _last_status = time.time()
    FIT_POOL.ex.shutdown(wait=True)
    log_resources("end")
    save_speeds()
    log(f"fit pool: {FIT_POOL.n_done} fits done, {FIT_POOL.n_failed} failed, {FIT_POOL.n_cancelled} cancelled")

PLAN, JOBS = refresh_plan()
if CORE_SEARCH_RESULT is not None:
    core_search_rescore(CORE_SEARCH_RESULT)
    PIPE_W = CORE_SEARCH_RESULT["best_workers"]          # the projection below uses the final choice
log(f"runner stopped: {STOP_REASON}. jobs this session: {n_jobs_run}; done {len(CACHE.done)}, "
    f"failed {len(CACHE.failed)}, pending {len(CACHE.pending)}; speeds {json.dumps(SPEED, default=_json_default)}")
TIME_PROJECTION = print_projection(JOBS, "BENCHMARK RESULT: projected time for everything that is left" if BENCHMARK
                                   else "Projected time for everything that is left")


## 15. Analysis of the redesign: F1 to F7 and the claims

In [ ]:
# Analysis of the redesign (plan, Statistics and decision rules): families F1 to F7 with Holm within each family,
# the claims A to E, the ladder's table, and summary_redesign.md. Tolerates partial results.
TBL2 = os.path.join(WORK_DIR, "tables", "redesign"); os.makedirs(TBL2, exist_ok=True)
from lsm2.stats import per_split_seed, compare, compare_diff, holm, verdict, family, NB_RATIO

def load_runs(kinds=None):
    rows = []
    for rid, v in CACHE.done.items():
        d = os.path.join(WORK_DIR, "runs", rid)
        try:
            cfg = json.load(open(os.path.join(d, "config.json"))); met = json.load(open(os.path.join(d, "metrics.json")))
        except Exception:
            continue
        s = cfg["spec"]
        if kinds and s.get("kind") not in kinds:
            continue
        r = {"run_id": rid, "group": cfg["group"], **{f"spec_{k}": (json.dumps(x) if isinstance(x, (dict, list)) else x) for k, x in s.items()},
             **{k: x for k, x in met.items() if not isinstance(x, (list, dict))}}
        p = os.path.join(d, "predictions.npz")
        if s.get("split") == "final" and os.path.exists(p):      # DS2 without record 232
            z = np.load(p); keep = z["patient_id"] != "232"
            m = compute_metrics(z["y_true"][keep].astype(int), z["y_pred"][keep].astype(int))
            r["macro_f1_no232"] = m["macro_f1"]
        rows.append(r)
    return pd.DataFrame(rows)

def _series(df, **sel):
    d = df
    for k, v in sel.items():
        col = f"spec_{k}"
        if col not in d:
            return None
        d = d[d[col].isna()] if v is None else d[d[col] == v]
    return d

def redesign_analysis():
    R2 = load_runs({"lsm2", "feat2", "hybrid", "feat", "lsm"})
    if R2.empty or "spec_kind" not in R2:
        print("redesign analysis: no runs yet"); return {}
    L = R2[R2.spec_kind == "lsm2"]
    def lsm2(liquid, readout, fe=CFG11, variant=None, loss="svm"):
        return _series(L, liquid=liquid, readout=readout, filter=fe[0], norm=fe[1], encoder=fe[2], variant=variant, loss=loss)
    old = R2[R2.spec_kind.isin(["feat", "lsm"])]
    elm = _series(old, kind="feat", model="elm", encoder="multi")
    noliq = _series(old, kind="feat", model="no_liquid", encoder="multi")
    elm_rr = _series(old, kind="feat", model="elm_rr", encoder="multi")
    fixed_rrliq = _series(old[old.spec_kind == "lsm"], mode="fixed", filter="morph", norm="robust", encoder="multi",
                          readout="svm", rr_in_liquid=True, recurrence=True) if "spec_rr_in_liquid" in old else None
    if fixed_rrliq is not None and "spec_rr_set" in fixed_rrliq:
        fixed_rrliq = fixed_rrliq[fixed_rrliq.spec_rr_set.isna()]
    if e3_partner_needed():
        fixed_rrliq = lsm2("X", "film", variant="rrliq")
    F2r = R2[R2.spec_kind == "feat2"]
    ctrl = lambda m, ro: _series(F2r, model=m, readout=ro)
    fams = []
    ros = ["film"] + list(SNAP_READOUTS)
    fams += family("F1 recurrence", [(ro, compare(lsm2("final", ro), lsm2("final_off", ro), True)) for ro in ros], True)
    fams += family("F2 added factors", [(ro, compare(lsm2("final", ro), lsm2("X", ro))) for ro in ros])
    fams += family("F3 other models", [("vs ELM", compare(lsm2("final", "film"), elm)), ("vs no liquid", compare(lsm2("final", "film"), noliq))])
    cfg = lambda fe, loss: lsm2("final", "film", fe, loss=loss)
    fams += family("F4 pipeline", [("multi vs rate", compare(cfg(CFG11, "svm"), cfg(("morph", "robust", "rate"), "svm"))),
                                   ("multi vs latency", compare(cfg(CFG11, "svm"), cfg(("morph", "robust", "latency"), "svm")))])
    fams += family("F5 with RR", [("RR in liquid vs paper's fixed + RR in liquid", compare(lsm2("final", "film", variant="rrliq"), fixed_rrliq)),
                                  ("RR in liquid vs ELM + RR", compare(lsm2("final", "film", variant="rrliq"), elm_rr)),
                                  ("RR readout vs film", compare(lsm2("final", "film_rr"), lsm2("final", "film")))])
    f6 = []
    for ro in SNAP_READOUTS:
        for m, lab in (("fade", "fading-memory baseline"), ("selm", "snapshot ELM"), ("noliq30", "no-liquid control")):
            f6.append((f"{ro} vs {lab}", compare(lsm2("final", ro), ctrl(m, ro))))
    fams += family("F6 beyond plain memory", f6)
    fams += family("F7 memory ablation", [(ro, compare_diff(lsm2("final", ro), lsm2("final_off", ro), lsm2("final", ro, variant="ablated"),
                                                            lsm2("final_off", ro, variant="ablated"))) for ro in SNAP_READOUTS])
    F = pd.DataFrame(fams)
    F.to_csv(os.path.join(TBL2, "families.csv"), index=False)
    # F4: Friedman, Iman-Davenport and Kendall's W over the 24 configurations, blocks = the 12 CV splits, per seed
    f4 = []
    E2 = L[(L.spec_readout == "film") & L.spec_variant.isna() & (L.spec_liquid == "final") & L.spec_split.astype(str).str.startswith("cv-")]
    for sd in sorted(E2.spec_seed.dropna().unique()) if not E2.empty else []:
        e = E2[E2.spec_seed == sd].copy()
        e["cid"] = [CONFIGS_BY_FE.get(((a, b, c), d)) for a, b, c, d in zip(e.spec_filter, e.spec_norm, e.spec_encoder, e.spec_loss)]
        M = e.pivot_table(index="spec_split", columns="cid", values="macro_f1")
        if M.shape[1] >= 3 and M.shape[0] >= 3 and not M.isna().any().any():
            st, ranks, _ = friedman_block(M.values, list(M.columns))
            f4.append({"seed": sd, **st, "n_configs": M.shape[1], "n_blocks": M.shape[0]})
    pd.DataFrame(f4).to_csv(os.path.join(TBL2, "F4_friedman.csv"), index=False)
    # claims (plan, Claims and what each needs)
    lad = json.load(open(os.path.join(WORK_DIR, "checks", "stage1", "ladder.json"))) if os.path.exists(os.path.join(WORK_DIR, "checks", "stage1", "ladder.json")) else {}
    conf = [s for s in lad.get("steps", []) if s.get("pass")]
    A_ = bool(conf) and all(r.get("level4", {}).get("pass") for k, r in lad.get("records", {}).items() if k.startswith("confirm|") and r.get("pass"))
    v = {(r["family"], r["comparison"]): r for r in fams}
    snaps_mem = ["snap342", "snap396"]
    B_at = [ro for ro in snaps_mem if v.get(("F1 recurrence", ro), {}).get("verdict") == "meaningful"
            and v.get(("F7 memory ablation", ro), {}).get("verdict") == "meaningful"
            and v.get(("F1 recurrence", ro), {}).get("cv_diff", -1) > v.get(("F1 recurrence", "snap288"), {}).get("cv_diff", np.inf)]
    C_at = [ro for ro in B_at if v.get(("F6 beyond plain memory", f"{ro} vs fading-memory baseline"), {}).get("verdict") == "meaningful"
            and v.get(("F6 beyond plain memory", f"{ro} vs snapshot ELM"), {}).get("verdict") == "meaningful"]
    D_ = (v.get(("F1 recurrence", "film"), {}).get("verdict") == "meaningful"
          and v.get(("F3 other models", "vs ELM"), {}).get("verdict") == "meaningful"
          and v.get(("F3 other models", "vs no liquid"), {}).get("verdict") == "meaningful")
    claims = {"A. the liquid remembers": A_, "B. its memory helps classify beats": bool(B_at), "B at": B_at,
              "C. more than plain memory": bool(C_at), "C at": C_at, "D. helps the paper's pipeline": bool(D_),
              "E. a better classifier overall": "not claimed",
              "wording": ("the LSM does help" if (A_ and B_at and C_at) else
                          "narrow wording (A and B)" if (A_ and B_at) else "neither")}
    jdump(claims, os.path.join(TBL2, "claims.json"))
    lines = ["# Redesign: results so far", f"_Updated {datetime.datetime.utcnow().isoformat(timespec='seconds')} UTC_", ""]
    if CAL:
        lines += [f"Final liquid: **{CAL['final_key']}** (n = {CAL['n_sub']}, g {CAL['g']}, Maass scale {CAL['maass_scale']}), "
                  f"confirmation attempts {CAL['attempts']}; check 1f {'passed' if CAL['level1f_pass'] else 'failed (partner runs redone)'}", ""]
    lines += ["## Families (Holm within each)", F.drop(columns=[c for c in F.columns if c in ("t",)], errors="ignore").round(4).to_string(index=False) if not F.empty else "none yet", ""]
    if f4:
        lines += ["## F4: Friedman over the 24 configurations (per seed)", pd.DataFrame(f4).round(4).to_string(index=False), ""]
    lines += ["## Claims", "```", json.dumps(claims, indent=2, default=str), "```"]
    atomic_write_text(os.path.join(WORK_DIR, "summary_redesign.md"), "\n".join(lines) + "\n")
    print("\n".join(lines[:4]))
    cols = [c for c in ("family", "comparison", "cv_diff", "ds2_diff", "verdict") if c in F]
    print(F[cols].to_string(index=False) if not F.empty else "")
    return {"families": F, "claims": claims}

def friedman_block(M, ids):
    n, k = M.shape
    chi2, p = sstats.friedmanchisquare(*[M[:, j] for j in range(k)])
    W = chi2 / (n * (k - 1)); FF = (n - 1) * chi2 / (n * (k - 1) - chi2)
    pFF = sstats.f.sf(FF, k - 1, (k - 1) * (n - 1))
    ranks = np.vstack([sstats.rankdata(-r) for r in M]).mean(0)
    return {"chi2": chi2, "p": p, "iman_davenport_F": FF, "p_F": pFF, "kendall_W": W, "best_by_rank": ids[int(np.argmin(ranks))]}, ranks, None

try:
    REDESIGN = redesign_analysis()
except Exception:
    print("redesign analysis failed:\n" + traceback.format_exc())


## 16. The replication's tables and figures (optional)

In [ ]:
if OLD_ANALYSIS:
    # Analysis: master table, paper-style tables and statistics, comparisons, figures, summary.md.
    # Every section tolerates partial results (it reports what exists so far).
    TBL, FIG = os.path.join(WORK_DIR, "tables"), os.path.join(WORK_DIR, "figures")
    INK, INK2, SURF, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e4e3df"
    SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]            # validated categorical slots 1-3
    plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "axes.edgecolor": GRID, "axes.labelcolor": INK2,
                         "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "axes.grid": True, "grid.color": GRID,
                         "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9,
                         "legend.frameon": False})

    def load_results():
        rows = []
        for rid in sorted(os.listdir(os.path.join(WORK_DIR, "runs"))):
            d = os.path.join(WORK_DIR, "runs", rid)
            try:
                cfg = json.load(open(os.path.join(d, "config.json"))); met = json.load(open(os.path.join(d, "metrics.json")))
            except Exception:
                continue
            liq = json.load(open(os.path.join(d, "liquid_stats.json"))) if os.path.exists(os.path.join(d, "liquid_stats.json")) else {}
            s = cfg["spec"]
            r = {"run_id": rid, "group": cfg["group"], "groups": ",".join(cfg.get("groups", [cfg["group"]])),
                 "config_id": cfg.get("config_id"), **{f"spec_{k}": v for k, v in s.items()},
                 **{k: v for k, v in met.items() if k not in ("confusion_matrix", "eval_macro_f1_by_epoch")},
                 "confusion_matrix": json.dumps(met["confusion_matrix"])}
            if isinstance(s.get("arch"), dict):
                r["spec_arch"] = json.dumps(s["arch"], sort_keys=True)
            sim = liq.get("DS1_simulation") or {}
            r.update({"A_in_used": liq.get("A_in_used"), "mu_in": liq.get("mu_in"),
                      "liquid_firing_prob": sim.get("liquid_firing_prob_per_step"),
                      "drive_over_target": sim.get("input_current_over_target"), "silent_neurons": sim.get("silent_neurons"),
                      "n_recurrent_synapses": (liq.get("structure") or {}).get("n_recurrent_synapses")})
            rows.append(r)
        return pd.DataFrame(rows)

    R = load_results()
    R.to_csv(os.path.join(WORK_DIR, "results_master.csv"), index=False)
    print(f"results_master.csv: {len(R)} runs" + ("  (nothing has run yet, so there is nothing to analyse)" if R.empty else ""))
    SUMMARY = []          # markdown lines for summary.md
    KEY = ["accuracy", "macro_f1", "recall_N", "recall_SVEB", "recall_VEB", "precision_N", "precision_SVEB",
           "precision_VEB", "f1_N", "f1_SVEB", "f1_VEB"]

    def _is_cv(df):
        return df["spec_split"].astype(str).str.startswith("cv-")

    def lsm_base(df):
        if df.empty or "spec_kind" not in df:
            return df.iloc[0:0]
        base = df[(df.spec_kind == "lsm") & df.config_id.notna()]
        if "spec_ain_override" in base:                        # k-stale runs saved with a config id before the fix
            base = base[base.spec_ain_override.isna()]
        return base

    def mean_sd(df, by, cols=("macro_f1", "accuracy", "recall_SVEB", "recall_VEB")):
        g = df.groupby(by)[list(cols)]
        out = g.mean().add_suffix("_mean").join(g.std(ddof=1).add_suffix("_sd")).join(g.size().rename("n"))
        return out.reset_index()

    def md_table(df, floatfmt=4, max_rows=40):
        if df is None or df.empty:
            return "_no results yet_\n"
        d = df.head(max_rows).copy()
        for c in ("config_id", "seed", "spec_seed", "n", "n_seeds", "n_blocks", "n_configs", "n_splits", "best_by_rank", "vs_config",
                  "spec_gain", "gain", "spec_num_levels", "num_levels", "folds", "identical", "runs", "jobs"):
            if c in d.columns and d[c].dtype.kind == "f" and d[c].dropna().apply(float.is_integer).all():
                d[c] = d[c].astype("Int64")
        for c in d.columns:
            if d[c].dtype.kind == "f":
                d[c] = d[c].map(lambda v: "" if pd.isna(v) else f"{v:.{floatfmt}f}")
        head = "| " + " | ".join(map(str, d.columns)) + " |\n|" + "---|" * len(d.columns) + "\n"
        return head + "".join("| " + " | ".join(map(str, r)) + " |\n" for r in d.values)

    def safe(section):
        def deco(fn):
            if R.empty:                       # nothing has run yet: nothing to analyse
                return fn
            try:
                fn()
            except Exception as ex:
                SUMMARY.append(f"\n> Section **{section}** could not be produced yet: `{ex.__class__.__name__}: {str(ex)[:200]}`\n")
                print(f"[analysis] {section}: {ex.__class__.__name__}: {ex}")
            return fn
        return deco

    def savefig(fig, name):
        fig.tight_layout(); fig.savefig(os.path.join(FIG, name), dpi=150); plt.close(fig)

    # ---------------- 1. main grid: final (Table 8 analogue) and CV (Tables 6, 10) ----------------
    @safe("main grid")
    def _main():
        B = lsm_base(R)
        fin = B[B.spec_split == "final"]
        if len(fin):
            t = mean_sd(fin, ["spec_mode", "config_id"]).sort_values(["spec_mode", "macro_f1_mean"], ascending=[True, False])
            t.to_csv(os.path.join(TBL, "main_final_by_config.csv"), index=False)
            fin[["spec_mode", "spec_seed", "config_id"] + KEY].sort_values(["spec_mode", "config_id", "spec_seed"]).to_csv(
                os.path.join(TBL, "main_final_all_seeds.csv"), index=False)
            SUMMARY.append("\n## Config 11 on DS2 (paper: acc 93.51, Macro-F1 63.92)\n")
            SUMMARY.append(md_table(t[t.config_id == 11]))
        cv = B[_is_cv(B)]
        if len(cv):
            per = cv.groupby(["spec_mode", "spec_seed", "config_id"])
            t = per[KEY].mean().add_suffix("_mean").join(per[KEY].std(ddof=1).add_suffix("_sd")).join(per.size().rename("n_folds")).reset_index()
            t.to_csv(os.path.join(TBL, "main_cv_summary.csv"), index=False)

    # ---------------- 2. paper statistics on the CV grid ----------------
    def friedman_block(M, ids):
        n, k = M.shape
        chi2, p = sstats.friedmanchisquare(*[M[:, j] for j in range(k)])
        W = chi2 / (n * (k - 1)); FF = (n - 1) * chi2 / (n * (k - 1) - chi2)
        pFF = sstats.f.sf(FF, k - 1, (k - 1) * (n - 1))
        ranks = np.vstack([sstats.rankdata(-r) for r in M]).mean(0)
        best = ids[int(np.argmin(ranks))]
        rows = []
        for j, cid in enumerate(ids):
            if cid == best:
                continue
            w = sstats.wilcoxon(M[:, ids.index(best)], M[:, j])
            rows.append({"vs_config": cid, "W": w.statistic, "p": w.pvalue})
        wdf = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
        m = len(wdf)
        wdf["p_holm"] = np.minimum(1, np.maximum.accumulate((m - np.arange(m)) * wdf.p.values)); wdf["significant"] = wdf.p_holm < 0.05
        return {"chi2": chi2, "p": p, "iman_davenport_F": FF, "p_F": pFF, "kendall_W": W, "best_by_rank": best}, \
               pd.DataFrame({"config_id": ids, "mean_rank": ranks}).sort_values("mean_rank"), wdf

    @safe("paper statistics")
    def _stats():
        cv = lsm_base(R); cv = cv[_is_cv(cv)]
        out = []
        for (mode, seed), g in cv.groupby(["spec_mode", "spec_seed"]):
            piv = g.pivot_table(index="spec_split", columns="config_id", values="macro_f1")
            if piv.shape[1] < 3 or piv.isna().any().any() or piv.shape[0] < 3:
                continue
            ids = [int(c) for c in piv.columns]
            res, ranks, wdf = friedman_block(piv.values, ids)
            out.append({"mode": mode, "seed": seed, "n_blocks": piv.shape[0], "n_configs": len(ids), **res})
            ranks.to_csv(os.path.join(TBL, f"friedman_ranks_{mode}_s{seed}.csv"), index=False)
            wdf.to_csv(os.path.join(TBL, f"wilcoxon_holm_{mode}_s{seed}.csv"), index=False)
        if out:
            df = pd.DataFrame(out); df.to_csv(os.path.join(TBL, "friedman_main_cv.csv"), index=False)
            SUMMARY.append("\n## Friedman / Iman-Davenport / Kendall's W on the CV grid (paper: chi2 169.63, F 17.54, W 0.615)\n")
            SUMMARY.append(md_table(df[["mode", "seed", "n_blocks", "n_configs", "chi2", "iman_davenport_F", "kendall_W", "best_by_rank"]]))

    # ---------------- 3. exact-replication attempt vs the authors' released results ----------------
    @safe("exact-replication comparison")
    def _exact():
        cv = lsm_base(R); cv = cv[_is_cv(cv) & (cv.spec_mode == "published") & (cv.spec_seed == 2026)]
        rows = []
        for cid, c in CONFIGS.items():
            p = os.path.join(REPO_DIR, "results/cv_24_configurations", c["authors_csv"])
            if not os.path.exists(p):
                continue
            A = pd.read_csv(p)
            for _, a in A.iterrows():
                sp = f"cv-r{int(a.rep)}f{int(a.fold)}"
                o = cv[(cv.config_id == cid) & (cv.spec_split == sp)]
                if o.empty:
                    continue
                o = o.iloc[0]
                acm = [[int(x) for x in row.split()] for row in re.findall(r"\[([\d\s]+)\]", a.confusion_matrix)]
                rows.append({"config_id": cid, "encoder": c["encoder"], "split": sp, "authors_macro_f1": a.macro_f1,
                             "ours_macro_f1": o.macro_f1, "diff": o.macro_f1 - a.macro_f1,
                             "identical_confusion_matrix": acm == json.loads(o.confusion_matrix)})
        if rows:
            df = pd.DataFrame(rows); df.to_csv(os.path.join(TBL, "exact_attempt_cv.csv"), index=False)
            s = df.groupby("encoder").agg(folds=("split", "size"), identical=("identical_confusion_matrix", "sum"),
                                          mean_abs_diff=("diff", lambda d: d.abs().mean())).reset_index()
            SUMMARY.append("\n## Exact-replication attempt (published, seed 2026) vs the authors' per-fold CSVs\n")
            SUMMARY.append(f"torch 2.4.0 active: {TORCH_PINNED}; GPU: {ENV.get('gpu', 'none')} (authors: RTX 4060 Laptop, CUDA 11.8).\n\n")
            SUMMARY.append(md_table(s))
            fig, ax = plt.subplots(figsize=(4.2, 4.2))
            for i, enc in enumerate(ENCS):
                d = df[df.encoder == enc]
                ax.scatter(d.authors_macro_f1, d.ours_macro_f1, s=18, color=SERIES[i], label=enc, edgecolor=SURF, linewidth=0.8)
            lo, hi = df[["authors_macro_f1", "ours_macro_f1"]].min().min() - 0.02, df[["authors_macro_f1", "ours_macro_f1"]].max().max() + 0.02
            ax.plot([lo, hi], [lo, hi], color=INK2, lw=1, ls="--")
            ax.set_xlabel("authors' Macro-F1 (per fold)"); ax.set_ylabel("ours, published code, seed 2026")
            ax.set_title("Per-fold Macro-F1: authors vs this run", color=INK, loc="left"); ax.legend()
            savefig(fig, "exact_attempt_per_fold.png")

    # ---------------- 4. encoders: published vs fixed ----------------
    @safe("encoders")
    def _encoders():
        B = lsm_base(R); B = B[(B.spec_filter == "morph") & (B.spec_norm == "robust") & (B.spec_readout == "svm")]
        if B.empty:
            return
        t = []
        for (mode, enc, part), g in B.assign(part=np.where(_is_cv(B), "cv", "final")).groupby(["spec_mode", "spec_encoder", "part"]):
            seed_means = g.groupby("spec_seed").macro_f1.mean()
            t.append({"mode": mode, "encoder": enc, "part": part, "macro_f1_mean": seed_means.mean(),
                      "macro_f1_sd_over_seeds": seed_means.std(ddof=1) if len(seed_means) > 1 else np.nan, "n_seeds": len(seed_means)})
        t = pd.DataFrame(t); t.to_csv(os.path.join(TBL, "encoders_morph_robust_svm.csv"), index=False)
        SUMMARY.append("\n## Encoders (morph + robust + SVM), published vs fixed\n"); SUMMARY.append(md_table(t))
        d = t[t.part == "final"]
        if d.empty:
            return
        fig, ax = plt.subplots(figsize=(5.2, 3.2))
        modes = [m for m in ("published", "fixed", "fix_gamma", "fix_ain") if m in d["mode"].values]
        w = 0.8 / len(modes)
        for i, mode in enumerate(modes):
            dd = d[d["mode"] == mode].set_index("encoder").reindex(ENCS)
            x = np.arange(3) + (i - (len(modes) - 1) / 2) * w
            ax.bar(x, dd.macro_f1_mean, w * 0.9, yerr=dd.macro_f1_sd_over_seeds.fillna(0), color=(SERIES + ["#eda100"])[i],
                   label=mode, error_kw={"ecolor": INK2, "lw": 0.8})
        ax.set_xticks(np.arange(3)); ax.set_xticklabels(ENCS); ax.set_ylabel("Macro-F1 on DS2 (mean ± sd over seeds)")
        ax.set_title("Encoder comparison, published vs fixed liquid", color=INK, loc="left"); ax.legend(ncol=len(modes))
        savefig(fig, "encoders_published_vs_fixed.png")

    # ---------------- 5. input-gain sweep ----------------
    @safe("gain sweep")
    def _gain():
        G = R[(R.spec_kind == "lsm") & (R.spec_mode == "fixed") & (R.spec_filter == "morph") & (R.spec_norm == "robust")
              & (R.spec_readout == "svm") & (R.spec_lambda_val == 0.8) & (R.spec_recurrence == True) & (R.spec_rr_in_liquid == False)
              & (R.spec_num_levels.isin([200]) | R.spec_num_levels.isna())]
        if G.empty:
            return
        G = G.assign(part=np.where(_is_cv(G), "cv", "final"))
        t = G.groupby(["spec_encoder", "spec_gain", "part"]).agg(macro_f1=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
                                                                 firing=("liquid_firing_prob", "mean"), drive=("drive_over_target", "mean"),
                                                                 n=("macro_f1", "size")).reset_index()
        t.to_csv(os.path.join(TBL, "gain_sweep.csv"), index=False)
        SUMMARY.append("\n## Input-gain sweep (fixed liquid; gain 1 = as the paper describes)\n"); SUMMARY.append(md_table(t))
        fig, axs = plt.subplots(1, 2, figsize=(9, 3.4))
        for i, enc in enumerate(ENCS):
            d = t[(t.spec_encoder == enc) & (t.part == "final")].sort_values("spec_gain")
            if d.empty:
                continue
            axs[0].errorbar(d.spec_gain, d.macro_f1, yerr=d.macro_f1_sd.fillna(0), color=SERIES[i], marker="o", ms=4, lw=2, label=enc, capsize=2)
            axs[1].plot(d.firing, d.macro_f1, color=SERIES[i], marker="o", ms=4, lw=2, label=enc)
        for ax in axs:
            ax.set_xscale("log"); ax.set_ylabel("Macro-F1 on DS2"); ax.legend()
        axs[0].set_xlabel("input gain (× the fixed liquid's weights)"); axs[1].set_xlabel("liquid firing probability per step")
        axs[0].set_title("Accuracy vs input strength", color=INK, loc="left"); axs[1].set_title("Accuracy vs spike activity", color=INK, loc="left")
        savefig(fig, "gain_sweep.png")

    # ---------------- 6. ablations, RR, baselines ----------------
    def _variant(r):
        k = r.spec_kind
        if k == "lsm":
            v = f"liquid {r.spec_mode}"
            if r.spec_recurrence is False or r.spec_recurrence == False: v += ", recurrence off"
            if r.spec_lambda_val != 0.8: v += f", lambda {r.spec_lambda_val}"
            if r.spec_rr_in_liquid: v += ", RR in liquid"
            if isinstance(getattr(r, "spec_rr_set", None), str): v += " (+ mean-RR ratio)"
            if r.spec_readout == "svm_rr": v += ", RR in readout"
            gr = getattr(r, "spec_g_rec", None)
            if gr is not None and not pd.isna(gr): v += f", g_rec {float(gr):g}"
            if r.spec_readout == "svm_late": v += ", last-window readout"
            if r.spec_gain != 1: v += f", gain {r.spec_gain}"
            if r.spec_num_levels not in (200, None) and not pd.isna(r.spec_num_levels): v += f", K {int(r.spec_num_levels)}"
            if r.spec_ain_override is not None and not pd.isna(r.spec_ain_override): v += ", A_in 0.05892"
            return v
        if k == "feat":
            return {"no_liquid": "no liquid (encoder counts)", "elm": "random features (ELM)", "raw_beat": "raw beat, linear",
                    "rr_only": "RR only, linear", "rr_shape": "RR + beat shape, linear", "lda": "RR + beat shape, LDA",
                    "elm_rr": "random features (ELM) + RR channels", "rr_only_mrr": "RR only + mean-RR ratio, linear",
                    "rr_shape_mrr": "RR + beat shape + mean-RR ratio, linear"}[r.spec_model]
        if k == "nn":
            t = getattr(r, "spec_trial", None)
            tag = "" if t is None or (isinstance(t, float) and pd.isna(t)) else " tuned" if t == "best" else f" trial {t}"
            rs = getattr(r, "spec_rr_set", None)
            return f"{r.spec_model.upper()}{tag}{' + RR' if r.spec_use_rr else ''}{' + mean-RR ratio' if isinstance(rs, str) else ''}"
        if k == "huang":
            return f"Huang 2014, gamma {float(r.spec_gamma):.3g}"
        if k == "hybrid":
            return f"hybrid ({r.spec_rule}): {r.spec_base_label}"
        return k

    @safe("ablations / RR / baselines")
    def _compare():
        D = R[(R.spec_filter == "morph") & (R.spec_norm == "robust")].copy()
        D = D[(D.spec_kind != "lsm") | ((D.spec_readout.isin(["svm", "svm_rr"])) & (D.spec_num_levels.isin([200]) | D.spec_num_levels.isna()))]
        if D.empty:
            return
        D["variant"] = D.apply(_variant, axis=1)
        D = D[~D.variant.str.contains(" trial ")]          # tuning trials are reported in their own table
        D["enc"] = D.spec_encoder.fillna("-")
        D["part"] = np.where(_is_cv(D), "cv", "final")
        t = D.groupby(["variant", "enc", "part"]).agg(macro_f1=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
                                                      recall_SVEB=("recall_SVEB", "mean"), f1_SVEB=("f1_SVEB", "mean"),
                                                      recall_VEB=("recall_VEB", "mean"), accuracy=("accuracy", "mean"),
                                                      firing=("liquid_firing_prob", "mean"), n=("macro_f1", "size")).reset_index()
        t = t[~t.variant.str.contains("gain|K |A_in", regex=True)]
        t.to_csv(os.path.join(TBL, "ablations_rr_baselines.csv"), index=False)
        SUMMARY.append("\n## Liquid ablations, RR timing, baselines (morph + robust front end)\n")
        SUMMARY.append(md_table(t[t.part == "final"].sort_values("macro_f1", ascending=False), max_rows=60))
        d = t[(t.part == "final") & (t.enc.isin(["multi", "-"]))].sort_values("macro_f1")
        if d.empty:
            return
        fig, ax = plt.subplots(figsize=(6.4, 0.32 * len(d) + 1.2))
        y = np.arange(len(d))
        ax.barh(y - 0.18, d.macro_f1, 0.34, color=SERIES[0], label="Macro-F1", xerr=d.macro_f1_sd.fillna(0), error_kw={"ecolor": INK2, "lw": 0.8})
        ax.barh(y + 0.18, d.recall_SVEB, 0.34, color=SERIES[1], label="SVEB recall")
        ax.set_yticks(y); ax.set_yticklabels(d.variant); ax.set_xlabel("DS2 test set (mean over seeds)")
        ax.set_title("What does the work? (Multi-Threshold where applicable)", color=INK, loc="left", fontsize=9); ax.legend(loc="lower right")
        savefig(fig, "ablations_rr_baselines.png")

    # ---------------- 7. corrected resampled t-tests (Nadeau & Bengio) ----------------
    def nb_test(a, b):
        """a, b: Series indexed by CV split (already averaged over seeds). Corrected repeated k-fold t-test."""
        j = a.index.intersection(b.index)
        if len(j) < 3:
            return None
        d = (a[j] - b[j]).values
        n_tr, n_te = 0.75, 0.25
        var = np.var(d, ddof=1) * (1 / len(d) + n_te / n_tr)
        t = d.mean() / math.sqrt(var) if var > 0 else np.inf
        return {"n_splits": len(d), "mean_diff": d.mean(), "t": t, "p": 2 * sstats.t.sf(abs(t), len(d) - 1)}

    @safe("corrected t-tests")
    def _ttests():
        C = R[_is_cv(R)].copy()
        if C.empty:
            return
        C["variant"] = C.apply(_variant, axis=1); C["enc"] = C.spec_encoder.fillna("-")
        def series(variant, enc="multi", cfg=None):
            d = C[(C.variant == variant) & (C.enc == enc)]
            if cfg is not None:
                d = d[d.config_id == cfg]
            else:
                d = d[(d.spec_filter == "morph") & (d.spec_norm == "robust")]
                if "spec_readout" in d:
                    d = d[d.spec_readout.isin(["svm", "svm_rr"]) | d.spec_readout.isna()]
            return d.groupby("spec_split").macro_f1.mean()
        pairs = [("published vs fixed (config 11)", series("liquid published", cfg=11), series("liquid fixed", cfg=11)),
                 ("fixed: multi vs rate", series("liquid fixed", "multi", 11), series("liquid fixed", "rate", 7)),
                 ("fixed: multi vs latency", series("liquid fixed", "multi", 11), series("liquid fixed", "latency", 9)),
                 ("fixed liquid vs no liquid", series("liquid fixed"), series("no liquid (encoder counts)")),
                 ("fixed liquid vs random features", series("liquid fixed"), series("random features (ELM)")),
                 ("fixed liquid vs recurrence off", series("liquid fixed"), series("liquid fixed, recurrence off")),
                 ("fixed + RR readout vs fixed", series("liquid fixed, RR in readout"), series("liquid fixed")),
                 ("fixed + RR in liquid vs fixed", series("liquid fixed, RR in liquid"), series("liquid fixed")),
                 ("CNN + RR vs fixed liquid", series("CNN + RR", "-"), series("liquid fixed")),
                 ("tuned CNN vs fixed liquid", series("CNN tuned", "-"), series("liquid fixed")),
                 ("tuned GRU vs fixed liquid", series("GRU tuned", "-"), series("liquid fixed")),
                 ("tuned CNN + RR vs fixed liquid + RR readout", series("CNN tuned + RR", "-"), series("liquid fixed, RR in readout")),
                 ("tuned GRU + RR vs fixed liquid + RR readout", series("GRU tuned + RR", "-"), series("liquid fixed, RR in readout")),
                 ("tuned GRU + RR vs published liquid + RR readout", series("GRU tuned + RR", "-"), series("liquid published, RR in readout"))]
        rows = []
        for name, a, b in pairs:
            res = nb_test(a, b)
            if res:
                rows.append({"comparison": name, **res})
        if rows:
            df = pd.DataFrame(rows); df.to_csv(os.path.join(TBL, "corrected_ttests.csv"), index=False)
            SUMMARY.append("\n## Corrected resampled t-tests on DS1 CV (Nadeau & Bengio; seeds averaged per split)\n")
            SUMMARY.append(md_table(df))

    # ---------------- 8. retune and k-stale diagnostics ----------------
    @safe("retune")
    def _retune():
        best, df = retune_choice()
        if df is not None:
            df.to_csv(os.path.join(TBL, "retune_stage1.csv"), index=False)
            SUMMARY.append("\n## Re-tuning the fixed liquid (stage 1, DS1 CV rep 1)\n"); SUMMARY.append(md_table(df.head(10)))

    @safe("NN tuning")
    def _nn_tune():
        rows, picks = [], []
        for m in ("cnn", "gru"):
            for use_rr in (False, True):
                best, df = nn_tune_choice(m, use_rr)
                if df is None:
                    continue
                rows.append(df)
                default = df[(df.trial == 0) & (df.epochs == df[df.trial == 0].sort_values("cv_macro_f1").epochs.iloc[-1])].iloc[0]
                picks.append({"model": m.upper() + (" + RR" if use_rr else ""), "best_trial": best["trial"], "epochs": best["epochs"],
                              "n_params": int(df[df.trial == best["trial"]].n_params.iloc[0]), "cv_macro_f1": best["cv_macro_f1"],
                              "untuned_arch_best_cv_macro_f1": float(default.cv_macro_f1), "arch": json.dumps(best["arch"], sort_keys=True)})
        if rows:
            pd.concat(rows).to_csv(os.path.join(TBL, "nn_tuning_stage1.csv"), index=False)
            P = pd.DataFrame(picks); P.to_csv(os.path.join(TBL, "nn_tuning_choice.csv"), index=False)
            SUMMARY.append("\n## Tuned CNN / GRU baselines (stage 1: DS1 CV rep 1 only; DS2 untouched)\n")
            SUMMARY.append(md_table(P.drop(columns="arch")))
            SUMMARY.append("\nChosen architectures:\n\n" + "".join(f"- {r['model']}: `{r['arch']}`\n" for r in picks))
            SUMMARY.append("\nNote: stage-2 CV numbers include the rep-1 folds used for the choice (slightly optimistic, as for the "
                           "liquid re-tuning); the DS2 numbers are clean.\n")

    @safe("k-stale diagnostic")
    def _kstale():
        B = R[(R.spec_kind == "lsm") & (R.spec_mode == "published") & (R.spec_seed == 2026) & _is_cv(R)]
        if B.empty:
            return
        B = B.assign(cid=B.apply(lambda r: next((cid for cid, c in CONFIGS.items() if (c["filter"], c["norm"], c["encoder"], c["readout"])
                                                 == (r.spec_filter, r.spec_norm, r.spec_encoder, r.spec_readout)), None), axis=1))
        B = B.assign(kind=np.where(B.spec_ain_override.notna(), "A_in 0.05892", "paper A_in"))
        rows = []
        for cid, c in CONFIGS.items():
            if c["encoder"] == "multi":
                continue
            p = os.path.join(REPO_DIR, "results/cv_24_configurations", c["authors_csv"])
            a = pd.read_csv(p).macro_f1.mean() if os.path.exists(p) else np.nan
            r = {"config_id": cid, "encoder": c["encoder"], "authors_cv_macro_f1": a}
            for kind, g in B[B.cid == cid].groupby("kind"):
                r[f"ours_{kind}"] = g.macro_f1.mean()
            rows.append(r)
        df = pd.DataFrame(rows); df.to_csv(os.path.join(TBL, "kstale_diagnostic.csv"), index=False)
        SUMMARY.append("\n## Which A_in did the authors' Rate/Latency runs use? (CV mean Macro-F1, published code, seed 2026)\n")
        SUMMARY.append(md_table(df))

    # ---------------- 8b. extensions: Huang baseline, hybrid rule, mean-RR feature, ELM + RR ----------------
    @safe("extensions")
    def _extensions():
        keep = R.spec_kind.isin(["huang", "hybrid"])
        if "spec_model" in R:
            keep |= R.spec_model.isin(["elm_rr", "rr_only_mrr", "rr_shape_mrr"])
        if "spec_rr_set" in R:
            keep |= R.spec_rr_set.notna()
        if "spec_g_rec" in R:
            keep |= R.spec_g_rec.notna()
        pdiag = os.path.join(WORK_DIR, "checks", "working_liquid_diag.json")
        if os.path.exists(pdiag):
            wd = json.load(open(pdiag))
            dt = pd.DataFrame([{k: v for k, v in r.items() if k != "perturbation_curve_from_t_perturb"} for r in wd["settings"]])
            dt["chosen"] = dt.candidate & (dt.g_rec == wd["chosen_g_rec"])
            dt.to_csv(os.path.join(TBL, "working_liquid_diag.csv"), index=False)
            SUMMARY.append(f"\n## Working liquid: perturbation diagnostic (chosen g_rec = {wd['chosen_g_rec']:g})\n")
            SUMMARY.append(md_table(dt[["label", "firing_per_step", "last_window_firing_per_step", "recurrent_over_input_current",
                                        "perturbation_early", "perturbation_late", "damage_ratio", "n_recurrent_synapses", "chosen"]]))
        E = R[keep].copy()
        if E.empty:
            return
        E["variant"] = E.apply(_variant, axis=1)
        E["enc"] = E.spec_encoder.fillna("-") if "spec_encoder" in E else "-"
        rows = []
        for (v, enc), g in E.groupby(["variant", "enc"]):
            cv, fin = g[_is_cv(g)], g[g.spec_split == "final"]
            ps = cv.groupby("spec_split").macro_f1.mean()            # seeds averaged per split
            r = {"variant": v, "encoder": enc, "cv_macro_f1": ps.mean(), "cv_splits": len(ps),
                 "cv_reps23": ps[[s for s in ps.index if not s.startswith("cv-r1")]].mean(),
                 "ds2_macro_f1": fin.macro_f1.mean(), "ds2_sd": fin.macro_f1.std(ddof=1), "ds2_runs": len(fin),
                 "ds2_recall_SVEB": fin.recall_SVEB.mean(), "ds2_precision_SVEB": fin.precision_SVEB.mean(),
                 "ds2_recall_VEB": fin.recall_VEB.mean(), "ds2_precision_VEB": fin.precision_VEB.mean()}
            if "vote_ave_N_VEB" in g and g.spec_kind.iloc[0] == "huang":   # Huang's own selection criterion, rep-1 folds only
                r["rep1_vote_ave_N_VEB"] = cv[cv.spec_split.str.startswith("cv-r1")].vote_ave_N_VEB.mean()
            rows.append(r)
        df = pd.DataFrame(rows)
        if "rep1_vote_ave_N_VEB" in df and df.rep1_vote_ave_N_VEB.notna().any():
            df["huang_primary"] = df.rep1_vote_ave_N_VEB == df.rep1_vote_ave_N_VEB.max()    # chosen on DS1 rep 1, never DS2
        df.to_csv(os.path.join(TBL, "extensions_summary.csv"), index=False)
        SUMMARY.append("\n## Extensions: Huang baseline, hybrid SVEB rule, mean-RR feature, ELM + RR (descriptive; verdicts come later)\n")
        SUMMARY.append(md_table(df.sort_values("ds2_macro_f1", ascending=False), max_rows=80))

    # ---------------- 9. config 11 confusion matrices and saved state vectors ----------------
    @safe("config 11 figures")
    def _cfg11():
        fin = lsm_base(R); fin = fin[(fin.spec_split == "final") & (fin.config_id == 11) & (fin.spec_seed == SEEDS[0])]
        modes = [m for m in ("published", "fixed") if m in fin.spec_mode.values]
        if not modes:
            return
        fig, axs = plt.subplots(1, len(modes), figsize=(3.6 * len(modes), 3.3))
        axs = np.atleast_1d(axs)
        for ax, mode in zip(axs, modes):
            cm = np.array(json.loads(fin[fin.spec_mode == mode].iloc[0].confusion_matrix))
            frac = cm / cm.sum(1, keepdims=True)
            ax.imshow(frac, cmap="Blues", vmin=0, vmax=1)
            for i in range(3):
                for j in range(3):
                    ax.text(j, i, f"{cm[i, j]}\n{frac[i, j]:.1%}", ha="center", va="center", fontsize=8, color=SURF if frac[i, j] > 0.6 else INK)
            ax.set_xticks(range(3)); ax.set_yticks(range(3)); ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
            ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.grid(False); ax.set_title(f"config 11, {mode}, seed {SEEDS[0]}", color=INK, loc="left")
        savefig(fig, "config11_confusion_matrices.png")

    @safe("state-vector PCA")
    def _pca():
        files = sorted(glob.glob(os.path.join(WORK_DIR, "states", "*.npz")))
        pick = [f for f in files if f"_s{SEEDS[0]}_" in os.path.basename(f)]
        if not pick:
            return
        fig, axs = plt.subplots(1, len(pick), figsize=(4 * len(pick), 3.6)); axs = np.atleast_1d(axs)
        rows = []
        for ax, f in zip(axs, pick):
            z = np.load(f); X = z["counts"].astype(np.float32) / WIN; y = z["y"]
            rng = np.random.default_rng(0)
            idx = np.concatenate([rng.choice(np.where(y == c)[0], min(1500, (y == c).sum()), replace=False) for c in range(3) if (y == c).any()])
            Xs = (X[idx] - X[idx].mean(0)) / (X[idx].std(0) + 1e-6)
            U, S, Vt = np.linalg.svd(Xs, full_matrices=False)
            P = Xs @ Vt[:2].T
            for c in range(3):
                m = y[idx] == c
                ax.scatter(P[m, 0], P[m, 1], s=4, color=SERIES[c], label=CLASS_NAMES[c], alpha=0.6, linewidths=0)
            mode = "published" if "published" in os.path.basename(f) else "fixed"
            ax.set_title(f"{mode}: liquid states, PCA", color=INK, loc="left"); ax.legend(markerscale=3)
            nz = (X > 0).mean()
            rows.append({"file": os.path.basename(f), "mode": mode, "nonzero_fraction": float(nz),
                         "pc1_pc2_variance_share": float((S[:2] ** 2).sum() / (S ** 2).sum())})
        savefig(fig, "state_vectors_pca.png")
        pd.DataFrame(rows).to_csv(os.path.join(TBL, "state_vectors_summary.csv"), index=False)

    # ---------------- summary.md ----------------
    done_by_group = pd.Series([v[4] if len(v) > 4 else "?" for v in CACHE.done.values()]).value_counts()
    pend_by_group = pd.Series([g for _, g in CACHE.pending]).value_counts()
    status = pd.DataFrame({"done": done_by_group, "pending": pend_by_group}).reindex(GROUP_ORDER).fillna(0).astype(int)
    head = [f"# LSM-ECG replication: results so far\n",
            f"_Updated {datetime.datetime.utcnow().isoformat(timespec='seconds')} UTC, session {SESSION_TAG}. "
            f"Runner stopped: {STOP_REASON}._\n",
            "\n## Environment\n", md_table(pd.DataFrame([{k: ENV.get(k) for k in ('torch', 'torch_pinned_2_4_0', 'gpu', 'cuda', 'python', 'snntorch', 'sklearn', 'smoke_test')}])),
            "\n## Checks\n",
            f"- DS1 beat order identical to the authors' file: **{CHECKS.get('ds1_matches_authors_order')}**; class counts {CHECKS.get('class_counts')}\n",
            f"- Authors' fold indices regenerate exactly: **{CHECKS.get('folds_regenerate_exactly')}**\n",
            f"- Checksums: {CHECKS.get('checksums', {}).get('files_checked')} files checked, "
            f"{len(CHECKS.get('checksums', {}).get('mismatches', []))} mismatches; missing from the input: "
            f"{CHECKS.get('checksums', {}).get('missing') or 'none'} (none of them used by the pipeline)\n"]
    if os.path.exists(os.path.join(WORK_DIR, "checks", "brian2_check.json")):
        b2r = json.load(open(os.path.join(WORK_DIR, "checks", "brian2_check.json")))
        for mode, v in b2r["liquids"].items():
            head.append(f"- Brian2 vs PyTorch ({mode} liquid, {len(v['per_beat'])} beats, {v['total_cells']:,} neuron-steps): "
                        f"{v['brian2_vs_torch64_mismatches']} mismatches vs float64 equations, {v['brian2_vs_authors_float32_mismatches']} vs the authors' float32 snnTorch neuron; "
                        f"equations vs snnTorch (float32): {v['equations_vs_snntorch_float32_mismatches']}\n")
    head += ["\n## Progress by group\n", md_table(status.reset_index().rename(columns={"index": "group"}))]
    for _f in ("ain_measured.csv", "liquid_structure.csv"):     # written once by the checks job; restore after a resume / merge
        if os.path.exists(os.path.join(WORK_DIR, "checks", _f)) and not os.path.exists(os.path.join(TBL, _f)):
            shutil.copy2(os.path.join(WORK_DIR, "checks", _f), os.path.join(TBL, _f))
    if MERGE_MODE:
        head += ["\n## Merged output\n", "".join(f"- from `{md}`\n" for md in MERGE_DIRS)]
    _stp = os.path.join(WORK_DIR, "checks", "self_test.json")
    if os.path.exists(_stp):
        _st = json.load(open(_stp))
        head += [f"\n## Self-test (session {_st.get('session', '?')}): {'all checks passed' if _st['passed'] else 'some checks FAILED'}\n"]
        head += [f"- {'PASS' if v['ok'] else ('NOTE' if k.startswith('torch 2.4.0') else 'FAIL')} {k}: {v['detail']}\n" for k, v in _st["checks"].items()]
    _res = sorted(glob.glob(os.path.join(WORK_DIR, "logs", "resources_*.csv")))
    if _res:
        try:
            _rl = pd.read_csv(_res[-1])
            head += ["\n## Resources this session\n",
                     f"- runner: {'pipeline, ' + str(PIPE_W) + ' parallel fits' if PIPELINE else 'one job at a time'}; CPU cores {N_CPU}; "
                     f"RAM {ENV.get('ram_gb')} GB; GPU {ENV.get('gpu', 'none')}\n",
                     f"- process peak RAM {_rl.peak_rss_gb.max():.1f} GB; lowest available RAM {_rl.available_gb.min():.1f} GB; "
                     f"peak GPU memory {_rl.gpu_mem_gb.max() if 'gpu_mem_gb' in _rl and _rl.gpu_mem_gb.notna().any() else '-'} GB "
                     f"(details: logs/{os.path.basename(_res[-1])})\n"]
        except Exception as ex:
            print("resources summary:", ex)
    _tp = os.path.join(TBL, "time_projection.txt")
    if os.path.exists(_tp):
        head += ["\n## Projected time remaining (from measured speeds)\n", "```\n" + open(_tp).read() + "```\n"]
    atomic_write_text(os.path.join(WORK_DIR, "summary.md"), "".join(head + SUMMARY))
    print("summary.md, tables/ and figures/ written:", sorted(os.listdir(TBL)), sorted(os.listdir(FIG)))

else:
    print('replication tables and figures not rebuilt (OLD_ANALYSIS = False)')


## 17. End of session

In [ ]:
# End of session: final cache write, disk use, what is left, how to continue.
CACHE.write()
gb = work_dir_gb(force=True)
left = pd.Series([g for _, g in CACHE.pending]).value_counts().reindex(GROUP_ORDER).dropna().astype(int)
print(f"session time: {elapsed_h():.2f} h | output size: {gb:.2f} GB | done {len(CACHE.done)} | failed {len(CACHE.failed)} | pending {len(CACHE.pending)}")
print(f"stopped because: {STOP_REASON}")
if MERGE_MODE:
    print(f"Merged {len(MERGE_DIRS)} output folders into {WORK_DIR}: {len(CACHE.done)} finished runs. "
          + (f"{len(CACHE.pending)} runs of the whole plan are not in either folder yet (pending by group below)." if len(left) else
             "Nothing in the plan is missing."))
    if len(left):
        print("pending by group:\n" + left.to_string())
elif SELF_TEST:
    print("Self-test only (see the SELF-TEST block above). Set SELF_TEST = False to run the smoke test, benchmark or real run.")
elif len(left):
    print("pending by group:\n" + left.to_string())
    if MACHINE == "kaggle":
        print("\nTo continue: Save Version -> attach this version's output as an input (Add Input -> Your Work / Notebook Output),\n"
              "put its folder path in RESUME_DIRS in the first cell, then Save & Run All again.")
    else:
        print(f"\nTo continue: run the notebook again with the same WORK_DIR. It reads {CACHE_PATH} and skips everything finished.")
else:
    print("Everything in the plan is done.")
if CACHE.failed:
    print(f"\n{len(CACHE.failed)} failed runs (tracebacks in logs/failures/); they are retried next session if RETRY_FAILED=True.")
